# GLNO para la ecuación de Schrödinger en muchas formas (etapa 3)

Notebook autónomo para Google Colab: escribe una copia de los módulos del proyecto, así que no hay que clonar nada.

- **Datos:** 200 formas de área $4\pi$ (esferas y toros deformados) con paquetes gaussianos, $V=0$, resueltos con Crank–Nicolson. Se generan y se empaquetan en su computador; aquí solo se desempaquetan y se verifican con SHA-256.
- **Red:** el `GLNONet` del código del paper. Entrada: $\psi_0$ por vértice $(\mathrm{Re},\mathrm{Im})$. Salida: los 10 estados guardados después de $t=0$, hasta $t=0.5$. En cada forma usa la base de Ritz, que se recalcula aquí y se compara con la calculada en su computador.
- **Entrenamiento:** el del paper (AdamW, lr $10^{-3}$ dividida por 2 cada 100 épocas, lotes de 4, 300 épocas), con signos y rotaciones al azar de la base y recorte del gradiente. Pérdida: error relativo $L^2$ ponderado con la masa.

**Antes de empezar**

1. En *Entorno de ejecución → Cambiar tipo de entorno*, elija una GPU. La corrida real usa unos 18 GiB (7.95 de datos y ~10 de activaciones con lotes de 4 toros): sirve una G4, o una L4 (cabe justo, ~19 de 22 GiB); una T4 no. Haga la prueba rápida en la misma GPU de la corrida real, porque mide su velocidad.
2. Suba a Drive el dataset empaquetado: un solo archivo `.zip` por dataset, directamente en `Mi unidad`. No hay que crear ninguna carpeta:

| En su computador | En Drive | Tamaño | Para |
|---|---|---|---|
| `data_packed/glno_shapes_small.zip` | `Mi unidad/glno_shapes_small.zip` | 420 MB | la prueba rápida (fase A) |
| `data_packed/glno_shapes.zip` | `Mi unidad/glno_shapes.zip` | 8.06 GB | la corrida real (fases C a E); subirlo es la fase B |

En lugar del zip también sirve la carpeta `data_packed/<nombre>` tal cual (los `.tar` y `shapes_tars.json`; 461 MB la de `shapes_small` y 8.83 GB la de `shapes`), subida a `Mi unidad/glno_schrodinger/packed/<nombre>`.

Ejecute las celdas en orden (*Entorno de ejecución → Ejecutar todo*). Los checkpoints se guardan en Drive en cada época: si la sesión se corta, vuelva a ejecutar todo y el entrenamiento continúa donde quedó; se pierde como mucho una época. Use una sola sesión por run: dos sesiones escribiendo en la misma carpeta mezclarían los checkpoints.


In [ ]:
# Va antes de cualquier "import torch": PyTorch lee esta variable una sola vez, al empezar a usar la GPU.
# Los lotes alternan esferas (10 242 vértices) y toros (15 679); con segmentos expandibles eso no fragmenta la memoria.
import os
import sys

ALLOC_CONF = "expandable_segments:True"
loaded = sys.modules.get("torch")           # solo importa si PyTorch ya empezó a usar la GPU sin la variable
if os.environ.get("PYTORCH_CUDA_ALLOC_CONF") != ALLOC_CONF and loaded is not None and loaded.cuda.is_initialized():
    raise RuntimeError("PyTorch ya usó la GPU sin PYTORCH_CUDA_ALLOC_CONF. Reinicie la sesión "
                       "(Entorno de ejecución → Reiniciar sesión) y ejecute el notebook desde la primera celda.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ALLOC_CONF
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])


In [ ]:
import importlib.util
import platform
import shutil
import subprocess
import tarfile
from pathlib import Path
import numpy as np
import scipy
import torch

if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError("Este código necesita NumPy >= 2; actualice el entorno de Colab.")
if not hasattr(tarfile, "data_filter"):
    raise RuntimeError(f"Python {platform.python_version()} no tiene los filtros de extracción de tarfile, que el "
                       "desempaquetado necesita (3.10.12+, 3.11.4+ o 3.12+).")
if importlib.util.find_spec("einops") is None:  # lo único que importan los módulos y que Colab puede no traer
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "einops"])

try:                                  # este paquete solo existe en Colab
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | NumPy {np.__version__} | "
      f"SciPy {scipy.__version__} | Colab: {IN_COLAB}")
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Esta sesión no tiene GPU. Elija una en Entorno de ejecución → Cambiar tipo de entorno "
                       "y ejecute el notebook desde la primera celda.")
DEVICE = "cuda" if IN_COLAB else "cpu"   # fuera de Colab, siempre la CPU: es para probar el flujo, no para entrenar
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
if DEVICE == "cuda":
    print(f"GPU: {GPU_NAME}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
else:
    print("Fuera de Colab todo corre en la CPU. Sirve para probar el flujo con un dataset diminuto, no para entrenar.")
DISK = Path("/content") if IN_COLAB else Path.cwd()
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.1f} GiB | "
      f"disco libre en {DISK}: {shutil.disk_usage(DISK).free / 1e9:.1f} GB")


In [ ]:
if "WORKDIR" not in globals():           # al repetir la celda fuera de Colab, Path.cwd() ya es esta carpeta
    WORKDIR = Path("/content/glno_shapes") if IN_COLAB else Path.cwd() / "colab_workspace" / "glno_shapes"
for folder in ('reference', 'src/schrodinger_glno', 'src/schrodinger_glno/glno'):
    (WORKDIR / folder).mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
if str(WORKDIR / "src") not in sys.path:
    sys.path.insert(0, str(WORKDIR / "src"))
print("Archivos de trabajo:", WORKDIR)


## Código del proyecto y huellas de referencia

Estas celdas escriben los módulos del proyecto que el entrenamiento importa y las dos huellas de referencia de la base de Ritz, calculadas en su computador (`artifacts/glno_shapes_fingerprint_*.json`). Los archivos Python correspondientes se conservan también en `src/`. Si cambia una de estas celdas, reinicie la sesión antes de seguir.


In [ ]:
%%writefile src/schrodinger_glno/__init__.py
"""Schrödinger equation on triangle meshes: operators, initial states and Crank–Nicolson."""


In [ ]:
%%writefile src/schrodinger_glno/data_utils.py
"""Dataset configuration, complex channels and manifest integrity for GLNO."""
from dataclasses import dataclass
from hashlib import sha256
import json
from pathlib import Path
import numpy as np
from numpy.typing import NDArray

@dataclass(frozen=True)
class DatasetConfig:
    train: int = 800
    val: int = 100
    test: int = 100
    seed: int = 0
    ood: int = 0

    def __post_init__(self) -> None:
        if min(self.train, self.val, self.test) < 1 or self.ood < 0:
            raise ValueError("train, val and test must be positive; ood >= 0")

def psi_to_channels(psi: NDArray, V: NDArray | None = None) -> NDArray:
    if V is not None and V.max() > 0:
        return np.stack([psi.real, psi.imag, V / V.max()], axis=-1)
    return np.stack([psi.real, psi.imag], axis=-1)

def _digest(path: Path) -> str:
    h = sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def _manifest_identity(manifest: dict) -> str:
    payload = {key: manifest[key] for key in ("format_version", "preset", "physics", "counts", "splits", "sha256")}
    return sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def dataset_identity(directory: Path, *, verify: bool = True) -> str:
    """Return the manifest identity and optionally verify every data file."""
    manifest = json.loads((directory / "manifest.json").read_text())
    if manifest["identity"] != _manifest_identity(manifest):
        raise ValueError("Dataset manifest identity is invalid")
    if verify:
        for relative, expected in manifest["sha256"].items():
            if _digest(directory / relative) != expected:
                raise ValueError(f"Dataset file differs from manifest: {relative}")
    return manifest["identity"]


In [ ]:
%%writefile src/schrodinger_glno/geodesic.py
"""Geodesic distance on a closed triangle mesh and the geodesic cutoff of the stage-3 packet.

The packet of ``initial_conditions.gaussian_packet`` uses straight-line (chordal) distances.
On a non-convex surface it puts amplitude on parts that are close in R³ but far along the
surface (a limb next to the body, the other face of a thin plate). Stage 3 multiplies it by
a smooth cutoff in the geodesic distance from its centre: 1 up to 7σ, 0 from 9σ. The
distance comes from the heat method (K. Crane, C. Weischedel and M. Wardetzky, "Geodesics
in Heat", ACM TOG 2013) with the cotangent Laplacian and lumped mass of ``mesh``.
"""
import numpy as np
import scipy.sparse as sp
import scipy.sparse.csgraph as csgraph
import scipy.sparse.linalg as sla
from numpy.typing import NDArray
from .initial_conditions import gaussian_packet, mass_norm
from .mesh import _check_mesh, cotan_laplacian, mean_edge_length, vertex_areas


class HeatGeodesics:
    """Heat-method geodesic distances from single vertices of one closed, connected mesh.

    ``(M + tL) u = δ_source`` with ``t = h²`` (h: mean edge length, M: lumped mass), the
    unit field ``X = -∇u/|∇u|`` on each face, then the least-squares ``φ`` with ``∇φ ≈ X``:
    ``L φ = Gᵀ A X``, where G is the per-face gradient and A the face areas. ``-Gᵀ A X`` is
    the integrated divergence of X at the vertices (the cotangent formula of the paper),
    with the sign of L, the weak ``-Δ``, and ``L = Gᵀ A G``. L is singular on a closed mesh;
    vertex 0 is pinned to 0 (``Gᵀ A X`` sums to zero, so the system stays consistent) and φ
    is shifted to vanish at the source. Both systems are factorized once per mesh, so a
    distance costs two pairs of triangular solves and a few sparse products.

    The mesh must have no negative cotangent weights (a Delaunay mesh, as stage 3 requires):
    then ``M + tL`` is an M-matrix and u > 0. Otherwise u can turn negative and the distance
    is wrong by tens of percent, so a negative u raises. u decays like ``e^{-d/h}`` (≈ 1e-52
    at the far side of the 25 346-vertex torus) and must stay a normal double, which limits d
    to about 700 h (stage-3 meshes reach about 100 h); an underflow raises too.

    Measured beyond 3h from the source: median relative error 0.9 % / 0.7 % on
    ``icosphere(4)`` / ``(5)`` against great circles; along the outer equator of the torus
    R = 1, r = 0.5, mean 2.1 % / 1.7 % / 1.4 % at edge 0.06 / 0.04 / 0.03. On those meshes
    ``d(a, b)`` and ``d(b, a)`` differ by less than one edge. Cost: factorization 0.5 s
    (10 242 vertices) to 1.5 s (25 346), then 0.03-0.04 s per distance.
    """

    def __init__(self, verts: NDArray, faces: NDArray) -> None:
        verts, faces = np.asarray(verts, float), np.asarray(faces)
        _check_mesh(verts, faces)
        n = len(verts)
        edges = faces[:, [0, 1, 1, 2, 2, 0]].reshape(-1, 2)
        adjacency = sp.coo_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(n, n))
        if csgraph.connected_components(adjacency, directed=False)[0] != 1:
            raise ValueError("the mesh must be a single connected piece with every vertex in a face")
        corners = verts[faces]
        cross = np.cross(corners[:, 1] - corners[:, 0], corners[:, 2] - corners[:, 0])
        double_area = np.linalg.norm(cross, axis=1)
        if np.any(double_area <= 0):
            raise ValueError("the mesh has degenerate faces")
        normal = cross / double_area[:, None]
        # Hat function of corner k: gradient N × e_k / (2A), e_k its opposite edge (k+1 → k+2).
        grads = [np.cross(normal, corners[:, (k + 2) % 3] - corners[:, (k + 1) % 3]) / double_area[:, None]
                 for k in range(3)]
        rows = np.tile((3 * np.arange(len(faces)))[:, None] + np.arange(3), 3).ravel()
        cols = np.repeat(faces, 3, axis=1).ravel()
        vals = np.stack(grads, axis=1).ravel()
        self.gradient = sp.csr_matrix((vals, (rows, cols)), shape=(3 * len(faces), n))
        self.face_areas = 0.5 * double_area
        self.t = mean_edge_length(verts, faces)**2
        # No ε in the cotangents: with it, the distance would depend on the mesh's units.
        L = cotan_laplacian(verts, faces, denom_eps=0.0)
        self._heat = sla.splu(sp.csc_matrix(sp.diags(vertex_areas(verts, faces)) + self.t * L))
        self._poisson = sla.splu(sp.csc_matrix(L[1:, 1:]))
        self._n = n

    def distance(self, source: int) -> NDArray:
        """Geodesic distance from vertex ``source`` to every vertex, exactly 0 at the source."""
        if not 0 <= source < self._n:
            raise ValueError("source must be a vertex index")
        delta = np.zeros(self._n)
        delta[source] = 1.0
        u = self._heat.solve(delta)
        if u.min() < 0:
            raise ValueError("the heat solution is negative somewhere: the mesh has negative "
                             "cotangent weights (it is not Delaunay)")
        if not (u.min() >= np.finfo(float).tiny and np.all(np.isfinite(u))):
            raise ValueError("the heat solution underflowed; the surface is too long for this edge length")
        grad = (self.gradient @ u).reshape(-1, 3)
        scale = np.abs(grad).max(axis=1)
        if not np.all(scale > 0):
            raise ValueError("the heat gradient vanishes on a face")
        grad /= scale[:, None]                        # so that squaring in the norm cannot underflow
        field = -grad / np.linalg.norm(grad, axis=1)[:, None]
        rhs = self.gradient.T @ (self.face_areas[:, None] * field).ravel()
        phi = np.concatenate([[0.0], self._poisson.solve(rhs[1:])])
        return phi - phi[source]


def geodesic_cutoff(distance: NDArray, sigma: float, inner: float = 7.0, outer: float = 9.0) -> NDArray:
    """Weights 1 for ``d ≤ inner·σ``, 0 for ``d ≥ outer·σ`` and a C² smoothstep in between.

    With ``x = (d - inner·σ) / ((outer - inner)·σ)`` the weight is ``1 - (6x⁵ - 15x⁴ + 10x³)``:
    value, slope and curvature match the constants at both ends.
    """
    if not (sigma > 0 and np.isfinite(sigma)) or not 0 <= inner < outer:
        raise ValueError("need sigma > 0 and 0 <= inner < outer")
    x = np.clip((np.asarray(distance, float) - inner * sigma) / ((outer - inner) * sigma), 0.0, 1.0)
    return 1 - x**3 * (10 + x * (6 * x - 15))


def cut_gaussian_packet(verts: NDArray, mass: NDArray | sp.spmatrix, normals: NDArray, center: int,
                        k0: float, sigma: float, direction: NDArray, distance: NDArray,
                        inner: float = 7.0, outer: float = 9.0) -> tuple[NDArray, float]:
    """``gaussian_packet`` times ``geodesic_cutoff(distance, sigma, inner, outer)``, normalized with ``mass``.

    ``distance`` holds the geodesic distances from ``center`` (``HeatGeodesics.distance``).
    Returns the packet, with ``ψᴴ M ψ = 1``, and the fraction of the chordal packet's
    ``‖ψ‖²_M`` that the cutoff removed, ``1 - ‖wψ‖²_M``: the part that had leaked to
    geodesically distant parts of the surface (with a non-diagonal mass it can be a
    round-off below 0 when nothing is cut).
    """
    psi = gaussian_packet(verts, mass, normals, center, k0, sigma, direction)
    distance = np.asarray(distance, float)
    if distance.shape != (len(verts),):
        raise ValueError("distance must hold one value per vertex")
    if abs(distance[center]) > 1e-3 * sigma:
        raise ValueError("distance must be measured from center")
    cut = geodesic_cutoff(distance, sigma, inner, outer) * psi
    kept = mass_norm(cut, mass)
    return cut / kept, 1.0 - kept**2


In [ ]:
%%writefile src/schrodinger_glno/glno/__init__.py
"""GLNO network adapted from the paper's code (PyTorch). Requires the optional ``glno`` extra."""


In [ ]:
%%writefile src/schrodinger_glno/glno/augment.py
"""Random gauge of the eigenbasis while training: sign flips and rotations inside eigenvalue groups.

The eigenvectors of a mesh are fixed only up to sign, and inside a group of nearly equal
eigenvalues only up to a rotation: the solver's choice is arbitrary. The periodic branch
of the GLNO layer, ``Φ diag(H(λ)) Φᵀ M``, does not see that choice (exactly for signs,
up to ``H'·gap`` for rotations), but the aperiodic branch (``layers.py``,
``output_residue2``) does. Training with a random gauge teaches the network not to depend
on it.

For every sample the basis ``E`` becomes ``E Q``, with ``Q`` block diagonal: one Haar-random
orthogonal block per group from ``spectral.eigen_clusters`` (a group of one mode gets
``±1``, i.e. a sign flip). Eigenvalues, mass, geo feature, input and labels are unchanged.
Inside a group the eigenvalues differ by less than the threshold τ, so the rotated basis
gives each mixed mode a slightly wrong frequency; with τ = 1e-4 that adds up to ~0.08 % to
the labels' mismatch at t = 0.5 over the stage-3 dataset (Plan_GLNO_etapa3.md §1.3), and
0.012-0.016 % on the step-4 gate meshes (000_sphere, 003_torus).

The random numbers come from a CPU generator seeded by ``(seed, epoch, step)``, so a
resumed run and another GPU model draw the same gauges.
"""
import numpy as np
import torch

__all__ = ["gauge_generator", "group_starts", "block_haar", "tf32_matmul_enabled", "augment_batch"]


def gauge_generator(seed: int, epoch: int, step: int) -> torch.Generator:
    """CPU generator for the gauge of one training step."""
    state = np.random.SeedSequence([seed, epoch, step]).generate_state(1, dtype=np.uint64)[0]
    return torch.Generator(device="cpu").manual_seed(int(state))


def group_starts(labels: np.ndarray) -> np.ndarray:
    """First mode of each group, from labels made by ``spectral.eigen_clusters``; checks they are contiguous."""
    labels = np.asarray(labels)
    k = np.arange(len(labels))
    if labels.ndim != 1 or len(labels) == 0 or not np.issubdtype(labels.dtype, np.integer) or labels[0] != 0 \
            or np.any(np.diff(labels) < 0) or np.any(labels > k) or np.any(labels[labels] != labels):
        raise ValueError("group labels must be non-decreasing first-mode indices, as eigen_clusters returns")
    return np.flatnonzero(labels == k)


def block_haar(clusters: torch.Tensor | np.ndarray, generator: torch.Generator, *, device=None,
               dtype: torch.dtype = torch.float32) -> torch.Tensor:
    """(B, K, K) block-diagonal orthogonal matrices, one Haar-random block per eigenvalue group.

    ``clusters`` is (B, K), the labels of ``eigen_clusters`` for each sample (keep them on
    the CPU: reading GPU labels forces a synchronisation). Entries off the blocks are
    exactly 0; a group of one mode gets ``±1``. The Gaussian entries are drawn in a fixed
    order (sample, then group, row by row). All blocks of one size go through one batched
    QR with the sign correction of Mezzadri (2007), which makes each block exactly Haar on
    O(s). Everything is built on the CPU and only the finished ``Q`` is moved: thousands of
    tiny QRs on a GPU cost ~90 ms per step against ~4 ms here, and the result does not
    depend on the device.
    """
    if torch.is_tensor(clusters):
        labels = clusters.detach().cpu().numpy()
        device = clusters.device if device is None else device
    else:
        labels = np.asarray(clusters)
    if labels.ndim != 2:
        raise ValueError("clusters must be (B, K)")
    batch, k = labels.shape
    starts = [group_starts(row) for row in labels]
    sample = np.concatenate([np.full(len(s), b) for b, s in enumerate(starts)])
    start = np.concatenate(starts)
    size = np.concatenate([np.diff(np.append(s, k)) for s in starts])
    squares = size * size
    offsets = np.cumsum(squares) - squares
    draws = torch.randn(int(squares.sum()), generator=generator, dtype=torch.float64)
    index, values = [], []                     # only the block entries go to the device, not the dense Q
    for n in np.unique(size):
        chosen = size == n
        gauss = draws[torch.from_numpy(offsets[chosen][:, None] + np.arange(n * n))].reshape(-1, n, n)
        q, r = torch.linalg.qr(gauss)
        signs = torch.sign(torch.diagonal(r, dim1=-2, dim2=-1))
        signs[signs == 0] = 1
        values.append((q * signs.unsqueeze(-2)).reshape(-1))
        i = np.arange(n)
        rows = start[chosen][:, None, None] + i[:, None]
        cols = start[chosen][:, None, None] + i
        index.append(np.stack(np.broadcast_arrays(sample[chosen][:, None, None], rows, cols)).reshape(3, -1))
    index = torch.from_numpy(np.concatenate(index, axis=1))
    values = torch.cat(values).to(dtype)
    Q = torch.zeros(batch, k, k, dtype=dtype, device=device)
    if device is not None:
        index, values = index.to(device), values.to(device)
    Q[index[0], index[1], index[2]] = values
    return Q


def tf32_matmul_enabled() -> bool:
    """Whether float32 CUDA matmuls run in TF32, with the new (``fp32_precision``) or legacy API.

    In PyTorch >= 2.9, reading the legacy ``allow_tf32`` after the new API was used raises.
    """
    precision = getattr(torch.backends.cuda.matmul, "fp32_precision", None)
    if precision is not None:
        return precision == "tf32"
    return torch.backends.cuda.matmul.allow_tf32 or torch.get_float32_matmul_precision() != "highest"


def augment_batch(batch: dict, generator: torch.Generator) -> dict:
    """A new batch whose ``evecs`` are ``evecs @ Q``; nothing in ``batch`` is modified.

    ``batch`` needs ``evecs`` (B, N, K) and ``clusters`` (B, K). Rows of zero padding stay
    exactly zero. On the GPU, TF32 matmuls must be off: they would round the rotated basis
    to ~1e-3 and break its orthonormality.
    """
    evecs = batch["evecs"]
    if evecs.is_cuda and tf32_matmul_enabled():
        raise RuntimeError("TF32 matmuls are on; turn them off with "
                           "torch.backends.cuda.matmul.fp32_precision = 'ieee' (PyTorch >= 2.9) or "
                           "torch.backends.cuda.matmul.allow_tf32 = False")
    if tuple(batch["clusters"].shape) != (evecs.shape[0], evecs.shape[2]):
        raise ValueError("clusters must be (B, K) for evecs (B, N, K)")
    Q = block_haar(batch["clusters"], generator, device=evecs.device, dtype=evecs.dtype)
    return {**batch, "evecs": torch.bmm(evecs, Q)}


In [ ]:
%%writefile src/schrodinger_glno/glno/data.py
"""Connect the sphere and torus datasets to the paper's ``GLNONet``.

``GLNONet.forward`` takes a dictionary with ``input`` (B, N, C), ``vertices``
(B, N, 3; concatenated to ``input`` inside the network), ``mass`` (B, N),
``evals`` (B, K), ``evecs`` (B, N, K) and ``geo_feat`` (B, N). Here the mesh is
the same for every sample, so those tensors are loaded once and expanded to
the batch without copying.

Shapes for one sample: ``input`` is ψ0 as (N, 2) = (Re, Im); ``labels`` are the
saved states flattened time by time to (N, 2 * n_t) =
(Re t0, Im t0, Re t1, Im t1, ...).
"""
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset


def glno_config(C_out: int, k_eig: int, device: str = "cpu", **overrides) -> dict:
    """Every key ``GLNONet`` reads (the repository's ``config/`` YAML files are missing).

    blocks, C_width, glno_poles, glno_sigma and the learning rate follow the
    paper's Table 5, Poisson column: one fixed mesh, as in this stage. The rest
    are not in the paper and use the repository's simplest options.
    """
    config = {
        # Table 5 (Poisson)
        "blocks": 4, "C_width": 64, "glno_poles": 1, "glno_sigma": 1,
        # Input and output: ψ0 (2 channels) + vertex coordinates (3) -> 2 channels per saved time
        "C_in": 2 + 3, "vertices_dim": 3, "C_out": C_out, "device": device,
        "k_eig": k_eig, "k_min": 0, "k_max": k_eig, "rotate": False, "geo_preprocess": None,
        "last_activation": None, "outputs_at": "vertices",
        # Encoder and decoder
        "encoder": "mlp", "encoder_hidden_layers": [64], "encoder_dropout": 0.0,
        "encoder_norm": None, "encoder_activation": "gelu",
        "decoder": "mlp", "decoder_hidden_layers": [64], "decoder_dropout": 0.0,
        "decoder_norm": None, "decoder_activation": "gelu",
        # Laplace transform layer (Eq. 17-18: Gaussian filter and Taylor expansion)
        "gaussian": "expx2", "gaussian_factor": 1.0, "exp_x2_appr": True,
        "normalize_pole": None, "glno_scale": None,
        "glno_period_norm": None, "glno_aperiod_norm": None, "glno_norm": None,
        # GLNO block
        "connect_method": "mlp", "mlp_hidden_layers": [64, 64], "mlp_dropout": 0.0,
        "mlp_norm": None, "mlp_activation": "gelu", "norm_feature": "layer", "skip": True,
        "learned_geo_feat": False, "high_fre": False,
    }
    unknown = set(overrides) - set(config)
    if unknown:
        raise KeyError(f"Unknown GLNO config keys: {sorted(unknown)}")
    config.update(overrides)
    return config


class SurfaceSplit(Dataset):
    """One split of a sphere or torus dataset, read lazily from the ``.npy`` memmaps."""

    def __init__(self, data_dir: Path, split: str, dtype: torch.dtype = torch.float32):
        data_dir = Path(data_dir)
        manifest = json.loads((data_dir / "manifest.json").read_text())
        if split not in manifest["splits"]:
            raise ValueError(f"Unknown split {split!r}; available: {manifest['splits']}")
        with np.load(data_dir / "mesh.npz") as mesh:
            self.mesh = {"vertices": torch.as_tensor(mesh["verts"], dtype=dtype),
                         "mass": torch.as_tensor(mesh["mass"], dtype=dtype),
                         "evals": torch.as_tensor(mesh["evals"], dtype=dtype),
                         "evecs": torch.as_tensor(mesh["evecs"], dtype=dtype),
                         "geo_feat": torch.as_tensor(mesh["geo_feat"], dtype=dtype)}
            self.times = mesh["t_out"]
        self.inputs = np.load(data_dir / split / "inputs.npy", mmap_mode="r")
        self.outputs = np.load(data_dir / split / "outputs.npy", mmap_mode="r")
        self.params = np.load(data_dir / split / "params.npy")
        self.dtype = dtype

    @property
    def n_out(self) -> int:
        return 2 * self.outputs.shape[2]

    def __len__(self) -> int:
        return len(self.inputs)

    def __getitem__(self, i: int) -> dict:
        n_vertices = self.outputs.shape[1]
        return {"input": torch.as_tensor(np.array(self.inputs[i]), dtype=self.dtype),
                "labels": torch.as_tensor(np.array(self.outputs[i]).reshape(n_vertices, -1), dtype=self.dtype)}

    def collate(self, samples: list[dict]) -> dict:
        """Stack samples and expand the shared mesh tensors to the batch size."""
        batch = {key: torch.stack([s[key] for s in samples]) for key in ("input", "labels")}
        size = len(samples)
        for key, value in self.mesh.items():
            batch[key] = value.unsqueeze(0).expand(size, *value.shape)
        return batch


SphereSplit = SurfaceSplit    # stage-1 name, kept for existing code


def mass_weighted_relative_l2(pred: torch.Tensor, target: torch.Tensor, mass: torch.Tensor) -> torch.Tensor:
    """Mean over the batch of ``‖pred - target‖_M / ‖target‖_M``, over all channels.

    ``‖u‖_M² = Σ_i m_i Σ_c u_ic²``: each vertex counts with its area, as in the
    physical norm, instead of the per-vertex Euclidean norm of the repository's ``LpLoss``.
    """
    weights = mass.unsqueeze(-1)
    error = torch.sqrt(torch.sum(weights * (pred - target)**2, dim=(-2, -1)))
    scale = torch.sqrt(torch.sum(weights * target**2, dim=(-2, -1)))
    return torch.mean(error / scale)


In [ ]:
%%writefile src/schrodinger_glno/glno/geometry.py
"""Spectral projections used by the GLNO layers.

Copied from the GLNO repository (github.com/JohnJoeZhu/Geometric-Neural-Operators-via-Aperiodic-Laplace-Learning-GLNO-, commit ca0fbb2), file model/geometry.py.
The repository has no license; this copy is for private university work. Changes are marked
with "# [adapted]" comments; the rest is unchanged.
"""
import torch
# import utils  # [adapted] removed: the repository's utils package is not needed here
import numpy as np
def to_basis(values, basis, massvec):
    """
    Transform data in to an orthonormal basis (where orthonormal is wrt to massvec)
    Inputs:
      - values: (B,S,V,C)
      - basis: (B,V,K)
      - massvec: (B,V)
    Outputs:
      - (B,S,K,C) transformed values
    """
    basisT = basis.transpose(-2, -1).unsqueeze(1).repeat(1, values.shape[1], 1, 1)
    # print("basisT", basisT.shape)
    # print((values * massvec.unsqueeze(1).unsqueeze(-1)).shape)
    return torch.matmul(basisT, (values * massvec.unsqueeze(1).unsqueeze(-1)))

def to_basis2(values, basis, massvec):
    """
    Transform data in to an orthonormal basis (where orthonormal is wrt to massvec)
    Inputs:
      - values: (B,S,g,V,C)
      - basis: (B,V,K)
      - massvec: (B,V)
    Outputs:
      - (B,S,g,K,C) transformed values
    """
    basisT = basis.transpose(-2, -1).unsqueeze(1).unsqueeze(1).repeat(1, values.shape[1],values.shape[2], 1, 1)
    # print("basisT", basisT.shape)
    # print((values * massvec.unsqueeze(1).unsqueeze(-1)).shape)
    return torch.matmul(basisT, (values * massvec.unsqueeze(1).unsqueeze(1).unsqueeze(-1)))

def from_basis(values, basis):
    """
    Transform data out of an orthonormal basis
        Inputs:
        - values: (B,...,K,C_in)
        - basis: (B,V,K)
        Outputs:
        - (B,...,V,C) reconstructed values
    """
    # basis = basis.unsqueeze(1).repeat(1, values.shape[1], 1, 1)
    if values.is_complex() or basis.is_complex():
        raise ValueError("from basis unsupport complex matmul")
        # return utils.cmatmul(utils.ensure_complex(basis), utils.ensure_complex(values))
    else:
        return torch.einsum('b...kc,bvk->b...vc', values, basis)
        # return torch.matmul(basis, values)
    

# Randomly rotate points.
# Torch in, torch out
# Note fornow, builds rotation matrix on CPU. 
def rotate(pts, randgen=None):
    R = random_rotation_matrix(randgen) 
    R = torch.from_numpy(R).to(device=pts.device, dtype=pts.dtype)
    return torch.matmul(pts, R) 

def rotate_y(pts):
    angles = torch.rand(1, device=pts.device, dtype=pts.dtype) * (2. * np.pi)
    rot_mats = torch.zeros(3, 3, device=pts.device, dtype=pts.dtype)
    rot_mats[0,0] = torch.cos(angles)
    rot_mats[0,2] = torch.sin(angles)
    rot_mats[2,0] = -torch.sin(angles)
    rot_mats[2,2] = torch.cos(angles)
    rot_mats[1,1] = 1.

    pts = torch.matmul(pts, rot_mats)
    return pts

def random_rotation_matrix(randgen=None):
    """
    Creates a random rotation matrix.
    randgen: if given, a np.random.RandomState instance used for random numbers (for reproducibility)
    """
    # adapted from http://www.realtimerendering.com/resources/GraphicsGems/gemsiii/rand_rotation.c
    
    if randgen is None:
        randgen = np.random.RandomState()
        
    theta, phi, z = tuple(randgen.rand(3).tolist())
    
    theta = theta * 2.0*np.pi  # Rotation about the pole (Z).
    phi = phi * 2.0*np.pi  # For direction of pole deflection.
    z = z * 2.0 # For magnitude of pole deflection.
    
    # Compute a vector V used for distributing points over the sphere
    # via the reflection I - V Transpose(V).  This formulation of V
    # will guarantee that if x[1] and x[2] are uniformly distributed,
    # the reflected points will be uniform on the sphere.  Note that V
    # has length sqrt(2) to eliminate the 2 in the Householder matrix.
    
    r = np.sqrt(z)
    Vx, Vy, Vz = V = (
        np.sin(phi) * r,
        np.cos(phi) * r,
        np.sqrt(2.0 - z)
        )
    
    st = np.sin(theta)
    ct = np.cos(theta)
    
    R = np.array(((ct, st, 0), (-st, ct, 0), (0, 0, 1)))
    # Construct the rotation matrix  ( V Transpose(V) - I ) R.

    M = (np.outer(V, V) - np.eye(3)).dot(R)
    return M

def compute_dis(verts, central):
    # verts: [number_of_points, 1, 3]
    # central: [1, number_of_central, 3]
    verts_expanded = verts.unsqueeze(-2)  # [n_points, 1, 3]
    central_expanded = central.unsqueeze(-3)  # [1, n_central, 3]
    
    distances = torch.sqrt(torch.sum((verts_expanded - central_expanded) ** 2, dim=-1))
    
    return distances


In [ ]:
%%writefile src/schrodinger_glno/glno/layers.py
"""GLNO layers: Laplace transform layer, GLNO block and GLNONet.

Copied from the GLNO repository (github.com/JohnJoeZhu/Geometric-Neural-Operators-via-Aperiodic-Laplace-Learning-GLNO-, commit ca0fbb2), file model/GLNO/layers.py.
The repository has no license; this copy is for private university work. Changes are marked
with "# [adapted]" comments; the rest is unchanged.
"""
import sys
import os
import random

# [adapted] removed two unused imports of modules missing from the repository:
# from Benchmark.CortexDiffusion.geometry import norm
# from model.GLNO.pointnet import FPSPointNetModule
import scipy
import scipy.sparse.linalg as sla

import numpy as np
import torch
import torch.nn as nn

from .geometry import to_basis, from_basis, rotate  # [adapted] local copy of model/geometry.py
from einops.layers.torch import Rearrange

DEBUG=os.getenv('DEBUG_MODE', '0') == '1'

class Laplace_Transform_Layer(nn.Module):
    """
    Laplace Transform with learnable non-linear basis and learnable system poles and residues on spectral domain.

    Inputs:
      - values: (V,C) in the spectral domain
      - L: (V,V) sparse laplacian
      - evals: (K) eigenvalues
      - mass: (V) mass matrix diagonal

      (note: L/evals may be omitted as None depending on method)
    Outputs:
      - (V,C) diffused values
    """

    def __init__(self, config):
        super(Laplace_Transform_Layer, self).__init__()
        self.C_inout = config["C_width"]
        self.in_channels = config["C_width"]
        self.out_channels = config["C_width"]
        self.num_sigma = config["glno_sigma"]
        self.num_poles = config["glno_poles"]
        
        self.exp_x2_appr=config['exp_x2_appr']
        self.gaussian=config['gaussian']
        self.gaussian_factor=config.get('gaussian_factor',1)

        self.LT_kmin = config.get('k_min',0)
        self.LT_kmax = config.get('k_max',config['k_eig'])

        self.period_norm=config['glno_period_norm']
        self.aperiod_norm= config['glno_aperiod_norm']
        self.glno_norm=config['glno_norm']
        self.norm_period=NormLayer(self.period_norm,self.C_inout)
        self.norm_aperiod=NormLayer(self.aperiod_norm,self.C_inout)
        self.norm_glno=NormLayer(self.glno_norm,self.C_inout)

        self.scale=config.get("glno_scale",None)
        self.system_poles = nn.Parameter(torch.rand(self.C_inout, self.C_inout, self.num_poles, dtype=torch.cfloat)) #*5-2.5for car
        self.system_residues = nn.Parameter(torch.randn(self.C_inout, self.C_inout, self.num_poles, dtype=torch.cfloat))
        self.sigma = nn.Parameter(-(torch.rand(self.C_inout, self.num_sigma, dtype=torch.float))) #default 0.02
        if self.scale:
            self.system_poles.data *= self.scale
            self.system_residues.data *= self.scale
            self.sigma.data *= self.scale
            
        self.normalize_pole=config["normalize_pole"]
        self.normalize_k=self.LT_kmax
        self.normalize_basis=config.get("normalize_basis",False)
        self.safe_mode=config.get("safe_mode",True)
        self.basis_norm=config.get("basis_norm",1)
        # [adapted] False keeps only the periodic (spectral) branch Φ diag(H(λ)) Φᵀ M, which does not
        # change when an eigenvector changes sign; the default (True) is the repository's layer
        self.aperiodic=config.get("aperiodic",True)
        # [adapted] True computes the normalised 'expx2' weights as a softmax: the same weights, but when
        # ω² is far from every eigenvalue (e.g. above μ_K/700 with gaussian_factor 700) every exp
        # underflows, and exp/Σexp is 0/0 with NaN gradients; the default (False) is the repository's
        self.gaussian_stable=config.get("gaussian_stable",False)

    def gaussian_spectral_filter(self, omega, evals, residue):
        """
        Parameter:
            omega: pole_freq [c_in,c_out,num_pole]
            sigma: [batch,c_in,num_eig]
            evals: [batch,num_eig]
            evecs: [batch,num_vectices,num_eig]
            residue: [batch,c_in,c_out,num_eig]
            
        Return:  [n_vertices]
        """
        # 计算目标特征值 λ = ω²
        omega_ = omega ** 2
        batch, _ = evals.shape

        # gaussian kernel weights
        scale = omega_.unsqueeze(0).unsqueeze(-1) - evals.reshape(batch, 1, 1, 1, -1)
        if self.gaussian=='expx2' and self.gaussian_stable:  # [adapted] see __init__
            weights = torch.softmax(-(scale*self.gaussian_factor)**2, dim=-1)
        else:
            if self.gaussian=='expx2':
                weights = torch.exp(-(scale*self.gaussian_factor)**2)
            elif self.gaussian=='x2':
                weights = scale**2
            elif self.gaussian=='expx':
                weights=torch.exp(-scale)
            else:
                raise ValueError("gaussian_x2 should be expx2/x2/expx")

            # normalize weights
            weights = weights / torch.sum(weights, dim=-1, keepdim=True)
        if self.safe_mode:
            weights= torch.where(torch.isnan(weights), torch.zeros_like(weights), weights)
        weights = torch.where(weights < 1e-5, torch.zeros_like(weights), weights)

        efuc = torch.einsum("biopk,biop->biopk", weights, torch.real(residue))
        
        return efuc

    def pole_to_operator(self, pole, evals, evecs, residue, geo_feat, **kwargs):
        """
            pole:  [c_in,c_out,num_pole]
            residue:  [batch,c_out,num_eig]
        """
        x, y = pole.real, pole.imag

        approx_func = self.gaussian_spectral_filter(y, evals, residue)

        # use Taylor expansion to approximate exponential operator
        term1=torch.einsum("biopk,iop->bok",approx_func,x)
        
        operator0 = torch.einsum("biopk,bnk->bon",approx_func,evecs)
        operator1 = torch.einsum("bok,bnk,bn->bon",term1,evecs,geo_feat)
        operator=operator0-operator1

        if self.exp_x2_appr:
            term2=torch.einsum("biopk,iop->bok",approx_func,x**2)
            operator2 = torch.einsum("bok,bnk,bn->bon",term2,evecs,geo_feat**2)
            operator+=operator2*0.5

        return operator#/self.num_poles

    def PoleRes(self, evals, x_spec, system_poles, system_residues):
        """
        Pole residue calculation

        parameter:
            evals: (batch,num_eigenvectors)
            sigma: (in_channels,sigma)
            x_spec: input spectral (batch_size, in_channels, num_eigenvectors*sigma)
            system_poles: (in_channels, out_channels, num_poles)
            system_residues: (in_channels, out_channels, num_poles)

        return:
            output_residue1: periodic_signal
            output_residue2: aperiodic_signal
        """
        # batch_size, in_channels, num_eigenvectors = x_spec.shape
        # out_channels = system_poles.shape[1]

        # evals_extended: (1, 1, 1, num_eigenvectors)
        batch_size, in_channels, _ = x_spec.shape

        evals_extended = torch.sqrt(torch.abs(evals)).unsqueeze(1).unsqueeze(-1)

        if DEBUG:
           assert torch.isnan(evals_extended).any()==False, "evals_extended contains NaN"

        sigma = self.sigma.unsqueeze(1).unsqueeze(0)
        pole_in = evals_extended*1j + sigma
        pole_in = pole_in.reshape(batch_size, in_channels, 1, 1, -1)
        # [batch,inc,1,1,k_eig*sigma]

        # system_poles_extended: (in_channels, out_channels, num_poles, 1)
        system_poles_extended = system_poles.unsqueeze(-1).unsqueeze(0)

        # Hw = residue / (eval - pole)
        term1 = torch.div(1, torch.sub(pole_in, system_poles_extended))
        if self.safe_mode:
            term1_real = torch.nan_to_num(term1.real, nan=0.0)
            term1_imag = torch.nan_to_num(term1.imag, nan=0.0)
            term1 = torch.complex(term1_real, term1_imag)
        if DEBUG:
            assert torch.isnan(term1).any()==False, "term1 contains NaN"
        # [batch,inc,outc,numpole,k_eig*sigma]
        Hw = system_residues.unsqueeze(-1).unsqueeze(0) * term1
        # [batch,inc,outc,numpole,k_eig*sigma]
        # print(Hw.shape,x_spec.shape)
        # output_residue1 = α * H(λ) (periodic)
        output_residue1 = torch.einsum("bix,biokx->biox", x_spec, Hw)
        # output_residue2 = α * H(μ) (aperiodic)
        output_residue2 = torch.einsum("bix,biokx->biok", x_spec, -Hw)
        return output_residue1, output_residue2

    def forward(self, x, mass, evals, evecs, geo_feat):
        if x.shape[-1] != self.C_inout:
            raise ValueError(
                "Tensor has wrong shape = {}. Last dim shape should have number of channels = {}".format(
                    x.shape, self.C_inout))

        # Transform to spectral
        batch_size = x.shape[0]

        if evals.shape[-1]<self.LT_kmax:
            raise ValueError("evals.shape[-1]<self.LT_kmax")

        if self.normalize_pole:
            evals = (evals / evals[...,self.normalize_k-1].unsqueeze(-1)) * self.normalize_pole

        evals=evals[...,self.LT_kmin:self.LT_kmax]
        evecs=evecs[...,self.LT_kmin:self.LT_kmax]       

        # print(self.sigma.shape,geo_feat.shape)
        basis = torch.exp(-torch.einsum("cs,bn->bcsn", self.sigma, geo_feat))
        if self.normalize_basis:
            basis = basis / torch.norm(basis, dim=-1, keepdim=True) #normalize basis
        x_scale = torch.einsum("bnc,bcsn->bsnc", x, basis)
        x_spec = to_basis(x_scale, evecs, mass)
        # x_spec=[batch,sigma,num_eig,inc]

        if DEBUG:
            assert torch.isnan(x_scale).any()==False, "x_scale contains NaN"
            assert torch.isnan(basis).any()==False, "basis contains NaN"
            assert torch.isnan(x_spec).any()==False, "x_spec contains NaN"

        num_eig = x_spec.shape[-2]
        x_spec = x_spec.permute(0, 2, 1, 3).reshape(batch_size, -1, self.C_inout)
        # x_spec=[batch,num_eig*sigma,inc]
        x_spec_lno = x_spec.transpose(1, 2).to(torch.cfloat)
        # x_spec_lno=[batch,inc,num_eig*sigma]

        # Compute output residues
        output_residue1, output_residue2 = self.PoleRes(
            evals, x_spec_lno, self.system_poles, self.system_residues
        )
        
        if DEBUG:
            assert torch.isnan(output_residue1).any()==False, "output_residue1 contains NaN"
            assert torch.isnan(output_residue2).any()==False, "output_residue2 contains NaN"

        # Transform back to per-vertex
        # output_residue1=[batch,outchannel,num_eig*sigma]
        x1_spec = torch.real(output_residue1).reshape(batch_size, self.C_inout, self.C_inout, num_eig, self.num_sigma).permute(0,4,2,3,1)
        x1 = from_basis(x1_spec, evecs)
        # x1=[batch,sigma,out,num_vec,in]
        x_period = torch.einsum("bsoni,bisn->bno", x1, basis)
        # print(torch.max(x1_spec),torch.max(basis),torch.max(x_period))
        if self.period_norm:
            x_period=self.norm_period(x_period)
        else:
            x_period=x_period/self.num_sigma/self.C_inout
        
        if self.aperiodic:  # [adapted] the aperiodic branch can be switched off (see __init__)
            x_aperiod = self.pole_to_operator(self.system_poles, evals, evecs, output_residue2, geo_feat)

            if self.aperiod_norm:
                x_aperiod=self.norm_aperiod(x_aperiod.transpose(1, 2))
            else:
                x_aperiod=x_aperiod.transpose(1, 2)/self.num_sigma/self.C_inout

            x_glno = x_period + x_aperiod
        else:
            x_glno = x_period
        # x_glno = torch.nan_to_num(x_glno, nan=0.0)
        if DEBUG:
            assert torch.isnan(x_glno).any()==False

        if self.glno_norm:
            x_glno=self.norm_glno(x_glno)
            
        return x_glno
    
class MLP(nn.Sequential):
    '''
    A simple MLP with configurable hidden layer sizes.
    '''
    def __init__(self, layer_sizes, dropout=False, norm='layer', act='relu', name="MLP"):
        super(MLP, self).__init__()
        activation = {'gelu': nn.GELU(), 'relu': nn.ReLU(), 'tanh': nn.Tanh(),
                      'sigmoid': nn.Sigmoid(), 'leaky_relu': nn.LeakyReLU(),
                      'elu': nn.ELU(), 'softplus': nn.Softplus()}.get(act, None)
        
        for i in range(len(layer_sizes) - 1):
            is_last = (i + 2 == len(layer_sizes))

            if dropout and i > 0:
                self.add_module(
                    name + "_layer_dropout_{:03d}".format(i),
                    nn.Dropout(p=dropout)
                )

            self.add_module(
                name + "_layer_{:03d}".format(i),
                nn.Linear(layer_sizes[i], layer_sizes[i + 1]),
            )

            if not is_last and norm: ##增加了not is_last
                self.add_module(
                    name + f"_{norm}norm_{i:03d}",
                    NormLayer(norm, layer_sizes[i + 1])
                )   #原来是layer

            # Nonlinearity
            # (but not on the last layer)
            if not is_last and activation is not None:
                self.add_module(
                    name + "_act_{:03d}".format(i),
                    activation
                )

class EdgeConvNet(nn.Module):  # [adapted] torch_geometric is imported only when this class is used
    """A simple GNN with configurable hidden layer sizes. Using MLP as MLP"""

    def __init__(self, in_channels, out_channels, hidden_dims, dropout=True,norm='layer', activation='relu'):
        super(EdgeConvNet, self).__init__()
        from torch_geometric.nn.conv.edge_conv import EdgeConv  # [adapted] moved from module level
        # print(in_channels,out_channels,hidden_dims,dropout,norm,activation)
        self.mlp = MLP([2*in_channels]+hidden_dims+[out_channels], dropout=dropout, norm=norm, act=activation, name="edge_conv_mlp")
        self.edge_conv = EdgeConv(nn=self.mlp, aggr='sum')

    def forward(self, x, edge_index=None):
        if edge_index is None:
            return x

        # KNN
        if x.shape[0] == 1:
            # edge_index_ = torch.tensor(edge_index[0],dtype=torch.long)
            x_out = self.edge_conv(x[0], edge_index[0]).unsqueeze(0)
        else:
            x_out = []
            for i in range(x.shape[0]):
                edge_index_ = torch.tensor(edge_index[i], dtype=torch.long).to(x.device)
                x_out.append(self.edge_conv(x[i], edge_index_))
            x_out = torch.stack(x_out)

        return x_out

class NormLayer(nn.Module):
    def __init__(self, method, num_features, affine=True):
        super(NormLayer, self).__init__()
        self.num_features=num_features
        if method == 'layer':
            self.norm = nn.LayerNorm(num_features, elementwise_affine=affine)
        elif method == 'batch':
            self.norm = nn.Sequential(
                        Rearrange('b n c -> b c n'),
                        nn.BatchNorm1d(num_features, track_running_stats=True, affine=affine),
                        Rearrange('b c n -> b n c')
                    )
        elif method == 'instance':
            self.norm=nn.Sequential(
                        Rearrange('b n c -> b c n'),
                        nn.InstanceNorm1d(num_features, affine=affine),
                        Rearrange('b c n -> b n c')
                    )
        elif method == 'group':
            self.norm=nn.Sequential(
                        Rearrange('b n c -> b c n'),
                        nn.GroupNorm(num_groups=16, num_channels=num_features, affine=affine),
                        Rearrange('b c n -> b n c')
                    )
        elif method is None:
            self.norm=None
        else:
            raise ValueError("invalid norm type")

    def forward(self, x):
        if x.shape[-1] != self.num_features and x.shape[-2] == self.num_features:
            x = x.transpose(-1, -2)
        elif x.shape[-1] != self.num_features:
            raise ValueError("input feature dimension does not match num_features")
        flag=False
        if len(x.shape)<3:
            x = x.unsqueeze(0)
            flag=True
        
        if self.norm is not None:
            x = self.norm(x)
        if flag:
            x = x.squeeze(0)
        return x

class GLNOBlock(nn.Module):
    def __init__(self, idx, config):
        super(GLNOBlock, self).__init__()

        self.C_width = config["C_width"]
        self.idx=idx

        # Operator
        self.LT = Laplace_Transform_Layer(config=config)

        self.learned_geo_feat=config['learned_geo_feat']
        if self.learned_geo_feat:
            self.geo_feat_dim=config['geo_feat_dim']
            self.geo_hidden_layers=config.get('geo_feat_hidden_layers',[self.C_width*2,self.C_width*2])
            self.geo_dropout=config.get('geo_feat_dropout',0.0)
            self.geo_norm=config.get('geo_feat_norm',None)
            self.geo_activation=config.get('geo_feat_activation',None)
            self.mlp_geo=MLP([self.geo_feat_dim]+self.geo_hidden_layers+[1], dropout=self.geo_dropout, norm=self.geo_norm,act=self.geo_activation)
            self.last_geo_norm=NormLayer(config.get('geo_feat_last_norm',None),1)

        self.high_fre=config['high_fre']
        if self.high_fre:
            self.k_eig_high_fre=config['k_eig_high_fre']
            self.high_fre_hidden_dims=config.get("high_fre_hidden_layers",[self.C_width])
            self.high_fre_dropout=config.get("high_fre_dropout",0.0)
            self.high_fre_norm=config.get("high_fre_norm",None)
            self.high_fre_activation=config.get("high_fre_activation",None)
            self.mlp_high=MLP([self.C_width]+ self.high_fre_hidden_dims + [self.C_width], dropout=self.high_fre_dropout, norm=self.high_fre_norm,act=self.high_fre_activation)

        self.connect_method = config["connect_method"]

        if self.connect_method:
            self.mlp_hidden_dims = config.get("mlp_hidden_layers",[self.C_width,self.C_width])
            self.mlp_dropout = config.get("mlp_dropout",0.0)
            self.mlp_norm = config.get("mlp_norm",None)
            if config.get('mlp_norm_last_disable',None)==True and idx < config['blocks']:
                self.mlp_norm=None
            self.mlp_activation = config.get("mlp_activation",None)
            self.MLP_C = 2* self.C_width+1
            # MLPs
            if self.connect_method=='mlp':
                self.mlp = MLP([self.MLP_C] + self.mlp_hidden_dims + [self.C_width], dropout=self.mlp_dropout, norm=self.mlp_norm,act=self.mlp_activation)
            elif self.connect_method=='gnn':
                self.edge_conv = EdgeConvNet(self.MLP_C, self.C_width, hidden_dims=self.mlp_hidden_dims, dropout=self.mlp_dropout, norm=self.mlp_norm,activation=self.mlp_activation)
            else:
                raise ValueError("invalid connect_method")

        self.norm_feature=NormLayer(config["norm_feature"], self.MLP_C)        
        
        self.skip=config["skip"]

        
    def forward(self, x_in, mass, evals, evecs, geo_feat, pos=None, edges=None):
        if x_in.shape[-1] != self.C_width:
            raise ValueError(
                "Tensor has wrong shape = {e}. Last dim shape should have number of channels = {}".format(
                    x_in.shape, self.C_width))
        if self.learned_geo_feat:
            if len(geo_feat.shape)==2:
                geo_feat=geo_feat.unsqueeze(-1)
            geo_feat=self.mlp_geo(geo_feat)
            if self.last_geo_norm:
                geo_feat=self.last_geo_norm(geo_feat)
            geo_feat=geo_feat.squeeze(-1)

        x_glno = self.LT(x_in, mass, evals, evecs, geo_feat)

        if self.high_fre:
            if evecs.shape[-1]<self.k_eig_high_fre:
                raise ValueError("k_eig_high_fre should be less than or equal to the number of eigenvectors")
            evecs=evecs[...,:self.k_eig_high_fre]
            x_low=from_basis(to_basis(x_in.unsqueeze(1), evecs, mass), evecs).squeeze(1)
            x_high=x_in-x_low
            x_high=self.mlp_high(x_high)
            x_glno+=x_high
        
        if DEBUG:
            assert torch.isnan(x_glno).any()==False, "LT output contains NaN"
        
        if len(geo_feat.shape)==2:
            geo_feat=geo_feat.unsqueeze(-1)
        feature_combined = torch.cat((x_in, x_glno, geo_feat), dim=-1) 
        feature_combined = self.norm_feature(feature_combined)         
        
        # Apply the connect method
        if self.connect_method=='mlp':
            x0_out = self.mlp(feature_combined)
        elif self.connect_method=='gnn':
            if edges is None:
                raise ValueError("edges must be provided for gnn connect_method")
            x0_out = self.edge_conv(feature_combined, edges)
        else:
            raise ValueError("invalid connect_method")

        # Skip connection
        if self.skip:
            x0_out = x0_out + x_in

        return x0_out


class GLNONet(nn.Module):
    def __init__(self, config):
        """
        Construct a GLNONet.

        Parameters loaed from config:
            C_in (int):                     input dimension
            C_out (int):                    output dimension
            last_activation (str)          a function to apply to the final outputs of the network, such as torch.nn.functional.log_softmax (default: None)
            outputs_at (string)             produce outputs at various mesh elements by averaging from vertices. One of ['vertices', 'edges', 'faces', 'global_mean']. (default 'vertices', aka points for a point cloud)
            C_width (int):                  dimension of internal DiffusionNet blocks (default: 128)
            N_block (int):                  number of DiffusionNet blocks (default: 4)
            mlp_hidden_dims (list of int):  a list of hidden layer sizes for MLPs (default: [C_width, C_width])
            dropout (bool):                 if True, internal MLPs use dropout (default: True)
            diffusion_method (str):      how to evaluate diffusion, one of ['spectral', 'implicit_dense']. If implicit_dense is used, can set k_eig=0, saving precompute.
            """

        super(GLNONet, self).__init__()

        # Load parameters
        self.device=config["device"]
        
        self.C_in = config["C_in"]
        self.C_width = config["C_width"]
        self.N_block = config["blocks"]
        self.C_out = config["C_out"]
        self.vertices_dim = config["vertices_dim"]
        self.rotate = config["rotate"]
        self.geo_pre = config.get("geo_preprocess",None)
        self.k_eig=config['k_eig']

        if config["last_activation"] is None:
            self.last_activation = None
        elif config["last_activation"] == "log_softmax":
            self.last_activation = nn.LogSoftmax(dim=-1)
        else:
            raise ValueError("invalid setting for last_activation")
        
        self.outputs_at = config["outputs_at"]
        if self.outputs_at not in ['vertices', 'edges', 'faces', 'global_mean']:
            raise ValueError("invalid setting for outputs_at")
        
        ## Set up the network
        self.encoder_method=config.get('encoder',None)
        if self.encoder_method is None:
            self.encoder = nn.Linear(self.C_in, self.C_width)
        else:
            self.encoder_dropout=config["encoder_dropout"]
            self.encoder_norm=config["encoder_norm"]
            self.encoder_hidden_dims=config["encoder_hidden_layers"]
            self.encoder_activation=config["encoder_activation"]
            if config["encoder"]=='mlp':
                self.encoder = MLP([self.C_in] + self.encoder_hidden_dims + [self.C_width], dropout=self.encoder_dropout, norm=self.encoder_norm,act=self.encoder_activation)
            elif config["encoder"]=='gnn':
                self.encoder = EdgeConvNet(self.C_width, self.C_width, hidden_dims=self.encoder_hidden_dims, dropout=self.encoder_dropout,norm=self.encoder_norm,act=self.encoder_activation)
            else:
                raise ValueError("invalid encoder method")
        
        if config["decoder"]=='mlp':
            self.decoder_dropout=config["decoder_dropout"]
            self.decoder_norm=config["decoder_norm"]
            self.decoder_hidden_dims=config["decoder_hidden_layers"]
            self.decoder_activation=config["decoder_activation"]
            # [adapted] the decoder receives the last block's output (C_width channels), not C_width*N_block:
            self.decoder = MLP([self.C_width] + self.decoder_hidden_dims + [self.C_out], dropout=self.decoder_dropout, norm=self.decoder_norm,act=self.decoder_activation)
        elif config['decoder'] is None:
           self.decoder=nn.Linear(self.C_width, self.C_out)
        else: 
            raise ValueError("invalid decoder method")
        
        self.blocks = []
        for i_block in range(self.N_block):
            block = GLNOBlock(i_block,config=config)
            self.blocks.append(block)
            self.add_module("block_" + str(i_block), self.blocks[-1])

    def forward(self, data):
        """
        In the notation in this document, dimension are:
            - C: channel dimension (C_in/C_out on construction)
            - N: the number of vertices/points, which CAN be different for each forward pass
            - B: is an OPTIONAL batch dimension
            - K/K_EIG: is the number of eigenvalues used for spectral acceleration
            - V is vertices_dim at most 3

        Parameters:
            data should be a dictionary with the following keys:
                vertice (required):   tensor of vertex positions, dimension [N,V] or [B,N,V]
                input:     tensor of input features, dimension [N,C] or [B,N,C]
                faces:     tensor of face indices, dimension [F,3] or [B,F,3]
                edges:     tensor of edge indices, dimension [E,2] or [B,E,2]
                mass (required):      Mass vector, dimension [N] or [B,N]
                evals (required):     Eigenvalues of Laplace matrix, dimension [K_EIG] or [B,K_EIG]
                evecs (required):     Eigenvectors of Laplace matrix, dimension [N,K_EIG] or [B,N,K_EIG]
                geo_feat (required):   tensor of geometric features, dimension [N,C] or [B,N,C] (C>0 if learned geometric features block is used)
            
        Returns:
            x_out (tensor):    Output with dimension [N,C_out] or [B,N,C_out]
        """
        ## Real data
        x_in=None
        if "input" in data:
            x_in=data["input"].to(self.device)
        if "mass" in data:
            mass = data["mass"].to(self.device)
        else:
            raise ValueError("input data must contain'mass' key")
        if "evals" in data:
            evals = data["evals"].to(self.device)
            if evals.shape[-1]<self.k_eig:
                raise ValueError("input evals must have at least k_eig elements")
        else:
            raise ValueError("input data must contain 'evals' key")
        if "evecs" in data:
            evecs = data["evecs"].to(self.device)
        else:
            raise ValueError("input data must contain 'evecs' key")
        if "geo_feat" in data:
            geo_feat = data["geo_feat"].to(self.device)
            if self.geo_pre=='exp':
                geo_feat=torch.exp(-geo_feat)
            elif self.geo_pre=='max':
                geo_feat=geo_feat/torch.max(geo_feat,dim=-1,keepdim=True)
            elif self.geo_pre is not None:
                raise ValueError("geometry feature preprocessing method not recognized")
        else:
            raise ValueError("input data must contain 'geo_feat' key")
        
        edges,faces,vertices=None,None,None
        if "edges" in data:
            edges = data["edges"].to(self.device)
        if "faces" in data:
            faces = data["faces"].to(self.device)
        if "vertices" in data:
            vertices = data["vertices"].to(self.device)
            vertices = vertices[...,:self.vertices_dim]

            if self.rotate:
                vertices=rotate(vertices)
            if x_in is not None:
                x_in=torch.cat([x_in,vertices],dim=-1)
            else:
                x_in=vertices

        if x_in is None:
            raise ValueError("input data must contain 'input' or 'vertices' key")

        ## Check dimensions, and append batch dimension if not given
        if x_in.shape[-1] != self.C_in:
            raise ValueError(
                "DiffusionNet was constructed with C_in={}, but x_in has last dim={}".format(self.C_in, x_in.shape[-1]))

        if len(x_in.shape) == 2: # add a batch dim
            appended_batch_dim = True
            x_in = x_in.unsqueeze(0)
            mass = mass.unsqueeze(0)
            evals = evals.unsqueeze(0)
            evecs = evecs.unsqueeze(0)
            geo_feat = geo_feat.unsqueeze(0)
            if vertices is not None: vertices = vertices.unsqueeze(0)
            if edges is not None: edges = edges.unsqueeze(0)
            if faces is not None: faces = faces.unsqueeze(0)
        elif len(x_in.shape) == 3:
            appended_batch_dim = False
        else:
            raise ValueError("x_in should be tensor with shape [N,C] or [B,N,C]")

        ## Forward pass through the network
        if DEBUG:
            assert torch.isnan(x_in).any() == False, "input contains NaNs"
            assert torch.isnan(mass).any() == False, "mass contains NaNs"
            assert torch.isnan(evals).any() == False, "evals contains NaNs"
            assert torch.isnan(evecs).any() == False, "evecs contains NaNs"
            assert torch.isnan(geo_feat).any() == False, "geo_feat contains NaNs"
        
        x = self.encoder(x_in)
        if DEBUG:
            assert torch.isnan(x).any() == False, "encoder output contains NaNs"
        for b in self.blocks:
            x = b(x, mass, evals, evecs, geo_feat, vertices, edges)
        if DEBUG:
            assert torch.isnan(x).any() == False, "block output contains NaNs"
        x = self.decoder(x)
        if DEBUG:
            assert torch.isnan(x).any() == False, "decoder output contains NaNs"

        ## Remap output to requested output type
        if self.outputs_at == 'vertices':
            x_out = x

        elif self.outputs_at == 'edges':
            if edges is None:
                raise ValueError("edges must be provided for outputs_at='edges'")
            # Remap to edges
            x_gather = x.unsqueeze(-1).expand(-1, -1, -1, 2)
            edges_gather = edges.unsqueeze(2).expand(-1, -1, x.shape[-1], -1)
            xe = torch.gather(x_gather, 1, edges_gather)
            x_out = torch.mean(xe, dim=-1)

        elif self.outputs_at == 'faces':
            if faces is None:
                raise ValueError("faces must be provided for outputs_at='faces'")
            # Remap to faces
            x_gather = x.unsqueeze(-1).expand(-1, -1, -1, 3)
            faces_gather = faces.unsqueeze(2).expand(-1, -1, x.shape[-1], -1)
            xf = torch.gather(x_gather, 1, faces_gather)
            x_out = torch.mean(xf, dim=-1)

        elif self.outputs_at == 'global_mean':
            # Produce a single global mean ouput using a weighted mean according to the point mass/area which is discretization-invariant.
            x_out = torch.sum(x * mass.unsqueeze(-1), dim=-2) / torch.sum(mass, dim=-1, keepdim=True)

        # Apply last nonlinearity if specified
        if self.last_activation is not None:
            x_out = self.last_activation(x_out)

        # Remove batch dim if we added it
        if appended_batch_dim:
            x_out = x_out.squeeze(0)

        return x_out


In [ ]:
%%writefile src/schrodinger_glno/glno/shapes_data.py
"""Stage-3 data for the GLNO: what the network sees of every shape, built once and kept on one device.

``ShapeStore`` reads a dataset of ``shapes_dataset`` (one directory per shape) and builds, shape by
shape, the tensors of Plan_GLNO_etapa3.md §1:

``evecs``     ``E = Φ U``, the lumped-orthonormal Ritz basis of ``spectral.ritz_polar_basis``
``evals``     ``max(μ, 0) / eval_scale``, the blended-mass Ritz values Crank–Nicolson used
``mass``      the lumped mass, the one ``to_basis`` and the loss use
``geo_feat``  ``‖x - c‖ / geo_scale``, ``c`` the area centroid
``input``     ``amplitude · (Re ψ0, Im ψ0)``
``labels``    ``amplitude · (Re ψ(t_1), Im ψ(t_1), Re ψ(t_2), …)``: the saved states without t = 0
``clusters``  the eigenvalue groups of ``spectral.eigen_clusters``, for ``augment.augment_batch``

There is no ``vertices`` key: the network runs with ``C_in = 2``.

Layout. Shapes of one family share their tensors, padded with zeros to the largest mesh of that
family in the whole dataset (10 242 vertices for the spheres and 15 679 for the tori of
``data/shapes``, at most 1 % of padding), so a store of some shapes pads as the store of all of
them. A padded vertex has mass 0 and a zero row in ``evecs``, so it adds nothing to ``to_basis`` or
to a mass-weighted error; the network's output there is meaningless and is dropped with
``n_verts``. The Ritz basis is computed on the CPU, one shape at a time, and copied into tensors
allocated beforehand on ``device``: the host stays under ~1 GB while ``data/shapes`` fills
7.95 GiB of the device. A batch holds packets of one family (``batch`` refuses a mix); there is no
``DataLoader`` and there are no worker processes.

Packets are numbered in the order of the shapes, each shape's in-distribution packets before its
out-of-distribution ones; ``table`` has one entry per packet and ``rows`` selects from it. The
numbers are positions in one store: a store of other sets or shapes numbers its packets
differently, so what is saved per packet is keyed by the table's ``(shape, set, packet)``.

The Ritz basis is recomputed wherever the store is built, and LAPACK may round differently on
another CPU, so ``fingerprint`` keeps a few numbers per shape that pin down the basis (signs
included) and ``fingerprint_mismatch`` compares them with a tolerance; a hash of the bytes would
differ between two machines that agree to 1e-9.
"""
from dataclasses import dataclass
import json
import math
from pathlib import Path
import time
from typing import Callable, Iterable
import numpy as np
from numpy.typing import NDArray
import torch
from ..mesh import vertex_areas
from ..shapes_dataset import FAMILIES, MANIFEST_FILE, ShapesConfig, shapes_dataset_identity
from .spectral import (RITZ_VERSION, centroid, eigen_clusters, geo_feature, ritz_polar_basis,
                       spectral_oracle_error)

__all__ = ["SETS", "GROUPS", "ShapesPreprocess", "ShapeStore", "shape_arrays", "family_batches", "group_means",
           "fingerprint_mismatch", "training_geo_scale"]

SETS = ("train", "val", "test", "ood")              # packet sets; "ood" exists on test shapes only
GROUPS = ("select", "sphere", "torus", "round")     # reporting groups, see ShapeStore.group_positions
FINGERPRINT_VERSION = 1
_FAMILY_CODE = {family: code for code, family in enumerate(FAMILIES)}
_PROBE_PRIMES = ((2, 3, 5, 7), (11, 13, 17, 19))


@dataclass(frozen=True)
class ShapesPreprocess:
    """How a stored shape becomes network inputs; part of a run's identity.

    ``basis``: ``"ritz"`` (``E = Φ U`` with ``μ``) or ``"lumped"`` (the stored ``Φ`` with ``λ``, an
    ablation: it misses the data by up to 27 %). ``eval_scale`` is one constant for every shape,
    never a shape's own largest eigenvalue. ``geo_scale`` is the largest ``‖x - c‖`` over the
    training shapes of ``data/shapes`` (``training_geo_scale`` gives 1.403019) and stays fixed for
    any other dataset. ``amplitude`` brings a typical ``|ψ|`` per vertex near 1 on a surface of
    area 4π. ``drop_initial`` leaves t = 0 out of the labels: it is the input.
    """
    basis: str = "ritz"
    eval_scale: float = 700.0
    cluster_gap: float = 1e-4
    geo_scale: float = 1.4030
    amplitude: float = math.sqrt(8 * math.pi)
    drop_initial: bool = True

    def __post_init__(self) -> None:
        if self.basis not in ("ritz", "lumped"):
            raise ValueError("basis must be 'ritz' or 'lumped'")
        for name in ("eval_scale", "geo_scale", "amplitude"):
            value = getattr(self, name)
            if not (math.isfinite(value) and value > 0):
                raise ValueError(f"{name} must be finite and positive")
        if not (math.isfinite(self.cluster_gap) and self.cluster_gap >= 0):
            raise ValueError("cluster_gap must be finite and >= 0")


def _shape_indices(shapes) -> NDArray:
    """Shape indices as an int64 vector; refuses what is not an integer or a list of integers."""
    if np.iterable(shapes) and not isinstance(shapes, (str, np.ndarray)):
        shapes = list(shapes)                                   # a set, a range, a generator
    indices = np.atleast_1d(np.asarray(shapes))
    if indices.ndim != 1 or (indices.size and not np.issubdtype(indices.dtype, np.integer)):
        raise ValueError("shapes must be an integer or a list of integers")
    return indices.astype(np.int64)


def _probes(U: NDArray | None, values: NDArray) -> NDArray:
    """16 numbers ``aᵀ U b`` that move by O(1) when a column of ``U`` changes sign, and 4 of the eigenvalues.

    ``a`` and ``b`` are fixed cosines, not random draws, so they do not depend on a NumPy version.
    """
    k = len(values)
    modes = np.arange(1, k + 1)[:, None]
    a, b = (np.cos(modes * np.sqrt(primes)) for primes in _PROBE_PRIMES)
    core = a.T @ b if U is None else a.T @ U @ b
    return np.concatenate([core.ravel(), [values[min(1, k - 1)], values[k // 2], values[-1], values.mean()]])


def shape_arrays(folder: Path, config: ShapesConfig, preprocess: ShapesPreprocess) -> dict:
    """The network's view of the mesh in ``folder``, unpadded, as NumPy arrays.

    ``evecs`` (N, K) float32, ``evals`` (K,) float32 already divided by ``eval_scale``, ``mass`` (N,)
    and ``geo_feat`` (N,) float32, ``clusters`` (K,) int64 and ``probes`` (20,) float64 for the
    fingerprint. Refuses a mesh whose files disagree with ``config``, whose area is not the
    config's 4π (``amplitude`` and ``geo_scale`` assume it), or whose largest eigenvalue is not
    near ``K`` (Weyl's law on area 4π: ``λ_K ≈ K``; a wide window, against eigenvalues in other
    units).
    """
    with np.load(Path(folder) / "mesh.npz") as stored:
        verts, faces, mass, evals, evecs, t_out = (stored[key] for key in
                                                   ("verts", "faces", "mass", "evals", "evecs", "t_out"))
    k = config.k_eig
    if evecs.shape != (len(verts), k) or evals.shape != (k,) or mass.shape != (len(verts),):
        raise ValueError(f"{folder}: mesh.npz does not hold {k} eigenpairs of its {len(verts)} vertices")
    if t_out.shape != (config.output_steps,) or not np.allclose(t_out, config.output_times(), rtol=0, atol=1e-12):
        raise ValueError(f"{folder}: the saved times differ from the dataset config")
    if not np.allclose(mass, vertex_areas(verts, faces), rtol=1e-12, atol=0):
        raise ValueError(f"{folder}: the stored mass is not the lumped mass of the mesh")
    if abs(mass.sum() / config.area - 1) > 1e-9:
        raise ValueError(f"{folder}: the area of the mesh is {mass.sum():.9g}, not {config.area:.9g}")
    if preprocess.basis == "ritz":
        ritz = ritz_polar_basis(verts, faces, evecs)
        basis, values, U = ritz.E, ritz.mu, ritz.U
    else:
        phi = evecs.astype(np.float64)
        if not np.allclose(phi.T @ (mass[:, None] * phi), np.eye(k), rtol=0, atol=1e-6):   # as ritz_polar_basis
            raise ValueError(f"{folder}: the stored eigenvectors are not orthonormal in the lumped mass")
        if not np.isfinite(evals).all() or np.any(np.diff(evals) < 0):
            raise ValueError(f"{folder}: the stored eigenvalues are not finite and ascending")
        basis, values, U = evecs.astype(np.float32, copy=False), np.maximum(evals, 0.0), None
    if not 0.5 < values[-1] / k < 2:
        raise ValueError(f"{folder}: largest eigenvalue / K = {values[-1] / k:.3g}, expected about 1 "
                         "(Weyl's law on area 4π)")
    return {"evecs": basis, "evals": (values / preprocess.eval_scale).astype(np.float32),
            "mass": mass.astype(np.float32), "geo_feat": geo_feature(verts, mass, preprocess.geo_scale),
            "clusters": eigen_clusters(values, preprocess.cluster_gap), "probes": _probes(U, values)}


class ShapeStore:
    """Every selected packet of a stage-3 dataset as network inputs on ``device``.

    ``sets`` picks the packet sets to load and ``shapes`` (indices) restricts the shapes; a shape
    without a selected set is skipped. ``verify`` checks the SHA-256 of every file of the dataset
    first. ``progress`` receives one line per shape.

    Attributes: ``identity`` (the dataset's), ``config`` (its ``ShapesConfig``), ``preprocess``,
    ``steps`` and ``times`` of the labels, ``dt``, ``k_eig``, ``c_out`` (label channels),
    ``n_max`` (padded vertices per family), ``shapes`` (one dict per loaded shape), ``table`` (one
    entry per packet: ``shape``, ``set`` as an index into ``SETS``, ``family`` as an index into
    ``shapes_dataset.FAMILIES``, ``round``, ``packet``, ``n_verts``, and the rows ``slot`` and
    ``position`` of its shape and of itself in the family tensors), ``params`` and
    ``diagnostics`` (the dataset's columns, per packet) and ``nbytes``.
    """

    def __init__(self, directory: Path, preprocess: ShapesPreprocess | None = None, *,
                 device: str | torch.device = "cpu", sets: Iterable[str] = SETS,
                 shapes: int | Iterable[int] | None = None, verify: bool = False,
                 progress: Callable[[str], None] | None = None) -> None:
        directory = Path(directory)
        self.preprocess = preprocess = ShapesPreprocess() if preprocess is None else preprocess
        self.device = torch.device(device)
        self.identity = shapes_dataset_identity(directory, verify=verify)
        manifest = json.loads((directory / MANIFEST_FILE).read_text())
        self.config = config = ShapesConfig(**manifest["config"])
        sets = (sets,) if isinstance(sets, str) else tuple(sets)
        if not set(sets) <= set(SETS):
            raise ValueError(f"unknown packet sets {sorted(set(sets) - set(SETS))}; there are {SETS}")
        entries = manifest["shapes"]
        if [entry["index"] for entry in entries] != list(range(config.n_shapes)):
            raise ValueError("the manifest does not list the shapes in order")
        wanted = None if shapes is None else set(_shape_indices(shapes).tolist())
        if wanted is not None and not wanted <= set(range(config.n_shapes)):
            raise ValueError(f"shape indices must be in 0..{config.n_shapes - 1}")
        selected = []
        for entry in entries:
            index = entry["index"]
            if (entry["family"], entry["split"], entry["round"]) != (config.family(index), config.split(index),
                                                                     config.is_round(index)):
                raise ValueError(f"{entry['id']}: the manifest entry disagrees with the dataset config")
            packet_sets = [(name, prefix, count) for name, prefix, count, _ in config.packet_sets(index)
                           if name in sets]
            if packet_sets and (wanted is None or index in wanted):
                selected.append((entry, packet_sets))
        if not selected:
            raise ValueError("no packets selected")

        first = 1 if preprocess.drop_initial else 0
        self.steps, self.times = config.output_indices()[first:], config.output_times()[first:]
        self.dt, self.k_eig, self.c_out = config.dt, config.k_eig, 2 * len(self.steps)
        self.n_max = {family: max(entry["n_verts"] for entry in entries if entry["family"] == family)
                      for family in FAMILIES if any(entry["family"] == family for entry in entries)}
        self._families = {}
        for family in FAMILIES:
            members = [packet_sets for entry, packet_sets in selected if entry["family"] == family]
            if not members:
                continue
            n_shapes, n = len(members), self.n_max[family]
            n_packets = sum(count for packet_sets in members for _, _, count in packet_sets)

            def zeros(*shape: int) -> torch.Tensor:
                return torch.zeros(shape, dtype=torch.float32, device=self.device)

            self._families[family] = {
                "evecs": zeros(n_shapes, n, self.k_eig), "evals": zeros(n_shapes, self.k_eig),
                "mass": zeros(n_shapes, n), "geo_feat": zeros(n_shapes, n),
                "clusters": torch.zeros((n_shapes, self.k_eig), dtype=torch.int64),      # CPU, for augment_batch
                "inputs": zeros(n_packets, n, 2), "labels": zeros(n_packets, n, self.c_out)}

        amplitude = np.float32(preprocess.amplitude)
        slots, positions = dict.fromkeys(FAMILIES, 0), dict.fromkeys(FAMILIES, 0)
        columns = {key: [] for key in ("shape", "set", "family", "round", "packet", "n_verts", "slot", "position")}
        params, diagnostics, self.shapes, self._probes = [], [], [], {}
        for entry, packet_sets in selected:
            start = time.perf_counter()
            family, n, folder = entry["family"], entry["n_verts"], directory / entry["id"]
            data, slot = self._families[family], slots[family]
            arrays = shape_arrays(folder, config, preprocess)
            if len(arrays["mass"]) != n:
                raise ValueError(f"{entry['id']}: the mesh has {len(arrays['mass'])} vertices, the manifest says {n}")
            for key in ("evecs", "mass", "geo_feat"):
                data[key][slot, :n] = torch.from_numpy(arrays[key])
            data["evals"][slot] = torch.from_numpy(arrays["evals"])
            data["clusters"][slot] = torch.from_numpy(arrays["clusters"])
            self._probes[entry["id"]] = arrays["probes"]
            for name, prefix, count in packet_sets:
                inputs, outputs = np.load(folder / f"{prefix}inputs.npy"), np.load(folder / f"{prefix}outputs.npy")
                if (inputs.shape != (count, n, 2) or outputs.shape != (count, n, config.output_steps, 2)
                        or inputs.dtype != np.float32 or outputs.dtype != np.float32):
                    raise ValueError(f"{entry['id']}: {prefix}inputs.npy or {prefix}outputs.npy has the wrong shape "
                                     "or is not float32")
                if not np.isfinite(outputs).all():
                    raise ValueError(f"{entry['id']}: {prefix}outputs.npy has values that are not finite")
                if not np.array_equal(inputs, outputs[:, :, 0]):
                    raise ValueError(f"{entry['id']}: {prefix}inputs.npy is not the stored state at t = 0")
                position = positions[family]
                data["inputs"][position:position + count, :n] = torch.from_numpy(amplitude * inputs)
                data["labels"][position:position + count, :n] = torch.from_numpy(
                    amplitude * outputs[:, :, first:].reshape(count, n, self.c_out))
                params.append(np.load(folder / f"{prefix}params.npy"))
                diagnostics.append(np.load(folder / f"{prefix}diagnostics.npy"))
                if len(params[-1]) != count or len(diagnostics[-1]) != count:
                    raise ValueError(f"{entry['id']}: {prefix}params.npy or {prefix}diagnostics.npy has the wrong length")
                for key, value in (("shape", entry["index"]), ("set", SETS.index(name)),
                                   ("family", _FAMILY_CODE[family]), ("round", entry["round"]), ("n_verts", n),
                                   ("slot", slot)):
                    columns[key].append(np.full(count, value))
                columns["packet"].append(np.arange(count))
                columns["position"].append(np.arange(position, position + count))
                positions[family] += count
            slots[family] += 1
            seconds = time.perf_counter() - start
            self.shapes.append({"index": entry["index"], "id": entry["id"], "family": family, "split": entry["split"],
                                "round": entry["round"], "n_verts": n, "slot": slot, "seconds": seconds})
            if progress is not None:
                progress(f"{entry['id']}: {n} vertices, {sum(c for _, _, c in packet_sets)} packets, {seconds:.1f} s")
        self.table = {key: np.concatenate(values) for key, values in columns.items()}
        self.params, self.diagnostics = np.concatenate(params), np.concatenate(diagnostics)
        self.nbytes = sum(tensor.numel() * tensor.element_size()
                          for data in self._families.values() for tensor in data.values())

    def __len__(self) -> int:
        return len(self.table["shape"])

    def rows(self, *, sets: str | Iterable[str] | None = None, family: str | None = None,
             is_round: bool | None = None, shapes: int | Iterable[int] | None = None) -> NDArray:
        """Packet numbers, ascending, of the packets that pass every given filter."""
        keep = np.ones(len(self), dtype=bool)
        if sets is not None:
            sets = (sets,) if isinstance(sets, str) else tuple(sets)
            if not set(sets) <= set(SETS):
                raise ValueError(f"unknown packet sets {sorted(set(sets) - set(SETS))}; there are {SETS}")
            keep &= np.isin(self.table["set"], [SETS.index(name) for name in sets])
        if family is not None:
            if family not in _FAMILY_CODE:
                raise ValueError(f"unknown family {family!r}; there are {FAMILIES}")
            keep &= self.table["family"] == _FAMILY_CODE[family]
        if is_round is not None:
            if not isinstance(is_round, (bool, np.bool_)):
                raise ValueError("is_round must be True, False or None")
            keep &= self.table["round"] == bool(is_round)
        if shapes is not None:
            keep &= np.isin(self.table["shape"], _shape_indices(shapes))
        return np.flatnonzero(keep)

    def _checked(self, rows, *, allow_empty: bool = False) -> NDArray:
        rows = np.asarray(rows)
        if rows.ndim != 1:
            raise ValueError("rows must be a list of packet numbers")
        if len(rows) == 0:
            if allow_empty:
                return np.zeros(0, dtype=np.int64)
            raise ValueError("rows is empty")
        if not np.issubdtype(rows.dtype, np.integer) or rows.min() < 0 or rows.max() >= len(self):
            raise ValueError(f"packet numbers must be integers in 0..{len(self) - 1}")
        return rows.astype(np.int64)

    def batch(self, rows, device: str | torch.device | None = None) -> dict:
        """The packets ``rows`` (all of one family) as one batch for ``GLNONet``, moved to ``device`` if given.

        ``input`` (B, N, 2), ``labels`` (B, N, c_out), ``mass`` and ``geo_feat`` (B, N), ``evals``
        (B, K) and ``evecs`` (B, N, K), with N the family's padded size; and on the CPU ``clusters``
        (B, K), ``n_verts`` (B,) and ``rows`` (B,). The tensors are copies: changing them leaves
        the store as it is.
        """
        rows = self._checked(rows)
        codes = np.unique(self.table["family"][rows])
        if len(codes) != 1:
            raise ValueError("a batch holds packets of one family")
        data = self._families[FAMILIES[codes[0]]]
        slots = torch.from_numpy(self.table["slot"][rows])
        shape, packet = slots.to(self.device), torch.from_numpy(self.table["position"][rows]).to(self.device)
        batch = {"input": data["inputs"][packet], "labels": data["labels"][packet], "mass": data["mass"][shape],
                 "evals": data["evals"][shape], "evecs": data["evecs"][shape], "geo_feat": data["geo_feat"][shape]}
        if device is not None:
            batch = {key: value.to(device) for key, value in batch.items()}
        return batch | {"clusters": data["clusters"][slots], "n_verts": torch.from_numpy(self.table["n_verts"][rows]),
                        "rows": torch.from_numpy(rows)}

    def oracle_error(self, batch: dict) -> NDArray:
        """(B, T) spectral oracle of a batch, in float64: the Crank–Nicolson factor of each mode, applied in
        the batch's own ``evecs`` and ``evals`` to its ``input``, against its ``labels``.

        It is ``spectral.spectral_oracle_error`` computed from what the network is given, so it also
        checks the scales, the padding and the layout of the labels. With the Ritz basis it is the
        error the network could reach (Plan §1.5).
        """
        amplitude, scale = self.preprocess.amplitude, self.preprocess.eval_scale
        errors = np.empty((len(batch["rows"]), len(self.steps)))
        for i, n in enumerate(batch["n_verts"].tolist()):
            evecs, evals, mass, start, labels = (batch[key][i].double().cpu().numpy()
                                                 for key in ("evecs", "evals", "mass", "input", "labels"))
            target = labels[:n].reshape(n, len(self.steps), 2)
            errors[i] = spectral_oracle_error(evecs[:n], evals * scale, mass[:n],
                                              (start[:n, 0] + 1j * start[:n, 1]) / amplitude,
                                              (target[..., 0] + 1j * target[..., 1]) / amplitude, self.steps, self.dt)
        return errors

    def group_positions(self, rows) -> dict:
        """Where each reporting group sits in ``rows``: positions into ``rows``, not packet numbers.

        ``sphere`` and ``torus`` leave the tori of revolution out, ``round`` holds them and
        ``select`` is everything else (their copies in validation and test are near-copies of the
        training ones, so they are reported apart). ``rows`` must be in the order of the values
        the positions will index: for results collected batch by batch that is the batches'
        order, see ``family_batches``.
        """
        rows = self._checked(rows, allow_empty=True)
        family, is_round = self.table["family"][rows], self.table["round"][rows]
        groups = {"select": ~is_round, "round": is_round}
        groups |= {name: (family == _FAMILY_CODE[name]) & ~is_round for name in FAMILIES}
        return {name: np.flatnonzero(groups[name]) for name in GROUPS}

    def fingerprint(self) -> dict:
        """What identifies the basis of the loaded shapes, and only the basis: the scales, the amplitude and
        the rest of ``preprocess`` belong in the run's meta.

        Compare two of them with ``fingerprint_mismatch``, never with ``==``: the numbers change in
        their last digits with the BLAS thread count and the CPU. In a checkpoint it goes beside
        the meta, not inside a dict that is compared for equality.
        """
        return {"version": FINGERPRINT_VERSION, "ritz_version": RITZ_VERSION, "identity": self.identity,
                "basis": self.preprocess.basis, "k_eig": self.k_eig,
                "shapes": {name: probes.tolist() for name, probes in self._probes.items()}}


def family_batches(store: ShapeStore, rows, batch_size: int, *, seed: int | None = None,
                   epoch: int = 0) -> list[NDArray]:
    """Cut ``rows`` into batches of one family each.

    With a ``seed``, the packets are shuffled inside each family, cut, and the batches shuffled,
    all from ``(seed, epoch)`` and the set of ``rows``, not their order: a resumed run sees the
    batches of an uninterrupted one. Without it the packets keep their order inside each family
    and the families follow one another (for evaluation). The last batch of a family may be
    smaller.

    Either way ``np.concatenate(batches)`` is not ``rows`` when ``rows`` mixes families. File
    per-packet results by ``batch["rows"]`` (packet numbers), or give ``np.concatenate(batches)``,
    not ``rows``, to ``group_positions``, ``params`` and ``diagnostics``.
    """
    rows = store._checked(rows, allow_empty=True)
    if batch_size < 1:
        raise ValueError("batch_size must be >= 1")
    if len(np.unique(rows)) != len(rows):
        raise ValueError("rows repeats a packet")
    rng = None if seed is None else np.random.default_rng([seed, epoch])
    family = store.table["family"][rows]
    batches = []
    for code in np.unique(family):
        members = rows[family == code]
        if rng is not None:
            members = rng.permutation(np.sort(members))
        batches += [members[start:start + batch_size] for start in range(0, len(members), batch_size)]
    if rng is not None:
        batches = [batches[i] for i in rng.permutation(len(batches))]
    return batches


def group_means(values, groups: dict) -> dict:
    """The mean over the first axis of ``values`` for each group of ``ShapeStore.group_positions``.

    A group without packets (a set with no torus, say) gives ``None`` instead of a NaN or an error.
    ``values`` holds one entry per row given to ``group_positions``, in the same order.
    """
    values = np.asarray(values)
    count = len(groups["select"]) + len(groups["round"])            # the two split the rows
    if len(values) != count:
        raise ValueError(f"{len(values)} values for groups of {count} rows")
    return {name: values[positions].mean(axis=0) if len(positions) else None for name, positions in groups.items()}


def fingerprint_mismatch(a: dict, b: dict, *, tolerance: float = 1e-3, eval_tolerance: float = 1e-6) -> str | None:
    """Why two fingerprints describe different bases, or ``None`` when they agree on the shapes they share.

    The 16 basis probes of a shape move by O(1) when one eigenvector changes sign and by ~1e-5
    between LAPACK drivers (round tori, whose nearly equal pairs rotate by ~1e-6), so they are
    compared to ``tolerance`` in absolute value; the 4 eigenvalue probes to ``eval_tolerance``,
    relative.
    """
    for fingerprint in (a, b):
        if not (isinstance(fingerprint, dict) and isinstance(fingerprint.get("shapes"), dict)):
            return "not a fingerprint"
    for key in ("version", "ritz_version", "identity", "basis", "k_eig"):
        if a.get(key) != b.get(key):
            return f"{key} differs: {a.get(key)!r} and {b.get(key)!r}"
    shared = [name for name in a["shapes"] if name in b["shapes"]]
    if not shared:
        return "no shape in common"
    for name in shared:
        first, second = np.asarray(a["shapes"][name], dtype=float), np.asarray(b["shapes"][name], dtype=float)
        if (first.shape != second.shape or first.ndim != 1 or len(first) <= 4
                or not (np.isfinite(first).all() and np.isfinite(second).all())):
            return f"{name}: malformed probes"
        change = np.abs(first[:-4] - second[:-4]).max()
        if change > tolerance:
            return f"{name}: the basis differs (probes move by {change:.2e})"
        scale = np.maximum(np.abs(first[-4:]), np.abs(second[-4:]))
        change = np.divide(np.abs(first[-4:] - second[-4:]), scale, out=np.zeros(4), where=scale > 0).max()
        if change > eval_tolerance:
            return f"{name}: the eigenvalues differ (relative change {change:.2e})"
    return None


def training_geo_scale(directory: Path) -> float:
    """Largest ``‖x - c‖`` over the training shapes: the ``geo_scale`` that puts their ``geo_feat`` in [0, 1]."""
    directory = Path(directory)
    manifest = json.loads((directory / MANIFEST_FILE).read_text())
    largest = 0.0
    for entry in manifest["shapes"]:
        if entry["split"] == "train":
            with np.load(directory / entry["id"] / "mesh.npz") as stored:
                verts, mass = stored["verts"], stored["mass"]
            largest = max(largest, float(np.linalg.norm(verts - centroid(verts, mass), axis=1).max()))
    if largest == 0.0:
        raise ValueError("the dataset has no training shape")
    return largest


In [ ]:
%%writefile src/schrodinger_glno/glno/shapes_train.py
"""Stage-3 training of the GLNO on many shapes: the data of ``ShapeStore``, one process, resumable.

The optimizer is the paper's (AdamW, weight decay 1e-4, learning rate 1e-3 halved every 100
epochs, 300 epochs, batches of 4) and the loss the mass-weighted relative L2 error over the saved
times after t = 0. What stage 3 adds:

- every batch holds packets of one family (``shapes_data.family_batches``) and, while training,
  its basis gets random signs and rotations inside the eigenvalue groups (``augment.py``);
  validation uses the canonical basis;
- a step whose loss or gradient is not finite is dropped without touching the weights or the
  optimizer, and counted; more than ``max_skipped`` of an epoch stops the run;
- the gradient is clipped at ``grad_clip`` times the median of its recent norms (the step-4 trial
  saw isolated spikes of up to 280 times the median);
- ``best.pt`` is chosen on ``val_select``, the validation error without the tori of revolution
  (their validation copies are near-copies of training shapes).

``run_dir`` holds ``last.pt`` (every epoch), ``last_prev.pt`` (the one before, used when
``last.pt`` cannot be read: a file cut short by Drive), ``best.pt``, ``epoch_<n>.pt`` for the
epochs in ``snapshots`` and ``history.json``. A resume checks that ``best.pt`` is the checkpoint
the state names; when it is not (Drive kept an older one, or it cannot be read) a line says so
and ``best.pt`` becomes the best from that resume on. Every checkpoint carries the run's meta (dataset,
preprocessing, network and settings) and, beside it, the fingerprint of the Ritz basis. Resuming
asks for the same meta, compared with ``==``, and for a compatible fingerprint, compared with
``shapes_data.fingerprint_mismatch`` (its last digits change with the CPU). ``epochs``,
``grad_clip``, ``max_skipped`` and ``snapshots`` are not part of the meta: asking for more epochs
continues the run, and the clip can be switched on to rescue a run that keeps failing on the same
batches. The batches, the gauges and the clip threshold depend only on ``(seed, epoch, step)`` and
on what the checkpoint keeps, so a resumed run repeats an uninterrupted one. One process per
``run_dir``: two sessions writing the same run would mix their checkpoints.
"""
from collections import deque
from dataclasses import asdict, dataclass
import json
import math
import os
from pathlib import Path
import reprlib
import statistics
import time
from typing import Callable
import numpy as np
from numpy.typing import NDArray
import torch
from .augment import augment_batch, gauge_generator, tf32_matmul_enabled
from .data import glno_config, mass_weighted_relative_l2
from .layers import GLNONet
from .shapes_data import ShapesPreprocess, ShapeStore, family_batches, fingerprint_mismatch, group_means

__all__ = ["VARIANTS", "ShapesTrainSettings", "shapes_glno_config", "check_config", "run_meta", "train_step",
           "clip_threshold", "packet_errors", "evaluate_rows", "validation_summary", "train_shapes",
           "load_checkpoint", "checkpoint_preprocess", "predict_shape", "calibrate", "ieee_matmuls"]

TRAINER_VERSION = 1
# The two networks of the plan (D2). "full" is the paper's layer, with the gaussian_factor that did best in
# the step-4 trial; "periodic" keeps only the branch Φ diag(H(λ)) Φᵀ M, which does not see the signs of the basis.
VARIANTS = {"periodic": {"aperiodic": False},
            "full": {"aperiodic": True, "gaussian_factor": 50.0}}
GRAD_WINDOW = 200       # recent gradient norms kept for the clip threshold: one epoch of data/shapes
GRAD_WARMUP = 20        # no clipping before this many norms: the median of a handful says little
LAST, LAST_PREVIOUS, BEST, HISTORY = "last.pt", "last_prev.pt", "best.pt", "history.json"
_UNTRACKED_SETTINGS = ("epochs", "grad_clip", "max_skipped", "snapshots")
_LAST_KEYS = {"model", "optimizer", "scheduler", "history", "best_val", "grad_norms", "epoch", "meta", "fingerprint"}


@dataclass(frozen=True)
class ShapesTrainSettings:
    epochs: int = 300                   # Table 5 (Poisson)
    batch_size: int = 4                 # 200 steps per epoch on data/shapes
    lr: float = 1e-3                    # Table 5 (Poisson)
    weight_decay: float = 1e-4          # AdamW, as in the repository's train.py
    step_size: int = 100                # the paper halves the learning rate every 100 epochs
    gamma: float = 0.5
    seed: int = 0
    augment: bool = True                # random signs and rotations of the basis while training
    grad_clip: float | None = 10.0      # times the median of the recent gradient norms; None: no clipping
    max_skipped: float = 0.02           # fraction of an epoch's steps that may be dropped as not finite
    snapshots: tuple[int, ...] = (100, 200)     # epochs kept apart: 100 full epochs at each learning rate

    def __post_init__(self) -> None:
        if self.epochs < 0 or self.batch_size < 1 or self.step_size < 1:
            raise ValueError("epochs must be >= 0, batch_size and step_size >= 1")
        if not (self.lr > 0 and self.weight_decay >= 0 and 0 < self.gamma <= 1):
            raise ValueError("lr must be > 0, weight_decay >= 0 and gamma in (0, 1]")
        if self.grad_clip is not None and not (math.isfinite(self.grad_clip) and self.grad_clip > 0):
            raise ValueError("grad_clip must be positive, or None")
        if not 0 <= self.max_skipped <= 1:
            raise ValueError("max_skipped is a fraction of an epoch")
        if any(epoch < 1 for epoch in self.snapshots):
            raise ValueError("snapshots are numbers of finished epochs, from 1")


def shapes_glno_config(store: ShapeStore, variant: str, device: str | torch.device | None = None,
                       **overrides) -> dict:
    """The network of stage 3 for the data of ``store``: the paper's (4 blocks, 64 channels, 1 pole), fed with
    ψ0 alone (``C_in = 2``, no coordinates) and giving the states after t = 0 (``C_out = store.c_out``).

    ``variant`` is a key of ``VARIANTS``. ``device`` (the store's by default) is where the network
    runs; ``overrides`` change ``glno_config`` keys or the two switches ``aperiodic`` and
    ``gaussian_stable`` (on here: the same weights as the repository's filter without its 0/0).
    """
    if variant not in VARIANTS:
        raise ValueError(f"unknown variant {variant!r}; there are {tuple(VARIANTS)}")
    options = {"C_in": 2, "gaussian_stable": True} | VARIANTS[variant] | overrides
    switches = {key: options.pop(key) for key in ("aperiodic", "gaussian_stable")}
    config = glno_config(C_out=store.c_out, k_eig=store.k_eig, device=str(device or store.device), **options)
    config.update(switches)
    check_config(config, store)
    return config


def check_config(config: dict, store: ShapeStore) -> None:
    """Refuse the network options that break on padded batches of many meshes.

    Batch, instance and group normalisations average over the vertices, the padded ones included;
    ``normalize_basis`` does the same to the basis (the padding is then exact only to ~4e-7). The
    eigenvalue groups of the store are computed over all its modes, so the network must use all
    of them. The rest needs mesh connectivity or coordinates, which a batch does not carry.
    """
    problems = [f"{key}={value!r}" for key, value in config.items()
                if "norm" in key and not key.startswith("normalize") and value not in (None, "layer")]
    required = {"connect_method": "mlp", "encoder": "mlp", "decoder": "mlp", "outputs_at": "vertices",
                "k_min": 0, "k_max": store.k_eig, "k_eig": store.k_eig, "C_in": 2, "C_out": store.c_out}
    problems += [f"{key}={config.get(key)!r} (must be {value!r})" for key, value in required.items()
                 if config.get(key, value) != value]
    problems += [f"{key}={config[key]!r}" for key in ("rotate", "normalize_pole", "geo_preprocess", "normalize_basis",
                                                      "learned_geo_feat", "high_fre") if config.get(key)]
    if config.get("gaussian", "expx2") not in ("expx2", "x2", "expx"):
        problems.append(f"gaussian={config['gaussian']!r}")
    if problems:
        raise ValueError("stage 3 cannot train a network with " + ", ".join(problems))


def run_meta(store: ShapeStore, config: dict, settings: ShapesTrainSettings) -> dict:
    """What a run is: a checkpoint is continued only by a call with the same meta.

    ``shapes`` are the ones that train and the ones that choose ``best.pt``; a store that also
    holds test shapes, or does not, is the same run.
    """
    loaded = {name: sorted(set(store.table["shape"][store.rows(sets=name)].tolist())) for name in ("train", "val")}
    return {"trainer_version": TRAINER_VERSION, "dataset": store.identity, "preprocess": asdict(store.preprocess),
            "shapes": loaded,
            "network": {key: value for key, value in config.items() if key != "device"},
            "settings": {key: value for key, value in asdict(settings).items() if key not in _UNTRACKED_SETTINGS}}


def ieee_matmuls() -> None:
    """Turn TF32 matmuls off on CUDA: they would round the rotated basis to ~1e-3 (``augment_batch`` refuses them)."""
    if hasattr(torch.backends.cuda.matmul, "fp32_precision"):
        torch.backends.cuda.matmul.fp32_precision = "ieee"
    else:
        torch.backends.cuda.matmul.allow_tf32 = False
    if tf32_matmul_enabled():
        raise RuntimeError("could not turn TF32 matmuls off")


def train_step(model: GLNONet, optimizer: torch.optim.Optimizer, batch: dict,
               clip: float | None = None) -> tuple[float | None, float | None, bool]:
    """One optimisation step on ``batch``: the loss, the gradient norm before clipping, and whether it was clipped.

    When the loss or the gradient is not finite the step is dropped: the weights and the optimizer
    stay as they were and ``(None, None, False)`` comes back. A gradient longer than ``clip`` is
    scaled down to that norm.
    """
    loss = mass_weighted_relative_l2(model(batch), batch["labels"], batch["mass"])
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    norm = torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(gradient) for gradient in gradients]))
    loss, norm = loss.item(), norm.item()
    if not (math.isfinite(loss) and math.isfinite(norm)):
        optimizer.zero_grad(set_to_none=True)
        return None, None, False
    clipped = clip is not None and norm > clip
    if clipped:
        for gradient in gradients:
            gradient.mul_(clip / norm)
    optimizer.step()
    return loss, norm, clipped


def clip_threshold(norms, factor: float | None) -> float | None:
    """``factor`` times the median of the recent gradient norms, once there are ``GRAD_WARMUP`` of them."""
    if factor is None or len(norms) < GRAD_WARMUP:
        return None
    return factor * statistics.median(norms)


def packet_errors(pred: torch.Tensor, target: torch.Tensor, mass: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """``‖pred - target‖_M / ‖target‖_M`` of (B, N, 2T) outputs: (B, T) at each saved time and (B,) over all
    of them. The second is the loss of each packet; padded vertices have mass 0 and do not count."""
    size, n, channels = pred.shape
    weights = mass.reshape(size, n, 1)
    missed = torch.sum(weights * (pred - target).reshape(size, n, channels // 2, 2).square().sum(-1), dim=1)
    norm = torch.sum(weights * target.reshape(size, n, channels // 2, 2).square().sum(-1), dim=1)
    return torch.sqrt(missed / norm), torch.sqrt(missed.sum(1) / norm.sum(1))


def _device(model: GLNONet) -> torch.device:
    return next(model.parameters()).device


def _batch(store: ShapeStore, rows, device: torch.device) -> dict:
    return store.batch(rows, None if device == store.device else device)


@torch.no_grad()
def evaluate_rows(model: GLNONet, store: ShapeStore, rows, batch_size: int = 4) -> dict:
    """Errors of ``model`` on the packets ``rows`` in the canonical basis: ``per_time`` (n, T) and ``total`` (n,),
    in the order of ``rows``.

    The batches come family by family, not in that order, so every result is filed by its packet
    number (``batch["rows"]``) and read back through ``rows``.
    """
    rows = store._checked(rows, allow_empty=True)
    per_time, total = np.full((len(store), len(store.steps)), np.nan), np.full(len(store), np.nan)
    was_training, device = model.training, _device(model)
    model.eval()
    try:
        for members in family_batches(store, rows, batch_size):
            batch = _batch(store, members, device)
            each, whole = packet_errors(model(batch), batch["labels"], batch["mass"])
            filed = batch["rows"].numpy()
            per_time[filed], total[filed] = each.double().cpu().numpy(), whole.double().cpu().numpy()
    finally:
        model.train(was_training)
    return {"per_time": per_time[rows], "total": total[rows]}


def validation_summary(store: ShapeStore, rows, errors: dict) -> dict:
    """Means of ``evaluate_rows`` by reporting group, as plain numbers: ``total`` and ``per_time`` for each of
    ``shapes_data.GROUPS`` (``None`` for a group without packets)."""
    groups = store.group_positions(rows)
    total, per_time = group_means(errors["total"], groups), group_means(errors["per_time"], groups)
    return {"total": {name: None if value is None else float(value) for name, value in total.items()},
            "per_time": {name: None if value is None else value.tolist() for name, value in per_time.items()}}


def _save(state: dict, path: Path) -> None:
    """Write to a temporary file first, so an interrupted save never leaves a broken checkpoint."""
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    os.replace(temporary, path)


def _save_last(state: dict, run_dir: Path) -> None:
    """The new ``last.pt`` is complete on disk before the old one becomes ``last_prev.pt``."""
    last, temporary = run_dir / LAST, run_dir / "last.tmp"
    torch.save(state, temporary)
    if last.exists():
        os.replace(last, run_dir / LAST_PREVIOUS)
    os.replace(temporary, last)


def _differences(saved: dict, asked: dict) -> list[str]:
    """The entries of the meta that differ, named one by one."""
    found, absent = [], "<absent>"
    for key, value in asked.items():
        old = saved.get(key)
        if old == value:
            continue
        if isinstance(old, dict) and isinstance(value, dict):
            found += [f"{key}.{name} ({reprlib.repr(old.get(name, absent))} in the checkpoint, "
                      f"{reprlib.repr(value.get(name, absent))} now)"
                      for name in sorted(set(old) | set(value)) if old.get(name, absent) != value.get(name, absent)]
        else:
            found.append(f"{key} ({reprlib.repr(old)} in the checkpoint, {reprlib.repr(value)} now)")
    return found


def _checked_best(run_dir: Path, state: dict, log: Callable[[str], None]) -> float:
    """``best_val`` of the state when ``best.pt`` is the checkpoint it names; otherwise ``inf``, so that the
    next epoch writes a ``best.pt`` that agrees with the history from here on."""
    best = state["best_val"]
    if not math.isfinite(best):
        return best
    try:
        held = torch.load(run_dir / BEST, map_location="cpu", weights_only=False)
        if held["epoch"] <= state["epoch"] and held["record"]["val_select"] == best:
            return best
        problem = f"is the end of epoch {held['epoch'] + 1} with val_select {held['record']['val_select']:.6g}"
    except Exception as error:
        problem = f"cannot be read ({type(error).__name__})"
    log(f"{BEST} {problem}, not the best of this state (val_select {best:.6g}): the next epoch replaces it")
    return float("inf")


def _write_history(history: list[dict], run_dir: Path) -> None:
    temporary = run_dir / "history.tmp"
    temporary.write_text(json.dumps(history, indent=1) + "\n")
    os.replace(temporary, run_dir / HISTORY)


def _read_last(run_dir: Path, device: torch.device, log: Callable[[str], None]) -> tuple[dict | None, str | None]:
    """The newest readable state of the run and the file it came from, or ``(None, None)`` for a new run; an
    unreadable one is never overwritten."""
    found = [run_dir / name for name in (LAST, LAST_PREVIOUS) if (run_dir / name).exists()]
    for path in found:
        try:
            state = torch.load(path, map_location=device, weights_only=False)
            if not (isinstance(state, dict) and _LAST_KEYS <= set(state)):
                raise ValueError("it is not a checkpoint of this trainer")
        except Exception as error:      # a file cut short by Drive fails in pickle, zipfile or torch
            log(f"{path.name} cannot be read ({type(error).__name__}: {error})")
            continue
        if path.name != LAST:
            log(f"continuing from {path.name}: one epoch is repeated")
        return state, path.name
    if found:
        raise ValueError(f"{run_dir}: {' and '.join(path.name for path in found)} cannot be read; "
                         f"use a new run_dir to start again ({BEST} and the snapshots here still hold weights)")
    if (run_dir / HISTORY).exists():
        raise ValueError(f"{run_dir} holds the {HISTORY} of a run but neither {LAST} nor {LAST_PREVIOUS} "
                         "(is Drive mounted and complete?); use a new run_dir to start again")
    return None, None


def _kept(epoch: int) -> str:
    """What is on disk when the run stops inside the epoch of index ``epoch``."""
    return f"{LAST} is the end of epoch {epoch}" if epoch else "no checkpoint was written yet"


def _epoch_seed(seed: int, epoch: int) -> int:
    return int(np.random.SeedSequence([seed, epoch, 1]).generate_state(1, dtype=np.uint64)[0])


def train_shapes(store: ShapeStore, run_dir: Path, config: dict,
                 settings: ShapesTrainSettings = ShapesTrainSettings(),
                 log: Callable[[str], None] = print) -> list[dict]:
    """Train (or resume) the network ``config`` on the ``train`` packets of ``store``; returns the history.

    ``config`` comes from ``shapes_glno_config`` and names the device. ``store`` must hold the
    ``train`` and ``val`` sets; with the four sets, the fingerprint in the checkpoints covers
    every shape an evaluation will load.
    """
    run_dir = Path(run_dir)
    check_config(config, store)
    device = torch.device(config["device"])
    if device.type == "cuda":
        ieee_matmuls()
    train_rows, val_rows = store.rows(sets="train"), store.rows(sets="val")
    if len(train_rows) == 0:
        raise ValueError("the store holds no training packets")
    if len(store.group_positions(val_rows)["select"]) == 0:
        raise ValueError("the store holds no validation packets outside the tori of revolution: "
                         "there is nothing to choose best.pt with")
    run_dir.mkdir(parents=True, exist_ok=True)
    probe = run_dir / "write_test.tmp"          # a folder that cannot be written fails here, not after an epoch
    probe.write_bytes(b"run_dir can be written")
    probe.unlink()
    meta, fingerprint = run_meta(store, config, settings), store.fingerprint()
    torch.manual_seed(settings.seed)
    model = GLNONet(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings.lr, weight_decay=settings.weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=settings.step_size, gamma=settings.gamma)
    history, best, start_epoch, norms = [], float("inf"), 0, deque(maxlen=GRAD_WINDOW)
    state, source = _read_last(run_dir, device, log)
    if state is not None:
        differ = _differences(state["meta"], meta)
        if differ:
            raise ValueError(f"{run_dir} holds a run with another {', '.join(differ)}; use a new run_dir")
        reason = fingerprint_mismatch(state["fingerprint"], fingerprint)
        if reason is not None:
            raise ValueError(f"{run_dir}: the Ritz basis built here is not the checkpoint's ({reason})")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, best, start_epoch = state["history"], _checked_best(run_dir, state, log), state["epoch"] + 1
        norms.extend(state["grad_norms"])
        if source != LAST and (run_dir / LAST).exists():    # the next save must not make it last_prev.pt
            os.replace(run_dir / LAST, run_dir / "last_unreadable.pt")
        _write_history(history, run_dir)                    # a cut between last.pt and history.json
        log(f"Resuming at epoch {start_epoch} (best val_select so far {state['best_val']:.4g})")
    else:
        log(f"New run in {run_dir}")
    log(f"device={device}  parameters={sum(p.numel() for p in model.parameters()):,}  "
        f"train={len(train_rows)}  val={len(val_rows)}  store={store.nbytes / 2**30:.2f} GiB")

    for epoch in range(start_epoch, settings.epochs):
        start = time.perf_counter()
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        torch.manual_seed(_epoch_seed(settings.seed, epoch))        # dropout, if any, repeats after a resume
        model.train()
        batches = family_batches(store, train_rows, settings.batch_size, seed=settings.seed, epoch=epoch)
        total, samples, clipped, skipped, epoch_norms = 0.0, 0, 0, 0, []
        for step, members in enumerate(batches):
            batch = _batch(store, members, device)
            if settings.augment:
                batch = augment_batch(batch, gauge_generator(settings.seed, epoch, step))
            loss, norm, was_clipped = train_step(model, optimizer, batch, clip_threshold(norms, settings.grad_clip))
            if loss is None:
                skipped += 1
                if skipped > settings.max_skipped * len(batches):
                    raise RuntimeError(
                        f"epoch {epoch + 1}: {skipped} of {len(batches)} steps had a loss or a gradient that is "
                        f"not finite (allowed: {settings.max_skipped:.0%}). The weights were not touched and "
                        f"{_kept(epoch)}; grad_clip can be changed without starting again")
                continue
            norms.append(norm)
            epoch_norms.append(norm)
            total, samples, clipped = total + loss * len(members), samples + len(members), clipped + was_clipped
        train_seconds = time.perf_counter() - start
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step()
        summary = validation_summary(store, val_rows, evaluate_rows(model, store, val_rows, settings.batch_size))
        val_select = summary["total"]["select"]
        record = {"epoch": epoch, "train_loss": total / samples if samples else None, "val_select": val_select,
                  "val_round": summary["total"]["round"], "val_sphere": summary["total"]["sphere"],
                  "val_torus": summary["total"]["torus"], "val_per_time": summary["per_time"],
                  "lr": lr, "steps": len(batches), "skipped": skipped,
                  "clipped": clipped, "grad_clip": settings.grad_clip,
                  "grad_norm_median": statistics.median(epoch_norms) if epoch_norms else None,
                  "grad_norm_max": max(epoch_norms) if epoch_norms else None,
                  "ms_per_sample": round(1000 * train_seconds / len(train_rows), 2),
                  "seconds": round(time.perf_counter() - start, 1)}
        if device.type == "cuda":
            record["reserved_gib"] = round(torch.cuda.max_memory_reserved(device) / 2**30, 2)
        history.append(record)
        if not all(torch.isfinite(parameter).all() for parameter in model.parameters()):
            raise RuntimeError(f"epoch {epoch + 1}: the weights are not finite; nothing was saved and "
                               f"{_kept(epoch)}")
        checkpoint = {"model": model.state_dict(), "config": config, "meta": meta, "fingerprint": fingerprint,
                      "epoch": epoch, "record": record, "torch": str(torch.__version__)}
        if not math.isfinite(val_select):
            log(f"epoch {epoch + 1}: val_select is not finite; {BEST} is not updated")
        if val_select < best:
            best = val_select
            _save(checkpoint, run_dir / BEST)
        if epoch + 1 in settings.snapshots:
            _save(checkpoint, run_dir / f"epoch_{epoch + 1}.pt")
        _save_last(checkpoint | {"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                                 "history": history, "best_val": best, "grad_norms": list(norms)}, run_dir)
        _write_history(history, run_dir)
        trained = "none" if record["train_loss"] is None else f"{record['train_loss']:.4f}"
        log(f"epoch {epoch + 1}/{settings.epochs}  train {trained}  val_select {val_select:.4f}  "
            f"lr {record['lr']:.1e}  skipped {skipped}  clipped {clipped}  {record['seconds']} s")
    return history


def load_checkpoint(path: Path, device: str | torch.device | None = None) -> tuple[GLNONet, dict]:
    """The network of a checkpoint, in evaluation mode on ``device``, and everything else the file holds."""
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    state = torch.load(Path(path), map_location=device, weights_only=False)
    model = GLNONet({**state["config"], "device": str(device)}).to(device)
    model.load_state_dict(state.pop("model"))
    return model.eval(), state


def checkpoint_preprocess(state: dict) -> ShapesPreprocess:
    """The preprocessing a checkpoint was trained with (``geo_scale`` and the scales are not recomputed)."""
    return ShapesPreprocess(**state["meta"]["preprocess"])


@torch.no_grad()
def predict_shape(model: GLNONet, store: ShapeStore, shape: int, *, sets=None,
                  batch_size: int = 4) -> tuple[NDArray, NDArray]:
    """Packet numbers and predicted states of the loaded packets of one shape, as the dataset stores them:
    complex (P, N, T) at ``store.config.output_times()``, without padding or amplitude.

    When the labels leave t = 0 out, the first state is the input ψ0.
    """
    if isinstance(shape, (bool, np.bool_)) or not isinstance(shape, (int, np.integer)):
        raise ValueError("predict_shape takes one shape index")       # a list would be cut to its first mesh
    rows = store.rows(sets=sets, shapes=shape)
    if len(rows) == 0:
        raise ValueError(f"the store holds no packets of shape {shape}")
    device, amplitude, was_training = _device(model), store.preprocess.amplitude, model.training
    model.eval()
    try:
        states = []
        for start in range(0, len(rows), batch_size):
            batch = _batch(store, rows[start:start + batch_size], device)
            n = int(batch["n_verts"][0])
            out = model(batch)[:, :n].double().reshape(len(batch["rows"]), n, len(store.steps), 2)
            if store.preprocess.drop_initial:
                out = torch.cat([batch["input"][:, :n].double().unsqueeze(2), out], dim=2)
            states.append(torch.complex(out[..., 0], out[..., 1]).cpu().numpy() / amplitude)
    finally:
        model.train(was_training)
    return rows, np.concatenate(states)


def calibrate(store: ShapeStore, config: dict, settings: ShapesTrainSettings = ShapesTrainSettings(), *,
              warmup: int = 3, steps: int = 10, epochs: tuple[int, ...] = (50, 300)) -> dict:
    """Milliseconds per sample and reserved memory by family, measured on real batches, and the hours that
    ``epochs`` epochs would take.

    A throwaway network runs ``warmup`` training steps and then ``steps`` timed ones on batches of
    each family, and one timed pass over validation batches; nothing is saved and the run that
    follows is not affected (``train_shapes`` seeds itself). On CUDA the clock starts and stops on
    a synchronised device. ``reserved_gib`` is the peak PyTorch reserved on the device during that
    family's steps, the store included when it lives there (subtract ``store.nbytes`` for the cost
    of the step alone); the same holds for the ``reserved_gib`` of each epoch in the history.
    """
    check_config(config, store)
    device = torch.device(config["device"])
    cuda = device.type == "cuda"
    if cuda:
        ieee_matmuls()
    torch.manual_seed(settings.seed)
    model = GLNONet(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings.lr, weight_decay=settings.weight_decay)

    def clock() -> float:
        if cuda:
            torch.cuda.synchronize(device)
        return time.perf_counter()

    families, epoch_seconds = {}, 0.0
    for family in store.n_max:
        train_rows = store.rows(sets="train", family=family)
        val_rows = store.rows(sets="val", family=family)
        if len(train_rows) == 0:
            continue
        if cuda:                # without the blocks cached for the other family, so each one's peak is its own
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
        batches = family_batches(store, train_rows, settings.batch_size, seed=settings.seed)
        model.train()
        seconds = samples = 0
        for step in range(warmup + steps):
            members = batches[step % len(batches)]
            start = clock()
            batch = _batch(store, members, device)
            if settings.augment:
                batch = augment_batch(batch, gauge_generator(settings.seed, 0, step))
            train_step(model, optimizer, batch)
            if step >= warmup:
                seconds, samples = seconds + clock() - start, samples + len(members)
        measured = {"train_ms_per_sample": 1000 * seconds / samples, "train_packets": len(train_rows),
                    "val_packets": len(val_rows), "eval_ms_per_sample": None,
                    "reserved_gib": torch.cuda.max_memory_reserved(device) / 2**30 if cuda else None}
        epoch_seconds += seconds / samples * len(train_rows)
        if len(val_rows):
            timed = val_rows[:settings.batch_size * max(1, steps // 2)]
            evaluate_rows(model, store, timed[:settings.batch_size], settings.batch_size)       # warm-up
            start = clock()
            evaluate_rows(model, store, timed, settings.batch_size)
            per_sample = (clock() - start) / len(timed)
            measured["eval_ms_per_sample"] = 1000 * per_sample
            epoch_seconds += per_sample * len(val_rows)
        families[family] = measured
    return {"device": str(device), "batch_size": settings.batch_size, "families": families,
            "epoch_seconds": epoch_seconds,
            "hours": {str(count): count * epoch_seconds / 3600 for count in epochs}}


In [ ]:
%%writefile src/schrodinger_glno/glno/spectral.py
"""What the GLNO sees of one mesh in the stage-3 dataset, computed when the data are loaded.

The stored eigenpairs ``(Φ, λ)`` solve ``L φ = λ M_l φ`` with the lumped mass ``M_l``,
but Crank–Nicolson made the data with the blended mass ``M_b = (M_l + M_c)/2``. Evolving
in ``(Φ, λ)`` misses the data by 1.4–27 % at t = 0.5. ``ritz_polar_basis`` fixes this
inside the stored 700 modes, without a new eigensolve:

1. Rayleigh–Ritz with the blended mass: ``A y = μ B y`` with ``A = Φᵀ L Φ`` and
   ``B = Φᵀ M_b Φ``, so ``yᵀ B y = I`` and ``μ`` are the frequencies CN actually used.
2. The orthogonal polar factor ``U = y (yᵀy)^(-1/2)``, which equals ``B^(1/2) y``.
   ``E = Φ U`` stays orthonormal in the lumped mass, ``Eᵀ diag(m) E = UᵀU = I``, so the
   network's diagonal-mass ``to_basis``/``from_basis`` stay exact projections, and each
   column ``k`` of ``E`` carries ``μ_k``. Evolving in ``(E, μ)`` is within 0.28 % (median)
   and 1.35 % (max) of the data, which is the data's own time-step error.

Because ``(5/8) M_l ⪯ M_b ⪯ M_l`` element by element and ``Φᵀ M_l Φ = I``, the
eigenvalues of ``B`` lie in ``[5/8, 1]`` and the singular values of ``y`` in
``[1, √(8/5)]`` (measured on the 200 shapes: ``[1 - 5e-8, 1.055]``).

The signs of the eigenvectors that LAPACK returns may change with the driver or the
number of BLAS threads, so the column signs of ``U`` are fixed (largest entry positive).
On the dataset the largest entry beats the second by at least 1.8e-5, and ``U`` agrees
across drivers and thread counts to 4e-10, or to ~1e-6 on round tori, whose nearly
equal eigenvalue pairs are anyway rotated at random in training (``augment.py``).

NumPy and SciPy only, float64 inside. See Plan_GLNO_etapa3.md, sections 1.1–1.5.
"""
from typing import NamedTuple
import numpy as np
import scipy.linalg
import scipy.sparse as sp
from numpy.typing import NDArray
from ..mesh import blended_mass, cotan_laplacian, vertex_areas

RITZ_VERSION = 1
SINGULAR_VALUE_BOUNDS = (1.0, float(np.sqrt(8 / 5)))


class RitzBasis(NamedTuple):
    mu: NDArray                  # (K,) float64, ascending, >= 0: blended-mass Ritz values
    U: NDArray                   # (K, K) float64, orthogonal, canonical column signs
    E: NDArray                   # (N, K) float32, Φ U
    singular_values: NDArray     # (K,) float64, of y, descending


def canonical_signs(U: NDArray) -> NDArray:
    """Flip each column so that its entry of largest magnitude is positive."""
    rows = np.argmax(np.abs(U), axis=0)
    signs = np.sign(U[rows, np.arange(U.shape[1])])
    signs[signs == 0] = 1
    return U * signs


def ritz_polar_basis(verts: NDArray, faces: NDArray, evecs: NDArray, *,
                     L: sp.spmatrix | None = None, blended: sp.spmatrix | None = None,
                     driver: str | None = None) -> RitzBasis:
    """Blended-mass Ritz values and the lumped-orthonormal polar basis inside ``span(evecs)``.

    ``evecs`` (N, K) must be orthonormal in the lumped mass ``vertex_areas(verts, faces)``,
    as stored in each shape's ``mesh.npz``. ``L`` and ``blended`` default to the operators
    the Crank–Nicolson solver used (``cotan_laplacian`` and ``blended_mass``). ``driver``
    is passed to ``scipy.linalg.eigh`` (tests only).
    """
    phi = np.asarray(evecs, dtype=np.float64)
    n, k = phi.shape
    if len(verts) != n or not 1 <= k < n:
        raise ValueError("evecs must be (number of vertices, K) with 1 <= K < number of vertices")
    L = cotan_laplacian(verts, faces) if L is None else L
    blended = blended_mass(verts, faces) if blended is None else blended
    lumped = vertex_areas(verts, faces)
    gram = phi.T @ (lumped[:, None] * phi)
    # 1e-6 keeps this check ahead of the singular-value one (s_min ≈ 1 - error/2); float32 storage gives 1e-7
    if not np.allclose(gram, np.eye(k), rtol=0, atol=1e-6):
        raise ValueError(f"evecs are not orthonormal in the lumped mass (max error "
                         f"{np.abs(gram - np.eye(k)).max():.1e})")
    A = phi.T @ (L @ phi)
    B = phi.T @ (blended @ phi)
    A, B = (A + A.T) / 2, (B + B.T) / 2
    mu, y = scipy.linalg.eigh(A, B, driver=driver)
    W, s, Vt = np.linalg.svd(y)
    U = canonical_signs(W @ Vt)
    if np.abs(U.T @ U - np.eye(k)).max() > 1e-10:
        raise ValueError("the polar factor is not orthogonal")
    low, high = SINGULAR_VALUE_BOUNDS
    if s.min() < low - 1e-6 or s.max() > high + 1e-6:
        raise ValueError(f"singular values of y outside [1, √(8/5)]: [{s.min():.6f}, {s.max():.6f}]")
    if np.any(np.diff(mu) < 0):
        raise ValueError("Ritz values are not ascending")
    return RitzBasis(np.maximum(mu, 0.0), U, (phi @ U).astype(np.float32), s)


def eigen_clusters(mu: NDArray, rel_gap: float) -> NDArray:
    """Label groups of consecutive modes whose relative gap is below ``rel_gap``.

    The gap between modes k and k+1 is ``(μ_{k+1} - μ_k) / ((μ_{k+1} + μ_k)/2)``; a new
    group starts where it is ``>= rel_gap``. Mode 0 (the constant) is always alone. Each
    mode gets the index of the first mode of its group, so labels are non-decreasing and
    ``labels[k] == k`` marks the start of a group.

    Groups chain: a group can span more than ``rel_gap`` from its first to its last mode
    (on the dataset, up to ~3× at 1e-4).
    """
    mu = np.asarray(mu, dtype=np.float64)
    if mu.ndim != 1 or len(mu) < 1 or not np.isfinite(mu).all() or np.any(np.diff(mu) < 0):
        raise ValueError("mu must be a non-empty, finite, ascending vector")
    if not rel_gap >= 0:
        raise ValueError("rel_gap must be >= 0")
    mean = (mu[1:] + mu[:-1]) / 2
    gap = np.divide(np.diff(mu), mean, out=np.zeros(len(mu) - 1), where=mean > 0)
    starts = np.concatenate([[True], gap >= rel_gap])
    if len(mu) > 1:
        starts[1] = True
    first = np.flatnonzero(starts)
    return first[np.cumsum(starts) - 1].astype(np.int64)


def centroid(verts: NDArray, mass: NDArray) -> NDArray:
    """Area-weighted centroid ``Σ m x / Σ m``."""
    mass = np.asarray(mass, dtype=np.float64)
    return (mass[:, None] * verts).sum(axis=0) / mass.sum()


def geo_feature(verts: NDArray, mass: NDArray, scale: float) -> NDArray:
    """``‖x - c‖ / scale`` per vertex (float32), ``c`` the area centroid; invariant to rigid motions."""
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("scale must be finite and positive")
    return (np.linalg.norm(verts - centroid(verts, mass), axis=1) / scale).astype(np.float32)


def multiplier(mu: NDArray, steps: NDArray, dt: float, scheme: str) -> NDArray:
    """(K, T) factor applied to each mode after ``steps`` time steps of size ``dt``.

    ``exact``: ``e^{-iμt/2}`` with ``t = steps·dt``, the solution of ``i M ψ' = (L/2) ψ``.
    ``cn``: Crank–Nicolson, ``r(μ)^n`` with ``r = (1 - iμΔt/4)/(1 + iμΔt/4)``, written as
    ``e^{-2in·atan(μΔt/4)}`` (``|r| = 1``), from ``(M + iΔt A/2) ψ^{n+1} = (M - iΔt A/2) ψ^n``
    with ``A = L/2``.
    """
    mu = np.asarray(mu, dtype=np.float64)
    steps = np.asarray(steps)
    if not np.issubdtype(steps.dtype, np.integer) or np.any(steps < 0):
        raise ValueError("steps must be non-negative integers")
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive")
    if scheme == "exact":
        phase = 0.5 * np.outer(mu, steps * dt)
    elif scheme == "cn":
        phase = 2 * np.outer(np.arctan(mu * dt / 4), steps)
    else:
        raise ValueError("scheme must be 'exact' or 'cn'")
    return np.exp(-1j * phase)


def propagate(E: NDArray, mu: NDArray, mass: NDArray, psi0: NDArray, steps: NDArray, dt: float,
              scheme: str = "cn") -> NDArray:
    """(N, T) evolution of ``psi0`` inside ``span(E)``: ``E (f ⊙ Eᵀ(m ψ0))``, ``f`` from ``multiplier``.

    ``E`` must be orthonormal in the lumped mass ``mass``.
    """
    E = np.asarray(E, dtype=np.float64)
    coeffs = E.T @ (np.asarray(mass, dtype=np.float64) * psi0)
    return E @ (coeffs[:, None] * multiplier(mu, steps, dt, scheme))


def mass_relative_error(pred: NDArray, target: NDArray, mass: NDArray) -> NDArray:
    """``‖pred - target‖_M / ‖target‖_M`` per column of (N, T) arrays, with the lumped mass."""
    mass = np.asarray(mass, dtype=np.float64)[:, None]
    return np.sqrt(np.sum(mass * np.abs(pred - target) ** 2, axis=0) / np.sum(mass * np.abs(target) ** 2, axis=0))


def spectral_oracle_error(E: NDArray, mu: NDArray, mass: NDArray, psi0: NDArray, outputs: NDArray,
                          steps: NDArray, dt: float) -> NDArray:
    """Error of the Crank–Nicolson multiplier applied in ``(E, μ)`` against the stored outputs.

    The labels are CN states, and in this basis CN is one factor per mode that the
    network could learn exactly, so this is the error of the best answer of that form
    (the "achievable error"). It is not a strict lower bound: the ``e^{-σg}`` weighting
    and the per-vertex layers can leave ``span(E)``. ``outputs`` is (N, T) complex at
    ``steps``; returns (T,).
    """
    return mass_relative_error(propagate(E, mu, mass, psi0, steps, dt, "cn"), outputs, mass)


In [ ]:
%%writefile src/schrodinger_glno/initial_conditions.py
"""Gaussian wave packets discretized on a triangle mesh.

The continuous packet is evaluated at each vertex (one complex value per vertex).
That is the P1 finite-element representation Crank–Nicolson uses, and the same
per-vertex format the GLNO datasets store as ``input``. The vector is then
normalized with the mass matrix used by the solver: ``ψᴴ M ψ`` approximates
``∫ |ψ|² dμ`` and is the quantity Crank–Nicolson conserves exactly. The plain sum
``Σ |ψ_i|²`` weights every vertex by 1 instead of by its area: normalizing with it
would leave ``∫ |ψ|² dμ`` close to the mean vertex area rather than 1, a factor
that also depends on the local vertex density on non-uniform meshes.

``mass`` may be one area per vertex (lumped) or a sparse matrix such as
``mesh.blended_mass``.
"""
import numpy as np
import scipy.sparse as sp
from numpy.typing import NDArray
from .mesh import as_mass_matrix, mean_edge_length


def mass_norm(psi: NDArray, mass: NDArray | sp.spmatrix) -> float:
    """Discrete L² norm on the surface: ``sqrt(ψᴴ M ψ)`` (``sqrt(Σ m_i |ψ_i|²)`` if lumped)."""
    if sp.issparse(mass):
        return float(np.sqrt(np.real(np.vdot(psi, mass @ psi))))
    return float(np.sqrt(np.sum(mass * np.abs(psi)**2)))


def tangent_direction(normal: NDArray, direction: NDArray) -> NDArray:
    """Unit vector of ``direction`` projected onto the tangent plane with this ``normal``."""
    normal = np.asarray(normal, float) / np.linalg.norm(normal)
    direction = np.asarray(direction, float)
    tangent = direction - (direction @ normal) * normal
    size = np.linalg.norm(tangent)
    if size < 1e-8 * max(np.linalg.norm(direction), 1e-300):
        raise ValueError("direction is (almost) normal to the surface; choose another one")
    return tangent / size


def gaussian_packet(verts: NDArray, mass: NDArray | sp.spmatrix, normals: NDArray, center: int,
                    k0: float, sigma: float, direction: NDArray) -> NDArray:
    """Gaussian packet centred at vertex ``center``, moving along ``direction``.

    ``ψ(x) ∝ exp(-|x - x0|² / (4σ²)) · exp(i k0 t0·(x - x0))``, where ``t0`` is
    ``direction`` projected onto the tangent plane at ``x0``. As in the 1D
    ``packet="paper"`` convention, σ is the standard deviation of ``|ψ|²`` and
    the momentum spread is ``1 / (2σ)``. Distances are chordal (straight lines in
    R³): smooth everywhere and accurate while σ is small compared with the
    curvature radius. Returns the vertex values normalized with ``mass``.
    """
    if not 0 <= center < len(verts):
        raise ValueError("center must be a vertex index")
    if sigma <= 0 or not np.isfinite(k0):
        raise ValueError("sigma must be positive and k0 finite")
    M = as_mass_matrix(mass)
    if M.shape != (len(verts), len(verts)) or np.any(M.diagonal() <= 0):
        raise ValueError("mass must hold one positive area per vertex or be a (V, V) mass matrix")
    offset = verts - verts[center]
    t0 = tangent_direction(normals[center], direction)
    psi = np.exp(-np.sum(offset**2, axis=1) / (4 * sigma**2)) * np.exp(1j * k0 * (offset @ t0))
    return psi / mass_norm(psi, mass)


def max_wavenumber(k0: float, sigma: float, n_std: float = 3.0) -> float:
    """Largest wavenumber with appreciable weight: ``|k0| + n_std / (2σ)``."""
    return abs(k0) + n_std / (2 * sigma)


def points_per_wavelength(verts: NDArray, faces: NDArray, k0: float, sigma: float,
                          n_std: float = 3.0) -> float:
    """Mean mesh edges per shortest wavelength of the packet; aim for about 10 or more.

    This only says whether the mesh represents ψ0 well. How well the phase is kept
    while the packet travels depends on the mass matrix (see ``mesh.blended_mass``).
    """
    return 2 * np.pi / (max_wavenumber(k0, sigma, n_std) * mean_edge_length(verts, faces))


def spectral_fraction(psi: NDArray, mass: NDArray | sp.spmatrix, evecs: NDArray) -> float:
    """Fraction of ``‖ψ‖²`` captured by the eigenvectors ``evecs``, M-orthonormal for this ``mass``."""
    coeffs = evecs.T @ (as_mass_matrix(mass) @ psi)
    return float(np.sum(np.abs(coeffs)**2) / mass_norm(psi, mass)**2)


In [ ]:
%%writefile src/schrodinger_glno/mesh.py
"""Triangle-mesh operators with the same definitions the GLNO repository uses.

``cotan_laplacian`` and ``vertex_areas`` reproduce ``potpourri3d.cotan_laplacian``
and ``potpourri3d.vertex_areas`` (used in ``GLNO-repo/utils/geometry/operator.py``),
so the solver and the network see the same discretization.
"""
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as sla
from numpy.typing import NDArray


def _check_mesh(verts: NDArray, faces: NDArray) -> None:
    if verts.ndim != 2 or verts.shape[1] != 3 or faces.ndim != 2 or faces.shape[1] != 3:
        raise ValueError("verts must be (V, 3) and faces (F, 3)")
    if faces.min() < 0 or faces.max() >= len(verts):
        raise ValueError("faces reference vertices that do not exist")


def icosphere(subdivisions: int, radius: float = 1.0) -> tuple[NDArray, NDArray]:
    """Icosahedron refined ``subdivisions`` times and projected onto the sphere.

    It has ``10 * 4**subdivisions + 2`` vertices and outward-oriented faces.
    """
    if subdivisions < 0 or radius <= 0:
        raise ValueError("subdivisions must be >= 0 and radius > 0")
    t = (1 + np.sqrt(5)) / 2
    verts = np.array([(-1, t, 0), (1, t, 0), (-1, -t, 0), (1, -t, 0), (0, -1, t), (0, 1, t),
                      (0, -1, -t), (0, 1, -t), (t, 0, -1), (t, 0, 1), (-t, 0, -1), (-t, 0, 1)], float)
    faces = np.array([(0, 11, 5), (0, 5, 1), (0, 1, 7), (0, 7, 10), (0, 10, 11), (1, 5, 9), (5, 11, 4),
                      (11, 10, 2), (10, 7, 6), (7, 1, 8), (3, 9, 4), (3, 4, 2), (3, 2, 6), (3, 6, 8),
                      (3, 8, 9), (4, 9, 5), (2, 4, 11), (6, 2, 10), (8, 6, 7), (9, 8, 1)])
    verts /= np.linalg.norm(verts, axis=1, keepdims=True)
    for _ in range(subdivisions):
        # Each face edge (01, 12, 20) gets one shared midpoint; each face splits into four.
        edges = np.sort(faces[:, [0, 1, 1, 2, 2, 0]].reshape(-1, 2), axis=1)
        unique, inverse = np.unique(edges, axis=0, return_inverse=True)
        mid = verts[unique].mean(axis=1)
        mid /= np.linalg.norm(mid, axis=1, keepdims=True)
        ab, bc, ca = (len(verts) + inverse.reshape(-1, 3)).T
        a, b, c = faces.T
        verts = np.vstack([verts, mid])
        faces = np.concatenate([np.stack(f, axis=1) for f in
                                ((a, ab, ca), (b, bc, ab), (c, ca, bc), (ab, bc, ca))])
    return radius * verts, faces


def cotan_laplacian(verts: NDArray, faces: NDArray, denom_eps: float = 1e-10) -> sp.csr_matrix:
    """Cotangent Laplacian: symmetric, positive semidefinite, the weak form of ``-Δ``.

    Edge ``ij`` has weight ``(cot α + cot β) / 2``, where α and β are the angles
    opposite to it. Rows sum to zero.
    """
    _check_mesh(verts, faces)
    corners = verts[faces]
    rows, cols, vals = [], [], []
    for k in range(3):
        i, j = (k + 1) % 3, (k + 2) % 3
        u = corners[:, i] - corners[:, k]
        w = corners[:, j] - corners[:, k]
        cot = np.einsum("fd,fd->f", u, w) / (np.linalg.norm(np.cross(u, w), axis=1) + denom_eps)
        half = 0.5 * cot
        fi, fj = faces[:, i], faces[:, j]
        rows += [fi, fj, fi, fj]
        cols += [fj, fi, fi, fj]
        vals += [-half, -half, half, half]
    n = len(verts)
    return sp.coo_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                         shape=(n, n)).tocsr()


def face_areas(verts: NDArray, faces: NDArray) -> NDArray:
    _check_mesh(verts, faces)
    corners = verts[faces]
    return 0.5 * np.linalg.norm(np.cross(corners[:, 1] - corners[:, 0], corners[:, 2] - corners[:, 0]), axis=1)


def vertex_areas(verts: NDArray, faces: NDArray) -> NDArray:
    """Lumped mass: one third of the area of the triangles around each vertex."""
    areas = face_areas(verts, faces)
    mass = np.zeros(len(verts))
    for k in range(3):
        np.add.at(mass, faces[:, k], areas / 3)
    return mass


def consistent_mass(verts: NDArray, faces: NDArray) -> sp.csr_matrix:
    """Consistent P1 mass matrix: ``M_ij = ∫ φ_i φ_j dA``, exact for piecewise-linear functions.

    On each triangle of area ``a`` the local matrix is ``a/12 · [[2,1,1],[1,2,1],[1,1,2]]``.
    Its rows sum to the lumped masses.
    """
    areas = face_areas(verts, faces)
    rows, cols, vals = [], [], []
    for i in range(3):
        for j in range(3):
            rows.append(faces[:, i])
            cols.append(faces[:, j])
            vals.append(areas * (2 if i == j else 1) / 12)
    n = len(verts)
    return sp.coo_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                         shape=(n, n)).tocsr()


def blended_mass(verts: NDArray, faces: NDArray) -> sp.csr_matrix:
    """Average of the lumped and consistent mass matrices, used by the time stepper.

    With P1 elements the lumped mass gives Laplacian eigenvalues that are too low and
    the consistent mass gives values that are too high, by almost the same amount; the
    average cancels most of it (Marfurt 1984; Christon 1999). On the 10 242-vertex
    icosphere the error at ℓ = 15 drops from -2.1 % to +0.02 %, so wave packets keep
    the right phase while they travel. The cancellation depends on the triangles: on
    the ``torus`` mesh it is only partial: for λ in 150–250, -1.7 % (lumped) becomes
    +0.13 % at edge 0.038, and -1.1 % becomes +0.09 % at edge 0.03 (second order in
    the edge), so there packets gain a small phase lead.
    """
    return (0.5 * (sp.diags(vertex_areas(verts, faces)) + consistent_mass(verts, faces))).tocsr()


def as_mass_matrix(mass: NDArray | sp.spmatrix) -> sp.csr_matrix:
    """A mass given as one area per vertex (lumped) or as a sparse matrix, as a sparse matrix."""
    return sp.csr_matrix(mass) if sp.issparse(mass) else sp.diags(np.asarray(mass, float)).tocsr()


def vertex_normals(verts: NDArray, faces: NDArray) -> NDArray:
    """Unit normals averaged from the incident faces, weighted by area."""
    _check_mesh(verts, faces)
    corners = verts[faces]
    face_normals = np.cross(corners[:, 1] - corners[:, 0], corners[:, 2] - corners[:, 0])
    normals = np.zeros_like(verts)
    for k in range(3):
        np.add.at(normals, faces[:, k], face_normals)
    return normals / np.linalg.norm(normals, axis=1, keepdims=True)


def laplacian_eigenpairs(L: sp.spmatrix, mass: NDArray, k: int) -> tuple[NDArray, NDArray]:
    """The ``k`` smallest solutions of ``L φ = λ M φ``, sorted, with ``Φᵀ M Φ = I``.

    Shift-invert around a small negative value avoids factoring the singular ``L``;
    tiny negative eigenvalues from round-off are clipped to zero, as in the GLNO repo.
    ARPACK starts from a seeded vector: with a random start, repeated eigenvalues
    (the sphere has 2ℓ+1 per level) come back in a different basis on every run.
    ``mass`` is one area per vertex (lumped) or a sparse mass matrix.
    """
    n = L.shape[0]
    if not 1 <= k < n:
        raise ValueError("k must satisfy 1 <= k < number of vertices")
    v0 = np.random.default_rng(0).standard_normal(n)
    evals, evecs = sla.eigsh(L.tocsc(), k=k, M=as_mass_matrix(mass).tocsc(), sigma=-1e-3, v0=v0)
    order = np.argsort(evals)
    return np.clip(evals[order], 0, None), evecs[:, order]


def mean_edge_length(verts: NDArray, faces: NDArray) -> float:
    _check_mesh(verts, faces)
    edges = np.unique(np.sort(faces[:, [0, 1, 1, 2, 2, 0]].reshape(-1, 2), axis=1), axis=0)
    return float(np.linalg.norm(verts[edges[:, 0]] - verts[edges[:, 1]], axis=1).mean())


def _torus_points(u: NDArray, v: NDArray, major_radius: float, minor_radius: float) -> NDArray:
    rho = major_radius + minor_radius * np.cos(v)
    return np.stack([rho * np.cos(u), rho * np.sin(u), minor_radius * np.sin(v)], axis=-1)


def _zip_rings(start_a: int, n_a: int, offset_a: float, start_b: int, n_b: int,
               offset_b: float) -> tuple[NDArray, NDArray, NDArray]:
    """Outward-oriented triangle strip between ring a and the next ring b (one step up in v).

    Point k of a ring sits at angle ``offset + 2πk/n``; edge k joins points k and k+1. Both
    rings are walked in angle order and each step closes the triangle on the ring edge whose
    midpoint comes first, which is the one with the shorter diagonal. Also returns, for every
    point of each ring, how many edges of the other ring it is the apex of (1 when staggered).
    """
    two_pi = 2 * np.pi
    mid_a = two_pi * (np.arange(n_a) + 0.5) / n_a
    mid_b = np.mod(offset_b - offset_a + two_pi * (np.arange(n_b) + 0.5) / n_b, two_pi)
    first_b = int(np.argmin(mid_b))
    on_a = np.argsort(np.concatenate([mid_a, np.roll(mid_b, -first_b)]), kind="stable") < n_a
    a = (np.cumsum(on_a) - on_a) % n_a                    # current point of each ring at each step
    b = (first_b + np.cumsum(~on_a) - ~on_a) % n_b
    faces = np.where(on_a[:, None],
                     np.stack([start_a + a, start_a + (a + 1) % n_a, start_b + b], axis=1),
                     np.stack([start_a + a, start_b + (b + 1) % n_b, start_b + b], axis=1))
    return faces, np.bincount(a[~on_a], minlength=n_a), np.bincount(b[on_a], minlength=n_b)


def _zip_ring_loop(counts: NDArray, starts: NDArray, first: int = 0) -> tuple[NDArray, NDArray, NDArray, NDArray]:
    """Offsets and faces of a closed loop of rings, visited in the order first, first + 1, ...

    Each ring is staggered by half a step from the one before. Where the two counts differ, four
    sub-step shifts are tried and the one keeping the previous ring's valences closest to 6 is
    kept; the last ring also scores the strip that closes the loop. Returns the ring offsets, the
    faces, valence - 6 of every vertex and the visiting order.
    """
    n_v = len(counts)
    order = (first + np.arange(n_v)) % n_v
    offsets = np.zeros(n_v)

    def strip(a, b):
        return _zip_rings(starts[a], counts[a], offsets[a], starts[b], counts[b], offsets[b])

    # up[j] / down[j]: per point of ring j, the edges of the next / previous ring it is the apex of.
    # Its valence is 4 + up + down (6 when both are 1). down[first] is a guess until the last strip.
    faces, up, down = [], {}, {first: np.ones(counts[first], int)}
    for i in range(1, n_v):
        j, prev = order[i], order[i - 1]
        base = offsets[prev] + np.pi / counts[j]
        shifts = [0.0] if counts[j] == counts[prev] else 2 * np.pi * np.arange(4) / (4 * counts[j])
        best = None
        for shift in shifts:
            offsets[j] = base + shift
            new = [strip(prev, j)] + ([strip(j, first)] if i == n_v - 1 else [])
            cost = np.sum((new[0][1] + down[prev] - 2)**2)
            if i == n_v - 1:
                cost += np.sum((new[1][1] + new[0][2] - 2)**2) + np.sum((up[first] + new[1][2] - 2)**2)
            if best is None or cost < best[0]:
                best = (cost, shift, new)
        offsets[j] = base + best[1]
        for (strip_faces, extra_a, extra_b), (a, b) in zip(best[2], [(prev, j), (j, first)]):
            faces.append(strip_faces)
            up[a], down[b] = extra_a, extra_b
    excess = np.concatenate([up[j] + down[j] - 2 for j in range(n_v)])
    return offsets, np.concatenate(faces), excess, order


def _stagger_closing_strip(offsets: NDArray, counts: NDArray, order: NDArray) -> NDArray:
    """Ring offsets with the strip that closes the loop staggered like the other equal-count strips.

    Strips between rings of equal count get a half-step stagger, except the closing one, which
    gets whatever phase the others add up to (aligned rings would make a seam of right triangles).
    When its two counts are equal, its phase error is shared by all equal-count strips, as the
    same fraction of each one's step, by rotating the rings after each of them. Equal-count
    strips zip the same way at any phase in (0, 1), so the faces stay as they are. Each one ends
    at most ½·s_c/Σs of its step off the half step (s: the steps of all equal-count strips, s_c
    the closing one's), a quarter step when there are two of the same count.
    """
    first, last = order[0], order[-1]
    c = counts[order]
    equal_steps = np.concatenate([[0.0], np.where(c[1:] == c[:-1], 2 * np.pi / c[1:], 0.0)])
    if counts[last] != counts[first] or not equal_steps.any():
        return offsets
    step = 2 * np.pi / counts[first]
    error = np.mod(offsets[first] - offsets[last], step) - step / 2
    out = offsets.copy()
    out[order] += error / (step + equal_steps.sum()) * np.cumsum(equal_steps)
    return out


def torus(target_edge: float, major_radius: float = 1.0, minor_radius: float = 0.5) -> tuple[NDArray, NDArray]:
    """Torus of revolution with near-equilateral triangles of side ≈ ``target_edge``.

    The surface is ``((R + r cos v) cos u, (R + r cos v) sin u, r sin v)``. Vertices lie on
    rings of constant v, a geodesic distance ≈ target_edge·√3/2 apart; ring j has
    round(2π ρ_j / target_edge) points (ρ = R + r cos v), so triangles keep their size on the
    inner side instead of narrowing by (R - r)/(R + r) as on a (u, v) grid. Consecutive rings
    are staggered by half a step and zipped together; the strip that closes the loop gets the
    same stagger through a slight twist (``_stagger_closing_strip``). Where two counts differ,
    each extra point makes a 5-7 pair with two near-right triangles (cotan weight ≈ 0 on their
    diagonal), the best a ring mesh can do; ring offsets are picked among four sub-step shifts
    to keep two of them off the same point (valence 4 or 8); if that still happens where the
    loop closes (coarse meshes), the loop is redone starting from the inner side. With
    R - r ≥ 2·target_edge this gives valences 5-7, angles of 38-92.5° and vertex areas within a
    factor 1.5. Nearer the horn torus (a dozen triangles or fewer around the inner equator) the
    mesh is still a valid embedding but its quality degrades. Faces are outward-oriented as in
    ``icosphere``, and the result is deterministic.
    """
    R, r, h = float(major_radius), float(minor_radius), float(target_edge)
    if not (np.isfinite(R) and R > r > 0):
        raise ValueError("need major_radius > minor_radius > 0")
    if not 0 < h < r:
        raise ValueError("need 0 < target_edge < minor_radius")
    n_v = max(3, int(round(2 * np.pi * r / (h * np.sqrt(3) / 2))))
    v = 2 * np.pi * np.arange(n_v) / n_v
    counts = np.maximum(3, np.rint(2 * np.pi * (R + r * np.cos(v)) / h).astype(int))
    starts = np.concatenate([[0], np.cumsum(counts)])

    def score(excess):                   # vertices of valence 4 or 8 first, then Σ (valence - 6)²
        return int(np.sum(np.abs(excess) > 1)), int(np.sum(excess**2))

    best = _zip_ring_loop(counts, starts)
    for first in dict.fromkeys((n_v // 2, n_v // 4, 3 * n_v // 4)):      # inner side first
        if score(best[2])[0] == 0:
            break
        c = counts[(first + np.arange(n_v)) % n_v]
        if first == 0 or (c[-1] == c[0] and not np.any(c[1:] == c[:-1])):
            continue                     # the first pass, or a closing strip that could not be staggered
        new = _zip_ring_loop(counts, starts, first)
        if score(new[2]) < score(best[2]):
            best = new
    offsets, faces, _, order = best
    offsets = _stagger_closing_strip(offsets, counts, order)
    ring = np.repeat(np.arange(n_v), counts)
    u = offsets[ring] + 2 * np.pi * (np.arange(starts[-1]) - starts[ring]) / counts[ring]
    return _torus_points(u, v[ring], R, r), faces


In [ ]:
%%writefile src/schrodinger_glno/reference.py
"""Exact solutions of the free Schrödinger equation on the continuous unit sphere and torus.

Sphere: with V = 0 and ħ = m = 1, ``ψ(x, t) = Σ_ℓ e^{-iℓ(ℓ+1)t/2} (Π_ℓ ψ0)(x)``, where the
projection onto degree ℓ follows from the addition theorem:
``(Π_ℓ f)(x) = (2ℓ+1)/(4π) ∫ P_ℓ(x·y) f(y) dμ(y)``. The integral uses a
Gauss–Legendre (in cos θ) × uniform (in φ) grid, exact for the band-limited
integrands involved. This is the reference that measures how far a mesh solution
is from the continuous sphere, not just from the mesh's own equation. The torus
reference, ``torus_exact_solution``, is described in the comment before ``torus_points``.
"""
import numpy as np
import scipy.linalg as sla
from numpy.typing import NDArray


def sphere_quadrature(n_theta: int = 100, n_phi: int = 200) -> tuple[NDArray, NDArray]:
    """Points on the unit sphere and weights summing to 4π."""
    z, wz = np.polynomial.legendre.leggauss(n_theta)
    phi = 2 * np.pi * np.arange(n_phi) / n_phi
    Z, PHI = np.meshgrid(z, phi, indexing="ij")
    r = np.sqrt(1 - Z**2)
    points = np.stack([r * np.cos(PHI), r * np.sin(PHI), Z], axis=-1).reshape(-1, 3)
    return points, np.repeat(wz, n_phi) * (2 * np.pi / n_phi)


def sphere_packet(points: NDArray, x0: NDArray, t0: NDArray, k0: float, sigma: float) -> NDArray:
    """The continuous Gaussian packet of ``initial_conditions.gaussian_packet``, unnormalized."""
    offset = points - x0
    return np.exp(-np.sum(offset**2, 1) / (4 * sigma**2)) * np.exp(1j * k0 * (offset @ t0))


def sphere_exact_solution(targets: NDArray, x0: NDArray, t0: NDArray, k0: float, sigma: float,
                          times: NDArray, l_max: int = 60, chunk: int = 512) -> NDArray:
    """ψ(targets, times) on the unit sphere, with ψ0 normalized so that ∫|ψ0|² dμ = 1.

    ``targets`` must lie on the unit sphere. ``l_max = 60`` reproduces packets with
    k0 + 3/(2σ) up to about 25 to round-off; raise it for sharper packets.
    """
    # Exact for integrands up to degree ~2·(l_max/2 + 30) - 1: P_ℓ (ℓ ≤ l_max) times the packet's band.
    points, weights = sphere_quadrature(l_max // 2 + 30, 2 * (l_max // 2 + 30))
    psi0 = sphere_packet(points, x0, t0, k0, sigma)
    source = weights * psi0 / np.sqrt(np.sum(weights * np.abs(psi0)**2))
    ell = np.arange(l_max + 1)
    phases = np.exp(-1j * np.outer(ell * (ell + 1) / 2, np.asarray(times, float)))   # (ℓ, n_t)
    out = np.zeros((len(targets), len(times)), complex)
    for start in range(0, len(targets), chunk):
        c = np.clip(targets[start:start + chunk] @ points.T, -1, 1)
        p_prev, p = np.ones_like(c), c.copy()
        proj = np.zeros((len(c), l_max + 1), complex)
        proj[:, 0] = (p_prev @ source) / (4 * np.pi)
        proj[:, 1] = 3 * (p @ source) / (4 * np.pi)
        for l in range(1, l_max):                       # Bonnet recurrence for P_{l+1}
            p_prev, p = p, ((2 * l + 1) * c * p - l * p_prev) / (l + 1)
            proj[:, l + 1] = (2 * l + 3) * (p @ source) / (4 * np.pi)
        out[start:start + chunk] = proj @ phases
    return out


# Torus of revolution x(u, v) = ((R + r cos v) cos u, (R + r cos v) sin u, r sin v), R > r > 0.
# Metric ds² = ρ² du² + r² dv² with ρ(v) = R + r cos v, area element dA = r ρ du dv. With
# f = e^{imu} g(v), -Δf = λf becomes the periodic Sturm–Liouville problem
# -(ρ g')'/r² + m² g/ρ = λ ρ g for every integer m, solved by Fourier collocation in v.
# ψ0 is expanded in these eigenfunctions (FFT in u, weighted projection in v), every term
# gets its phase e^{-iλt/2}, and ψ is evaluated anywhere by exact trigonometric
# interpolation in u and v. All steps are spectrally accurate.


def _check_torus(major_radius: float, minor_radius: float) -> None:
    if not (np.isfinite(major_radius) and 0 < minor_radius < major_radius):
        raise ValueError("the torus needs finite radii with major_radius > minor_radius > 0")


def _wrap_angle(angle: NDArray) -> NDArray:
    """Angle in [0, 2π); ``np.mod`` alone returns 2π for tiny negative inputs."""
    angle = np.mod(angle, 2 * np.pi)
    return np.where(angle >= 2 * np.pi, 0.0, angle)


def torus_points(u: NDArray, v: NDArray, major_radius: float = 1.0, minor_radius: float = 0.5) -> NDArray:
    """Points ``x(u, v)`` on the torus, shape ``broadcast(u, v).shape + (3,)``."""
    u, v = np.broadcast_arrays(np.asarray(u, float), np.asarray(v, float))
    rho = major_radius + minor_radius * np.cos(v)
    return np.stack([rho * np.cos(u), rho * np.sin(u), minor_radius * np.sin(v)], axis=-1)


def torus_angles(points: NDArray, major_radius: float = 1.0) -> tuple[NDArray, NDArray]:
    """Angles ``(u, v)`` in [0, 2π) of points on the torus, the inverse of ``torus_points``.

    ``u`` turns around the z axis and ``v`` around the tube, from the outer equator
    (v = 0) through the top (v = π/2) to the inner equator (v = π). A point off the
    surface gets the angles of its nearest surface point, so ``r`` is not needed.
    """
    points = np.asarray(points, float)
    u = np.arctan2(points[..., 1], points[..., 0])
    v = np.arctan2(points[..., 2], np.hypot(points[..., 0], points[..., 1]) - major_radius)
    return _wrap_angle(u), _wrap_angle(v)


def torus_resolution(k0: float, sigma: float, major_radius: float = 1.0,
                     minor_radius: float = 0.5) -> tuple[int, int]:
    """Default odd grid sizes ``(n_u, n_v)`` of ``torus_exact_solution``.

    With ``K = |k0| + 6.5/σ`` (beyond it the packet's spectral weight ``e^{-σ²k²}``
    is below 1e-18), u modes go up to ``|m| ≤ ⌈K (R + r)⌉ + 20`` (a ring of radius
    ρ ≤ R + r carries wavenumber m/ρ) and v modes up to
    ``⌈1.5 K r⌉ + ⌈30 / arccosh(R/r)⌉``. The margin in v keeps the eigenfunctions
    resolved: their Fourier coefficients decay like ``e^{-arccosh(R/r) |k|}`` (1/ρ has
    poles at cos v = -R/r), slowly on fat tori, where the cost grows as r → R. On
    R = 1, r = 0.5 this gives 241 × 147 for k0 = 12, σ = 0.12 and 143 × 99 for
    k0 = 12, σ = 0.3. Margin, measured there for k0 ≤ 12, σ ∈ [0.12, 0.3], t ≤ 0.5:
    20 fewer u modes still give round-off (≤ 3e-12), half the v grid ≤ 1e-9.
    """
    _check_torus(major_radius, minor_radius)
    if not (sigma > 0 and np.isfinite(sigma) and np.isfinite(k0)):
        raise ValueError("sigma must be positive and k0 finite")
    K = abs(k0) + 6.5 / sigma
    m_max = int(np.ceil(K * (major_radius + minor_radius))) + 20
    strip = np.arccosh(major_radius / minor_radius)
    k_max = int(np.ceil(1.5 * K * minor_radius)) + int(np.ceil(30 / strip))
    return 2 * m_max + 1, 2 * k_max + 1


def _tube_grid(n_v: int, major_radius: float, minor_radius: float) -> tuple[NDArray, NDArray]:
    """``v_j = 2πj/n_v`` and ``ρ(v_j)``, with ρ exactly even (``ρ_j = ρ_{n_v - j}``)."""
    j = np.arange(n_v)
    return 2 * np.pi * j / n_v, major_radius + minor_radius * np.cos(2 * np.pi * np.minimum(j, n_v - j) / n_v)


def _fourier_derivative(n: int) -> NDArray:
    """d/dv on ``n`` (odd) equispaced periodic points: ``D_jk = ½ (-1)^{j-k} / sin((j-k)π/n)``.

    Built as an exactly antisymmetric circulant, so it anticommutes exactly with the
    reflection v → -v.
    """
    p = np.arange(1, (n + 1) // 2)
    d = np.zeros(n)
    d[p] = 0.5 * (-1.0)**p / np.sin(p * np.pi / n)
    d[n - p] = -d[p]
    j = np.arange(n)
    return d[(j[:, None] - j[None, :]) % n]


def _tube_operator(n_v: int, major_radius: float, minor_radius: float) -> tuple[NDArray, NDArray, NDArray]:
    """``ρ(v_j)`` and ``diag(ρ)^{-1/2} Dᵀ diag(ρ) D diag(ρ)^{-1/2} / r²`` in the even and odd bases.

    The bases ``e_0, (e_j + e_{n-j})/√2`` and ``(e_j - e_{n-j})/√2`` (j = 1 .. (n_v - 1)/2)
    are orthonormal and block-diagonalize every operator that commutes with v → -v.
    """
    _, rho = _tube_grid(n_v, major_radius, minor_radius)
    D = _fourier_derivative(n_v)
    scale = 1 / np.sqrt(rho)
    A = scale[:, None] * (D.T @ (rho[:, None] * D)) * scale[None, :] / minor_radius**2
    A = 0.5 * (A + A.T)
    j = np.arange(1, (n_v + 1) // 2)
    same = A[np.ix_(j, j)] + A[np.ix_(n_v - j, n_v - j)]
    cross = A[np.ix_(j, n_v - j)] + A[np.ix_(n_v - j, j)]
    even = np.empty((len(j) + 1, len(j) + 1))
    even[0, 0] = A[0, 0]
    even[0, 1:] = even[1:, 0] = (A[0, j] + A[0, n_v - j]) / np.sqrt(2)
    even[1:, 1:] = 0.5 * (same + cross)
    return rho, even, 0.5 * (same - cross)


def _tube_eigenpairs(m: int, rho: NDArray, even: NDArray, odd: NDArray,
                     minor_radius: float) -> tuple[NDArray, NDArray]:
    """``torus_eigenpairs`` from the output of ``_tube_operator``: two half-size ``eigh``."""
    n_v, h = len(rho), len(odd)
    j = np.arange(1, h + 1)
    q = float(m)**2 / rho**2                     # diag(ρ)^{-1/2} diag(m²/ρ) diag(ρ)^{-1/2}
    # MRRR (driver "evr"): as accurate as the default here and several times faster on these
    # small matrices when the BLAS runs multithreaded.
    lam_even, y_even = sla.eigh(even + np.diag(q[:h + 1]), driver="evr", check_finite=False)
    lam_odd, y_odd = sla.eigh(odd + np.diag(q[1:h + 1]), driver="evr", check_finite=False)
    G = np.zeros((n_v, n_v))
    G[0, :h + 1] = y_even[0]
    G[j, :h + 1] = G[n_v - j, :h + 1] = y_even[1:] / np.sqrt(2)
    G[j, h + 1:] = y_odd / np.sqrt(2)
    G[n_v - j, h + 1:] = -y_odd / np.sqrt(2)
    evals = np.concatenate([lam_even, lam_odd])
    order = np.argsort(evals, kind="stable")
    G /= np.sqrt(rho * (2 * np.pi)**2 * minor_radius / n_v)[:, None]
    return evals[order], G[:, order]


def torus_eigenpairs(m: int, n_v: int, major_radius: float = 1.0,
                     minor_radius: float = 0.5) -> tuple[NDArray, NDArray]:
    """Eigenpairs of ``-Δ (e^{imu} g) = λ e^{imu} g`` for one Fourier mode ``m`` in u.

    ``g`` solves ``-(ρ g')'/r² + m² g/ρ = λ ρ g`` on the circle, by Fourier collocation
    on ``v_j = 2πj/n_v`` (``n_v`` odd, so there is no Nyquist mode): the symmetric
    positive semidefinite ``K = Dᵀ diag(ρ) D / r² + diag(m²/ρ)`` against the weight
    ``diag(ρ)``. ρ is even in v, so the even and the odd eigenfunctions are computed
    separately (exact parity, half-size problems). Returns the ``n_v`` eigenvalues in
    ascending order and the real eigenvectors ``g(v_j)`` as columns, normalized so that
    ``∫∫ |e^{imu} g|² r ρ du dv = 1`` by the periodic trapezoid rule:
    ``(2π)² r / n_v · Gᵀ diag(ρ) G = I``. Only m² enters, so ``λ_{-m,n} = λ_{m,n}``.
    Modes whose v wavenumber is well below n_v/2 are accurate to round-off; the top of
    the discrete spectrum is not.
    """
    _check_torus(major_radius, minor_radius)
    if int(n_v) != n_v or n_v < 3 or n_v % 2 == 0:
        raise ValueError("n_v must be an odd integer >= 3")
    return _tube_eigenpairs(m, *_tube_operator(int(n_v), major_radius, minor_radius), minor_radius)


def torus_exact_solution(targets: NDArray, x0: NDArray, t0: NDArray, k0: float, sigma: float,
                         times: NDArray, major_radius: float = 1.0, minor_radius: float = 0.5, *,
                         n_u: int | None = None, n_v: int | None = None, chunk: int = 2048) -> NDArray:
    """ψ(targets, times) on the torus, with ψ0 normalized so that ∫|ψ0|² dA = 1.

    ψ0 is ``sphere_packet`` (the chordal packet of ``initial_conditions.gaussian_packet``)
    with centre ``x0`` on the torus and unit tangent ``t0`` there, used as given. It is
    sampled on an ``n_u × n_v`` (u, v) grid (default ``torus_resolution``; even sizes are
    rounded up to odd), normalized with the periodic trapezoid rule and the exact area
    element, expanded in ``e^{imu} g_{m,n}(v)`` (FFT in u, then the eigenvectors of
    ``torus_eigenpairs`` with weight ρ) and evolved with ``e^{-iλt/2}``. ``targets``
    (N, 3) are mapped to (u, v) by ``torus_angles`` and evaluated by trigonometric
    interpolation, at most ``chunk`` targets at a time (fewer for many times, to bound
    memory). Returns (N, len(times)) complex.

    Measured at the default resolution on R = 1, r = 0.5 (80 packets: outer and inner
    equator, top, generic; four directions; k0 ≤ 12, σ ∈ [0.12, 0.3]; t ≤ 0.5): ψ0 is
    reproduced to ≤ 4e-14 (max |ψ0| ≈ 1.2-3.3), doubling n_u and n_v changes ψ by
    ≤ 3e-12 (round-off in the eigenvalues) and ∫|ψ(t)|² dA = 1 to ≤ 1e-14. 16 000
    targets × 11 times take about 1 s for k0 = 12, σ = 0.12.
    """
    _check_torus(major_radius, minor_radius)
    default_u, default_v = torus_resolution(k0, sigma, major_radius, minor_radius)
    n_u, n_v = (int(n) | 1 for n in (n_u or default_u, n_v or default_v))
    if n_u < 3 or n_v < 3 or chunk < 1:
        raise ValueError("n_u and n_v must be at least 3 and chunk positive")
    targets, x0, t0 = (np.asarray(a, float) for a in (targets, x0, t0))
    if targets.ndim != 2 or targets.shape[1] != 3 or x0.shape != (3,) or t0.shape != (3,):
        raise ValueError("targets must be (N, 3), x0 and t0 3-vectors")
    times = np.atleast_1d(np.asarray(times, float))
    if len(targets) == 0 or len(times) == 0:
        return np.zeros((len(targets), len(times)), complex)
    u = 2 * np.pi * np.arange(n_u) / n_u
    v, rho = _tube_grid(n_v, major_radius, minor_radius)
    grid = torus_points(u[:, None], v[None, :], major_radius, minor_radius).reshape(-1, 3)
    psi0 = sphere_packet(grid, x0, t0, k0, sigma).reshape(n_u, n_v)
    psi0 /= np.sqrt((2 * np.pi)**2 * minor_radius / (n_u * n_v) * np.sum(rho * np.abs(psi0)**2))
    modes = np.fft.fft(psi0, axis=0) / n_u                      # row m: ψ0 = Σ_m modes[m](v) e^{imu}
    weight = (2 * np.pi)**2 * minor_radius / n_v * rho          # coefficients a = Gᵀ (weight ⊙ f)
    operator = _tube_operator(n_v, major_radius, minor_radius)
    evolved = np.empty((n_v, n_u, len(times)), complex)         # ψ(t) at (v_j, mode m)
    for m in range(n_u // 2 + 1):
        lam, G = _tube_eigenpairs(m, *operator, minor_radius)
        rows = [m] if m == 0 else [m, n_u - m]                  # +m and -m share the eigenpairs
        # Coefficients and evolution as real products (complex data viewed as re/im pairs).
        data = np.ascontiguousarray(weight[:, None] * modes[rows].T)
        coeffs = (G.T @ data.view(float)).view(complex)          # (n, len(rows))
        terms = coeffs[:, :, None] * np.exp(-0.5j * np.outer(lam, times))[:, None, :]
        evolved[:, rows] = (G @ terms.reshape(n_v, -1).view(float)).view(complex).reshape(terms.shape)
    coef = (np.fft.fft(evolved, axis=0) / n_v).reshape(n_v, -1)  # ψ = Σ coef[k, m] e^{ikv} e^{imu}
    k_of, m_of = np.fft.fftfreq(n_v, 1.0 / n_v), np.fft.fftfreq(n_u, 1.0 / n_u)
    target_u, target_v = torus_angles(targets, major_radius)
    out = np.empty((len(targets), len(times)), complex)
    step = max(1, min(chunk, (1 << 22) // max(1, n_u * len(times))))   # in_v below: ≤ 64 MB
    for start in range(0, len(targets), step):
        part = slice(start, start + step)
        in_v = (np.exp(1j * np.outer(target_v[part], k_of)) @ coef).reshape(-1, n_u, len(times))
        out[part] = np.matmul(np.exp(1j * np.outer(target_u[part], m_of))[:, None, :], in_v)[:, 0]
    return out


In [ ]:
%%writefile src/schrodinger_glno/sampling.py
"""Random Gaussian-packet parameters on a mesh.

Each packet is ``(center, phi, k0, sigma)``: a vertex chosen with probability
proportional to its area (uniform over the surface, not over vertices), an
angle ``phi`` for the direction in the tangent plane, and ``(k0, sigma)`` drawn
uniformly and kept only if the mesh resolves the packet.
"""
import numpy as np
from numpy.typing import NDArray
from .initial_conditions import max_wavenumber, tangent_direction

PARAM_COLUMNS = ("center", "phi", "k0", "sigma")


def tangent_basis(normal: NDArray) -> tuple[NDArray, NDArray]:
    """Orthonormal ``(e1, e2)`` spanning the plane orthogonal to ``normal``."""
    normal = np.asarray(normal, float) / np.linalg.norm(normal)
    axis = np.eye(3)[np.argmin(np.abs(normal))]  # the axis least aligned with the normal
    e1 = tangent_direction(normal, axis)
    return e1, np.cross(normal, e1)


def direction_from_angle(normal: NDArray, phi: float) -> NDArray:
    """Unit tangent vector ``cos(phi) e1 + sin(phi) e2`` at a point with this ``normal``."""
    e1, e2 = tangent_basis(normal)
    return np.cos(phi) * e1 + np.sin(phi) * e2


def allowed_wavenumber(mean_edge: float, min_points_per_wavelength: float) -> float:
    """Largest wavenumber with at least ``min_points_per_wavelength`` edges per wavelength."""
    return 2 * np.pi / (min_points_per_wavelength * mean_edge)


def resolved_fraction(k0_range: tuple[float, float], sigma_range: tuple[float, float], k_allowed: float,
                      n_std: float = 3.0) -> float:
    """Exact area fraction of ``k0_range x sigma_range`` where ``k0 + n_std / (2 sigma) <= k_allowed``.

    It is the probability that ``sample_packets`` accepts a uniform draw; the
    empirical acceptance of a few dozen draws is a noisy estimate of it.
    """
    (k_lo, k_hi), (s_lo, s_hi) = k0_range, sigma_range
    c = n_std / 2

    def integral(a: float, b: float) -> float:          # ∫ (k_allowed - k_lo - c/σ) dσ over [a, b]
        return (k_allowed - k_lo) * (b - a) - c * np.log(b / a) if b > a else 0.0

    # For each σ the accepted k0 run from k_lo to min(k_hi, k_allowed - c/σ): none below s1, all above s2.
    s1 = c / (k_allowed - k_lo) if k_allowed > k_lo else np.inf
    s2 = c / (k_allowed - k_hi) if k_allowed > k_hi else np.inf
    a, b = min(max(s1, s_lo), s_hi), min(max(s2, s_lo), s_hi)
    area = integral(a, b) + (k_hi - k_lo) * (s_hi - b)
    return float(area / ((k_hi - k_lo) * (s_hi - s_lo)))


def sample_packets(n: int, mass: NDArray, rng: np.random.Generator, k0_range: tuple[float, float],
                   sigma_range: tuple[float, float], k_allowed: float, stats: dict | None = None) -> NDArray:
    """Return ``(n, 4)`` rows ``(center, phi, k0, sigma)`` that the mesh resolves.

    ``(k0, sigma)`` is uniform on the part of ``k0_range x sigma_range`` where
    ``k0 + 3 / (2 sigma) <= k_allowed``. If ``stats`` is given, it receives the
    number of ``(k0, sigma)`` pairs ``tried`` and ``accepted``: an acceptance below 1
    means part of the requested rectangle was left out.
    """
    if n < 0 or np.any(mass <= 0):
        raise ValueError("n must be >= 0 and mass positive")
    if max_wavenumber(k0_range[0], sigma_range[1]) > k_allowed:
        raise ValueError("no (k0, sigma) in these ranges is resolved by the mesh")
    centers = rng.choice(len(mass), size=n, p=mass / mass.sum())
    phi = rng.uniform(0, 2 * np.pi, size=n)
    k0, sigma = np.empty(n), np.empty(n)
    filled, rounds, tried, accepted = 0, 0, 0, 0
    while filled < n:
        rounds += 1
        if rounds > 1000:
            raise ValueError("almost no (k0, sigma) in these ranges is resolved by the mesh")
        k_try = rng.uniform(*k0_range, size=2 * (n - filled) + 8)
        s_try = rng.uniform(*sigma_range, size=k_try.size)
        resolved = k_try + 3 / (2 * s_try) <= k_allowed
        tried += k_try.size
        accepted += int(resolved.sum())
        ok = np.flatnonzero(resolved)[: n - filled]
        k0[filled:filled + ok.size], sigma[filled:filled + ok.size] = k_try[ok], s_try[ok]
        filled += ok.size
    if stats is not None:
        stats.update(tried=tried, accepted=accepted)
    return np.column_stack([centers, phi, k0, sigma])


In [ ]:
%%writefile src/schrodinger_glno/shapes.py
"""Stage-3 shapes: deformed spheres and tori, made by moving the vertices of our meshes.

Both families start from a validated near-equilateral mesh (``mesh.icosphere``,
``mesh.torus``) and move every vertex onto a smooth random surface, so the connectivity
is kept. On a torus the triangles degrade only as much as the surface stretches; on a
sphere the vertices are then spread evenly over the surface (``deformed_sphere``). Every
shape is centred and scaled to area 4π, so k0, σ, T and the number of modes mean the same
on all of them (Generacion_datos_etapa3.md, §3 and §8). ``delaunay_flip`` removes the
negative cotangent weights a deformation leaves, ``mesh_quality``, the smallest radius of
curvature of a sphere (``smallest_curvature_radius``, from the analytic surface) and the tube
numbers of a torus (``tube_quantities``) feed ``check_gates``, which decides whether a shape is
kept, ``make_shape`` draws until one passes and
``refine_shape`` subdivides a kept shape on its own surface, for a refinement study.
"""
from dataclasses import dataclass
import numpy as np
import scipy.sparse as sp
from numpy.typing import NDArray
from scipy.interpolate import CubicSpline
from scipy.sparse.csgraph import connected_components
from scipy.spatial.distance import cdist
from .mesh import _check_mesh, cotan_laplacian, icosphere, torus, vertex_areas
from .reference import torus_angles
from .sampling import allowed_wavenumber

AREA = 4 * np.pi
SPINE_HARMONICS = (2, 3)                     # k of the spine terms a_k, b_k
# (j, k) of c_jk: one of each pair ±(j, k), so helical modes (j k ≠ 0) twist both ways.
TUBE_MODES = ((0, 1), (0, 2)) + tuple((j, k) for j in range(1, 4) for k in range(-2, 3))
RELAX_ITERATIONS = (150, 100, 100)           # relaxation steps on icosphere levels n - 2, n - 1 and n


def _normalisation(verts: NDArray, faces: NDArray, area: float = AREA) -> tuple[float, NDArray]:
    """Scale and centre of ``normalize_area``, which returns ``scale · (verts - centre)``."""
    mass = vertex_areas(verts, faces)
    return float(np.sqrt(area / mass.sum())), np.sum(mass[:, None] * verts, axis=0) / mass.sum()


def normalize_area(verts: NDArray, faces: NDArray, area: float = AREA) -> tuple[NDArray, float]:
    """Mesh moved to its area-weighted centroid and scaled to total lumped area ``area``.

    Returns the new vertices and the scale factor applied.
    """
    scale, centroid = _normalisation(verts, faces, area)
    return scale * (verts - centroid), scale


def _sh_degrees(l_min: int, l_max: int) -> tuple[NDArray, NDArray]:
    """(ℓ, m) of every real spherical harmonic, by ℓ = l_min..l_max, then m = -ℓ..ℓ."""
    ell = np.concatenate([np.full(2 * l + 1, l) for l in range(l_min, l_max + 1)])
    m = np.concatenate([np.arange(-l, l + 1) for l in range(l_min, l_max + 1)])
    return ell, m


def real_spherical_harmonics(l_max: int, directions: NDArray, l_min: int = 2) -> NDArray:
    """Real orthonormal spherical harmonics at ``directions`` (N, 3), shape (N, (l_max+1)² - l_min²).

    Columns go by ℓ = l_min..l_max, then m = -ℓ..ℓ: ``√2 N_ℓ|m| Q_ℓ|m|(z) Im (x + iy)^|m|`` for
    m < 0, ``N_ℓ0 Q_ℓ0(z)`` and ``√2 N_ℓm Q_ℓm(z) Re (x + iy)^m`` for m > 0 at the unit vector
    (x, y, z), with ``N_ℓm² = (2ℓ+1)/4π · (ℓ-m)!/(ℓ+m)!`` and ``Q_ℓm = P_ℓ^m / sin^m θ`` (P without
    the Condon–Shortley phase), a polynomial in z by the usual three-term recurrence. These are
    ``√2 (-1)^m Im Y_ℓ^|m|``, ``Y_ℓ^0`` and ``√2 (-1)^m Re Y_ℓ^m`` of ``scipy.special.sph_harm_y``,
    eight times faster (the relaxation of a deformed sphere evaluates them 350 times).
    ``∫ Y_ℓm Y_ℓ'm' dω = δ_ℓℓ' δ_mm'``.
    """
    directions = np.asarray(directions, float)
    x, y, z = (directions / np.linalg.norm(directions, axis=1, keepdims=True)).T
    columns = {}
    re, im = np.ones_like(x), np.zeros_like(x)                  # (x + iy)^m
    for m in range(l_max + 1):
        q_prev, q = 0.0, np.full_like(z, np.prod(np.arange(1.0, 2 * m, 2)))    # Q_mm = (2m - 1)!!
        for l in range(m, l_max + 1):
            if l > m:
                q, q_prev = ((2 * l - 1) * z * q - (l + m - 1) * q_prev) / (l - m), q
            if l >= l_min:
                norm = np.sqrt((2 * l + 1) / (4 * np.pi) / np.prod(np.arange(l - m + 1.0, l + m + 1)))
                columns[l, m] = norm * q if m == 0 else np.sqrt(2) * norm * q * re
                if m > 0:
                    columns[l, -m] = np.sqrt(2) * norm * q * im
        re, im = re * x - im * y, re * y + im * x
    return np.stack([columns[key] for key in zip(*_sh_degrees(l_min, l_max))], axis=1)


@dataclass(frozen=True)
class SphereDeformation:
    """Star-shaped surface ``x(ω) = e^{s(ω)} ω`` with ``s = Σ_{ℓ=2}^{l_max} Σ_m s_ℓm Y_ℓm``.

    ℓ = 0 (size) and ℓ = 1 (almost a translation) are left out: the normalisation fixes both.
    ``coefficients`` are ordered as the columns of ``real_spherical_harmonics``.
    """
    coefficients: tuple[float, ...]

    def __post_init__(self):
        if len(self.coefficients) < 5 or (self.l_max + 1)**2 - 4 != len(self.coefficients):
            raise ValueError("need (l_max + 1)² - 4 coefficients with l_max >= 2")
        object.__setattr__(self, "coefficients", tuple(float(c) for c in self.coefficients))

    @property
    def l_max(self) -> int:
        return int(round(np.sqrt(len(self.coefficients) + 4))) - 1

    @property
    def rms(self) -> float:
        """Root mean square of s over the unit sphere, ``(Σ s_ℓm² / 4π)^½``."""
        return float(np.sqrt(np.sum(np.square(self.coefficients)) / (4 * np.pi)))


def sample_sphere_deformation(rng: np.random.Generator, l_max: int = 4, slope: float = 2.0,
                              tau_range: tuple[float, float] = (0.05, 0.15)) -> SphereDeformation:
    """Random ``s_ℓm ~ N(0, ℓ^-slope)`` (variance), rescaled so that s has RMS τ ~ U(tau_range).

    Draws, in this order: the coefficients, then τ. The defaults come from the planning study:
    at the same τ, l_max = 6 halves the smallest radius of curvature, and τ = 0.15 is the largest
    RMS whose relaxed meshes pass the gates with the data error near the round sphere's.
    """
    ell, _ = _sh_degrees(2, l_max)
    s = rng.standard_normal(len(ell)) * ell**(-slope / 2)
    tau = rng.uniform(*tau_range)
    return SphereDeformation(tuple(s * tau / np.sqrt(np.sum(s**2) / (4 * np.pi))))


def sphere_radius(deformation: SphereDeformation, directions: NDArray) -> NDArray:
    """``ρ(ω) = exp(s(ω))`` at unit ``directions`` (N, 3).

    s is summed with NumPy, not BLAS, whose rounding depends on the number of threads: the
    shape must be the same bit for bit with any thread count.
    """
    Y = real_spherical_harmonics(deformation.l_max, directions)
    return np.exp(np.sum(Y * np.asarray(deformation.coefficients), axis=1))


def _onto_sphere_surface(deformation: SphereDeformation, points: NDArray) -> NDArray:
    """``points`` (N, 3) moved along their rays onto the surface ``ρ(ω) ω``."""
    omega = points / np.linalg.norm(points, axis=1, keepdims=True)
    return sphere_radius(deformation, omega)[:, None] * omega


def _sphere_midpoints(deformation: SphereDeformation, points: NDArray, edges: NDArray) -> NDArray:
    """Surface points at the midpoint directions of ``edges`` (E, 2) of ``points`` on the surface.

    The midpoint direction is the normalised mean of the two directions, as in ``mesh.icosphere``.
    """
    omega = points / np.linalg.norm(points, axis=1, keepdims=True)
    mid = omega[edges].mean(axis=1)
    mid /= np.linalg.norm(mid, axis=1, keepdims=True)
    return sphere_radius(deformation, mid)[:, None] * mid


def _midpoint_subdivision(faces: NDArray, n_verts: int) -> tuple[NDArray, NDArray]:
    """Every face split in four at its edge midpoints, by the rule of ``mesh.icosphere``.

    Returns the edges (E, 2) whose midpoints become vertices ``n_verts``, ``n_verts + 1``, ... in
    that order, and the new faces; on the faces of ``icosphere(n)`` they are those of ``icosphere(n + 1)``.
    """
    edges = np.sort(faces[:, [0, 1, 1, 2, 2, 0]].reshape(-1, 2), axis=1)
    unique, inverse = np.unique(edges, axis=0, return_inverse=True)
    ab, bc, ca = (n_verts + inverse.reshape(-1, 3)).T
    a, b, c = faces.T
    return unique, np.concatenate([np.stack(f, axis=1) for f in ((a, ab, ca), (b, bc, ab), (c, ca, bc), (ab, bc, ca))])


def _relax(deformation: SphereDeformation, points: NDArray, faces: NDArray, iterations: int,
           step: float = 0.3, umbrella: float = 0.5) -> NDArray:
    """``iterations`` steps that even out the edges of a mesh whose vertices lie on the surface.

    Each step moves vertex i by ``step · F_i`` without its component along the vertex normal, then
    back onto the surface along its ray. ``F_i = Σ_j (1 - l0/|e_ij|) e_ij / deg_i + umbrella ·
    (mean of the neighbours - x_i)``, with e_ij = x_j - x_i and l0 the current mean edge: springs
    pull every edge toward the mean. Umbrella smoothing alone converges to a near-conformal map,
    whose vertex density follows the conformal factor (≈ e^s), so the edges keep the spread of
    the radial map. Sparse products and elementwise sums only, never BLAS: the result does not
    depend on the number of threads.
    """
    n = len(points)
    _, _, keys = _edge_table(faces, n)
    i, j = np.divmod(keys, n)
    edge = np.arange(len(keys))
    spring = sp.csr_matrix((np.r_[np.ones(len(keys)), -np.ones(len(keys))], (np.r_[i, j], np.r_[edge, edge])),
                           shape=(n, len(keys)))                       # row i: Σ over its edges, as e_ij
    adjacency = sp.csr_matrix((np.ones(2 * len(keys)), (np.r_[i, j], np.r_[j, i])), shape=(n, n))
    corner = sp.csr_matrix((np.ones(faces.size), (faces.ravel(), np.repeat(np.arange(len(faces)), 3))),
                           shape=(n, len(faces)))                      # vertex-face incidence
    degree = np.diff(adjacency.indptr)[:, None]
    x = points
    for _ in range(iterations):
        e = x[j] - x[i]
        length = np.linalg.norm(e, axis=1, keepdims=True)
        force = spring @ ((1 - length.mean() / length) * e) / degree
        force += umbrella * (adjacency @ x / degree - x)
        p = x[faces]
        normal = corner @ np.cross(p[:, 1] - p[:, 0], p[:, 2] - p[:, 0])          # area-weighted
        normal /= np.linalg.norm(normal, axis=1, keepdims=True)
        force -= np.sum(force * normal, axis=1, keepdims=True) * normal
        x = _onto_sphere_surface(deformation, x + step * force)
    return x


def _sphere_surface(deformation: SphereDeformation, subdivisions: int, relax: bool = False,
                    iterations: tuple[int, ...] = RELAX_ITERATIONS) -> tuple[NDArray, NDArray]:
    """Mesh of the surface with the faces of ``icosphere(subdivisions)``, before normalisation.

    Without ``relax`` each vertex ω of the icosphere moves to ρ(ω) ω. With it, the same is done
    on the icosphere ``len(iterations) - 1`` levels coarser, which is relaxed (``_relax``) for
    ``iterations[0]`` steps, then subdivided (``_midpoint_subdivision``, new vertices at
    ``_sphere_midpoints``) and relaxed for ``iterations[1]`` steps, and so on up to level
    ``subdivisions``. The coarse levels move the vertex density across the whole surface in few
    steps; at the finest level alone the springs would need many more.
    """
    levels = min(len(iterations), subdivisions + 1) if relax else 1
    omega, faces = icosphere(subdivisions - levels + 1)
    x = sphere_radius(deformation, omega)[:, None] * omega
    for k, steps in enumerate(iterations[len(iterations) - levels:] if relax else ()):
        if k:
            edges, faces = _midpoint_subdivision(faces, len(x))
            x = np.vstack([x, _sphere_midpoints(deformation, x, edges)])
        x = _relax(deformation, x, faces, steps)
    return x, faces


def deformed_sphere(deformation: SphereDeformation, subdivisions: int = 5,
                    relax: bool = True) -> tuple[NDArray, NDArray]:
    """Mesh of the surface ρ(ω) ω with the faces of ``icosphere(subdivisions)``, after ``normalize_area``.

    Moving each icosphere vertex ω radially to ρ(ω) ω (``relax=False``) keeps the vertex density
    of the sphere: where ρ grows the edges stretch, and from an RMS τ of about 0.1 the edge spread
    (p95/mean), then the angles, fail the gates. ``relax`` spreads the vertices evenly over the same
    surface (``_sphere_surface``): every vertex stays on it to rounding, so the shape is kept (the
    first 20 eigenvalues move by less than 0.2 %, within the discretisation error) while the
    triangles improve. Unrelaxed levels nest: the first vertices of level n + 1 are those of level
    n, on the same surface (up to each level's normalisation, a similarity a chordal-area deficit
    away from the identity). Relaxed levels do not; ``refine_shape`` gives a nested finer mesh of either.
    """
    verts, faces = _sphere_surface(deformation, subdivisions, relax)
    return normalize_area(verts, faces)[0], faces


def _harmonic_polynomial(deformation: SphereDeformation, omega: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """``p = Σ s_ℓm Y_ℓm`` with each Y_ℓm the polynomial of ``real_spherical_harmonics`` in u = (x, y, z), not
    normalised: its value (N,), gradient (3, N) and Hessian (3, 3, N) in u at unit ``omega`` (3, N), where p = s.

    With w = (x + iy)^m, the m and -m terms are ``Q_ℓm(z) Re(c w)``, ``c = √2 N_ℓm (s_ℓm - i s_ℓ,-m)`` (``N_ℓ0
    s_ℓ0`` for m = 0); ``∂_y w = i ∂_x w``, and Q_ℓm' and Q_ℓm'' follow the recurrence of Q_ℓm, differentiated.
    Elementwise products only, never BLAS.
    """
    x, y, z = omega
    coefficients = dict(zip(zip(*_sh_degrees(2, deformation.l_max)), deformation.coefficients))
    powers = [np.ones_like(x) + 0j]                                    # (x + iy)^k
    for _ in range(deformation.l_max):
        powers.append(powers[-1] * (x + 1j * y))
    value, grad, hess = np.zeros_like(x), np.zeros((3,) + x.shape), np.zeros((3, 3) + x.shape)
    for m in range(deformation.l_max + 1):
        w = powers[m]
        w1 = m * powers[m - 1] if m >= 1 else 0 * w                    # ∂_x w
        w2 = m * (m - 1) * powers[m - 2] if m >= 2 else 0 * w          # ∂_x² w; ∂_x ∂_y w = i w2, ∂_y² w = -w2
        q_prev, dq_prev, ddq_prev = 0.0, 0.0, 0.0
        q, dq, ddq = np.full_like(z, np.prod(np.arange(1.0, 2 * m, 2))), np.zeros_like(z), np.zeros_like(z)
        for l in range(m, deformation.l_max + 1):
            if l > m:
                a, b = (2 * l - 1) / (l - m), (l + m - 1) / (l - m)
                q, q_prev, dq, dq_prev, ddq, ddq_prev = (a * z * q - b * q_prev, q, a * (q + z * dq) - b * dq_prev, dq,
                                                         a * (2 * dq + z * ddq) - b * ddq_prev, ddq)
            if l < 2:
                continue
            norm = np.sqrt((2 * l + 1) / (4 * np.pi) / np.prod(np.arange(l - m + 1.0, l + m + 1)))
            c = norm * (np.sqrt(2) * (coefficients[l, m] - 1j * coefficients[l, -m]) if m else coefficients[l, 0])
            B, Bx, By, Bxx, Bxy = (c * w).real, (c * w1).real, (1j * c * w1).real, (c * w2).real, (1j * c * w2).real
            value += q * B
            grad += [q * Bx, q * By, dq * B]
            hess += [[q * Bxx, q * Bxy, dq * Bx], [q * Bxy, -q * Bxx, dq * By], [dq * Bx, dq * By, ddq * B]]
    return value, grad, hess


def _outer(u: NDArray, v: NDArray) -> NDArray:
    return u[:, None] * v[None]


def _matvec(A: NDArray, v: NDArray) -> NDArray:
    return np.sum(A * v[None], axis=1)


def sphere_curvatures(deformation: SphereDeformation, directions: NDArray) -> tuple[NDArray, NDArray]:
    """Principal curvatures κ1 ≥ κ2 of the surface at ρ(ω) ω for the ``directions`` ω (N, 3), before normalisation.

    The surface is the zero set of ``G(x) = log|x| - p(x/|x|)``, p the polynomial of ``_harmonic_polynomial`` with
    gradient a and Hessian ∇²p. At x = ρω, ``∇G = (ω - g)/ρ`` with g = Pa the gradient of s on the unit sphere
    (P = I - ωωᵀ), and ``ρ² ∇²G = B = I - 2ωωᵀ - ∇²S``, where ``∇²S = P ∇²p P - aωᵀ - ωaᵀ - (a·ω)(I - 3ωωᵀ)`` is the
    Hessian of S(x) = p(x/|x|) at |x| = 1. The curvatures are the eigenvalues of ``Q ∇²G Q / |∇G|`` on the tangent
    plane, Q = I - nnᵀ with n = (ω - g)/√(1 + |g|²): those of ``C = QBQ`` over ``ρ √(1 + |g|²)``. C is 0 along n, so
    they are ``tr C/2 ± |C - (tr C/2) Q|_F / √2``, without the cancellation of ``√((tr C/2)² - det)``: 1/ρ to rounding
    on a round sphere. Positive where the surface bends away from its outward normal, as a sphere does. Elementwise
    products only, never BLAS.
    """
    omega = (np.asarray(directions, float) / np.linalg.norm(directions, axis=1, keepdims=True)).T     # (3, N)
    value, a, hess_p = _harmonic_polynomial(deformation, omega)
    eye = np.eye(3)[:, :, None]
    a_omega = np.sum(a * omega, 0)
    g = a - a_omega * omega
    h_omega, radial = _matvec(hess_p, omega), _outer(omega, omega)
    hess_S = (hess_p - _outer(omega, h_omega) - _outer(h_omega, omega) + np.sum(omega * h_omega, 0) * radial
              - _outer(a, omega) - _outer(omega, a) - a_omega * (eye - 3 * radial))
    B = eye - 2 * radial - hess_S
    stretch = np.sqrt(1 + np.sum(g * g, 0))
    n = (omega - g) / stretch
    Bn = _matvec(B, n)
    C = B - _outer(n, Bn) - _outer(Bn, n) + np.sum(n * Bn, 0) * _outer(n, n)
    mean = (C[0, 0] + C[1, 1] + C[2, 2]) / 2
    half_gap = np.sqrt(np.sum(np.square(C - mean * (eye - _outer(n, n))), axis=(0, 1)) / 2)
    factor = 1 / (np.exp(value) * stretch)
    return factor * (mean + half_gap), factor * (mean - half_gap)


def _largest_curvature(deformation: SphereDeformation, directions: NDArray) -> NDArray:
    """``max(|κ1|, |κ2|)`` at ρ(ω) ω (``sphere_curvatures``)."""
    k1, k2 = sphere_curvatures(deformation, directions)
    return np.maximum(np.abs(k1), np.abs(k2))


def smallest_curvature_radius(deformation: SphereDeformation, scale: float = 1.0, subdivisions: int = 6,
                              margin: float = 0.8, max_starts: int = 32, tol: float = 1e-10) -> float:
    """``scale / max |κ|``: the smallest radius of curvature of the surface ρ(ω) ω, times ``scale``.

    With the ``scale`` of ``build_shape`` it is in the units of the stored mesh (area 4π); 1 on the round sphere.
    ``max(|κ1|, |κ2|)`` (``sphere_curvatures``) on the directions of ``icosphere(subdivisions)``, then a compass search
    from each of its local maxima on the icosphere's edges that is within ``margin`` of the largest (at most
    ``max_starts``, the largest first): a 5 × 5 grid of tangent steps h around the best point, which moves to the
    grid's best when that is larger beyond rounding and keeps h if it moved to the grid's edge; otherwise h halves,
    from half an edge down to ``tol``. The grid alone is up to 1 % low; over 85 draws (ℓ ≤ 8, τ up to 0.3) the search
    was never below an icosphere(8) grid (655k directions), and agreed with it to 5e-4. Elementwise products only,
    never BLAS: the gate decision does not depend on the number of threads.
    """
    omega, faces = icosphere(subdivisions)
    kappa = _largest_curvature(deformation, omega)
    _, _, keys = _edge_table(faces, len(omega))
    i, j = np.divmod(keys, len(omega))
    neighbours = np.full(len(omega), -np.inf)
    np.maximum.at(neighbours, i, kappa[j])
    np.maximum.at(neighbours, j, kappa[i])
    start = np.flatnonzero((kappa >= neighbours) & (kappa >= margin * kappa.max()))
    start = start[np.argsort(-kappa[start], kind="stable")[:max_starts]]
    p, best = omega[start], kappa[start]
    step = np.full(len(p), 0.5 * np.linalg.norm(omega[i[0]] - omega[j[0]]))
    a, b = (g.ravel() for g in np.meshgrid(np.arange(-2, 3), np.arange(-2, 3), indexing="ij"))
    rows = np.arange(len(p))
    for _ in range(200):                                       # about 30 rounds; each move gains beyond rounding
        if step.max() <= tol:
            break
        axis = np.eye(3)[np.argmin(np.abs(p), axis=1)]          # a tangent frame (e1, e2) at p
        e1 = axis - np.sum(axis * p, 1, keepdims=True) * p
        e1 /= np.linalg.norm(e1, axis=1, keepdims=True)
        e2 = np.cross(p, e1)
        trial = p[:, None] + step[:, None, None] * (a[:, None] * e1[:, None] + b[:, None] * e2[:, None])
        trial /= np.linalg.norm(trial, axis=2, keepdims=True)
        values = _largest_curvature(deformation, trial.reshape(-1, 3)).reshape(len(p), -1)
        k = np.argmax(values, axis=1)
        better = values[rows, k] > best * (1 + 1e-13)
        p = np.where(better[:, None], trial[rows, k], p)
        best = np.where(better, values[rows, k], best)
        step = np.where(better & ((np.abs(a[k]) == 2) | (np.abs(b[k]) == 2)), step, step / 2)
    return float(scale / best.max())


@dataclass(frozen=True)
class TorusDeformation:
    """Tube ``x(u, v) = c(u) + r(u, v) (cos v e1(u) + sin v e2(u))`` around a closed spine c.

    Spine ``c = (R cos u, R sin u, z)`` with ``R = R0 (1 + Σ_k a_k cos(k u + φ_k))`` and
    ``z = R0 Σ_k b_k sin(k u + ψ_k)``, k in ``SPINE_HARMONICS``. Tube radius
    ``r = r0 exp(t)``, ``t = Σ c_jk cos(j u + k v + θ_jk)`` over ``TUBE_MODES`` (j = 0..3,
    k = -2..2: bumps that wind around the tube either way as they go along it). ``alpha = r0/R0``
    with ``R0 r0 = 1/π``, so the round torus (every coefficient zero, the default) has area 4π.
    The frame (e1, e2) is described in ``spine_frame``.
    """
    alpha: float = 0.5
    a: tuple[float, ...] = (0.0,) * len(SPINE_HARMONICS)
    phi: tuple[float, ...] = (0.0,) * len(SPINE_HARMONICS)
    b: tuple[float, ...] = (0.0,) * len(SPINE_HARMONICS)
    psi: tuple[float, ...] = (0.0,) * len(SPINE_HARMONICS)
    c: tuple[float, ...] = (0.0,) * len(TUBE_MODES)
    theta: tuple[float, ...] = (0.0,) * len(TUBE_MODES)

    def __post_init__(self):
        if not 0 < self.alpha < 1:
            raise ValueError("alpha = r0/R0 must be in (0, 1)")
        for name, size in [("a", len(SPINE_HARMONICS)), ("phi", len(SPINE_HARMONICS)), ("b", len(SPINE_HARMONICS)),
                           ("psi", len(SPINE_HARMONICS)), ("c", len(TUBE_MODES)), ("theta", len(TUBE_MODES))]:
            if len(getattr(self, name)) != size:
                raise ValueError(f"{name} needs {size} values")
            object.__setattr__(self, name, tuple(float(x) for x in getattr(self, name)))
        object.__setattr__(self, "alpha", float(self.alpha))

    @property
    def major_radius(self) -> float:
        return float(np.sqrt(1 / (np.pi * self.alpha)))

    @property
    def minor_radius(self) -> float:
        return self.alpha * self.major_radius

    @property
    def tube_rms(self) -> float:
        """Root mean square of t over (u, v): ``(Σ c_jk² / 2)^½``.

        The mean of ``cos(j u + k v + θ) cos(j' u + k' v + θ')`` over the torus is half the mean
        of ``cos((j-j') u + (k-k') v + ...) + cos((j+j') u + (k+k') v + ...)``, which is ½ when
        (j', k') = (j, k) and 0 otherwise: neither (0, 0) nor both of a pair ±(j, k) is a mode.
        """
        return float(np.sqrt(np.sum(np.square(self.c)) / 2))


def sample_torus_deformation(rng: np.random.Generator, alpha_range: tuple[float, float] = (0.45, 0.55),
                             a_max: float = 0.04, b_max: float = 0.06,
                             tau_range: tuple[float, float] = (0.0, 0.075)) -> TorusDeformation:
    """Random deformed torus: α, a_k, b_k uniform, phases uniform in [0, 2π), c_jk ~ N(0, 1)
    rescaled so that t has RMS τ ~ U(tau_range) over (u, v).

    Draws, in this order: α, a, φ, b, ψ, c, θ, τ. The defaults come from the planning study:
    above α = 0.55 the hole narrows (round α = 0.6 is at the hole gate) and a packet on the inner
    side reaches across it, and bends fold the inner side more often.
    """
    n_s, n_t = len(SPINE_HARMONICS), len(TUBE_MODES)
    alpha = rng.uniform(*alpha_range)
    a, phi = rng.uniform(-a_max, a_max, n_s), rng.uniform(0, 2 * np.pi, n_s)
    b, psi = rng.uniform(-b_max, b_max, n_s), rng.uniform(0, 2 * np.pi, n_s)
    c, theta = rng.standard_normal(n_t), rng.uniform(0, 2 * np.pi, n_t)
    tau = rng.uniform(*tau_range)
    c = c * tau / np.sqrt(np.sum(c**2) / 2)
    return TorusDeformation(alpha, tuple(a), tuple(phi), tuple(b), tuple(psi), tuple(c), tuple(theta))


def _spine(deformation: TorusDeformation, u: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """c(u), c'(u) and c''(u), shape ``u.shape + (3,)``, from their closed forms."""
    u = np.asarray(u, float)
    R0 = deformation.major_radius
    k = np.array(SPINE_HARMONICS, float)
    a, b = np.array(deformation.a), np.array(deformation.b)
    in_plane = k * u[..., None] + np.array(deformation.phi)
    height = k * u[..., None] + np.array(deformation.psi)
    R = R0 * (1 + np.sum(a * np.cos(in_plane), -1))
    R1 = -R0 * np.sum(a * k * np.sin(in_plane), -1)
    R2 = -R0 * np.sum(a * k**2 * np.cos(in_plane), -1)
    z = R0 * np.sum(b * np.sin(height), -1)
    z1 = R0 * np.sum(b * k * np.cos(height), -1)
    z2 = -R0 * np.sum(b * k**2 * np.sin(height), -1)
    cu, su = np.cos(u), np.sin(u)
    return (np.stack([R * cu, R * su, z], -1),
            np.stack([R1 * cu - R * su, R1 * su + R * cu, z1], -1),
            np.stack([R2 * cu - 2 * R1 * su - R * cu, R2 * su + 2 * R1 * cu - R * su, z2], -1))


def _vertical_frame(c1: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """T = c'/|c'|, e2⁰ = the vertical without its T component, e1⁰ = T × e2⁰ (outward on the round torus)."""
    T = c1 / np.linalg.norm(c1, axis=-1, keepdims=True)
    e2 = -T[..., 2:] * T
    e2[..., 2] += 1
    e2 /= np.linalg.norm(e2, axis=-1, keepdims=True)
    return T, np.cross(T, e2), e2


def _twist(deformation: TorusDeformation, n_grid: int) -> CubicSpline:
    """``θ(u) = -∫_0^u (ω - ω̄) du'`` with ``ω = (de1⁰/du)·e2⁰``, as a periodic cubic spline.

    ω is the rate at which the vertical frame turns about T; it has zero mean once ω̄ is
    taken out, so θ is periodic. Derivative and integral are spectral on ``n_grid`` points.
    """
    u = 2 * np.pi * np.arange(n_grid) / n_grid
    _, e1, e2 = _vertical_frame(_spine(deformation, u)[1])
    k = np.arange(n_grid // 2 + 1, dtype=float)
    k[-1] = 0                                                   # no Nyquist mode (n_grid is even)
    de1 = np.fft.irfft(1j * k[:, None] * np.fft.rfft(e1, axis=0), n_grid, axis=0)
    omega_hat = np.fft.rfft(np.sum(de1 * e2, -1))
    theta_hat = np.zeros_like(omega_hat)
    theta_hat[1:-1] = -omega_hat[1:-1] / (1j * k[1:-1])
    theta = np.fft.irfft(theta_hat, n_grid)
    theta -= theta[0]
    return CubicSpline(np.append(u, 2 * np.pi), np.append(theta, theta[0]), bc_type="periodic")


def spine_frame(deformation: TorusDeformation, u: NDArray,
                n_grid: int = 4096) -> tuple[NDArray, NDArray, NDArray, NDArray]:
    """Spine point c, unit tangent T and the tube frame (e1, e2) at ``u``, each ``u.shape + (3,)``.

    The vertical frame (``_vertical_frame``) closes up around the loop but twists about T when
    the spine goes up and down, which shears the tube and its triangles. It is rotated by
    ``θ = -∫ (ω - ω̄)`` (``_twist``), which leaves the constant rate ω̄ (zero twist plus a uniform
    rotation, so the frame still closes): ``(de1/du)·e2 = ω̄``. On the round torus ω = 0, θ = 0
    and ``e1 = (cos u, sin u, 0)``, ``e2 = ẑ``.
    """
    if n_grid < 8 or n_grid % 2:
        raise ValueError("n_grid must be even and at least 8")
    u = np.asarray(u, float)
    c, c1, _ = _spine(deformation, u)
    T, e1, e2 = _vertical_frame(c1)
    theta = _twist(deformation, n_grid)(u)[..., None]
    return c, T, np.cos(theta) * e1 + np.sin(theta) * e2, np.cos(theta) * e2 - np.sin(theta) * e1


def tube_log_radius(deformation: TorusDeformation, u: NDArray, v: NDArray) -> NDArray:
    """``t(u, v) = Σ c_jk cos(j u + k v + θ_jk)``, so that ``r = r0 e^t``."""
    u, v = np.broadcast_arrays(np.asarray(u, float), np.asarray(v, float))
    j, k = np.array(TUBE_MODES, float).T
    phase = j * u[..., None] + k * v[..., None] + np.array(deformation.theta)
    return np.sum(np.array(deformation.c) * np.cos(phase), -1)


def torus_surface(deformation: TorusDeformation, u: NDArray, v: NDArray) -> NDArray:
    """Points ``x(u, v)`` of the deformed torus before normalisation, ``broadcast(u, v).shape + (3,)``.

    v = 0 is the outer side and v = π/2 the top, as in ``reference.torus_points``.
    """
    u, v = np.broadcast_arrays(np.asarray(u, float), np.asarray(v, float))
    c, _, e1, e2 = spine_frame(deformation, u)
    r = deformation.minor_radius * np.exp(tube_log_radius(deformation, u, v))
    return c + r[..., None] * (np.cos(v)[..., None] * e1 + np.sin(v)[..., None] * e2)


def deformed_torus(deformation: TorusDeformation, target_edge: float = 0.037) -> tuple[NDArray, NDArray, float]:
    """``torus(target_edge, R0, r0)`` with each vertex moved from its (u, v) to x(u, v), normalised.

    Returns the vertices, the faces and the normalisation scale (``tube_quantities`` needs
    it). target_edge = 0.037 gives about 10.6k vertices with the mean edge of icosphere(5).
    """
    faces, u, v = _base_torus(deformation, target_edge)
    verts, scale = normalize_area(torus_surface(deformation, u, v), faces)
    return verts, faces, scale


def _base_torus(deformation: TorusDeformation, target_edge: float) -> tuple[NDArray, NDArray, NDArray]:
    """Faces of ``torus(target_edge, R0, r0)`` and the angles (u, v) of its vertices."""
    R0 = deformation.major_radius
    verts, faces = torus(target_edge, R0, deformation.minor_radius)
    return (faces, *torus_angles(verts, R0))


def tube_quantities(deformation: TorusDeformation, scale: float = 1.0, sigmas: tuple[float, ...] = (0.15, 0.2),
                    n_u: int = 1024, n_v: int = 128) -> dict:
    """Thickness, folding, hole and self-clearance of a deformed torus, lengths times ``scale``.

    - ``r_min``: the thinnest tube radius.
    - ``fold``: ``max_u κ(u) r_max(u)``, κ the spine curvature and r_max the largest radius of
      the section at u. At 1 the inner side of a bend folds onto itself.
    - ``hole``: ``2 min_u (|c_xy(u)| - r_max(u))``, the diameter of the widest vertical cylinder
      about the z axis (the spine's axis) that misses the ball of radius r_max(u) around every
      c(u), which holds the section at u. A lower bound of the surface's own clearance from the
      axis (the bulges of t need not point inward) and ``2 (R0 - r0)`` on the round torus. The
      planning study calibrated the hole gate on it: a packet on the inner side puts more of its
      norm across the hole as the hole narrows, and ``check_gates``'s 0.58 (area 4π) keeps that
      below about 6e-3 (σ = 0.2, beyond geodesic 4σ), as on the round α = 0.6 torus.
    - ``gap[str(σ)]`` (G7): the smallest ``|c(u1) - c(u2)| - r_max(u1) - r_max(u2)`` over
      sections at least 7σ apart along the surface, by the lower bound ``∫ |c'| (1 - κ r_max) du``
      of any surface path between them (the shorter way round). Each section lies in the ball
      of radius r_max around c(u), so this bounds the straight-line distance between parts of
      the surface the geodesic cut (§4: 1 up to 7σ) separates. ``None`` if no two sections are
      that far apart, which is common on thick tori: then the opposite sides of the hole are
      within 7σ of each other and only ``hole`` bounds them.

    Evaluated on an ``n_u × n_v`` grid in (u, v). Keys ``str(float(σ))`` and ``None`` (not inf)
    keep the dict strict JSON, unchanged by a round trip through a manifest.
    """
    u = 2 * np.pi * np.arange(n_u) / n_u
    v = 2 * np.pi * np.arange(n_v) / n_v
    c, c1, c2 = _spine(deformation, u)
    speed = np.linalg.norm(c1, axis=-1)
    kappa = np.linalg.norm(np.cross(c1, c2), axis=-1) / speed**3
    t = tube_log_radius(deformation, u[:, None], v[None, :])
    r0 = deformation.minor_radius
    r_max = r0 * np.exp(t.max(1))
    path = speed * np.clip(1 - kappa * r_max, 0, None)
    s = scale * np.concatenate([[0], np.cumsum(0.5 * (path + np.roll(path, -1)) * 2 * np.pi / n_u)])
    along = np.abs(s[:-1, None] - s[None, :-1])
    along = np.minimum(along, s[-1] - along)
    clearance = scale * (cdist(c, c) - r_max[:, None] - r_max[None, :])
    gap = {float(sigma): clearance[along >= 7 * sigma] for sigma in sigmas}
    return {"r_min": float(scale * r0 * np.exp(t.min())),
            "fold": float(np.max(kappa * r_max)),
            "hole": float(2 * scale * np.min(np.hypot(c[:, 0], c[:, 1]) - r_max)),
            "gap": {str(sigma): float(g.min()) if g.size else None for sigma, g in gap.items()}}


def _edge_table(faces: NDArray, n_verts: int) -> tuple[NDArray, NDArray, NDArray]:
    """Every edge of a closed, consistently oriented manifold mesh once, as its two half-edges.

    Half-edge ``3f + c`` is the edge of face f opposite its corner c, from ``faces[f, c+1]`` to
    ``faces[f, c+2]``. Returns both half-edges of every edge and the edge keys ``min·V + max``.
    """
    start, end = faces[:, [1, 2, 0]].ravel(), faces[:, [2, 0, 1]].ravel()
    key = np.minimum(start, end) * n_verts + np.maximum(start, end)
    order = np.argsort(key, kind="stable")
    key = key[order]
    first, second = order[0::2], order[1::2]
    if (len(key) % 2 or np.any(key[0::2] != key[1::2]) or np.any(key[2::2] == key[1:-1:2])
            or np.any(start[first] != end[second])):
        raise ValueError("the mesh must be closed, manifold and consistently oriented")
    return first, second, key[0::2]


def _cot(apex: NDArray, p: NDArray, q: NDArray, denom_eps: float = 1e-10) -> NDArray:
    """Cotangent of the angle at ``apex`` of the triangles (apex, p, q), computed as in ``cotan_laplacian``."""
    u, w = p - apex, q - apex
    return np.einsum("fd,fd->f", u, w) / (np.linalg.norm(np.cross(u, w), axis=1) + denom_eps)


def _corner_cotangents(verts: NDArray, faces: NDArray) -> NDArray:
    """Cotangent of every corner angle, (F, 3); edge weights from it equal ``-L_ij`` bit for bit."""
    corners = verts[faces]
    return np.stack([_cot(corners[:, k], corners[:, (k + 1) % 3], corners[:, (k + 2) % 3]) for k in range(3)], 1)


def delaunay_flip(verts: NDArray, faces: NDArray, tol: float = 1e-12, max_rounds: int = 100) -> tuple[NDArray, int]:
    """Flip edges with negative cotangent weight ``(cot α + cot β)/2`` until none is left.

    Extrinsic flips on a closed, consistently oriented manifold mesh: the edge ij of the faces
    (k, i, j) and (l, j, i) becomes kl, with faces (k, i, l) and (l, j, k); vertices stay. A
    weight is negative below ``-tol`` times the largest |weight|. Each round flips, in every
    face, at most its most negative edge, and only when the flip is legal: kl is not already an
    edge (no duplicate edge, the mesh stays a manifold), the new faces keep the orientation of
    the old pair with non-zero area, and the new weight is larger than the old one (no flip
    back and forth). In the plane these are Lawson's flips, which end at the Delaunay
    triangulation. Stops when no weight is negative, no legal flip is left or after
    ``max_rounds``. Returns the new faces and the number of flips.
    """
    _check_mesh(verts, faces)
    faces = np.array(faces, dtype=np.int64)
    n_flips = 0
    for _ in range(max_rounds):
        first, second, keys = _edge_table(faces, len(verts))
        cot = _corner_cotangents(verts, faces).ravel()
        weight = 0.5 * cot[first] + 0.5 * cot[second]
        bad = np.flatnonzero(weight < -tol * np.abs(weight).max())
        if bad.size == 0:
            break
        f1, c1 = np.divmod(first[bad], 3)
        f2, c2 = np.divmod(second[bad], 3)
        k, i, j, l = faces[f1, c1], faces[f1, (c1 + 1) % 3], faces[f1, (c1 + 2) % 3], faces[f2, c2]
        new_key = np.minimum(k, l) * len(verts) + np.maximum(k, l)
        pk, pi, pj, pl = verts[k], verts[i], verts[j], verts[l]
        old_normal = np.cross(pi - pk, pj - pk) + np.cross(pj - pl, pi - pl)
        size = np.einsum("fd,fd->f", old_normal, old_normal)
        n_a, n_b = np.cross(pi - pk, pl - pk), np.cross(pj - pl, pk - pl)
        new_weight = 0.5 * _cot(pi, pl, pk) + 0.5 * _cot(pj, pk, pl)
        legal = ((k != l) & ~np.isin(new_key, keys)
                 & (np.einsum("fd,fd->f", n_a, old_normal) > 1e-8 * size)
                 & (np.einsum("fd,fd->f", n_b, old_normal) > 1e-8 * size)
                 & (new_weight > weight[bad]))
        if not legal.any():
            break
        # An independent set: each face keeps its most negative legal edge.
        candidates = np.flatnonzero(legal)
        rank = np.empty(len(candidates), np.int64)
        rank[np.argsort(weight[bad][candidates], kind="stable")] = np.arange(len(candidates))
        best = np.full(len(faces), len(candidates))
        np.minimum.at(best, f1[candidates], rank)
        np.minimum.at(best, f2[candidates], rank)
        chosen = candidates[(best[f1[candidates]] == rank) & (best[f2[candidates]] == rank)]
        chosen = chosen[np.unique(new_key[chosen], return_index=True)[1]]      # one flip per new edge
        faces[f1[chosen]] = np.stack([k[chosen], i[chosen], l[chosen]], 1)
        faces[f2[chosen]] = np.stack([l[chosen], j[chosen], k[chosen]], 1)
        n_flips += len(chosen)
    return faces, n_flips


def mesh_quality(verts: NDArray, faces: NDArray, tol: float = 1e-12) -> dict:
    """Quality numbers of a triangle mesh, as plain Python values (for a manifest).

    Angles in degrees; ``n_negative`` counts the off-diagonal ``L_ij > tol · max |L_ij|`` of
    ``cotan_laplacian`` (negative edge weights), ``min_weight`` is the most negative weight
    ``-L_ij``; edge spreads are relative to the mean edge; ``k_allowed`` is
    ``allowed_wavenumber(mean edge, 10)``; ``closed``: every edge in exactly two faces;
    ``oriented``: every directed edge used once; genus from the Euler characteristic, which
    assumes a closed, orientable, connected mesh.
    """
    _check_mesh(verts, faces)
    corners = verts[faces]
    angles = []
    for k in range(3):
        u, w = corners[:, (k + 1) % 3] - corners[:, k], corners[:, (k + 2) % 3] - corners[:, k]
        angles.append(np.arctan2(np.linalg.norm(np.cross(u, w), axis=1), np.einsum("fd,fd->f", u, w)))
    angles = np.degrees(np.concatenate(angles))
    n = len(verts)
    start, end = faces.ravel().astype(np.int64), faces[:, [1, 2, 0]].ravel().astype(np.int64)
    keys, count = np.unique(np.minimum(start, end) * n + np.maximum(start, end), return_counts=True)
    edges = np.stack(np.divmod(keys, n), axis=1)
    lengths = np.linalg.norm(verts[edges[:, 0]] - verts[edges[:, 1]], axis=1)
    mean_edge = lengths.mean()
    L = cotan_laplacian(verts, faces).tocoo()
    off = L.data[L.row < L.col]
    negative = off > tol * np.abs(off).max()
    graph = sp.coo_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(n, n))
    euler = n - len(edges) + len(faces)
    return {"n_verts": len(verts), "n_faces": len(faces), "n_edges": len(edges),
            "min_angle": float(angles.min()), "max_angle": float(angles.max()),
            "n_negative": int(negative.sum()), "negative_fraction": float(negative.mean()),
            "min_weight": float(-off.max()),
            "mean_edge": float(mean_edge), "edge_p95_ratio": float(np.percentile(lengths, 95) / mean_edge),
            "edge_max_ratio": float(lengths.max() / mean_edge),
            "k_allowed": float(allowed_wavenumber(mean_edge, 10)),
            "closed": bool(np.all(count == 2)), "oriented": len(np.unique(start * n + end)) == len(start),
            "components": int(connected_components(graph, directed=False)[0]),
            "euler": int(euler), "genus": (2 - euler) / 2, "area": float(vertex_areas(verts, faces).sum())}


def check_gates(quality: dict, genus: int, tube: dict | None = None, *, min_angle: float = 30.0,
                max_angle: float = 120.0, max_negative: int = 0, max_edge_p95: float = 1.3,
                min_wavenumber: float = 16.0, min_curvature_radius: float = 0.18, min_tube: float = 0.30,
                max_fold: float = 0.8, min_hole: float = 0.58, min_gap: float = 4.0) -> tuple[bool, list[str]]:
    """Whether a mesh is kept for the dataset, and the reasons when it is not.

    ``quality`` comes from ``mesh_quality`` (after ``delaunay_flip``), ``genus`` is the expected
    one (0 sphere, 1 torus) and ``tube`` the ``tube_quantities`` of a torus. Lengths are in the
    units of the normalised mesh (area 4π). ``min_wavenumber`` bounds ``k0 + 3/(2σ)``: 16.0 covers
    the training ranges (k0 ≤ 6, σ ≥ 0.15); the out-of-distribution k0 = 8 with σ ≥ 0.18 needs
    16.33. ``min_curvature_radius`` bounds ``quality["min_curvature_radius"]``, the smallest radius
    of curvature of a sphere (``smallest_curvature_radius``, recorded by ``build_shape``), when
    ``quality`` has it. The planning study found that the data error and the number of modes follow
    it, not τ: draws with radius ≥ 0.19 kept the error within 1.6 % (k0 = 6) and 2.2 % (k0 = 8), and
    a τ = 0.2 draw with radius 0.11 that passed every other gate reached 3.45 %. ``min_hole`` is a
    length, not a multiple of σ: 2.9σ at σ = 0.2, near the plan's 3σ_max, calibrated in the
    planning study (``tube_quantities``). The gaps must be at least ``min_gap · σ``
    at each σ of ``tube["gap"]``; a ``None`` gap (no two sections 7σ apart) passes. Both dicts may
    come back from a JSON manifest.
    """
    q = quality
    checks = [(q["closed"] and q["oriented"], "not a closed, oriented manifold"),
              (q["components"] == 1, f"{q['components']} components"),
              (q["genus"] == genus, f"genus {q['genus']}, expected {genus}"),
              (q["min_angle"] >= min_angle, f"min angle {q['min_angle']:.1f}° < {min_angle}°"),
              (q["max_angle"] <= max_angle, f"max angle {q['max_angle']:.1f}° > {max_angle}°"),
              (q["n_negative"] <= max_negative, f"{q['n_negative']} negative cotangent weights"),
              (q["edge_p95_ratio"] <= max_edge_p95, f"p95/mean edge {q['edge_p95_ratio']:.3f} > {max_edge_p95}"),
              (q["k_allowed"] >= min_wavenumber, f"allowed wavenumber {q['k_allowed']:.2f} < {min_wavenumber}")]
    if "min_curvature_radius" in q:
        radius = q["min_curvature_radius"]
        checks.append((radius >= min_curvature_radius, f"curvature radius {radius:.3f} < {min_curvature_radius}"))
    if tube is not None:
        checks += [(tube["r_min"] >= min_tube, f"tube radius {tube['r_min']:.3f} < {min_tube}"),
                   (tube["fold"] < max_fold, f"fold {tube['fold']:.3f} >= {max_fold}"),
                   (tube["hole"] >= min_hole, f"hole {tube['hole']:.3f} < {min_hole}")]
        checks += [(gap >= min_gap * float(sigma), f"gap {gap:.3f} < {min_gap}σ at σ = {float(sigma)}")
                   for sigma, gap in tube["gap"].items() if gap is not None]
    reasons = [reason for ok, reason in checks if not ok]
    return not reasons, reasons


SAMPLERS = {"sphere": sample_sphere_deformation, "torus": sample_torus_deformation}
GENUS = {"sphere": 0, "torus": 1}


@dataclass(frozen=True, eq=False)
class Shape:
    """A shape that passed the gates: its mesh, the deformation that rebuilds it and its quality.

    ``quality`` is ``mesh_quality`` plus the number of ``flips``, the normalisation ``scale`` and
    ``offset``, the mesh options (``build_shape``) and, for a sphere, the ``min_curvature_radius``
    or, for a torus, the ``tube_quantities``: plain values, strict JSON. ``rejections`` holds the
    gate failures of the earlier draws.
    """
    family: str
    verts: NDArray
    faces: NDArray
    deformation: SphereDeformation | TorusDeformation
    quality: dict
    attempts: int
    rejections: tuple[tuple[str, ...], ...]


def build_shape(deformation: SphereDeformation | TorusDeformation, *, subdivisions: int = 5,
                target_edge: float = 0.037, sigmas: tuple[float, ...] = (0.15, 0.2),
                relax: bool = True) -> tuple[NDArray, NDArray, dict]:
    """Mesh of a deformation after ``delaunay_flip`` (renormalised to area 4π), and its quality.

    ``subdivisions`` and ``relax`` apply to spheres (``deformed_sphere``) and ``target_edge`` to
    tori. The mesh is ``scale · (x - offset)`` with x the vertices on the surface of the
    deformation, ``quality["scale"]`` and ``quality["offset"]`` (a list of 3 floats), the total
    factor and the shift of the normalisations before and after the flips: a revolution torus
    has radii ``R0 · scale`` and ``r0 · scale``, and ``refine_shape`` maps new surface points the
    same way. ``quality`` also records the mesh options, ``subdivisions`` and ``relax`` for a
    sphere and ``target_edge`` for a torus, so that the deformation and its quality alone rebuild
    and refine the mesh, and the shape's own numbers: ``min_curvature_radius``
    (``smallest_curvature_radius``) for a sphere, the ``tube_quantities`` for a torus, both in the
    units of the mesh. The result depends only on the arguments, bit for bit.
    """
    if isinstance(deformation, SphereDeformation):
        x, faces = _sphere_surface(deformation, subdivisions, relax)
        options = {"subdivisions": int(subdivisions), "relax": bool(relax)}
    else:
        faces, u, v = _base_torus(deformation, target_edge)
        x = torus_surface(deformation, u, v)
        options = {"target_edge": float(target_edge)}
    scale, offset = _normalisation(x, faces)
    verts = scale * (x - offset)
    faces, flips = delaunay_flip(verts, faces)
    if flips:                                      # flips change the area slightly
        rescale, shift = _normalisation(verts, faces)
        verts = rescale * (verts - shift)
        scale, offset = scale * rescale, offset + shift / scale
    quality = mesh_quality(verts, faces) | {"flips": flips, "scale": scale, "offset": [float(a) for a in offset]}
    quality |= options
    if isinstance(deformation, TorusDeformation):
        quality |= tube_quantities(deformation, scale, sigmas)
    else:
        quality["min_curvature_radius"] = smallest_curvature_radius(deformation, scale)
    return verts, faces, quality


def refine_shape(deformation: SphereDeformation | TorusDeformation, verts: NDArray, faces: NDArray, quality: dict,
                 *, target_edge: float | None = None, flip: bool = True) -> tuple[NDArray, NDArray]:
    """The mesh of ``build_shape`` midpoint-subdivided once, with the new vertices on the same surface.

    Each face splits in four (``_midpoint_subdivision``) and each edge gets a vertex at the midpoint
    of its ends' parameters: the normalised mean of their directions on a sphere (the rule of
    ``mesh.icosphere``), the mean of their angles (u, v) on a torus, the shorter way round
    (``target_edge``, by default ``quality["target_edge"]``, rebuilds the angles of the torus mesh).
    New points x are mapped like the stored ones, ``scale · (x - offset)`` with ``quality["scale"]``
    and ``quality["offset"]``, so ``verts`` are exactly the first V rows of the fine mesh and a
    solution on both meshes compares without interpolation. Then ``delaunay_flip`` (``flip``),
    which moves no vertex: splitting a curved mesh makes negative weights even where the coarse mesh
    has none (about 220 of 127k edges on the round torus), and the fine mesh must be as free of them.
    """
    scale, offset = quality["scale"], np.asarray(quality["offset"], float)
    edges, fine_faces = _midpoint_subdivision(np.asarray(faces), len(verts))
    if isinstance(deformation, SphereDeformation):
        new = _sphere_midpoints(deformation, verts / scale + offset, edges)
    else:
        target_edge = quality["target_edge"] if target_edge is None else target_edge
        _, u, v = _base_torus(deformation, target_edge)
        if len(u) != len(verts):
            raise ValueError(f"{len(verts)} vertices, but torus({target_edge}, R0, r0) has {len(u)}")

        def middle(angle):
            start, step = angle[edges[:, 0]], np.mod(angle[edges[:, 1]] - angle[edges[:, 0]] + np.pi, 2 * np.pi) - np.pi
            return np.mod(start + step / 2, 2 * np.pi)

        new = torus_surface(deformation, middle(u), middle(v))
    fine = np.vstack([verts, scale * (new - offset)])
    if flip:
        fine_faces = delaunay_flip(fine, fine_faces)[0]
    return fine, fine_faces


def make_shape(family: str, rng: np.random.Generator, *, max_attempts: int = 100, subdivisions: int = 5,
               target_edge: float = 0.037, sigmas: tuple[float, ...] = (0.15, 0.2), relax: bool = True,
               gates: dict | None = None, **params) -> Shape:
    """Draw ``family`` ("sphere" or "torus") deformations until one passes ``check_gates``.

    ``params`` go to the family's sampler (``sample_sphere_deformation``,
    ``sample_torus_deformation``) and ``gates`` to ``check_gates``. The same ``rng`` state
    gives the same shape bit for bit, and ``build_shape(shape.deformation)`` with the mesh
    options recorded in ``shape.quality`` rebuilds it. Raises ``RuntimeError`` after ``max_attempts`` rejections.
    """
    if family not in SAMPLERS:
        raise ValueError(f"family must be one of {sorted(SAMPLERS)}")
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")
    rejections = []
    for attempt in range(1, max_attempts + 1):
        deformation = SAMPLERS[family](rng, **params)
        verts, faces, quality = build_shape(deformation, subdivisions=subdivisions, target_edge=target_edge,
                                            sigmas=sigmas, relax=relax)
        tube = quality if family == "torus" else None
        passed, reasons = check_gates(quality, GENUS[family], tube, **(gates or {}))
        if passed:
            return Shape(family, verts, faces, deformation, quality, attempt, tuple(rejections))
        rejections.append(tuple(reasons))
    raise RuntimeError(f"no {family} passed the gates in {max_attempts} draws; last: {rejections[-1]}")


In [ ]:
%%writefile src/schrodinger_glno/shapes_archive.py
"""A stage-3 dataset as deterministic tars, for the trip to Google Drive and onto the disk of a Colab machine.

``pack`` turns a complete dataset directory (``shapes_dataset``) into a few uncompressed tars and an index,
``shapes_tars.json``; ``unpack`` rebuilds the directory from them and hands it over only when every SHA-256 of
its manifest holds. The link is slow and Drive, a FUSE mount, can serve a file that is truncated, stale or
still syncing, so every step is checked against a hash and nothing appears under the dataset's name before
the last check has passed.

**What travels.** The manifest decides (``dataset_files``): ``manifest.json``, ``config.json`` and, for every
shape, the files the manifest hashes plus ``shape.json``. Anything else in the directory (a ``.partial`` left
by a run, a log) is not part of the dataset and is not packed.

**Packing** (``pack``) verifies the dataset in full first, and then does not rely on that pass for what it
archives minutes later. Every data file is hashed again while it goes into its tar and must still have the
SHA-256 of the manifest. The JSON files, which the manifest does not hash, are packed from memory, as bytes
that were the same before and after the verification. A file that changes, or is read wrong, in between
stops the pack, instead of being found in Colab after the upload.

**Tars.** One per split for val and test, and the training split in ``train_parts`` tars of whole, consecutive
shapes, so an upload or a copy that is cut costs one tar and each tar is a complete archive of its own (not a
slice of a larger one)::

    shapes_<identity[:16]>_train_<k>of<N>.tar     shapes_<identity[:16]>_val.tar     shapes_<identity[:16]>_test.tar

The identity is in the name because Drive keeps two files of one name side by side: the tars of a dataset
that was generated again can then never be taken for the old ones. Members are paths from the dataset's
root (``000_sphere/mesh.npz``), with no directory entries, so ``tar -xf`` of any tar into a folder works too.

**manifest.json and config.json travel in every tar,** as its first two members. A tar extracted alone then
says which dataset it belongs to and which hash each of its files must have, and there is no extra small
file whose absence would make the others useless. The copies are the same bytes, so extracting several tars
overwrites them with themselves; the cost is 0.66 MB per tar of data/shapes (0.05 % in all).

**Determinism.** Packing twice, on any machine and at any time, gives the same bytes, so the SHA-256 of a tar
identifies an upload. Of a file only its name, its size and its bytes reach the tar: the members go in a
fixed order (the two metadata files, then the shapes by index, the files of a shape by name) with mode 0644,
uid = gid = 0, empty user and group names and time 0. The format is ustar (POSIX.1-1988), which every tool
reads: its header has no field beyond those (pax would add records, GNU its own), and a name or a size that
does not fit in it is an error, not an extension. There is no compression (zstd would save about 9 %), hence no
compressor version either, and the index holds no date and no path.

**Index** (``shapes_tars.json``, in the directory of the tars)::

    format_version   of this module (ARCHIVE_VERSION)
    identity         of the dataset (its manifest's), 64 hex characters
    files, bytes     of the extracted dataset: what the free-space check asks for
    tars             a list, in the order above, of
        name, split, shapes (ids)     the file and what it holds
        size, sha256                  of the tar
        members                       {path: size} of every member

**Unpacking** (``unpack``), in this order:

1. The index is read and checked; an expected identity, or its first 16 characters, is enforced.
2. If the target already holds that dataset, verified (every SHA-256), nothing is done: the tars are not even
   looked for. A target that holds anything else is refused, never deleted: a wrong path would otherwise cost
   a dataset of hours. An empty directory is used; a symbolic link that does not lead to the dataset is
   refused, since the temporary directory of step 5 has to be a sibling of the directory itself.
3. Every tar must exist with exactly its listed size, before anything is extracted; all the missing or
   truncated ones are named at once. ``check_sha256`` also hashes them first, at the price of reading them twice.
4. A ``<target>.unpacking`` left by an interrupted run is removed, and the free space is checked.
5. The tars are extracted into ``<target>.unpacking``, a sibling of the target and so on its file system
   (the caller gives a target on the local disk), with tarfile's ``data`` filter. They are read straight from
   where they are, in one sequential pass that hashes the bytes as tarfile consumes them: the SHA-256 and the
   size of each tar are checked against the index as soon as it is extracted, and a bad one stops the run by
   its name. Copying them to the local disk first would take twice the space (17.7 GB instead of 8.8) and
   one more pass, and would catch nothing this does not. The price is that a failed run keeps nothing, and
   the next one reads every tar again; so a tar that cannot be read, or whose bytes are not the index's, is
   read a second time before the run gives up, and a hiccup of Drive costs one tar instead of all of them.
6. The extracted directory is verified in full (identity, every SHA-256, ``shape.json``, ``config.json``, and
   that the tars held exactly the manifest's files); only then is it renamed to the target. Any failure
   removes ``<target>.unpacking``, so a failed run leaves no target at all.

**What a tar may hold.** Only regular files whose path the index lists with that size, each once, and the
index may only list paths of the dataset's layout (``manifest.json``, ``config.json``, ``<shape id>/<file>`` with
plain names). An absolute path, a ``..``, a link, a device, a directory entry or an unlisted file is refused
with the tar's name, before it is extracted and although the ``data`` filter would also stop most of them:
nothing is skipped or repaired, since a tar that is not what ``pack`` wrote should not be trusted in part.

**One file to upload** (``zip_tars``, ``read_zip_index``, ``unzip_tars``). A folder of several files is awkward to drag into Drive,
so the tars and their index can also travel inside one zip, ``glno_<dataset>.zip`` (``zip_name``). The zip is
only a container: it holds exactly ``shapes_tars.json`` and the tars it lists, each hashed on its way in.
Deflate takes about 9 % off these tars at any level, so the fastest level is used. In Colab ``unzip_tars``
copies the tars out of the zip onto the local disk, where each must again have the size and the SHA-256 of
the index, and ``unpack`` then works on that folder as on one uploaded as it is. The tars are on the local
disk twice for a moment (in the zip's folder and extracted): 17.7 GB for data/shapes, which the free-space
check asks for before anything is written. As with ``unpack``, the folder appears under its name only when
it is complete, a second call does nothing, and a folder that holds something else is refused, not deleted.

Only the standard library and ``shapes_dataset``: the module is copied into the Colab notebook's package.
"""
from contextlib import contextmanager
from hashlib import sha256
import io
import json
import os
from pathlib import Path
import re
import shutil
import sys
import tarfile
import time
from typing import Callable
import zipfile
import zlib
from .shapes_dataset import (CONFIG_FILE, FORMAT_VERSION, MANIFEST_FILE, SHAPE_FILE, SPLITS, _write_json,
                             shapes_dataset_identity)

ARCHIVE_VERSION = 1
INDEX_FILE = "shapes_tars.json"
METADATA_FILES = (MANIFEST_FILE, CONFIG_FILE)
TEMPORARY_SUFFIX = ".unpacking"
ZIP_TEMPORARY_SUFFIX = ".unzipping"
ZIP_LEVEL = 1                   # deflate leaves 91 % of these tars at level 1 and at level 6
MEMBER_MODE, MEMBER_MTIME = 0o644, 0
BLOCK = 2**20
# Room for what a file system adds to the bytes of the files: each one rounded up to a block, and its own records.
SPACE_MARGIN = 64 * 2**20
_NAME = re.compile(r"[A-Za-z0-9_][A-Za-z0-9_.-]*")
_HASH = re.compile(r"[0-9a-f]{64}")


class _Hashed:
    """A binary file that hashes and counts the bytes passing through it, so a tar is written, or read, only once."""

    def __init__(self, raw) -> None:
        self.raw, self.sha256, self.size = raw, sha256(), 0

    def _count(self, block: bytes) -> bytes:
        self.sha256.update(block)
        self.size += len(block)
        return block

    def write(self, block: bytes) -> int:
        return self.raw.write(self._count(block))

    def read(self, size: int) -> bytes:
        return self._count(self.raw.read(size))

    def tell(self) -> int:
        return self.size

    def drain(self) -> None:
        """Read to the end: tarfile stops at the end-of-archive blocks, before the padding that follows them."""
        while self.read(BLOCK):
            pass


def _in_layout(name) -> bool:
    """Whether ``name`` is a path of a dataset: a metadata file, or ``<shape id>/<file>`` with plain names.

    A plain name starts with a letter, a digit or ``_``, which leaves out ``..``, absolute paths (an empty
    first part), backslashes and drive letters.
    """
    parts = name.split("/") if isinstance(name, str) else []
    return name in METADATA_FILES if len(parts) == 1 else len(parts) == 2 and all(map(_NAME.fullmatch, parts))


def dataset_files(manifest: dict) -> dict[str, list[str]]:
    """``{shape id: its files}`` as sorted paths from the dataset's root: those the manifest hashes, and ``shape.json``.

    A manifest that names a file outside the layout is refused: ``pack`` would write a tar ``unpack`` rejects.
    """
    files = {entry["id"]: [] for entry in manifest["shapes"]}
    for relative in [*manifest["sha256"], *(f"{shape_id}/{SHAPE_FILE}" for shape_id in files)]:
        shape_id = relative.split("/", 1)[0]
        if not _in_layout(relative) or shape_id not in files or relative in files[shape_id]:
            raise ValueError(f"{MANIFEST_FILE} lists a file outside the dataset layout: {relative!r}")
        files[shape_id].append(relative)
    return {shape_id: sorted(names) for shape_id, names in files.items()}


@contextmanager
def _dataset_errors(directory: Path):
    """Turn what a missing file, or a manifest of another form, raises into a ValueError that names the file."""
    try:
        yield
    except FileNotFoundError as error:
        raise ValueError(f"Dataset file is missing: {error.filename}") from error
    except (KeyError, TypeError, AttributeError) as error:
        raise ValueError(f"{directory / MANIFEST_FILE} is not the manifest of a shapes dataset: {error!r}") from error


def _json(path: Path, data: bytes | None = None):
    """What the JSON file ``path`` says (``data``: its bytes, if already read); a damaged one is named.

    ``json`` alone gives a line and a column, which among the 200 ``shape.json`` of a dataset say nothing.
    """
    try:
        return json.loads(path.read_bytes() if data is None else data)
    except ValueError as error:                         # also the UnicodeDecodeError of bytes that are not text
        raise ValueError(f"{path} is not JSON: {error}") from error


def verified_identity(directory: Path) -> str:
    """The identity of the dataset in ``directory``, after checking every SHA-256, ``shape.json`` and ``config.json``.

    ``shapes_dataset_identity`` does not read ``config.json`` (``dataset_summary`` does), so it is compared
    here with the manifest's copy of the config. The JSON files are parsed first: a damaged one is then
    named, and found before the pass over the data instead of after it.
    """
    directory = Path(directory)
    with _dataset_errors(directory):
        manifest, stored = (_json(directory / name) for name in METADATA_FILES)
        for entry in manifest["shapes"]:
            _json(directory / entry["id"] / SHAPE_FILE)
        identity = shapes_dataset_identity(directory, verify=True)
        if stored != {"format_version": FORMAT_VERSION, "config": manifest["config"]}:
            raise ValueError(f"Dataset file differs from manifest: {CONFIG_FILE}")
    return identity


def _json_files(directory: Path) -> tuple[dict, dict[str, bytes]]:
    """The manifest and ``{path: bytes}`` of a dataset's JSON files: the manifest, the config, every ``shape.json``.

    The manifest hashes none of them, so ``pack`` cannot check them while it archives them, as it does the
    data files: it reads them before the verification and again after it, and packs the bytes it holds.
    """
    held = {MANIFEST_FILE: (directory / MANIFEST_FILE).read_bytes()}
    manifest = _json(directory / MANIFEST_FILE, held[MANIFEST_FILE])
    for name in (CONFIG_FILE, *(f"{entry['id']}/{SHAPE_FILE}" for entry in manifest["shapes"])):
        held[name] = (directory / name).read_bytes()
    return manifest, held


def _tar_groups(manifest: dict, train_parts: int) -> list[tuple[str, str, list[str]]]:
    """``(tar name, split, shape ids)`` of each tar: train in ``train_parts`` runs of consecutive shapes, val, test."""
    shapes = sorted(manifest["shapes"], key=lambda entry: entry["index"])
    ids = {split: [entry["id"] for entry in shapes if entry["split"] == split] for split in SPLITS}
    train = ids["train"]
    if not 1 <= train_parts <= max(1, len(train)):
        raise ValueError(f"train_parts must be between 1 and the number of training shapes ({len(train)})")
    prefix = f"shapes_{manifest['identity'][:16]}"
    cuts = [len(train) * k // train_parts for k in range(train_parts + 1)]
    groups = [(f"{prefix}_train_{k + 1}of{train_parts}.tar", "train", train[cuts[k]:cuts[k + 1]])
              for k in range(train_parts)]
    groups += [(f"{prefix}_{split}.tar", split, ids[split]) for split in SPLITS if split != "train"]
    return [group for group in groups if group[2]]


def _member(name: str, size: int) -> tarfile.TarInfo:
    """The header of a member: a regular file with nothing of the machine or the clock (see the module docstring)."""
    info = tarfile.TarInfo(name)
    info.size, info.mode, info.mtime, info.type = size, MEMBER_MODE, MEMBER_MTIME, tarfile.REGTYPE
    info.uid = info.gid = 0
    info.uname = info.gname = ""
    return info


def _add(tar: tarfile.TarFile, directory: Path, name: str, hashes: dict, held: dict) -> int:
    """Add the file ``name`` to the tar and return its size: the bytes in ``held``, or else those on the disk.

    The verification read the disk a while ago, and what is archived is read now: a data file is hashed on
    its way into the tar, and one that no longer has the SHA-256 of the manifest stops the pack.
    """
    if name in held:
        tar.addfile(_member(name, len(held[name])), io.BytesIO(held[name]))
        return len(held[name])
    with (directory / name).open("rb") as raw:
        source, size = _Hashed(raw), os.fstat(raw.fileno()).st_size
        tar.addfile(_member(name, size), source)
    if source.sha256.hexdigest() != hashes[name]:
        raise ValueError(f"Dataset file changed while it was packed: {name}")
    return size


def _write_tar(directory: Path, path: Path, names: list[str], hashes: dict, held: dict) -> dict:
    """Write the files ``names`` of ``directory`` as a deterministic tar; return its size, sha256 and members.

    ``hashes`` is the manifest's ``{path: sha256}`` and ``held`` the files to take from memory (``_add``).
    The temporary file is removed if the writing fails: after a full disk it would be what keeps it full.
    """
    temporary = path.with_name(path.name + ".tmp")
    try:
        with temporary.open("wb") as raw:
            stream = _Hashed(raw)
            with tarfile.open(fileobj=stream, mode="w", format=tarfile.USTAR_FORMAT, copybufsize=BLOCK) as tar:
                members = {name: _add(tar, directory, name, hashes, held) for name in names}
        os.replace(temporary, path)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise
    return {"size": stream.size, "sha256": stream.sha256.hexdigest(), "members": members}


def pack(directory: Path, out: Path, *, train_parts: int = 1,
         progress: Callable[[str], None] | None = None) -> dict:
    """Verify the dataset in ``directory`` (every SHA-256) and write its tars and ``shapes_tars.json`` in ``out``.

    Returns the index. ``train_parts`` is the number of tars of the training split (4 for data/shapes: about
    1.75 GB each). Tars of the same names are replaced, each written under a temporary name and renamed; the
    index is removed first and written last, so an interrupted run leaves no index of tars it did not finish.
    A file that is not the verified one when it is archived is a ValueError (see the module docstring).
    ``progress`` gets one line for the verification and one per tar.
    """
    directory, out = Path(directory), Path(out)
    say = progress or (lambda message: None)
    if out.resolve().is_relative_to(directory.resolve()):
        raise ValueError(f"{out} is inside the dataset {directory}: the tars would be taken for a part of it")
    with _dataset_errors(directory):
        manifest, held = _json_files(directory)
        # Before the verification, which reads the whole dataset: a wrong train_parts should not cost that pass.
        files, groups = dataset_files(manifest), _tar_groups(manifest, train_parts)
    identity = verified_identity(directory)
    say(f"verified {directory}: identity {identity[:16]}")
    with _dataset_errors(directory):
        changed = [name for name, data in _json_files(directory)[1].items() if held.get(name) != data]
    if changed:
        raise ValueError(f"Dataset file changed while it was verified: {changed[0]}")
    out.mkdir(parents=True, exist_ok=True)
    (out / INDEX_FILE).unlink(missing_ok=True)
    tars = []
    for name, split, shape_ids in groups:
        names = [*METADATA_FILES, *(relative for shape_id in shape_ids for relative in files[shape_id])]
        tars.append({"name": name, "split": split, "shapes": shape_ids}
                    | _write_tar(directory, out / name, names, manifest["sha256"], held))
        say(f"{name}: {len(shape_ids)} {split} shapes, {tars[-1]['size'] / 1e9:.3f} GB")
    extracted = {name: size for tar in tars for name, size in tar["members"].items()}
    index = {"format_version": ARCHIVE_VERSION, "identity": identity, "files": len(extracted),
             "bytes": sum(extracted.values()), "tars": tars}
    _write_json(out / INDEX_FILE, index)
    return index


def _index_problem(index) -> str | None:
    """What is wrong with a parsed index, or None. Its paths are checked here because they end up on the disk."""
    def is_size(value) -> bool:
        return isinstance(value, int) and not isinstance(value, bool) and value >= 0

    def is_hash(value) -> bool:
        return isinstance(value, str) and _HASH.fullmatch(value) is not None

    try:
        if index["format_version"] != ARCHIVE_VERSION:
            return f"format_version is {index['format_version']!r}, not {ARCHIVE_VERSION}"
        if not is_hash(index["identity"]):
            return "identity is not a SHA-256"
        names = [entry["name"] for entry in index["tars"]]
        if not names or len(set(names)) != len(names):
            return "it lists no tar, or one twice"
        files = {}
        for entry in index["tars"]:
            name, members = entry["name"], entry["members"]
            if not (isinstance(name, str) and _NAME.fullmatch(name) and name.endswith(".tar")):
                return f"{name!r} is not the plain name of a tar"
            if not (is_size(entry["size"]) and is_hash(entry["sha256"])):
                return f"{name} has no valid size and sha256"
            if not set(METADATA_FILES) <= set(members):
                return f"{name} lacks {' or '.join(METADATA_FILES)}"
            for member, size in members.items():
                if not (_in_layout(member) and is_size(size)):
                    return f"{name} lists {member!r}, which is outside the dataset layout"
                if member in files and (member not in METADATA_FILES or files[member] != size):
                    return f"{member} is listed in two tars"
                files[member] = size
        if (index["files"], index["bytes"]) != (len(files), sum(files.values())):
            return "files and bytes are not those of its members"
    except (KeyError, TypeError, AttributeError) as error:
        return f"{type(error).__name__}: {error}"
    return None


def _index(path: Path, data: bytes | None = None) -> dict:
    """The index in ``path``, parsed and checked (``data``: its bytes, if already read)."""
    index = _json(path, data)
    problem = _index_problem(index)
    if problem:
        raise ValueError(f"{path} is not a valid index of shapes tars: {problem}")
    return index


def read_index(tars_dir: Path) -> dict:
    """The index ``shapes_tars.json`` of the tars in ``tars_dir``; a missing or malformed one is a ValueError."""
    path = Path(tars_dir) / INDEX_FILE
    try:
        return _index(path)
    except FileNotFoundError:
        raise ValueError(f"Missing index of the tars: {path}") from None


def _check_expected(identity: str, expected: str | None, source: Path) -> None:
    """Refuse an index of another dataset than the ``expected`` identity (all of it, or 16 characters or more)."""
    if expected is None:
        return
    if not (isinstance(expected, str) and re.fullmatch(r"[0-9a-f]{16,64}", expected)):
        raise ValueError(f"The expected identity must be its 64 hex characters or the first 16, not {expected!r}")
    if not identity.startswith(expected):
        raise ValueError(f"{source} is of the dataset {identity[:16]}, not of the expected {expected[:16]}")


def _target_problem(target: Path, identity: str) -> str | None:
    """Why ``target`` is not the verified dataset ``identity``, or None when it is."""
    try:
        found = verified_identity(target)
    except (OSError, ValueError) as error:
        return str(error)
    return None if found == identity else f"it holds the dataset {found[:16]}"


def _tar_digest(path: Path) -> str:
    with path.open("rb") as raw:
        stream = _Hashed(raw)
        stream.drain()
    return stream.sha256.hexdigest()


def _tar_problem(path: Path, entry: dict, check_sha256: bool) -> str | None:
    """What is wrong with a tar before it is extracted, to follow its name in a message; None if nothing is."""
    try:
        if not path.is_file():
            return "is missing"
        size = path.stat().st_size
        if size != entry["size"]:
            return f"has {size} bytes, not {entry['size']} (truncated, or still uploading?)"
        if check_sha256 and _tar_digest(path) != entry["sha256"]:
            return "has another SHA-256 than the index"
    except OSError as error:
        return f"could not be read: {error}"
    return None


def _check_tars(tars_dir: Path, index: dict, check_sha256: bool) -> None:
    """Every tar of the index is there with its exact size (and SHA-256, when asked), or a ValueError names them all.

    All at once because, over a slow link, each upload that has to be repeated costs a while.
    """
    problems = [f"{entry['name']} {problem}" for entry in index["tars"]
                if (problem := _tar_problem(tars_dir / entry["name"], entry, check_sha256))]
    if problems:
        raise ValueError(f"The tars in {tars_dir} are not those of {INDEX_FILE}: " + "; ".join(problems))


def _check_space(folder: Path, needed: int) -> None:
    free = shutil.disk_usage(folder).free
    if free < needed + SPACE_MARGIN:
        raise ValueError(f"Not enough disk space in {folder}: the dataset needs {(needed + SPACE_MARGIN) / 1e9:.2f} GB "
                         f"and {free / 1e9:.2f} GB are free")


def _check_member(path: Path, member: tarfile.TarInfo, pending: dict) -> None:
    """Refuse a member that is not a regular file the index still expects, with that size; take it off ``pending``."""
    if member.type != tarfile.REGTYPE:
        raise ValueError(f"{path} holds {member.name!r}, which is not a regular file (a link, a device, a directory?); "
                         "the tar is refused")
    if not _in_layout(member.name) or pending.pop(member.name, None) != member.size:
        raise ValueError(f"{path} holds {member.name!r} ({member.size} bytes), which is not a file that {INDEX_FILE} "
                         "lists for it, once and with that size; the tar is refused")


class _ReadError(ValueError):
    """A tar that could not be read, or whose bytes are not those of the index: what a second read may cure."""


def _extract_tar(path: Path, entry: dict, folder: Path) -> None:
    """Extract one tar into ``folder`` in a single sequential read, and check it against its index ``entry``.

    Each member is checked before it is extracted (``_check_member``); the size and the SHA-256 of the tar
    are known when its last byte has gone by. Extracting it again overwrites its own files.
    """
    pending = dict(entry["members"])
    try:
        with path.open("rb") as raw:
            stream = _Hashed(raw)
            # Mode "r|" reads forward only, and a plain tar only. tarfile's defaults read the file 10 kB at a
            # time and copy a member 16 kB at a time; one larger block for both keeps the reads from Drive few.
            with tarfile.open(fileobj=stream, mode="r|", bufsize=BLOCK, copybufsize=BLOCK) as tar:
                for member in tar:
                    _check_member(path, member, pending)
                    tar.extract(member, folder, filter="data")
            stream.drain()
    except (tarfile.TarError, OSError) as error:
        raise _ReadError(f"{path} could not be extracted: {error}") from error
    # The bytes before the files: tarfile takes a damaged header, or a tar cut between two members, for the end
    # of the archive, and that is a read that went wrong, not a tar that lacks files.
    if (stream.size, stream.sha256.hexdigest()) != (entry["size"], entry["sha256"]):
        raise _ReadError(f"{path} differs from {INDEX_FILE}: {stream.size} bytes with SHA-256 "
                         f"{stream.sha256.hexdigest()[:16]}, not {entry['size']} with {entry['sha256'][:16]} "
                         "(a stale or damaged copy?)")
    if pending:
        raise ValueError(f"{path} lacks {len(pending)} of the files {INDEX_FILE} lists, e.g. {next(iter(pending))}")


def _extract_with_retry(path: Path, entry: dict, folder: Path, say: Callable[[str], None]) -> None:
    """Extract one tar, and once more if it could not be read; a tar with a member that is refused is not read again."""
    try:
        _extract_tar(path, entry, folder)
    except _ReadError as error:
        say(f"{error}; reading {path.name} again")
        _extract_tar(path, entry, folder)


def _verify_extracted(folder: Path, index: dict) -> None:
    """The extracted dataset is the index's: identity, every SHA-256, and exactly the files of its manifest."""
    identity = verified_identity(folder)
    if identity != index["identity"]:
        raise ValueError(f"The tars hold the dataset {identity[:16]}, not {index['identity'][:16]} of {INDEX_FILE}")
    manifest = _json(folder / MANIFEST_FILE)
    listed = {*METADATA_FILES, *(name for names in dataset_files(manifest).values() for name in names)}
    odd = sorted(listed ^ {name for entry in index["tars"] for name in entry["members"]})
    if odd:
        raise ValueError(f"The tars and {MANIFEST_FILE} differ in {len(odd)} files, e.g. {odd[0]}")


def _clear(path: Path) -> None:
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path)
    else:
        path.unlink(missing_ok=True)


def unpack(tars_dir: Path, target_dir: Path, *, identity: str | None = None, check_sha256: bool = False,
           progress: Callable[[str], None] | None = None) -> dict:
    """Rebuild in ``target_dir`` the dataset packed in ``tars_dir``, verified; do nothing if it is already there.

    ``target_dir`` goes on a local disk (``/content/data/shapes``); ``tars_dir`` may be on Drive. ``identity``
    is the expected one, or its first 16 characters or more. ``check_sha256`` hashes every tar before
    extracting any (they are hashed while they are extracted anyway). Returns ``identity``, ``directory``,
    ``files``, ``bytes``, ``seconds`` and ``extracted``: False when the target already held the verified
    dataset and nothing was done. Every refusal is a ValueError and leaves ``target_dir`` as it was; the
    steps and their reasons are in the module docstring.
    """
    if not hasattr(tarfile, "data_filter"):
        raise RuntimeError("unpack needs the extraction filters of tarfile (Python 3.12 or later, 3.11.4+ or "
                           f"3.10.12+); this is Python {sys.version.split()[0]}")
    tars_dir, target = Path(tars_dir), Path(target_dir)
    say = progress or (lambda message: None)
    start = time.perf_counter()
    index = read_index(tars_dir)
    _check_expected(index["identity"], identity, tars_dir / INDEX_FILE)
    result = {key: index[key] for key in ("identity", "files", "bytes")} | {"directory": str(target)}
    link = target.is_symlink()                          # also one that leads nowhere, which exists() does not see
    if link or target.exists():
        problem = _target_problem(target, index["identity"])
        if problem is None:
            say(f"{target} already holds the dataset {index['identity'][:16]}, verified: nothing to do")
            return result | {"extracted": False, "seconds": time.perf_counter() - start}
        # A link cannot be replaced by the rename, and the directory it leads to may be on another file system.
        if link or not target.is_dir() or any(target.iterdir()):
            raise ValueError(f"{target} {'is a symbolic link' if link else 'exists'} and is not the dataset "
                             f"{index['identity'][:16]} ({problem}); it is left as it is: delete it, or give "
                             "another target")
    _check_tars(tars_dir, index, check_sha256)
    temporary = target.with_name(target.name + TEMPORARY_SUFFIX)
    target.parent.mkdir(parents=True, exist_ok=True)
    _clear(temporary)
    _check_space(target.parent, index["bytes"])
    try:
        temporary.mkdir()
        for entry in index["tars"]:
            lap = time.perf_counter()
            _extract_with_retry(tars_dir / entry["name"], entry, temporary, say)
            say(f"{entry['name']}: {entry['size'] / 1e9:.3f} GB extracted in {time.perf_counter() - lap:.1f} s")
        _verify_extracted(temporary, index)
        os.replace(temporary, target)
    except BaseException:
        shutil.rmtree(temporary, ignore_errors=True)
        raise
    say(f"{target}: dataset {index['identity'][:16]} verified, {index['files']} files, {index['bytes'] / 1e9:.3f} GB")
    return result | {"extracted": True, "seconds": time.perf_counter() - start}


def zip_name(dataset: str) -> str:
    """The name of the single file that carries the tars of the dataset ``dataset`` (``shapes``, ``shapes_small``)."""
    return f"glno_{dataset}.zip"


def zip_tars(tars_dir: Path, zip_path: Path, *, progress: Callable[[str], None] | None = None) -> dict:
    """Write the tars of ``tars_dir`` and their index into the single file ``zip_path``, to upload instead of the folder.

    Every tar must be there with its listed size before anything is written, and is hashed while it goes in:
    one that does not have the SHA-256 of the index stops the run. The zip is written under a temporary
    name and renamed, so an interrupted run leaves no zip, and an older one stays as it was until then.
    Returns ``identity``, ``path``, ``size`` (of the zip) and ``bytes`` (of the tars). ``progress`` gets one line
    per tar.
    """
    tars_dir, zip_path = Path(tars_dir), Path(zip_path)
    say = progress or (lambda message: None)
    if zip_path.resolve().parent == tars_dir.resolve():
        raise ValueError(f"{zip_path} is inside the folder of the tars: whoever uploads the folder would upload "
                         "them twice")
    try:
        data = (tars_dir / INDEX_FILE).read_bytes()         # the bytes that are checked are the ones that travel
    except FileNotFoundError:
        raise ValueError(f"Missing index of the tars: {tars_dir / INDEX_FILE}") from None
    index = _index(tars_dir / INDEX_FILE, data)
    _check_tars(tars_dir, index, check_sha256=False)
    temporary = zip_path.with_name(zip_path.name + ".partial")
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_LEVEL) as archive:
            with archive.open(INDEX_FILE, "w") as member:
                member.write(data)
            for entry in index["tars"]:
                with (tars_dir / entry["name"]).open("rb") as raw, \
                        archive.open(entry["name"], "w", force_zip64=True) as member:
                    stream = _Hashed(raw)
                    while block := stream.read(BLOCK):
                        member.write(block)
                if (stream.size, stream.sha256.hexdigest()) != (entry["size"], entry["sha256"]):
                    raise ValueError(f"{tars_dir / entry['name']} is not the tar of {INDEX_FILE}: {stream.size} bytes "
                                     f"with SHA-256 {stream.sha256.hexdigest()[:16]}, not {entry['size']} with "
                                     f"{entry['sha256'][:16]}; pack the dataset again")
                say(f"{entry['name']}: {entry['size'] / 1e9:.3f} GB into {zip_path.name}")
        os.replace(temporary, zip_path)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise
    return {"identity": index["identity"], "path": str(zip_path), "size": zip_path.stat().st_size,
            "bytes": sum(entry["size"] for entry in index["tars"])}


def _holds_tars(folder: Path, data: bytes, index: dict) -> bool:
    """Whether ``folder`` holds this very index and each of its tars with its size."""
    try:
        return ((folder / INDEX_FILE).read_bytes() == data
                and all((folder / entry["name"]).stat().st_size == entry["size"] for entry in index["tars"]))
    except OSError:
        return False


def _unzip_tar(archive: zipfile.ZipFile, zip_path: Path, entry: dict, path: Path) -> None:
    """Copy one tar out of the zip to ``path`` in a single read, and check it against its index ``entry``."""
    try:
        with archive.open(entry["name"]) as member, path.open("wb") as out:
            stream = _Hashed(member)
            while block := stream.read(BLOCK):
                out.write(block)
    except (zipfile.BadZipFile, zlib.error, EOFError, OSError) as error:
        raise _ReadError(f"{entry['name']} could not be read from {zip_path}: {error}") from error
    if (stream.size, stream.sha256.hexdigest()) != (entry["size"], entry["sha256"]):
        raise _ReadError(f"{entry['name']} in {zip_path} differs from {INDEX_FILE}: {stream.size} bytes with SHA-256 "
                         f"{stream.sha256.hexdigest()[:16]}, not {entry['size']} with {entry['sha256'][:16]} "
                         "(a damaged upload?)")


@contextmanager
def _open_zip(zip_path: Path):
    """The zip, open for reading; a missing one, or one that is not a whole zip, is a ValueError."""
    try:
        archive = zipfile.ZipFile(zip_path)
    except FileNotFoundError:
        raise ValueError(f"Missing zip of the tars: {zip_path}") from None
    except (zipfile.BadZipFile, OSError) as error:
        raise ValueError(f"{zip_path} is not a complete zip (truncated, or still uploading?): {error}") from error
    with archive:
        yield archive


def _zip_index(archive: zipfile.ZipFile, zip_path: Path) -> tuple[bytes, dict]:
    """The index inside an open zip, as bytes and parsed; the zip must hold exactly it and its tars, with their sizes."""
    members = [(info.filename, info.file_size) for info in archive.infolist()]
    if INDEX_FILE not in dict(members):
        raise ValueError(f"{zip_path} holds no {INDEX_FILE}: it is not a zip of shapes tars")
    try:
        data = archive.read(INDEX_FILE)
    except (zipfile.BadZipFile, zlib.error, EOFError, OSError) as error:
        raise ValueError(f"{INDEX_FILE} could not be read from {zip_path}: {error}") from error
    index = _index(zip_path / INDEX_FILE, data)
    listed = [(INDEX_FILE, len(data)), *((entry["name"], entry["size"]) for entry in index["tars"])]
    if sorted(members) != sorted(listed):
        names = [name for name, _ in members]
        odd = sorted({name for name, _ in set(members) ^ set(listed)}
                     | {name for name in names if names.count(name) > 1})
        raise ValueError(f"{zip_path} does not hold exactly {INDEX_FILE} and its tars, each once and with its "
                         f"size: {', '.join(odd)}")
    return data, index


def read_zip_index(zip_path: Path) -> dict:
    """The index of the tars inside the zip of ``zip_tars``: what ``read_index`` gives for a folder. No tar is read."""
    zip_path = Path(zip_path)
    with _open_zip(zip_path) as archive:
        return _zip_index(archive, zip_path)[1]


def unzip_tars(zip_path: Path, tars_dir: Path, *, identity: str | None = None,
               progress: Callable[[str], None] | None = None) -> dict:
    """Copy the tars and the index out of the zip of ``zip_tars`` into ``tars_dir``, for ``unpack``; do nothing if they are there.

    ``tars_dir`` goes on a local disk (``/content/packed/shapes``), the one of the dataset that ``unpack`` will
    extract from it; ``zip_path`` may be on Drive. ``identity`` is the expected one, or its first 16 characters
    or more, and is enforced before any tar is read. In this order:

    1. The zip must be complete and hold exactly the index and the tars it lists, each with its size.
    2. If ``tars_dir`` already holds that index and those tars, nothing is done (``unpack`` hashes the tars when it
       extracts them). A folder that holds anything else is refused, never deleted; an empty one is used.
    3. There must be room for the tars and for the dataset.
    4. Each tar is copied into ``<tars_dir>.unzipping`` and must have the size and the SHA-256 of the index; one
       that reads wrong is read a second time before the run gives up. The folder is renamed last.

    Returns ``identity``, ``directory``, ``bytes`` (of the tars), ``seconds`` and ``extracted`` (False when nothing
    was done). Every refusal is a ValueError and leaves ``tars_dir`` as it was.
    """
    zip_path, target = Path(zip_path), Path(tars_dir)
    say = progress or (lambda message: None)
    start = time.perf_counter()
    with _open_zip(zip_path) as archive:
        data, index = _zip_index(archive, zip_path)
        _check_expected(index["identity"], identity, zip_path)
        result = {"identity": index["identity"], "directory": str(target),
                  "bytes": sum(entry["size"] for entry in index["tars"])}
        link = target.is_symlink()
        if link or target.exists():
            if not link and _holds_tars(target, data, index):
                say(f"{target} already holds the tars of {zip_path.name}: nothing to do")
                return result | {"extracted": False, "seconds": time.perf_counter() - start}
            if link or not target.is_dir() or any(target.iterdir()):
                raise ValueError(f"{target} {'is a symbolic link' if link else 'exists'} and does not hold the tars "
                                 f"of {zip_path.name}; it is left as it is: delete it, or give another folder")
        temporary = target.with_name(target.name + ZIP_TEMPORARY_SUFFIX)
        target.parent.mkdir(parents=True, exist_ok=True)
        _clear(temporary)
        needed, free = result["bytes"] + index["bytes"] + SPACE_MARGIN, shutil.disk_usage(target.parent).free
        if free < needed:
            raise ValueError(f"Not enough disk space in {target.parent}: the tars and the dataset that unpack "
                             f"extracts from them need {needed / 1e9:.2f} GB and {free / 1e9:.2f} GB are free")
        try:
            temporary.mkdir()
            for entry in index["tars"]:
                lap = time.perf_counter()
                try:
                    _unzip_tar(archive, zip_path, entry, temporary / entry["name"])
                except _ReadError as error:
                    say(f"{error}; reading {entry['name']} again")
                    _unzip_tar(archive, zip_path, entry, temporary / entry["name"])
                say(f"{entry['name']}: {entry['size'] / 1e9:.3f} GB out of {zip_path.name} in "
                    f"{time.perf_counter() - lap:.1f} s")
            (temporary / INDEX_FILE).write_bytes(data)
            os.replace(temporary, target)
        except BaseException:
            shutil.rmtree(temporary, ignore_errors=True)
            raise
    return result | {"extracted": True, "seconds": time.perf_counter() - start}


In [ ]:
%%writefile src/schrodinger_glno/shapes_dataset.py
"""Stage-3 datasets: Gaussian packets on many deformed spheres and tori, one directory per shape.

Every shape is its own sample of geometry (Generacion_datos_etapa3.md, §5 and §8): shape ``i``
is a sphere when i is even and a torus when it is odd, and every ``round_torus_every``-th
torus (``ShapesConfig.is_round``) is a torus of revolution with a random aspect ratio, whose
exact solution (``reference.torus_exact_solution``, with the radii and centre recorded as
``exact_torus`` by ``round_torus_geometry``) calibrates the data error. Splits go by index,
train first, then val and test, so both families are balanced in each and no shape is shared
between splits. Shape ``i``
depends only on ``(seed, i)``: ``np.random.default_rng([seed, i, stream])`` with one stream for
the shape (``shapes.make_shape``), one for its packets and one for its out-of-distribution
packets, so the data do not depend on the number of workers or on the order they finish in.

Per shape: the mesh of ``shapes.make_shape`` (area 4π), the cotangent Laplacian, the lumped
mass with its ``k_eig`` eigenpairs (the GLNO basis) and the blended mass of Crank–Nicolson,
factored at ``dt`` and at ``dt/2``. Each packet draws ``(center, phi, k0, sigma)`` as
``sampling.sample_packets`` does (centre ∝ lumped area, φ uniform, (k0, σ) uniform on the part
of the rectangle the mesh resolves) and is drawn again, with the reason counted, when

a. on a torus, the cross-section through the centre is thinner than ``min_tube_sigmas · σ``
   (``tube_section_radii``): in a thinner tube the packet wraps around it (§3, §4);
b. the chordal packet puts more than ``max_leak`` of its lumped norm beyond the geodesic
   distance ``leak_radius · σ`` from the centre: it reaches across the hole of a torus (§4);
c. the packet cut at 7σ–9σ (``geodesic.cut_gaussian_packet``) keeps less than
   ``min_spectral_fraction`` (``min_ood_spectral_fraction`` out of distribution) of its norm in
   the ``k_eig`` lumped eigenvectors, computed in float64 (§8.9: the packet is drawn again, not
   the shape).

The cut packet, normalised in the blended mass, is then evolved by Crank–Nicolson at ``dt``
(stored) and at ``dt/2`` (for the error estimate, §8.9), with the saved times of
``SurfaceConfig``: ``output_steps`` states including t = 0.

Layout of ``directory``::

    config.json                   the ShapesConfig; a run with another one is refused
    <id>/mesh.npz                 verts, faces, mass (lumped), evals, evecs, geo_feat, t_out
    <id>/inputs.npy               (n, V, 2)                  float32  Re, Im of ψ0
    <id>/outputs.npy              (n, V, output_steps, 2)    float32  saved states, t = 0 included
    <id>/norms.npy                (n, output_steps)          float64  ψᴴ M ψ, M the blended mass
    <id>/params.npy               (n, 4)                     float64  center, phi, k0, sigma
    <id>/diagnostics.npy          (n, 7)                     float64  DIAGNOSTIC_COLUMNS
    <id>/ood_*.npy                the same for the out-of-distribution packets (test shapes only)
    <id>/shape.json               deformation, quality, draws, packet statistics, file hashes, timings
    manifest.json                 settings, the shapes (shape.json without its timings and packet
                                  ranges), SHA-256 of every data file and an identity hash

``<id>`` is ``f"{i:03d}_{family}"``; ``n`` is ``packets_per_shape`` (``ood_packets_per_shape``
for ``ood_*``). ``mesh.npz`` has the keys of the stage-1 and stage-2 datasets: ``evals`` are
float64, but ``evecs`` are stored as float32 to halve the largest file of a shape (700 modes:
44 MB per torus instead of 87); rounding them changed the spectral fractions of a production
sphere and torus by less than 1e-8, far below the gates. ``geo_feat`` is the distance to the mean vertex, as before.

``DIAGNOSTIC_COLUMNS``:

- ``spectral_fraction``: of the cut ψ0 in the float64 lumped eigenbasis (rule c).
- ``cn_error_estimate``: ``(4/3) max_t ‖ψ_dt - ψ_dt/2‖ / ‖ψ_dt/2‖`` in the lumped-mass norm,
  Richardson's estimate of the time-stepping error of the stored trajectory (CN is second order).
- ``removed_fraction``: the part of the chordal packet's norm the geodesic cutoff removed.
- ``leak_fraction``: rule b's fraction beyond ``leak_radius · σ``.
- ``thin_draws``, ``leak_draws``, ``spectral_draws``: draws rejected by rules a, b and c before
  this packet was kept.

Writes resume: a shape is built in ``<id>.partial`` and renamed when complete, so an
interrupted run never leaves a complete-looking shape; a new run with the same config skips
the complete shapes and clears stale ``.partial`` directories. A run holds an exclusive
``flock`` on ``directory`` from start to end, so a second run on it (another ``families``,
say) is refused instead of clearing the ``.partial`` directories the first one is writing.
``manifest.json`` is written only when every shape exists. Its identity (as in
``schrodinger_fno.dataset``) hashes the settings, the shapes (seed, deformation, quality and
packet statistics of each) and the SHA-256 of every data file; ``shape.json`` is not hashed,
since it holds the timings, but ``shapes_dataset_identity(verify=True)`` checks it against
the manifest.

Shapes are built in spawned worker processes with single-threaded BLAS: as in stages 1 and 2,
ARPACK rotates the eigenbasis inside degenerate eigenspaces (the round tori have them) when
the thread count changes. With the same NumPy, SciPy and BLAS builds, 1 or N workers and a
resumed run give the same identity. Spawned workers import the caller's main module, so a
script that calls ``generate_shapes_dataset`` needs the ``if __name__ == "__main__":`` guard
and must not set the BLAS variables at import time (a worker that sees them changed refuses to
build). A worker that dies (killed, out of memory) stops the run with an error; run it again to
resume.
"""
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool
from dataclasses import asdict, dataclass
import fcntl
from hashlib import sha256
import json
from math import isfinite
import multiprocessing
import os
from pathlib import Path
import re
import shutil
import time
from typing import Callable, Iterable
import numpy as np
from numpy.typing import NDArray
from schrodinger_glno.data_utils import _digest, psi_to_channels
from .geodesic import HeatGeodesics, cut_gaussian_packet
from .initial_conditions import spectral_fraction
from .mesh import blended_mass, cotan_laplacian, laplacian_eigenpairs, vertex_areas, vertex_normals
from .sampling import PARAM_COLUMNS, allowed_wavenumber, direction_from_angle, resolved_fraction, sample_packets
from .shapes import AREA, TorusDeformation, _base_torus, make_shape, torus_surface, tube_log_radius
from .solver import build_cn_matrices, solve_CN

FORMAT_VERSION = 1
FAMILIES = ("sphere", "torus")
SPLITS = ("train", "val", "test")
SHAPE_STREAM, PACKET_STREAM, OOD_STREAM = 0, 1, 2
DIAGNOSTIC_COLUMNS = ("spectral_fraction", "cn_error_estimate", "removed_fraction", "leak_fraction",
                      "thin_draws", "leak_draws", "spectral_draws")
# Points per wavelength of the k_allowed in shapes.mesh_quality, which make_shape's min_wavenumber gate reads.
GATE_POINTS_PER_WAVELENGTH = 10
# shape.json entries the manifest copies (and its identity hashes), besides the packet statistics.
RECORD_KEYS = ("index", "id", "family", "split", "round", "seed", "exact_torus", "n_verts", "k_allowed", "attempts",
               "deformation", "quality")
PACKET_KEYS = ("packets", "draws", "rejected", "acceptance", "resolved_fraction")
BLAS_VARIABLES = ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS")
CONFIG_FILE, MANIFEST_FILE, SHAPE_FILE = "config.json", "manifest.json", "shape.json"
_IDENTITY_KEYS = ("format_version", "config", "counts", "time_convention", "param_columns", "diagnostic_columns",
                  "shapes", "sha256")
_INT_FIELDS = ("seed", "train_shapes", "val_shapes", "test_shapes", "packets_per_shape", "ood_packets_per_shape",
               "round_torus_every", "sphere_subdivisions", "sphere_l_max", "max_shape_draws", "output_steps", "k_eig",
               "max_packet_draws")
_FLOAT_FIELDS = ("sphere_slope", "torus_target_edge", "torus_a_max", "torus_b_max", "min_wavenumber",
                 "min_curvature_radius", "area", "final_time", "dt", "min_points_per_wavelength", "cutoff_inner",
                 "cutoff_outer", "min_tube_sigmas", "leak_radius", "max_leak", "min_spectral_fraction",
                 "min_ood_spectral_fraction")
_DEFORMATION_RANGES = ("sphere_tau_range", "torus_alpha_range", "torus_tau_range")
_PACKET_RANGES = ("k0_range", "sigma_range", "ood_k0_range", "ood_sigma_range")


@dataclass(frozen=True)
class ShapesConfig:
    """Settings of a stage-3 dataset; the defaults are the numbers adopted in §8.9.

    Shapes: ``train_shapes + val_shapes + test_shapes``, alternating sphere / torus; torus
    number j (= i // 2) is round when j is a multiple of ``round_torus_every`` (0: none).
    The sphere and torus fields go to ``shapes.make_shape`` and its samplers; round tori keep
    ``torus_alpha_range`` with no other deformation. Gates are the ``shapes.check_gates``
    defaults with ``min_wavenumber`` (k0 + 3/(2σ) = 8 + 3/0.36 ≈ 16.33 at the out-of-distribution corner),
    the wavenumber the mesh must resolve at ``min_points_per_wavelength`` like the packets, and
    ``min_curvature_radius`` (spheres only). ``area`` is fixed by ``shapes.build_shape``
    and only recorded. Time settings follow ``SurfaceConfig``. Packet rules: see the module docstring.
    """
    seed: int = 0
    train_shapes: int = 160
    val_shapes: int = 20
    test_shapes: int = 20
    packets_per_shape: int = 5
    ood_packets_per_shape: int = 5
    round_torus_every: int = 4
    sphere_subdivisions: int = 5
    sphere_relax: bool = True
    sphere_l_max: int = 4
    sphere_slope: float = 2.0
    sphere_tau_range: tuple[float, float] = (0.05, 0.15)
    torus_target_edge: float = 0.0305
    torus_alpha_range: tuple[float, float] = (0.45, 0.55)
    torus_a_max: float = 0.04
    torus_b_max: float = 0.06
    torus_tau_range: tuple[float, float] = (0.0, 0.075)
    min_wavenumber: float = 16.33
    min_curvature_radius: float = 0.18
    max_shape_draws: int = 100
    area: float = AREA
    final_time: float = 0.5
    dt: float = 0.001
    output_steps: int = 11
    k_eig: int = 700
    min_points_per_wavelength: float = 10.0
    k0_range: tuple[float, float] = (1.0, 6.0)
    sigma_range: tuple[float, float] = (0.15, 0.2)
    ood_k0_range: tuple[float, float] = (6.0, 8.0)
    ood_sigma_range: tuple[float, float] = (0.18, 0.2)
    cutoff_inner: float = 7.0
    cutoff_outer: float = 9.0
    min_tube_sigmas: float = 1.75
    leak_radius: float = 4.0
    max_leak: float = 5e-3
    min_spectral_fraction: float = 1 - 1e-5
    min_ood_spectral_fraction: float = 1 - 6e-5
    max_packet_draws: int = 200

    def __post_init__(self) -> None:
        # Accept JSON lists and NumPy scalars, but reject non-integral "integers" such as 5.5.
        for name in _INT_FIELDS:
            value = getattr(self, name)
            if isinstance(value, bool) or float(value) != int(value):
                raise ValueError(f"{name} must be an integer")
            object.__setattr__(self, name, int(value))
        if not isinstance(self.sphere_relax, (bool, np.bool_)):
            raise ValueError("sphere_relax must be true or false")
        object.__setattr__(self, "sphere_relax", bool(self.sphere_relax))
        for name in _FLOAT_FIELDS:
            value = float(getattr(self, name))
            if not isfinite(value):
                raise ValueError(f"{name} must be finite")
            object.__setattr__(self, name, value)
        for name in _DEFORMATION_RANGES + _PACKET_RANGES:
            low, high = (float(x) for x in getattr(self, name))
            if not (isfinite(low) and isfinite(high) and 0 <= low <= high):
                raise ValueError(f"{name} must satisfy 0 <= low <= high")
            if name in _PACKET_RANGES and not low < high:
                raise ValueError(f"{name} must satisfy low < high")
            object.__setattr__(self, name, (low, high))
        if self.seed < 0 or min(self.train_shapes, self.val_shapes, self.test_shapes) < 0 or self.n_shapes < 1:
            raise ValueError("seed and shape counts must be >= 0, with at least one shape")
        if self.packets_per_shape < 1 or self.ood_packets_per_shape < 0 or self.round_torus_every < 0:
            raise ValueError("need packets_per_shape >= 1, ood_packets_per_shape >= 0 and round_torus_every >= 0")
        if self.sphere_subdivisions < 0 or self.sphere_l_max < 2:
            raise ValueError("need sphere_subdivisions >= 0 and sphere_l_max >= 2")
        if not (0 < self.torus_alpha_range[0] and self.torus_alpha_range[1] < 1):
            raise ValueError("torus_alpha_range must lie in (0, 1)")
        if self.torus_target_edge <= 0 or min(self.torus_a_max, self.torus_b_max) < 0:
            raise ValueError("need torus_target_edge > 0 and torus_a_max, torus_b_max >= 0")
        if min(self.min_wavenumber, self.min_curvature_radius) < 0:
            raise ValueError("min_wavenumber and min_curvature_radius must be >= 0")
        if self.max_shape_draws < 1 or self.max_packet_draws < 1:
            raise ValueError("max_shape_draws and max_packet_draws must be positive")
        if self.area != AREA:
            raise ValueError("shapes.build_shape scales every shape to area 4π")
        if self.final_time <= 0 or self.dt <= 0:
            raise ValueError("final_time and dt must be positive")
        n = self.final_time / self.dt
        if abs(n - round(n)) > 1e-9 * max(1.0, n) or round(n) < 1:
            raise ValueError("final_time must be a whole number of dt steps")
        if self.output_steps < 2 or (self.steps - 1) % (self.output_steps - 1):
            raise ValueError("output_steps - 1 must divide the number of time steps")
        if self.k_eig < 1 or self.min_points_per_wavelength <= 0:
            raise ValueError("k_eig and min_points_per_wavelength must be positive")
        if self.sigma_range[0] <= 0 or self.ood_sigma_range[0] <= 0:
            raise ValueError("sigma must be positive")
        if self.ood_k0_range[0] < self.k0_range[1]:
            raise ValueError("ood_k0_range must start at or above the training k0 range")
        if not 0 <= self.cutoff_inner < self.cutoff_outer:
            raise ValueError("need 0 <= cutoff_inner < cutoff_outer")
        if self.min_tube_sigmas < 0 or self.leak_radius <= 0 or not 0 < self.max_leak <= 1:
            raise ValueError("need min_tube_sigmas >= 0, leak_radius > 0 and 0 < max_leak <= 1")
        if not (0 < self.min_spectral_fraction <= 1 and 0 < self.min_ood_spectral_fraction <= 1):
            raise ValueError("spectral fractions must be in (0, 1]")

    @property
    def n_shapes(self) -> int:
        return self.train_shapes + self.val_shapes + self.test_shapes

    @property
    def steps(self) -> int:
        return int(round(self.final_time / self.dt)) + 1

    def output_indices(self) -> NDArray:
        return np.linspace(0, self.steps - 1, self.output_steps).round().astype(int)

    def output_times(self) -> NDArray:
        return self.output_indices() * self.dt

    def _check_index(self, index: int) -> None:
        if not 0 <= index < self.n_shapes:
            raise ValueError(f"shape index {index} is outside 0..{self.n_shapes - 1}")

    def family(self, index: int) -> str:
        self._check_index(index)
        return FAMILIES[index % 2]

    def is_round(self, index: int) -> bool:
        every = self.round_torus_every
        return self.family(index) == "torus" and every > 0 and (index // 2) % every == 0

    def split(self, index: int) -> str:
        self._check_index(index)
        if index < self.train_shapes:
            return "train"
        return "val" if index < self.train_shapes + self.val_shapes else "test"

    def shape_id(self, index: int) -> str:
        return f"{index:03d}_{self.family(index)}"

    def shape_options(self, index: int) -> dict:
        """Keyword arguments of ``shapes.make_shape`` for shape ``index`` (without its family and rng).

        The gap gate of a torus (``shapes.tube_quantities``) is evaluated at the smallest and the largest σ
        of any packet. The wavenumber gate reads ``shapes.mesh_quality``'s k_allowed, at 10 points per
        wavelength, so ``min_wavenumber`` is rescaled to ``min_points_per_wavelength`` (a factor 1 by default).
        """
        sigmas = (min(self.sigma_range[0], self.ood_sigma_range[0]), max(self.sigma_range[1], self.ood_sigma_range[1]))
        wavenumber = self.min_wavenumber * (self.min_points_per_wavelength / GATE_POINTS_PER_WAVELENGTH)
        options = {"max_attempts": self.max_shape_draws, "sigmas": sigmas,
                   "gates": {"min_wavenumber": wavenumber, "min_curvature_radius": self.min_curvature_radius}}
        if self.family(index) == "sphere":
            return options | {"subdivisions": self.sphere_subdivisions, "relax": self.sphere_relax,
                              "l_max": self.sphere_l_max, "slope": self.sphere_slope,
                              "tau_range": self.sphere_tau_range}
        options |= {"target_edge": self.torus_target_edge, "alpha_range": self.torus_alpha_range}
        if self.is_round(index):
            return options | {"a_max": 0.0, "b_max": 0.0, "tau_range": (0.0, 0.0)}
        return options | {"a_max": self.torus_a_max, "b_max": self.torus_b_max, "tau_range": self.torus_tau_range}

    def packet_sets(self, index: int) -> list[tuple[str, str, int, int]]:
        """``(name, file prefix, packets, rng stream)`` of each packet set of shape ``index``."""
        split = self.split(index)
        sets = [(split, "", self.packets_per_shape, PACKET_STREAM)]
        if split == "test" and self.ood_packets_per_shape:
            sets.append(("ood", "ood_", self.ood_packets_per_shape, OOD_STREAM))
        return sets

    def packet_rules(self, name: str) -> tuple[tuple[float, float], tuple[float, float], float]:
        """``(k0_range, sigma_range, min_spectral_fraction)`` of a packet set."""
        if name == "ood":
            return self.ood_k0_range, self.ood_sigma_range, self.min_ood_spectral_fraction
        return self.k0_range, self.sigma_range, self.min_spectral_fraction


def tube_section_radii(deformation: TorusDeformation, quality: dict, verts: NDArray, n_v: int = 128,
                       chunk: int = 1024) -> NDArray:
    """Smallest tube radius of the cross-section through each vertex of a torus shape, in the units of ``verts``.

    ``shapes.build_shape`` puts vertex i at x(u_i, v_i), with (u_i, v_i) the angles of vertex i of the
    round base torus (``shapes._base_torus``), and ``delaunay_flip`` only changes faces, so u_i is read
    from that torus; the vertices are checked against ``scale · (x(u_i, v_i) - offset)``. The radius
    is ``scale · r0 · e^{min_v t(u_i, v)}`` with the minimum over ``n_v`` angles v (t has |k| ≤ 2 in v).
    """
    _, u, v = _base_torus(deformation, quality["target_edge"])
    scale = quality["scale"]
    expected = scale * (torus_surface(deformation, u, v) - np.asarray(quality["offset"], float))
    if expected.shape != np.shape(verts) or np.abs(expected - verts).max() > 1e-9:
        raise RuntimeError("the torus vertices are not x(u, v) of the base torus, in its order")
    grid = 2 * np.pi * np.arange(n_v) / n_v
    t_min = np.concatenate([tube_log_radius(deformation, part[:, None], grid[None, :]).min(axis=1)
                            for part in np.array_split(u, max(1, -(-len(u) // chunk)))])
    return scale * deformation.minor_radius * np.exp(t_min)


def round_torus_geometry(deformation: TorusDeformation, quality: dict) -> dict:
    """Radii and centre of a round torus shape in the units of its mesh, for ``reference.torus_exact_solution``.

    The mesh is ``scale · (x - offset)`` with x on the torus (R0, r0) about the origin, so its radii are
    ``R0 · scale`` and ``r0 · scale`` (§8.9: not the nominal ones) and its centre is ``-scale · offset``,
    to be subtracted from the vertices and the packet centre before calling the reference.
    """
    if any(deformation.a + deformation.b + deformation.c):
        raise ValueError("not a torus of revolution")
    scale = quality["scale"]
    return {"major_radius": deformation.major_radius * scale, "minor_radius": deformation.minor_radius * scale,
            "center": [-scale * x for x in quality["offset"]]}


def _shape_mesh(config: ShapesConfig, shape) -> dict:
    """Operators, GLNO eigenpairs (lumped mass, float64) and the solver's blended mass of one shape."""
    verts, faces = shape.verts, shape.faces
    L, lumped = cotan_laplacian(verts, faces), vertex_areas(verts, faces)
    evals, evecs = laplacian_eigenpairs(L, lumped, config.k_eig)
    return {"verts": verts, "faces": faces, "L": L, "mass": lumped, "blended": blended_mass(verts, faces),
            "normals": vertex_normals(verts, faces), "evals": evals, "evecs": evecs,
            "k_allowed": allowed_wavenumber(shape.quality["mean_edge"], config.min_points_per_wavelength)}


def _draw_packet(config: ShapesConfig, mesh: dict, rng: np.random.Generator, name: str,
                 stats: dict) -> tuple[NDArray, NDArray, NDArray]:
    """One packet of set ``name`` that passes rules a-c: its row, the cut ψ0 and its first diagnostics.

    Every rejected draw is counted in ``stats`` and in the returned diagnostics; the whole row
    is drawn again, since each rule depends on the centre and σ.
    """
    k0_range, sigma_range, min_fraction = config.packet_rules(name)
    verts, lumped, sections = mesh["verts"], mesh["mass"], mesh["sections"]
    rejected = {"thin": 0, "leak": 0, "spectral": 0}
    for _ in range(config.max_packet_draws):
        stats["draws"] += 1
        drawn = {}
        row = sample_packets(1, lumped, rng, k0_range, sigma_range, mesh["k_allowed"], drawn)[0]
        stats["kd_tried"] += drawn["tried"]
        stats["kd_accepted"] += drawn["accepted"]
        center, phi, k0, sigma = int(row[0]), *row[1:]
        if sections is not None and sections[center] < config.min_tube_sigmas * sigma:
            rejected["thin"] += 1
            continue
        distance = mesh["geodesics"].distance(center)
        # |ψ|² of the chordal packet, which does not depend on k0 or φ
        density = lumped * np.exp(-np.sum((verts - verts[center])**2, axis=1) / (2 * sigma**2))
        leak = float(density[distance > config.leak_radius * sigma].sum() / density.sum())
        if leak > config.max_leak:
            rejected["leak"] += 1
            continue
        direction = direction_from_angle(mesh["normals"][center], phi)
        psi0, removed = cut_gaussian_packet(verts, mesh["blended"], mesh["normals"], center, k0, sigma, direction,
                                            distance, config.cutoff_inner, config.cutoff_outer)
        fraction = spectral_fraction(psi0, lumped, mesh["evecs"])
        if fraction < min_fraction:
            rejected["spectral"] += 1
            continue
        for reason, count in rejected.items():
            stats["rejected"][reason] += count
        return row, psi0, np.array([fraction, removed, leak, rejected["thin"], rejected["leak"], rejected["spectral"]])
    raise RuntimeError(f"no {name} packet passed the rules in {config.max_packet_draws} draws: {rejected}")


def _cn_error_estimate(coarse: NDArray, fine: NDArray, mass: NDArray) -> float:
    """``(4/3) max_t ‖ψ_dt - ψ_dt/2‖_M / ‖ψ_dt/2‖_M`` (lumped M): Richardson for a second-order scheme."""
    difference = np.sqrt(np.sum(mass[:, None] * np.abs(coarse - fine)**2, axis=0))
    return float(4 / 3 * np.max(difference / np.sqrt(np.sum(mass[:, None] * np.abs(fine)**2, axis=0))))


def _write_packets(config: ShapesConfig, mesh: dict, solvers: tuple, folder: Path, index: int, name: str,
                   prefix: str, n: int, stream: int) -> dict:
    """Draw, evolve and save one packet set of a shape through memmaps; return its statistics."""
    rng = np.random.default_rng([config.seed, index, stream])
    (LU, U1), (LU_half, U1_half) = solvers
    steps, indices = config.steps, config.output_indices()
    n_vertices, n_out = len(mesh["verts"]), config.output_steps
    inputs = np.lib.format.open_memmap(folder / f"{prefix}inputs.npy", mode="w+", dtype=np.float32,
                                       shape=(n, n_vertices, 2))
    outputs = np.lib.format.open_memmap(folder / f"{prefix}outputs.npy", mode="w+", dtype=np.float32,
                                        shape=(n, n_vertices, n_out, 2))
    params, norms = np.zeros((n, len(PARAM_COLUMNS))), np.zeros((n, n_out))
    diagnostics = np.zeros((n, len(DIAGNOSTIC_COLUMNS)))
    stats = {"draws": 0, "rejected": {"thin": 0, "leak": 0, "spectral": 0}, "kd_tried": 0, "kd_accepted": 0}
    for i in range(n):
        params[i], psi0, (fraction, *checks) = _draw_packet(config, mesh, rng, name, stats)
        trajectory = solve_CN(psi0, LU, U1, steps, indices)
        half_steps = solve_CN(psi0, LU_half, U1_half, 2 * steps - 1, 2 * indices)
        norms[i] = np.real(np.einsum("vt,vt->t", trajectory.conj(), mesh["blended"] @ trajectory))
        diagnostics[i] = [fraction, _cn_error_estimate(trajectory, half_steps, mesh["mass"]), *checks]
        inputs[i], outputs[i] = psi_to_channels(psi0), psi_to_channels(trajectory)
    inputs.flush(); outputs.flush()
    del inputs, outputs
    np.save(folder / f"{prefix}norms.npy", norms)
    np.save(folder / f"{prefix}params.npy", params)
    np.save(folder / f"{prefix}diagnostics.npy", diagnostics)
    k0_range, sigma_range, _ = config.packet_rules(name)
    column = dict(zip(DIAGNOSTIC_COLUMNS, diagnostics.T))
    return stats | {"packets": n, "acceptance": stats["kd_accepted"] / stats["kd_tried"],
                    "resolved_fraction": resolved_fraction(k0_range, sigma_range, mesh["k_allowed"]),
                    "k0": [float(params[:, 2].min()), float(params[:, 2].max())],
                    "sigma": [float(params[:, 3].min()), float(params[:, 3].max())],
                    "min_spectral_fraction": float(column["spectral_fraction"].min()),
                    "max_cn_error_estimate": float(column["cn_error_estimate"].max()),
                    "max_removed_fraction": float(column["removed_fraction"].max()),
                    "max_leak_fraction": float(column["leak_fraction"].max()),
                    "max_norm_drift": float(np.abs(norms - norms[:, :1]).max())}


def write_shape(config: ShapesConfig, directory: Path, index: int) -> dict:
    """Build shape ``index`` with its packets in ``directory/<id>`` and return its ``shape.json`` record.

    Works in ``<id>.partial`` (cleared first) and renames it when every file is written. A
    complete ``<id>`` is left as it is and its stored record returned. Neither limits the BLAS
    threads nor locks ``directory``: ``generate_shapes_dataset`` does both and calls it in its workers.
    """
    directory = Path(directory)
    shape_id, family, split = config.shape_id(index), config.family(index), config.split(index)
    target, folder = directory / shape_id, directory / f"{shape_id}.partial"
    if target.is_dir():
        return json.loads((target / SHAPE_FILE).read_text())
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    clock = {"start": time.perf_counter()}

    def lap(name: str) -> None:
        now = time.perf_counter()
        clock[name], clock["last"] = now - clock.get("last", clock["start"]), now

    shape = make_shape(family, np.random.default_rng([config.seed, index, SHAPE_STREAM]), **config.shape_options(index))
    lap("shape")
    mesh = _shape_mesh(config, shape)
    verts = mesh["verts"]
    np.savez(folder / "mesh.npz", verts=verts, faces=mesh["faces"], mass=mesh["mass"], evals=mesh["evals"],
             evecs=mesh["evecs"].astype(np.float32), geo_feat=np.linalg.norm(verts - verts.mean(axis=0), axis=1),
             t_out=config.output_times())
    lap("eigenpairs")
    free = np.zeros(len(verts))                                    # V = 0
    solvers = (build_cn_matrices(mesh["L"], mesh["blended"], free, config.dt),
               build_cn_matrices(mesh["L"], mesh["blended"], free, config.dt / 2))
    mesh["geodesics"] = HeatGeodesics(verts, mesh["faces"])
    mesh["sections"] = tube_section_radii(shape.deformation, shape.quality, verts) if family == "torus" else None
    lap("factorizations")
    packets = {name: _write_packets(config, mesh, solvers, folder, index, name, prefix, n, stream)
               for name, prefix, n, stream in config.packet_sets(index)}
    lap("packets")
    hashes = {path.name: _digest(path) for path in sorted(folder.iterdir())}
    lap("hashes")
    record = {"format_version": FORMAT_VERSION, "index": index, "id": shape_id, "family": family, "split": split,
              "round": config.is_round(index), "seed": [config.seed, index],
              "exact_torus": round_torus_geometry(shape.deformation, shape.quality) if config.is_round(index) else None,
              "deformation": asdict(shape.deformation), "quality": shape.quality, "attempts": shape.attempts,
              "rejections": [list(reasons) for reasons in shape.rejections], "n_verts": len(verts),
              "k_allowed": mesh["k_allowed"], "max_eigenvalue": float(mesh["evals"][-1]), "packets": packets,
              "sha256": hashes,
              "timings": {key: round(value, 3) for key, value in clock.items() if key not in ("start", "last")}
              | {"total": round(time.perf_counter() - clock["start"], 3)}}
    (folder / SHAPE_FILE).write_text(json.dumps(record, indent=2, allow_nan=False) + "\n")
    os.replace(folder, target)
    return record


def _shape_task(config: ShapesConfig, directory: str, index: int) -> dict:
    """Worker entry point: ``write_shape``, with the shape's id in any error.

    Refuses to build when the BLAS variables are not the "1" ``generate_shapes_dataset`` set
    before spawning: the caller's main module, which every spawned worker imports, changed them.
    """
    changed = {name: os.environ.get(name) for name in BLAS_VARIABLES if os.environ.get(name) != "1"}
    if changed:
        raise RuntimeError(f"shape {config.shape_id(index)}: the worker's BLAS is not single-threaded ({changed}); "
                           "the caller's main module must not set these variables at import time")
    try:
        record = write_shape(config, Path(directory), index)
    except Exception as error:
        raise RuntimeError(f"shape {config.shape_id(index)}: {error!r}") from error
    return {key: record[key] for key in ("id", "split", "attempts", "n_verts", "timings")}


def _lock(directory: Path) -> int:
    """An exclusive ``flock`` on ``directory`` (its descriptor); it ends when that is closed or the process exits."""
    descriptor = os.open(directory, os.O_RDONLY)
    try:
        fcntl.flock(descriptor, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        os.close(descriptor)
        raise BlockingIOError(f"{directory} is in use by another run; run the families one after the other, "
                              "not at the same time") from None
    return descriptor


def _claim(directory: Path, config: ShapesConfig) -> None:
    """Store the config in an empty ``directory`` (which exists), or check that it matches the one stored there."""
    path = directory / CONFIG_FILE
    if path.exists():
        stored = json.loads(path.read_text())
        try:
            same = stored.get("format_version") == FORMAT_VERSION and ShapesConfig(**stored["config"]) == config
        except (TypeError, ValueError, KeyError):
            same = False
        if not same:
            raise FileExistsError(f"{directory} holds a shapes dataset made with a different config; "
                                  "use another directory to make a new one")
        return
    if any(directory.iterdir()):
        raise FileExistsError(f"Destination is not empty: {directory}")
    _write_json(path, {"format_version": FORMAT_VERSION, "config": asdict(config)})


def _write_json(path: Path, data: dict) -> None:
    """Strict JSON, written to a temporary file and renamed, so it is never seen half-written."""
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")
    os.replace(temporary, path)


def _identity(manifest: dict) -> str:
    payload = {key: manifest[key] for key in _IDENTITY_KEYS}
    return sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def _shape_entry(record: dict) -> dict:
    """The manifest entry of a shape: its ``shape.json`` record without the timings, hashes and packet ranges.

    The deformation and the quality (with ``scale``, ``offset`` and the mesh options) are what rebuild
    and refine the shape, so the identity covers them.
    """
    return {key: record[key] for key in RECORD_KEYS} | {
        "packets": {name: {key: stats[key] for key in PACKET_KEYS} for name, stats in record["packets"].items()}}


def _manifest(config: ShapesConfig, directory: Path) -> dict:
    """The manifest, from the ``shape.json`` of every shape."""
    shapes, hashes = [], {}
    for index in range(config.n_shapes):
        record = json.loads((directory / config.shape_id(index) / SHAPE_FILE).read_text())
        shapes.append(_shape_entry(record))
        hashes |= {f"{record['id']}/{name}": digest for name, digest in record["sha256"].items()}
    splits = [config.split(i) for i in range(config.n_shapes)]
    packets = {split: splits.count(split) * config.packets_per_shape for split in SPLITS}
    counts = {"shapes": {split: splits.count(split) for split in SPLITS},
              "families": {f: sum(config.family(i) == f for i in range(config.n_shapes)) for f in FAMILIES},
              "round_tori": sum(config.is_round(i) for i in range(config.n_shapes)),
              "packets": packets | {"ood": splits.count("test") * config.ood_packets_per_shape}}
    manifest = {"format_version": FORMAT_VERSION, "config": asdict(config), "counts": counts,
                "time_convention": "states include t=0", "param_columns": list(PARAM_COLUMNS),
                "diagnostic_columns": list(DIAGNOSTIC_COLUMNS), "shapes": shapes, "sha256": hashes}
    manifest["identity"] = _identity(manifest)
    return manifest


def shapes_dataset_identity(directory: Path, *, verify: bool = True) -> str:
    """Return the manifest identity; ``verify`` also checks every data file against its SHA-256 and every
    ``shape.json`` (not hashed: it holds the timings) against its manifest entry and hashes."""
    directory = Path(directory)
    manifest = json.loads((directory / MANIFEST_FILE).read_text())
    if manifest.get("format_version") != FORMAT_VERSION or manifest["identity"] != _identity(manifest):
        raise ValueError("Dataset manifest identity is invalid")
    if verify:
        for relative, expected in manifest["sha256"].items():
            if _digest(directory / relative) != expected:
                raise ValueError(f"Dataset file differs from manifest: {relative}")
        for entry in manifest["shapes"]:
            record = json.loads((directory / entry["id"] / SHAPE_FILE).read_text())
            hashes = {name: digest for name, digest in manifest["sha256"].items()
                      if name.split("/", 1)[0] == entry["id"]}
            try:
                same = _shape_entry(record) == entry and {
                    f"{entry['id']}/{name}": digest for name, digest in record["sha256"].items()} == hashes
            except (KeyError, TypeError, AttributeError):
                same = False
            if not same:
                raise ValueError(f"Dataset file differs from manifest: {entry['id']}/{SHAPE_FILE}")
    return manifest["identity"]


def _set_environment(values: dict) -> dict:
    """Set (or, for None, remove) environment variables; return their previous values."""
    previous = {name: os.environ.get(name) for name in values}
    for name, value in values.items():
        if value is None:
            os.environ.pop(name, None)
        else:
            os.environ[name] = value
    return previous


def generate_shapes_dataset(config: ShapesConfig, directory: Path, *, workers: int = 1,
                            families: Iterable[str] | None = None,
                            progress: Callable[[str], None] | None = None) -> dict | None:
    """Build the missing shapes of ``families`` (default: all) with ``workers`` processes; return the manifest.

    The manifest is written, and returned, only when every shape of the dataset exists;
    otherwise this returns None. A directory made with another config is refused
    (``FileExistsError``), as is a non-empty directory that is not a shapes dataset and one that
    another run holds (``BlockingIOError``). Workers are spawned processes with single-threaded
    BLAS, even for ``workers=1``; each one builds whole shapes (``write_shape``) and ``progress``
    gets one line per finished shape, with its timings. A failed shape stops the run once the
    shapes in progress are complete; a worker that dies stops it at once (``RuntimeError``).
    """
    directory = Path(directory)
    families = FAMILIES if families is None else tuple(families)
    if not set(families) <= set(FAMILIES) or not families:
        raise ValueError(f"families must be some of {FAMILIES}")
    if workers < 1:
        raise ValueError("workers must be at least 1")
    directory.mkdir(parents=True, exist_ok=True)
    lock = _lock(directory)
    try:
        return _generate(config, directory, workers, families, progress or (lambda message: None))
    finally:
        os.close(lock)


def _generate(config: ShapesConfig, directory: Path, workers: int, families: tuple[str, ...],
              say: Callable[[str], None]) -> dict | None:
    """``generate_shapes_dataset`` in a locked ``directory``."""
    _claim(directory, config)
    for stale in sorted(directory.glob("*.partial")):
        shutil.rmtree(stale)
        say(f"cleared {stale.name}, left by an interrupted run")
    missing = [i for i in range(config.n_shapes) if not (directory / config.shape_id(i)).is_dir()]
    pending = [i for i in missing if config.family(i) in families]
    if missing:
        (directory / MANIFEST_FILE).unlink(missing_ok=True)
    say(f"{config.n_shapes - len(missing)} of {config.n_shapes} shapes complete; " + (
        f"building {len(pending)} ({', '.join(families)}) with {min(workers, len(pending))} workers" if pending
        else "nothing to build"))
    if pending:
        start = time.perf_counter()
        previous = _set_environment(dict.fromkeys(BLAS_VARIABLES, "1"))     # inherited by the spawned workers
        try:
            # Not multiprocessing.Pool: it replaces a killed worker and waits forever for its shape.
            context = multiprocessing.get_context("spawn")
            with ProcessPoolExecutor(min(workers, len(pending)), mp_context=context) as pool:
                futures = [pool.submit(_shape_task, config, str(directory), index) for index in pending]
                try:
                    for done, future in enumerate(as_completed(futures), 1):
                        result = future.result()
                        t = result["timings"]
                        left = (time.perf_counter() - start) / done * (len(pending) - done) / 60
                        say(f"{result['id']} ({result['split']}, {result['n_verts']} vertices): {t['total']:.1f} s = "
                            f"shape {t['shape']:.1f} s ({result['attempts']} draws) + "
                            f"eigenpairs {t['eigenpairs']:.1f} s + factorizations {t['factorizations']:.1f} s + "
                            f"packets {t['packets']:.1f} s; {done}/{len(pending)} built, ~{left:.1f} min left")
                except BrokenProcessPool as error:
                    raise RuntimeError("a worker process died (killed, or out of memory?); "
                                       "run again to resume") from error
                except BaseException:
                    pool.shutdown(cancel_futures=True)      # finish the shapes in progress, start no other
                    raise
        finally:
            _set_environment(previous)
    if any(not (directory / config.shape_id(i)).is_dir() for i in missing):
        say(f"{sum(not (directory / config.shape_id(i)).is_dir() for i in missing)} shapes are still missing; "
            "the manifest is written once every shape exists")
        return None
    manifest = _manifest(config, directory)
    _write_json(directory / MANIFEST_FILE, manifest)
    say(f"manifest written: identity {manifest['identity'][:16]}")
    return manifest


def _reason_kind(reason: str) -> str:
    """A gate failure without its numbers, e.g. ``"min angle #° < #°"``."""
    return re.sub(r"(?<![A-Za-z\d])-?\d+(\.\d+)?(e[-+]?\d+)?", "#", reason)


def dataset_summary(directory: Path) -> dict:
    """Counts, rejection rates of shapes and packets, diagnostic ranges and timings of the complete shapes."""
    directory = Path(directory)
    config = ShapesConfig(**json.loads((directory / CONFIG_FILE).read_text())["config"])
    records = [json.loads((directory / config.shape_id(i) / SHAPE_FILE).read_text())
               for i in range(config.n_shapes) if (directory / config.shape_id(i)).is_dir()]
    shapes = {family: {"built": 0, "draws": 0, "reasons": {}, "seconds": 0.0} for family in FAMILIES}
    packets, ranges, drift = {}, {}, {}
    for record in records:
        entry = shapes[record["family"]]
        entry["built"] += 1
        entry["draws"] += record["attempts"]
        entry["seconds"] += record["timings"]["total"]
        for reasons in record["rejections"]:
            for reason in reasons:
                kind = _reason_kind(reason)
                entry["reasons"][kind] = entry["reasons"].get(kind, 0) + 1
        for name, prefix, _, _ in config.packet_sets(record["index"]):
            stats = record["packets"][name]
            total = packets.setdefault(name, {"packets": 0, "draws": 0, "thin": 0, "leak": 0, "spectral": 0})
            total["packets"] += stats["packets"]
            total["draws"] += stats["draws"]
            for reason, count in stats["rejected"].items():
                total[reason] += count
            columns = np.load(directory / record["id"] / f"{prefix}diagnostics.npy")
            low, high = columns.min(axis=0), columns.max(axis=0)
            if name in ranges:
                low, high = np.minimum(low, ranges[name][0]), np.maximum(high, ranges[name][1])
            ranges[name] = (low, high)
            norms = np.load(directory / record["id"] / f"{prefix}norms.npy")
            drift[name] = max(drift.get(name, 0.0), float(np.abs(norms - norms[:, :1]).max()))
    for entry in shapes.values():
        entry["rejection_rate"] = 1 - entry["built"] / entry["draws"] if entry["draws"] else None
        entry["mean_seconds"] = entry.pop("seconds") / entry["built"] if entry["built"] else None
    for name, total in packets.items():
        total["rejection_rate"] = 1 - total["packets"] / total["draws"]
        total["diagnostics"] = {column: [float(low), float(high)]
                                for column, low, high in zip(DIAGNOSTIC_COLUMNS, *ranges[name])}
        total["max_norm_drift"] = drift[name]
    summary = {"shapes": {"total": config.n_shapes, "built": len(records),
                          "by_split": {s: sum(r["split"] == s for r in records) for s in SPLITS},
                          "round_tori": sum(r["round"] for r in records), **shapes},
               "packets": packets}
    if (directory / MANIFEST_FILE).exists():
        summary["identity"] = json.loads((directory / MANIFEST_FILE).read_text())["identity"]
    return summary


In [ ]:
%%writefile src/schrodinger_glno/solver.py
"""Crank–Nicolson on a triangle mesh.

The semi-discrete equation is ``i M dψ/dt = A ψ`` with ``A = ½ L + M diag(V)``.
Crank–Nicolson becomes ``(M + i dt/2 A) ψ^{n+1} = (M - i dt/2 A) ψ^n``, which is
the 1D scheme with the identity replaced by ``M``. Time stepping reuses
``schrodinger_fno.solver.solve_CN`` unchanged.

``mass`` may be one area per vertex (lumped, a diagonal ``M``) or a sparse matrix
such as ``mesh.blended_mass``. The scheme conserves ``ψᴴ M ψ`` for that ``M``.
"""
import numpy as np
import scipy.sparse as sp
from numpy.typing import NDArray
from scipy.sparse.linalg import SuperLU, splu
from schrodinger_glno.time_stepping import solve_CN, solve_CN_full
from .mesh import as_mass_matrix

__all__ = ["hamiltonian_matrix", "build_cn_matrices", "energy", "exact_evolution", "spectral_evolution",
           "solve_CN", "solve_CN_full"]


def _checked_mass(L: sp.spmatrix, mass: NDArray | sp.spmatrix) -> sp.csr_matrix:
    n = L.shape[0]
    M = as_mass_matrix(mass)
    if L.shape != (n, n) or M.shape != (n, n):
        raise ValueError("L and the mass must be (V, V) or one value per vertex")
    if np.any(M.diagonal() <= 0):
        raise ValueError("mass must be positive")
    return M


def hamiltonian_matrix(L: sp.spmatrix, mass: NDArray | sp.spmatrix, V: NDArray) -> sp.csr_matrix:
    """``A = ½ L + M diag(V)``: real, symmetric for diagonal M, the weak form of ``-½Δ + V``."""
    M = _checked_mass(L, mass)
    if np.shape(V) != (L.shape[0],) or not np.isfinite(V).all():
        raise ValueError("V must hold one finite value per vertex")
    return (0.5 * L + M @ sp.diags(np.asarray(V, float))).tocsr()


def build_cn_matrices(L: sp.spmatrix, mass: NDArray | sp.spmatrix, V: NDArray, dt: float) -> tuple[SuperLU, sp.spmatrix]:
    """Factor ``M + i dt/2 A`` once and return it with ``M - i dt/2 A``."""
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive")
    A = hamiltonian_matrix(L, mass, V)
    M = _checked_mass(L, mass)
    U2 = (M + (0.5j * dt) * A).tocsc()
    U1 = (M - (0.5j * dt) * A).tocsr()
    return splu(U2), U1


def energy(psi: NDArray, L: sp.spmatrix, mass: NDArray | sp.spmatrix, V: NDArray) -> float:
    """``⟨ψ, Hψ⟩ / ⟨ψ, ψ⟩`` in the mass inner product (``ψᴴ A ψ / ψᴴ M ψ``)."""
    A = hamiltonian_matrix(L, mass, V)
    M = _checked_mass(L, mass)
    return float(np.real(np.vdot(psi, A @ psi)) / np.real(np.vdot(psi, M @ psi)))


def exact_evolution(L: sp.spmatrix, mass: NDArray | sp.spmatrix, V: NDArray, psi0: NDArray,
                    times: NDArray) -> NDArray:
    """Exact solution of ``i M dψ/dt = A ψ`` at ``times`` (evenly spaced from 0): ``e^{-it M⁻¹A} ψ0``.

    No time-stepping error, so it isolates the error of Crank–Nicolson. For a
    diagonal mass ``M⁻¹A`` is sparse; otherwise ``M⁻¹`` is applied with a sparse LU,
    without forming the dense matrix. Returns (V, len(times)).
    """
    from scipy.sparse.linalg import LinearOperator, expm_multiply
    times = np.asarray(times, float)
    if times[0] != 0 or (len(times) > 2 and not np.allclose(np.diff(times), times[1] - times[0])):
        raise ValueError("times must start at 0 and be evenly spaced")
    A = hamiltonian_matrix(L, mass, V)
    if not sp.issparse(mass):
        generator = (-1j * sp.diags(1 / np.asarray(mass, float)) @ A).tocsr()
        trace = None
    else:
        lu = splu(_checked_mass(L, mass).tocsc())
        At = A.T.tocsr()
        solve = lambda b: lu.solve(np.ascontiguousarray(b.real)) + 1j * lu.solve(np.ascontiguousarray(b.imag))
        solve_t = lambda b: (lu.solve(np.ascontiguousarray(b.real), trans="T")
                             + 1j * lu.solve(np.ascontiguousarray(b.imag), trans="T"))
        n = A.shape[0]
        # B = -i M⁻¹A and its adjoint Bᴴ = i Aᵀ M⁻ᵀ (A and M are real); trace(B) only sets a shift.
        generator = LinearOperator((n, n), dtype=complex,
                                   matvec=lambda x: -1j * solve(A @ np.ravel(x)),
                                   rmatvec=lambda y: 1j * (At @ solve_t(np.ravel(y))))
        trace = 0.0
    if len(times) == 1:
        return psi0[:, None].astype(complex)
    return expm_multiply(generator, psi0.astype(complex), start=0, stop=times[-1], num=len(times),
                         endpoint=True, traceA=trace).T


def spectral_evolution(psi0: NDArray, mass: NDArray | sp.spmatrix, energies: NDArray, evecs: NDArray,
                       times: NDArray) -> NDArray:
    """Exact evolution inside the span of ``evecs``: ``Φ e^{-iEt} Φᵀ M ψ0``.

    ``evecs`` must be M-orthonormal eigenvectors of ``A φ = E M φ`` for the same ``M``.
    With V = 0 these are the Laplacian eigenpairs and ``E = λ / 2``. Returns (V, len(times)).
    """
    coeffs = evecs.T @ (as_mass_matrix(mass) @ psi0)
    phases = np.exp(-1j * np.outer(energies, np.asarray(times, float)))
    return evecs @ (coeffs[:, None] * phases)


In [ ]:
%%writefile src/schrodinger_glno/time_stepping.py
"""Crank–Nicolson time stepping for factored surface operators."""
import numpy as np
from numpy.typing import NDArray
import scipy.sparse as spa
from scipy.sparse.linalg import SuperLU

def solve_CN(psi0: NDArray, LU: SuperLU, U1: spa.spmatrix, N_steps: int, t_out_idx: NDArray) -> NDArray:
    """Save requested states, numbered 0 through N_steps-1."""
    idx = np.asarray(t_out_idx)
    if N_steps < 1 or idx.ndim != 1 or idx.size < 1 or not np.issubdtype(idx.dtype, np.integer):
        raise ValueError("steps and output indices must be nonempty integers")
    if idx[0] < 0 or idx[-1] >= N_steps or np.any(np.diff(idx) <= 0):
        raise ValueError("output indices must be unique, increasing and within the simulated range")
    if np.asarray(psi0).shape != (U1.shape[0],):
        raise ValueError("psi0 length must match the CN matrices")
    psi = psi0.copy()
    snapshots = np.zeros((len(psi0), len(idx)), dtype=complex)
    out_count = 0
    for n in range(N_steps):
        if out_count < len(idx) and n == idx[out_count]:
            snapshots[:, out_count] = psi
            out_count += 1
        if n < N_steps - 1:
            psi = LU.solve(U1.dot(psi))
    return snapshots

def solve_CN_full(psi0: NDArray, LU: SuperLU, U1: spa.spmatrix, N_steps: int) -> NDArray:
    """Return every state, including the initial state."""
    return solve_CN(psi0, LU, U1, N_steps, np.arange(N_steps))


In [ ]:
%%writefile reference/glno_shapes_fingerprint_shapes.json
{"version": 1, "ritz_version": 1, "identity": "edba875cbb0ce818d5be24cca3f9aed00b07856b4cedc60fa64215d309b29a4e", "basis": "ritz", "k_eig": 700, "shapes": {"000_sphere": [3.5466464186381463, 6.093694979253594, -11.814724591221768, -2.36012903195795, -8.61799593617241, -10.041789954367982, 7.115071145632544, -6.406707367888949, -9.889978402105285, 8.263098135344924, -9.04939060273436, -15.681301320795685, -0.9213374258232334, -2.218229765212536, 9.86643849039324, -6.636453651982074, 1.917213637465609, 349.71429182204105, 698.6963114346307, 349.28262857698013], "001_torus": [-3.5456123533197186, 12.956359237288968, -7.00817159775877, 3.1520815291362556, 2.124221426330785, -1.636311083879466, -15.69859700169699, 7.558782176156052, -12.683275356083167, -24.976885666453608, 7.195103304778811, 8.731507080260268, 16.410806690132468, -13.125154998560806, 3.954813177655324, -4.2407523233914475, 1.6508252686626868, 353.29482117020655, 706.2781117057067, 350.60067466167527], "002_sphere": [4.424618797018675, -1.2589972793487165, -16.502328821342395, 30.0130026286189, -14.179094985128339, -8.947808541655602, -4.013790179579505, -4.2471516840028585, -8.958506246638292, -8.616314510270701, -11.50964095448831, -3.987446576122912, -14.76870263239479, -5.23904175102115, 5.73580708718988, 10.062318630615643, 1.8709209728107683, 347.0029652208124, 697.5128222281387, 349.2919480402235], "003_torus": [-11.832732963202005, -5.560092858976201, -3.266473699356851, 6.053904926375425, -2.397285246733979, 4.350524086593241, -5.953915852532441, 10.649577999219659, 3.09094094700502, 5.682670863410931, 17.184538170460634, -1.5420472177426623, -0.022345253454493452, -4.87617914549236, -9.599821357491816, 3.941824297370119, 1.6362933812349165, 355.23050413156597, 703.7933272362238, 350.5394596136162], "004_sphere": [6.732887477636352, -16.679143731737362, -6.068000261552514, -4.5815450226042636, 10.650138858045619, -11.023367953261545, -13.676046660358107, 7.103435664237391, 4.989421094220859, 6.331833582409374, -1.3526311013199284, 1.9935521931715832, -9.087721411525479, -17.297958085472853, 6.588623970042969, 9.851751899492264, 1.8203613203181233, 349.5462246846122, 698.7186602766039, 349.3259965647495], "005_torus": [7.039964702838979, -1.2096522571729773, -9.562613056622919, -18.006800372739765, 6.735899587980718, 4.00792228658324, 0.8095122632959901, -4.609682200337204, 1.8981887121191185, 1.5631120714255196, 4.558922076576691, -2.7859334427328175, -4.860005907673177, 7.662201725366854, -1.0430153372404045, 0.1016372446122773, 1.532360103129293, 346.80817812766514, 701.2146407971908, 350.51337294445557], "006_sphere": [8.916573679348925, -1.9339493542045654, -12.030750840264156, 10.990412315365994, 5.874704803299607, 7.693848130795175, 16.03154543969124, 0.5906078938180621, -0.18003301999535992, 24.765678810077397, 25.261043374095415, 4.36892099493703, 2.0358043173913334, -8.154184232384644, 2.7293309938100965, -15.452234300404779, 1.6698874319045696, 349.55352103965515, 701.7322710995371, 349.31397307648365], "007_torus": [-6.348928177962587, 0.8582419689694536, 2.2836731031136415, -1.962803210955782, -18.768976153491778, -7.733627747764729, -6.275696738445625, -5.041661059231177, 18.605687222306848, 6.010758271042198, 10.216741335739707, -8.949290276617587, -11.014555678550574, -2.0470025485591403, -1.3652040790522935, 4.002108955980701, 1.4192675942618882, 351.9482605234423, 702.8565842426589, 350.5678393789484], "008_sphere": [-6.938782089669877, -0.5680862607286015, -9.28017203738417, -11.277791434740358, -0.7936181766373416, 6.8885515408143405, -11.967965568909392, -2.081483315880065, 3.9467840366380713, -7.211107294168681, -1.5200415214396148, -4.3146543134634445, -0.762129552725809, -8.329519643965128, 10.672708529304224, -13.157215277898993, 1.9430831223059908, 348.8768528074517, 700.8692005748285, 349.2943356872492], "009_torus": [1.4515591262683059, -15.893360863310606, 4.532956175165268, 0.10212198773550174, 2.821289346204228, 8.286967281035917, 10.036387027838494, 0.1048099550991104, -4.837899663044175, -2.298635835187329, -13.128639758525704, -8.991683189734818, 0.1428247595300487, -6.491093459100931, 15.470667221978808, 1.4834207821902496, 1.5824406857518531, 350.106288622024, 697.1675188541686, 350.4368854705948], "010_sphere": [-13.194980120008019, 6.944096964182231, -9.881320940851332, -2.5008985009306413, -6.977206292209733, -5.935255502138219, 6.336005499189467, 9.715250055922223, -6.84614341661472, 1.3801667599123744, -8.662087968529402, 16.76930589868367, 8.722699525468379, -22.13104076170259, -0.1938131030556387, -10.890889896378432, 1.6796481324731356, 351.0940179889504, 704.29327334576, 349.3410456140161], "011_torus": [-1.6183407209373826, 6.067813439376447, -8.253716940964388, 4.263848642105582, -4.550239301397021, 7.871573780014572, 13.601240035999954, 7.097220027812427, -3.476221762293817, -6.628037849916799, -7.370972335257889, 12.482784760261787, -3.203312775761087, -4.728545272326631, 2.2618734493798867, -0.05547284024462562, 1.5529584345547007, 350.96840654965865, 697.6732622123055, 350.43314152466917], "012_sphere": [5.570865236780279, 2.835125268634936, 14.710088106120509, 7.788687789599132, 1.2822813879141446, 5.893108168103222, -1.685570039292244, -13.186008364133706, 4.624189039857021, 0.7433368508690936, -10.747023879742086, -7.04451302591964, 9.28105945742977, 2.4786911409838286, 1.1164828679251149, 0.5770705858179532, 1.7931216364669673, 350.45434536765225, 702.4457223760103, 349.3216793042468], "013_torus": [5.628983022807761, 2.629745975803624, 10.740062258605844, 3.043251556392292, 6.491425965902652, 1.0539852525594147, -1.4485781676044693, -11.915444335395087, -10.153050897439254, 8.166388926703934, 21.778097040230897, -23.234259599119053, 0.23279487791767695, -19.91164168305808, -9.205081330210072, 1.895965409922903, 1.5635639837597202, 351.16251177044313, 702.6840003266745, 350.50807838318843], "014_sphere": [-4.403086417460058, 6.0071894659726, 6.227820299130386, -9.33171307543363, -16.01944744398254, -2.817251543048399, -19.810566505126275, -3.3228139532659124, -2.332418542771267, -4.484880691586119, 2.0707998648841706, 3.870406681621212, 15.925846503656174, -2.1379415680949503, -3.6848059428235715, 4.290677770898769, 1.7795182481340714, 351.4783275032611, 700.0231742484766, 349.30815933018897], "015_torus": [2.050491182611015, -11.499073224388288, 8.312911317512091, -7.969156878329635, 6.408487988836953, -12.377212438330309, -0.9642420891957189, -4.416099933215261, -16.648773059591758, -3.6121360502347333, -3.0566666367566646, 5.173320908343199, -16.558249866228614, 3.207603799983973, 1.789685077228901, -8.659363746021592, 1.5649438469453505, 349.4491519895376, 701.8755086602382, 350.5067550231889], "016_sphere": [1.3153495572859248, 11.125530155668018, -20.611703585784678, 7.0098272811147035, -14.113177075648355, -7.696005629610887, 0.938365341232386, -7.371330848769155, -4.2652660225421855, 10.846408045255549, -0.813011151025997, -8.832656527698584, -6.80136268512741, -6.541468127771571, -1.5930038827422617, 17.026768301607625, 1.8346049849318515, 347.84863037197147, 699.2074919305866, 349.3035747921385], "017_torus": [11.81983881186579, 4.47885237548605, -6.670902642898522, 5.227041216854769, -0.6710964574406697, 1.3133934878529505, -17.207007774705765, -21.95787838667647, -10.882227782722117, -19.597111544916718, -6.142071376171369, -5.327555751627569, -11.995294620181546, -12.004637189246163, 13.7028879594107, 7.959116344018401, 1.5334742808037813, 352.0000477537383, 696.4473352049083, 350.44248107688924], "018_sphere": [-1.8395177130608054, 15.564821440963893, -7.530304180786336, -7.499655296577819, -3.9262863012826337, -3.08860058399293, 6.194744558411495, 9.969476349795212, 0.9108154471092189, -2.398728736468771, 4.576363516162736, -9.13127546270818, 3.2289824458418916, -2.3706704030793055, 8.133871911577767, 3.5615753168368705, 1.8146351341530884, 349.80488409074763, 699.8442503550193, 349.3406010863563], "019_torus": [-2.207275338345015, 0.5854404333893024, -7.067956910392105, -5.16536995382187, -1.959121642618963, 2.7310959938367096, -3.213163207469765, 10.892261489892645, -4.488922632107611, -1.1782868354876435, 1.3142146410747917, 0.656740272131439, 7.15353333235902, 8.301560939036637, 4.734016719518232, 6.517705465428742, 1.4467046202481566, 352.0932823799877, 705.0049020929869, 350.57130070610634], "020_sphere": [1.7420646964876627, 2.918531951144406, -0.03749055502191112, -4.412346503315814, 17.317273377830908, 7.855634963894764, 1.9306661601731099, -5.188772290297215, 11.83906839316797, 7.802703810135091, -2.614993089918304, -9.556419206858815, -3.594733956720672, 2.976771878926972, 2.583730720665537, -1.5753308266277717, 1.8241737295865468, 349.3777821866705, 699.0131286208268, 349.3324627245976], "021_torus": [2.2872228456145813, -6.411230753350271, 4.1564503185412285, -6.995929248304737, -1.2440441223324439, -3.206093763288072, -5.968540530774658, 11.97647326500257, 17.04336835613226, 9.712564284995691, -5.126551297567452, 4.159763195411797, -1.9236473267103298, -2.8959367748969456, 3.092602052012213, -14.934460382471064, 1.5089787924947313, 350.5249455019656, 702.8405496727468, 350.5061491190835], "022_sphere": [-6.560437674043152, 8.505665107020118, -5.425772569197521, -4.742294359704843, -2.1487436708930785, -11.126107275793279, -2.730056087415217, -11.147126454466157, -3.8802615639096585, -0.10786941824354646, -3.5768330908399113, 1.5700566641691143, -14.585317463696718, -2.5236802203236683, -8.020233888504805, -12.939907853441165, 1.7787355847043842, 350.8293125201693, 699.6890088175738, 349.3649587634454], "023_torus": [-11.205492668592479, 13.90205102937443, 2.565250090193132, 13.038942665941644, -7.586512206587181, 5.675795658435246, 3.8480682825665085, -1.312957625239879, -4.503718023283292, 3.3716375009719046, -4.92442504727122, 5.3033575496386085, 1.6164160045173455, 5.730538825195502, -4.892317926036193, -18.046933382390964, 1.5389051711623227, 349.46790667672195, 699.5355951167764, 350.47309603483905], "024_sphere": [-5.52768625450326, 10.221301724558067, 15.61068162437333, -6.556543478068966, 7.071342787683499, -14.051243213624879, 2.9483439553365978, 3.7584631154015113, 17.223076539244897, 10.263134964516997, -20.613806328236166, 16.455220568806084, -9.531336834880651, 14.940824648260879, 0.338113488524419, 5.937482689844906, 1.5338378888431556, 349.66846051347056, 701.9756768652352, 349.3853799257018], "025_torus": [21.792089751436627, 9.812445731130449, -4.015848017927052, 24.495103536232072, 3.1994078324226436, 10.317479674829821, 2.3396568781097464, -12.452402279640046, -10.22601859545773, 2.430367711708695, 12.57522833091049, -3.3979681520717264, -5.919405103745311, -9.188378239548035, 8.839285324135256, 8.009648422429999, 1.5274910717130616, 351.50215736513877, 695.8658524883754, 350.4619835839268], "026_sphere": [-12.001819126173338, 10.5439890090448, -1.160921584841328, -12.86456065933414, -6.763002208503101, -7.548650420425796, -6.558158419980816, 3.3122837886953964, -0.2755917123753324, 4.272475339659761, 16.274490295780218, 9.805412870106387, -7.9610178314964894, -10.39116927732748, -18.328138902008323, 16.28732965447033, 1.8676094917213615, 349.85424352044123, 698.3886238893011, 349.3155514117222], "027_torus": [1.7020935093992224, 7.074196437129145, -3.78581955826352, 1.441451876270091, -0.9290180292561505, 9.26908853895116, -1.157850299015068, -11.545709381692031, -0.2831464471214611, -0.9765155701460029, -2.6746030310526563, -8.421403745254228, -3.139722305315221, -9.157081703778115, -3.086596433814539, 0.8931410905445811, 1.6256858658560407, 353.7730651617102, 708.5716317696197, 350.52685259043125], "028_sphere": [1.4418732469988065, 2.782303396252723, -2.6619545878755195, -4.118298893692373, 8.868330779518152, -1.6859332996165928, -8.421877127055376, 4.054543230290072, 19.34683134688648, 0.8057739531881314, 1.9989914503576731, 8.145373778417436, 16.43637461321299, -8.012955653078743, 0.4603677690086132, 5.835341031106419, 1.6662028220663498, 349.20821464684406, 703.050993873235, 349.3886547580606], "029_torus": [6.830259800619623, 3.126677237006151, 10.88793310965521, 2.891811170894605, -12.214126597799979, -1.5239981124758708, -2.449482822390082, 0.3320654512481189, 7.116640643431509, 1.0324899141268595, 5.544366031232857, -1.994249885170815, -4.158471086951517, -14.197633454614044, 1.5679489085152802, -5.726424143684652, 1.4330970004955923, 352.7205033650005, 704.7591718385111, 350.65003129547955], "030_sphere": [4.449917398896592, 4.527114815287133, -13.1887163309011, 12.28734803073612, 4.1938760994608995, -16.2808757644354, -7.250052600312262, -9.319484768353636, -3.8748353412011647, 0.8169272057568033, 0.77690731958337, 6.816445803670419, 1.186405613620412, 10.857179135787746, 10.661268093517387, 4.975235099926751, 1.6609323841021895, 351.7400848288613, 703.1676657079503, 349.3521501620795], "031_torus": [-8.120496140525585, 4.568036655963864, -6.006745910112649, 9.75164156602058, 9.44267206485652, 9.515596902908777, 1.115572237689026, 4.32624239834509, 9.101907081311925, 1.9133904560745796, 5.499042270016751, -5.8234468677727484, -7.66038712026785, -0.015790864201427723, -1.8274402226102575, -6.145958685316961, 1.4500701945490169, 348.4386168237431, 699.8037779961277, 350.5007409886945], "032_sphere": [5.416309912292407, 21.417540563140108, 3.8067318878091, 14.19392908391671, 6.1798234484014944, 9.032894629864657, 11.673645342230351, 1.067024590963407, -3.844211426025722, -7.7850600808471935, -2.8664002103515003, 3.303866706669451, -2.419660017053907, -2.271097033289611, -0.7866208844912821, 8.301776279130937, 1.608991265763195, 350.5554193972116, 699.5847332697447, 349.36317968002703], "033_torus": [0.5053326906113802, 7.157794026686369, -1.2440153904700426, 3.2529284432580767, 8.560175249840812, -9.782864330498747, 6.704042748291658, -1.7277488666864342, 2.436627310628374, 9.802687536657512, -9.853803187065289, -1.1751721986601957, -3.4958555395007123, -2.2150919694450697, 2.1333468224832366, 2.9433367589229977, 1.5525080715849615, 350.8971447977447, 698.1485642012001, 350.44333539742047], "034_sphere": [-2.6401463634752655, 5.593518169471491, -1.7928253026931527, -5.919964952737178, 3.119040042860279, 11.62551947248091, 0.033059110862924346, -9.56624253344232, -18.01513862686904, 1.013210483963313, 0.12547213820561054, -6.064168153910263, -19.003916049769497, 5.0305689052364, -6.791891680502376, -3.4154436148470024, 1.8243572488235509, 351.31548926440877, 703.2558802555517, 349.2932659113611], "035_torus": [4.348355864934918, -10.752705111958653, -7.460925364633432, -2.794206777920407, 6.323916068110579, -7.251916286688496, -0.4778823875415732, 3.386783705747689, 8.579969063642412, -19.722608652323267, 1.9924452833173456, 8.301587414028598, -5.080125238581711, 1.8194453195786404, 4.11368219025964, 6.573847382179627, 1.4723045249545539, 351.20575403533445, 698.1353377224824, 350.61510808853865], "036_sphere": [4.57186669518955, -2.0567658189202795, -4.121036866857164, 4.366659826667901, -5.141720344834966, -5.1739750812510135, -9.120033749340406, -3.2660522146275754, -13.566398778946343, 3.932146391390515, 0.041494560467625785, -0.02978942403005913, 8.119399020481325, -2.1231548936668867, -11.842767877173891, 3.2006154282194217, 1.7578193560667856, 351.27261476557715, 699.3514855692663, 349.3613412186443], "037_torus": [2.9922607211118653, 2.9984227706645816, -2.0872773613400057, -2.3797607214483154, -8.468893646337484, -3.2350199834798876, -9.187284162690602, -10.448668903048898, -5.171688290498588, -10.939324649825565, -9.65732848629954, 0.3362047627097954, -18.20059779540287, -4.370289178305262, 15.794420049390173, -11.320448887325512, 1.5306902733746983, 348.38299644669223, 700.6207189599371, 350.5260987763605], "038_sphere": [-5.532161656928354, -5.290614090286612, 3.506924165748506, 9.261635746184265, -11.569230502230313, -15.785594840197795, 5.069206552993322, -2.891853439311675, -13.091288360596904, 4.1507312217740235, 6.215078703713747, 11.776013796681092, -8.840451800009093, -4.197924566842026, 3.515618872671816, -18.932563643498934, 1.8010488103122102, 351.1915666030215, 699.209670867315, 349.30611716253776], "039_torus": [-8.08319935157315, 5.404389219761955, -0.8299220941984196, 6.844796838225426, -9.788285610429106, -1.0746990566764199, 2.3266151107745747, -1.8242393449280945, 7.187283090516057, 2.1947219699408005, -10.98118278809308, 5.682919941928149, -7.818555273506033, -5.958914887123768, -8.613314785622498, 0.336171748183963, 1.6438962209870258, 354.39254153201824, 702.595678676977, 350.554568583426], "040_sphere": [-0.2739571451299829, -5.759418129001027, -10.22099900859136, -6.720327433195704, -4.885360123117815, 4.4122060866764325, 9.545794881639551, 3.3281256880638814, -0.9198398328369768, 4.227029193735559, -1.0554170160347887, 7.052152369843515, -5.54458285770882, 3.734788317638419, -21.684153199579143, -5.38117917649175, 1.6623919869634536, 350.47630199589474, 702.0645454095536, 349.3435837643235], "041_torus": [-1.7081268463042543, 5.491580236779466, -25.191660549218845, -10.30884776545357, 11.3948133877753, -4.17563560282797, -5.114184218073686, 1.499221477224702, 15.282192251949542, -16.023013372434647, -4.226094267344541, 0.5779778345622453, -20.229569284398107, 7.537737963345421, -6.049233145903381, -12.854974776991217, 1.5069407413104778, 349.84772241410457, 696.5479171270017, 350.5226910065094], "042_sphere": [2.884128153074874, -0.9281697523774506, 1.64143568371808, -1.2287508116591221, -7.638776361298105, 0.14176123407071062, 2.0988561314258725, -5.703000877379827, 7.153235422097046, -9.569048729962708, 3.1273179867981424, -11.401774526890076, -8.746800259035151, -11.707798143308441, 9.066646962525262, -12.135340871064908, 1.647867229106747, 349.3106117373086, 700.754530526194, 349.34709682145785], "043_torus": [6.217236855106244, -11.377915016957427, -11.158602976128552, 7.237544421025922, -10.212356316182452, -3.3964940024775823, -5.051719061621447, -2.441159897637214, 5.441518021150753, 4.982162119437848, 6.414857637448192, -0.6896532802338009, -1.1191733044657575, 1.8472620809554838, -11.928300080826691, -5.680391446925553, 1.5694670276936502, 348.6797929623924, 702.5294197229737, 350.4648872120868], "044_sphere": [2.2737911053104924, -0.4900364691894725, 15.196842745021893, 2.948596166341015, 3.968661953471698, 7.458096698021571, -9.82549502370488, -9.403131831197983, 17.34968853983431, 1.2076134670274525, 14.711985298886235, 9.83112488599059, -10.455834987039985, -22.957604154077718, 10.93232210186714, -3.023590135537162, 1.8185653394489898, 349.93608838797905, 697.3955799868942, 349.29905892057224], "045_torus": [9.1193962646444, 6.82176551053038, 0.9413287085811266, 5.137851148169094, 6.822598914179083, -0.7571707191126094, -1.711060268216385, -18.502569236329247, -0.8168843843125995, 1.5351902945280034, -4.689953389667209, -6.339003500278103, 6.253821089644769, 2.2681071279556444, -7.071452343399322, -1.8382635471819144, 1.5019720199147866, 349.140736517008, 699.4832538811113, 350.5992818860278], "046_sphere": [-13.66159871729451, 1.4593049168881134, -3.2594914147492364, -10.954862562707445, 3.5391647203458056, -0.6273343002006246, 6.496788431906792, -2.72378410024246, -1.8331694960565148, 13.979816755273342, -4.220429527731643, -2.965348310785834, -0.8886511032315254, 2.793756516038224, 13.823919584318238, -18.68220576820716, 1.8200628678667081, 348.72856582395525, 700.9576110449959, 349.2819999378687], "047_torus": [0.1514899887525738, -9.386810689879912, -5.038975215781534, 1.5979859390934754, 1.1106101610241623, -7.837147116961422, -2.5542734103774367, -5.362123311956111, 9.231972519647835, -2.4159573823338985, -1.6571099405037248, 4.613097736054629, -0.016562578249220028, -0.3574451395165017, -6.035504019400009, -7.821230328320079, 1.617299954520337, 352.9751198176275, 705.9113025580195, 350.52781194794244], "048_sphere": [-2.2774764328346886, 7.62353043013607, -10.026172697841485, 3.0127238599301984, -0.2775466826675901, -19.68564115713335, -6.0685193293755635, 2.9343304683294105, 10.492226342742207, -7.40784475081404, -0.31628336507140986, -1.5564828511414242, -0.9548580299401852, -1.9609175242185453, -1.8995896527409708, -2.465887703906587, 1.907132425780545, 347.1424965928207, 698.1635296548212, 349.27038441715814], "049_torus": [-10.908345303434084, -5.760745492589516, -8.927277150918451, 0.4914892454767221, 5.376526131502827, -0.64486466373274, -1.3352124661414484, -2.607798304119499, 10.038123347570473, -0.36091132951963, 2.8342620210485756, 11.317727366952841, 4.1172597144878065, 9.298768734380708, -7.151558556056951, 5.602293224398821, 1.5602536149339694, 349.38906299975105, 695.2007136048815, 350.44600238732977], "050_sphere": [14.38250035547061, -7.657376488012002, 26.173733370533235, -14.37005597183832, -6.430001760596176, 10.221821652035661, 7.869625730150185, -0.8384830160702048, -2.099905614027519, 10.172651512053282, 11.899714187242772, 11.571007465595443, 6.300933819026828, 10.036281642517295, 14.660639921536347, 12.00598016972339, 1.8549800979515738, 348.97061009431576, 700.0715913832973, 349.2980217902844], "051_torus": [-7.701191552442695, 4.969871040625929, -1.9828252805846072, 2.098293191992867, 4.131351956299038, -12.366324313376698, 10.57794517486387, -1.0119952500159968, -5.300059348118079, -3.5866659672132415, 0.4853061183087486, -19.340840497950662, 11.307349211295186, -3.9907608026288246, 2.4130697635188874, 3.8129954679471902, 1.5271525494645721, 349.4053958649664, 700.3702020131653, 350.50133365159815], "052_sphere": [0.5122724231674773, 4.0048366070210335, 14.358416685625915, 2.245392604410651, -4.21992165683221, -6.221234838266229, -10.879332470855072, 3.771491899209918, 2.8472618580143916, 5.555954460495979, 5.516740753132895, 0.5161731228100994, 6.6946158311934205, -12.720793950964925, -16.299108298734435, 5.290227616703847, 1.8892399863776481, 351.4494683253339, 694.251452150379, 349.2331716612286], "053_torus": [-2.4753142853685777, 4.999859430965463, -13.06825972161785, -2.738802879393678, 3.9231398476676858, 0.9856152850700153, -5.158360977591496, -11.006260862518944, -1.3759089255074088, -8.284362563443755, -11.330771808676522, -2.491371376860079, -6.578333816385577, -12.319589973912851, -2.1335419783236986, -3.8957143397562266, 1.4028102610866828, 352.0918608860354, 705.015077797912, 350.63076877895793], "054_sphere": [12.201089459741754, 5.904864270134174, 9.471677554097834, -2.8766981049746425, -6.412713411102403, 11.709643323282808, 6.261804194203421, -12.687663741697563, -8.90580673197481, -1.39718161270341, -1.484424065188937, 2.6743906223415985, -1.7316250193504077, 8.031501655465682, -4.249788454562391, -8.347300576271575, 1.761827816819021, 350.5190476558471, 700.5413760444964, 349.3068527734532], "055_torus": [-1.719340233851172, -4.280470699424773, -3.3435300538118073, 6.241640471601178, 6.897664376800811, 5.638926576265894, -1.5161918440934086, 7.681366238045147, -0.28366487364796544, -7.686589162470267, -5.8270343046237665, -4.480469627967934, 3.13946858834707, 1.409161904251372, -1.307331355269186, -9.826086997846412, 1.5737089991573896, 350.782266825373, 701.6588246315564, 350.5049151003173], "056_sphere": [-1.4235360911649675, 4.195775457023305, 10.776052321245997, -7.448986476054728, -5.894621860337514, -6.454472461468056, 16.692208575844457, -4.713676743643479, -0.19845444311186178, -3.6364346556505494, 2.497042866918683, 2.7909752358986473, -1.128008530130673, -9.377336388760622, 13.459591677309334, -1.2017927266302273, 1.8996694752259753, 350.36857190311474, 697.5788674837996, 349.2832718699411], "057_torus": [-0.9937602960639609, 1.2799111497274347, -7.23834397521059, -2.354631267674728, 7.4646580478374185, 6.0938478692082345, -13.391478922950515, -2.4508816653914396, -7.450888912848647, -4.56875942082162, -4.613957154487912, -18.68108534100388, -12.66122547895656, -6.929907934783289, -5.02671479016425, -1.4628422430428598, 1.5787475890182776, 349.9436900324179, 696.3847433086844, 350.4196748856146], "058_sphere": [7.945712065523282, -3.055696797810704, -11.848053002654845, 3.4788699501244498, -20.326357675404466, 28.193876653814687, -12.662825025554787, -2.6653269554818735, -2.037581951385911, -7.593602245385005, -4.378930292859499, 12.703928320075335, -3.162566823353496, -13.087980073984307, -11.265350095624171, -15.903869507996099, 1.6937468546198498, 349.35365645590565, 701.2733874229575, 349.3388526522868], "059_torus": [8.113092371281985, -7.3668894774687725, 3.4479269977887865, 1.2804683009302646, 6.738502758766035, -3.2104210408190252, 1.968472128346563, 1.6757068986596033, 6.89602014488586, 2.4834975266505275, -3.476084224878731, -12.588198333013077, 3.4939805413835314, 7.633953718757059, 1.1756237921098416, 0.631082252615266, 1.4825691341128824, 349.1510683063074, 700.8472183958096, 350.57990038644147], "060_sphere": [-12.564876505058756, -24.865737865863025, -17.21339331555223, 3.6954926881784687, -3.739194571161547, -3.2428722235240848, -27.089324570728778, -8.51256978238212, -10.131509529296604, -8.053894887037515, 17.528833765248123, 17.467193214092, -4.065472880518829, -18.416074803582298, 3.0926784434787518, 13.230113250451922, 1.7103034368409902, 350.08999457498857, 699.4512817748674, 349.3127720795832], "061_torus": [-10.778660050161164, 5.412658359429904, -7.3892719242624665, -2.6908060798462867, 1.7891763056826653, -12.086846103790577, 5.410014113977734, 2.972673618029224, 3.1758980224013076, -5.056967066670798, 1.0345519969417154, 4.474947883633088, 11.488731401305309, -4.785422635021718, -1.5655044965630887, 1.6134126110878153, 1.5399195826463152, 348.86907847598445, 700.2786668969138, 350.523883201095], "062_sphere": [-2.6828348126504427, -4.5614435663753135, 3.327444605563503, 9.74981693461384, -8.363048433142739, 1.765960444904524, 8.479126643639745, -4.86648224774094, -4.344094742703782, -6.855020031940946, 9.105656636551384, -5.345217447852546, 10.354994769356392, -8.406109756164955, -1.9075000891859242, 9.970702889080176, 1.7590701376360125, 350.54805078307305, 702.8197665929777, 349.37550236341895], "063_torus": [5.452066219481533, 6.097958825947342, -5.436739145719583, -0.8626268881384146, 1.3086403072235546, 12.505630268067675, -12.815484377968351, -8.35759947734865, -4.454849856021692, 2.9812492998568088, -1.014724764148817, -0.3213237504300519, -3.285117495494363, 7.129384910312748, -7.6131016732068115, -8.34349519252941, 1.561099855343566, 350.46325679116865, 700.3066110642848, 350.5068428210303], "064_sphere": [-13.721451433070364, -21.44498446607738, -1.8220876127568442, 2.0950771266539947, -12.281488927592834, 11.377768013711309, -5.713065018551014, 3.7610427738780903, 1.384126893724846, 6.458341472167083, 1.2509837618900415, -4.263805499415847, 12.96286276273589, 9.713540148594126, 3.454072462034084, -7.219726139626855, 1.6762787293317476, 350.34148023119616, 700.0023216937067, 349.35846065744903], "065_torus": [-0.16493895401657932, 6.41887429199334, -2.865891054206064, 10.547213147594166, 10.12107474720569, 7.630925752060315, -0.06394113713459859, -1.8599492550561294, 5.779619046797956, 0.7629451608638771, -5.957019554829817, 11.085237978599615, -2.111920054530837, 4.429885984506971, 3.7340132448459205, -3.5546590919368475, 1.491086854419678, 348.4697172197813, 699.9494904644142, 350.5312588216366], "066_sphere": [-2.0100613417658675, -10.508019036294638, -14.602078783337722, -5.603160171007929, 2.2013627005204963, 6.6525597640657566, 15.456093646685943, -9.131973330801635, 4.098151210751983, -10.82824001660693, -9.376083408405007, -9.031693746748138, -6.259887717920975, 3.237048376363993, -18.14382538448237, -9.828891527859238, 1.5497316928531835, 350.81793009195275, 700.8418250167845, 349.3806664299507], "067_torus": [6.354945018729741, -6.095384839179832, 12.917071937344133, -22.752556335348018, -2.05018705940467, -10.861646412558562, -5.1793084061699775, 4.3481593013496465, -2.609897901274259, 4.1655613554892605, -4.108344555516131, -2.283563560982258, -3.8394311660883167, -1.0006702283587687, -2.548013933228561, 7.104682211003463, 1.5375793614507103, 351.7011889453703, 696.9653694208624, 350.43809962651187], "068_sphere": [12.800187032970227, -8.743048933140704, 9.687688909668381, -0.6979680174426495, -1.7673170640084033, -19.21625954045904, 4.362988347322405, -5.13391235375252, -10.374070555826638, -14.855068031273142, 3.16737700334943, 11.510069448761705, -4.373586227226749, 6.144203586466667, -7.146810911911834, -23.556324826248137, 1.737670087895285, 348.05240829180264, 703.604289496314, 349.37248536618944], "069_torus": [-7.82144954232093, -2.061282282290387, 2.224401040118784, 2.022132874400984, -2.1795137438784407, 8.885366104228044, 1.5645622887952149, -1.8265663122538673, 10.535321993905665, -6.86040031746864, -3.316372611595046, -11.628208504364688, -5.456989687520064, 3.816276431965918, 3.436808400375823, -5.253068119493214, 1.538792917146072, 349.0502022521827, 701.715754678434, 350.5306994117589], "070_sphere": [6.438968850347717, -5.712472586855398, 7.477128970478793, -14.967305804077336, -12.86959469161413, 3.352957731747429, -11.778334169474157, 5.805749901142283, -15.718641684740167, -3.0074560754602833, -0.9445492715519253, -10.902469035200735, -5.982228131147843, 17.343148392856047, -15.08136082466119, -30.26918558715237, 1.7242507096371893, 348.2578999688721, 698.4997234862777, 349.3535993692145], "071_torus": [-7.217670309929776, -5.180642969926543, -3.503178148024114, -6.2145482228132725, -14.275813384550876, 11.798833450014325, -2.4359618037869484, -12.772707337887155, 7.573594260639357, -0.11585992254258759, 1.386182586593844, -11.852163563404275, -7.452766200157392, -5.327022551927097, -4.80737987963309, 3.255289087338122, 1.6044447442325114, 352.6216399564289, 701.1183532051223, 350.5716422362126], "072_sphere": [10.767061551887785, -14.70876642703395, 2.975406144429555, -7.065825744802884, 8.583370109640965, 6.528442705748029, -22.605369340816424, -1.0853727894217067, 26.861238973291115, -3.8926645510470204, -3.9887859127701595, 1.2251031773935905, 0.7036592741058829, -6.357092406328618, -8.462911842559208, -15.096516131410542, 1.7603255025530833, 350.55171486981214, 699.4205635145231, 349.30223591217975], "073_torus": [-7.750117382413908, 6.136945503398996, -1.7792433982843527, -2.743564782859555, -0.480903171260469, 1.2747631845154375, 3.6152618750543137, 0.606352160911888, -2.6503534210952946, 2.3689652160684007, -3.1652820637266426, 18.32029321339158, -7.684154255065819, 21.573497308958963, 6.637297030861785, -2.5049832421124876, 1.5030946937917746, 349.538495015254, 698.322926687778, 350.5201199388693], "074_sphere": [-2.9035601556342927, 0.4318964035670134, 13.557767316579879, 4.712861258723578, -12.941214244538072, -13.197637087376258, -7.343936592871454, -6.677541624098035, 3.5365994751792122, -1.6224999168444487, -13.603728398165558, -4.01769191892291, -9.779125550661163, -13.6525667494015, -5.475875609532652, 3.2032333911336783, 1.8237001608453989, 348.04017529807953, 701.1533870291908, 349.3041314030668], "075_torus": [-3.182325258158454, -2.024460346196668, -4.934593804709013, 3.8358413051018836, 1.2175731573904978, -14.299558890643723, -2.0719389515927675, 2.2617442906229726, -0.780994497491351, -11.279411842173069, -26.477254502768705, -0.20891704560490787, -9.120417842639911, -6.43928853917349, -1.1165168475914617, 0.9469898185047871, 1.4809985102453593, 349.51584556074124, 700.2865827337384, 350.56858634692054], "076_sphere": [-6.492655101665568, 6.097718625756033, 25.15915278222176, -11.004608547920467, -5.266952178020165, 14.555220629234562, 11.14601111419191, 6.933639708122021, -9.321791271571758, -1.4328603721595823, -7.8710472171444374, -12.92279634638183, 10.21073116164364, -6.373755811010318, -24.724918771217848, 3.1571460807461125, 1.799838500999103, 347.95464632953195, 699.4820351800099, 349.2840295413063], "077_torus": [-3.1980481413158075, 9.462874520609223, -0.09229069323924133, -1.468484118639942, -1.1434066686792543, 4.948150963768926, 6.3392388131273005, -2.1238125855739436, -8.763765715863038, -12.477670947009152, 3.726995875962598, 8.847668051187725, -1.978544924423801, -3.991790408852548, 6.7220085742859395, -5.037884430223711, 1.4965820915277643, 347.59371117718547, 699.271485389027, 350.5376871083381], "078_sphere": [-4.866155683145753, -3.281867988144672, -13.569516953405937, 15.420803791717631, -5.0120362956174915, 16.40604241938381, 0.34331397512673467, 2.7736834224209304, 4.172977491255751, -4.600534384065033, 11.014652977484555, -7.282809764108571, 1.3887441526391369, 8.828947522606756, -0.2913048326336747, -13.204909389731744, 1.8576417910833891, 351.59543475452085, 697.0140426933232, 349.2784410663444], "079_torus": [-2.592105328313641, -7.450699602354647, 1.2583814368283646, -5.736363207677789, 5.949068231678588, 5.0529718041224205, 6.427186848653053, -11.169731642990257, -5.603041386497226, 5.3167908343991925, 1.1257375776489553, 0.4442908595485884, -2.6097817941129375, -0.7229298500355528, 8.942125462284281, -7.7144570107534545, 1.4210482893338754, 349.6017560788046, 702.0127950343908, 350.5860639244053], "080_sphere": [9.073993572051583, -10.73206085628599, 2.5419333188738893, 15.258987286435083, 2.7902543402130657, 15.63009654264225, -0.6169556500573194, -23.97577143915812, 1.2469323086918367, 0.8981085413807897, -0.4901574582613559, 9.22350857247047, -9.394277979799472, -4.63998290001424, -15.880042307156176, 2.034333880364778, 1.848808416062862, 348.99887005670183, 699.3320768645631, 349.28478163336916], "081_torus": [-9.011110046267946, -9.633507578327063, 2.7082132034014945, 7.907449219628594, -15.446570653164862, -7.393487488601236, -15.343260462795412, 8.71368579998196, 2.6611749583669186, -15.92113567146071, -1.1535954932300787, 7.167854011337084, 6.886559516864576, 4.270176391265707, -0.08712935905999952, 10.352923437668903, 1.6503740176268582, 353.27484208651316, 706.5232163646531, 350.60075436647037], "082_sphere": [-2.023557332002966, -4.668938513780869, 4.083099715502655, 0.2631531758431249, 0.6696071493158584, -10.395309253755949, 4.840828386368202, -4.051400881239772, 9.56782109529758, 5.250762965068946, 0.7779063810179263, 1.6049576851161331, -9.399512919513368, 4.893426661941035, 1.4657306817124902, 5.262793808969336, 1.7034983757667688, 349.68014345539206, 700.9149143070467, 349.3052391309751], "083_torus": [-4.660363915803534, -17.09539775749969, -3.720543189255066, -13.871276336079763, -11.14090791610425, 11.185634682312436, 0.42186736334539887, 5.675276752662794, -13.777079221769412, 4.194656293603076, -2.0131217496971914, -5.7297797016125696, -8.139212255833506, 1.710801294829679, 5.834069888030415, 7.257666034029028, 1.564207670699917, 352.37574590486247, 700.4484676873275, 350.54245814751044], "084_sphere": [-2.6996166491398625, 2.338783758230366, -9.213114644323822, 6.759009338639296, -7.039305838920438, -15.378281233893981, 10.805639889912547, -1.6647480373474344, -11.539806037953872, 2.9291255663893523, 17.232947361529376, -11.35321557796557, 22.928698127814197, -8.95534560823561, -35.04755044848891, 6.489850560818392, 1.8911277497439363, 348.7762991057846, 697.5242805026655, 349.3113493258716], "085_torus": [12.606605419611121, 5.826797338826337, -7.076126403706488, -0.9141227189072181, -3.83384645468033, -10.758540991353332, 0.5732974859592124, -4.853479878577075, 7.387050930373057, -4.1263959899372695, 6.223158447292114, 0.5736882427247769, -9.645641982048279, -16.40605241493898, 2.122574062380001, 4.7233377167393416, 1.4757513319463205, 351.0690632623913, 698.3180967511029, 350.611047198525], "086_sphere": [16.803775394233533, 6.10704938160921, 6.1087594936088765, 3.948640648166167, -6.298458907422382, -4.518400088415981, -3.287633592081751, -6.544678972355731, 2.953309229959805, -13.995348330445987, 3.8843483244252663, 4.113571447405782, -5.899606227503788, -6.297518112176467, -2.021675302546721, -12.012402238839368, 1.905016339104229, 349.7338449975659, 696.6416174694567, 349.294022326577], "087_torus": [-7.807359968181617, 2.131028611271648, 3.5931153382014642, 6.675755523655244, -5.801007597649313, -9.885090367388758, -3.4420573741866445, 6.403542700192784, -12.10737247436268, -4.136336612994072, 1.4459596662111842, 18.97523879591096, 3.9174958906213355, 9.490338601079813, 2.297274093894054, -3.5631362653267757, 1.5177023478478813, 350.0346625529993, 700.5546238920967, 350.55281808408677], "088_sphere": [-8.70347362783355, -8.637373436744365, 1.1268698701422444, -0.48078766561620156, -10.928294258309577, -8.768277438894366, -6.279501829845044, -5.231012471839605, -4.651186040863206, -12.587647524905945, -8.82134575968174, -5.672025211307149, -7.6403950029262075, 0.3058388422595333, -0.9340601079380395, 3.65092388127189, 1.7401157458591354, 349.87780766416984, 706.1736258659812, 349.3435661469626], "089_torus": [3.8811854039105267, -7.462557229393255, -0.6952716515702715, 2.815710744835499, 6.206373586287119, 4.277709516141494, 3.487438746113142, -1.3075500373015547, 9.347165063162171, -6.640708908333461, -5.042341857859062, -10.174908070639455, 0.5032128958736104, -2.2137920676219522, -8.030894334345529, -14.72695902808869, 1.6266722535035691, 352.1842293432581, 704.1467781915551, 350.5075485369588], "090_sphere": [-0.8348563391850337, 10.212658001717497, 2.3912224364975163, 12.121160654819779, -3.6489979960198093, 17.57302022469048, 2.9637156067972374, 5.954160781926275, 5.1481794325252475, -6.261933507200383, -0.28916637637789844, 1.704607306241431, 4.152068134944928, 11.570929677550662, -8.447920340134244, -0.23180453322500938, 1.7462178855825194, 348.4446764661416, 700.047983083009, 349.3524626475739], "091_torus": [6.702024831869494, -3.0948998926292566, 6.335660541042593, 5.160117781377535, 5.880900567055587, -1.1965146941004172, 6.627959265599314, 7.373727474692398, -7.368179706703867, 0.3673843274431081, -3.3043010965678943, 5.563258956374711, 7.931111244519951, -7.0901519951871155, -17.496795017061704, 7.6774280263384895, 1.444423524686664, 348.1633552999101, 700.9917594435148, 350.6586865200471], "092_sphere": [-3.958787232047177, 3.7789614515799035, -27.95674849233756, -6.025063115991788, -7.1009194830894415, 3.082662881860221, -8.924329927538531, 2.9649948463020745, 9.44019176525838, -3.9834192068027168, 3.3959715720700068, -2.6396955679952585, -4.178626288104591, -2.2893273553955966, -6.736783960388397, 11.54289946932921, 1.7655386281399605, 348.4224992655591, 698.8948095571265, 349.31925323055714], "093_torus": [0.9512784698299739, 8.771230358490966, 0.21025693011477387, -10.011514991403812, 5.575974192078071, -1.4164411197867717, 5.3604290039645175, -2.8419213413415574, -4.155100304403835, -3.0255051885731943, -1.9976243317071631, -3.726166923573877, -5.74060869476148, -2.3865608163547183, -19.76113616101422, -7.298321293562474, 1.4421543663098109, 348.57134728007736, 698.9937732246497, 350.5819801343308], "094_sphere": [1.7778814046049236, -9.080137910152976, -3.5154422488955475, -14.452824974288811, -0.13257293611547905, -11.974146378365699, -2.4940248742737983, -4.877555970948171, -2.3021115694706533, 14.49809079757486, -3.7194293278599186, 9.291948536508485, -11.141010044642272, 2.1386543050206246, -5.0727313733422585, -0.13248096693833222, 1.8755537091782688, 352.1939175580621, 699.4361064863944, 349.302120656622], "095_torus": [0.1428850715990908, 11.583985228431908, -8.459986634634268, 0.6748653807616538, -15.813015095275318, -2.1575625339894042, -9.529538149399427, 4.035901401486248, 17.340487114573232, 6.564748776830507, 6.958439703043764, 0.7504739896192882, -4.9709097415264125, -8.824527423318239, -12.321999633095277, -2.3568493944351414, 1.6167089546604152, 355.12812360335874, 705.757204549177, 350.5421006340112], "096_sphere": [-10.24630503427758, -12.996918646505733, -7.698621005129056, -25.38877825955472, -8.786655190923396, -7.5249932587649315, 0.4326845079727213, 13.588500799395515, 21.382265452939627, -9.61048227653508, -7.986987015578309, 21.874367439591133, 1.1897883030379293, -10.450495801846568, 4.106385051695769, -8.398099133486813, 1.767169550243484, 349.70169373222035, 701.2876001612584, 349.297477830788], "097_torus": [7.98468844912597, -4.1942473593555185, -3.734288129782863, 22.121133747085025, -2.7414235489544394, -20.63981171724446, -13.159182883664302, 7.470037411282932, 5.791299516760967, -3.8770066776049132, 1.6691290446878084, 5.607167842066912, 5.859714573956456, -12.029836039376871, -1.6247175207740758, -5.130466267564672, 1.607609380043709, 351.23620800866206, 701.6698697163612, 350.46960021570635], "098_sphere": [-6.386589755792306, 1.2637478639682709, 2.2122363749871194, 5.299205806827133, -2.6499925039833325, -11.323513331548948, -5.9067351140098765, -7.758253927139714, 17.297310817518742, 4.8389550583139105, 2.915593040012632, -14.709376306509867, 1.1237583933716757, -14.11833095119117, -14.627134070167955, -5.59222825860669, 1.8787836849230284, 349.77082143538036, 698.4710978222541, 349.3536955729662], "099_torus": [5.98236428399381, 1.3446703853528197, 1.6529223336260248, 5.275839229295736, 5.892382891364473, -2.2898118080108842, -8.369180406077103, -9.151332345476476, -9.917831581422877, 8.056569708930063, -4.921811681722232, -9.03703700567024, 6.277621915891128, 0.16836208909034234, 4.866828813022612, 6.001749670283762, 1.4823601927167864, 349.0800470792155, 700.9962964583559, 350.61245534473676], "100_sphere": [11.423433181174712, -5.584125833571663, -0.4016903096585782, 7.0047763563698044, -6.682930182201545, -14.035707880820876, 15.583785290944434, -2.5470469325927545, -3.8834790539951385, 8.135929526254444, 14.746844919473059, 13.685929535042522, -28.97910457504311, -14.433408924538249, 11.73410829567613, -7.482796049111741, 1.8366873493438456, 348.1993737195139, 698.6547126427545, 349.2838967588603], "101_torus": [-2.148124229537255, 6.026935235099752, 5.968045416795819, -10.423565735601068, 5.171104494096129, 0.9757784422830134, -4.510830539691616, -8.971549530051453, -2.617756528387016, -1.4736016659773772, -13.05110401339967, 3.2608246549770583, -5.089493953328844, 2.2240228136937192, 6.339481065896409, -4.4263429216501216, 1.5223678339601854, 347.8527123283271, 699.6178026709873, 350.52756642888016], "102_sphere": [2.437496214947945, -8.909322642812604, 20.130497874144726, -7.882405475994195, -7.2243027795533425, 1.7786454381216656, -9.931040389561907, -1.5294396041331266, -8.584368596606257, -3.976652654029442, -1.89368186927539, 5.818442397290899, -4.442452413204629, -4.869216208940736, -3.714706865016389, -6.8400684210011855, 1.6602548967976596, 350.4086113130154, 700.1537691458251, 349.3574520095464], "103_torus": [-1.9109777791461577, -0.4731779128614606, -10.655981548800712, 3.0329459578413847, -8.124446657417637, -11.764839458595736, 10.815481588118113, -0.39487738216828483, -2.3497306461843004, 1.1636369178973256, 6.846549411142222, -8.776589507209113, 5.88898720664651, 10.231560993295822, 1.8652295179526066, -3.1988531407077536, 1.6305551048384486, 354.0320750799201, 703.4463989522766, 350.56100449625654], "104_sphere": [-2.053033039337919, -1.9440292307868687, 4.239090092277609, -9.852419237353818, -17.32985012612083, 11.03649401617617, 7.8465365472367035, 16.417729040112857, 15.782627074616276, 6.891617720406121, -27.924787863469405, 6.217235374367365, -2.2015597611723097, -11.297363147463436, 17.15457347581302, 9.136901536378453, 1.8981208037274906, 349.4870283576459, 694.9286618889163, 349.28055834106505], "105_torus": [-3.7695598441957343, 0.1577830035817651, 8.304230803196015, 8.817462118388518, -3.7274360480497135, 7.36403752794958, 18.443576331700246, 7.569818967253628, -7.158667319091823, 5.297220321479461, 3.7791231833494243, 7.715127421771147, -8.106582485672595, -0.65407273253077, 5.312122496813354, -4.072887628729855, 1.5953236592493567, 350.77082094291455, 699.3017010314917, 350.4722199531416], "106_sphere": [-5.824279463668794, -0.43601953432942864, 1.9114517060009977, 23.439314329932337, -1.365120154271219, -8.251186486689019, 1.0416067014721158, -8.631256726169717, -6.514313028345426, -10.793536799498057, 17.176816449459956, -16.765540301708604, 0.29250325954379797, 11.87501931797395, 15.014356829562924, 5.597513274925719, 1.851505265507832, 349.6039136395298, 700.3007580967546, 349.2900298900324], "107_torus": [-0.13008698259788876, -6.058158567433165, -1.7065715168680278, 0.22182494865035274, 19.942374056578725, -0.05038319306159389, -5.454343989949962, -7.088106078238598, -9.654838571410837, 3.292224309708124, -3.319288569267291, 16.955493149066438, 4.817082884379714, 0.5050398425481162, -8.273901242249368, -10.636604525472245, 1.440490354557838, 349.8916219055442, 703.2581996809921, 350.5975517465205], "108_sphere": [14.54396547325293, -4.940557595836556, -9.390539061840071, -11.531696255527066, 9.79035730825692, -3.1899674575020285, 18.263134223116126, -20.117521825238995, -11.32404503263777, -2.433898995716529, -3.535520412499219, 9.977533786984088, 1.6358663761643994, -1.3145422360515648, -4.5668405981228934, 6.541603243191205, 1.8357010811272707, 347.4227796306821, 699.9445250789119, 349.33034438115385], "109_torus": [0.6186899115054265, -2.6859199954506314, 1.1332115261331133, -3.218718439367303, 6.289557643796636, -6.8015181801551075, 7.170159818264501, 2.9158760994291715, 1.5796664554754627, -5.100395104947251, -3.1321837578797784, -0.8505475577056361, -8.92591181353145, 5.625122940326891, 4.030279791923517, 7.056769956319642, 1.6118250252889479, 354.609036476771, 700.9176081012733, 350.5404765814562], "110_sphere": [15.795695623462962, -5.685853526129527, -9.049809235230263, 10.049816998284717, -11.761148954630736, -4.407347747091187, 7.905338692038479, 9.620772005495255, -0.4116224702898479, 1.0152940236227295, 2.3454468168339506, 0.9186552706709912, 14.229425446912614, 5.1549698089296685, -26.25590971648961, -9.36969588185921, 1.828738415427639, 348.6040038602847, 698.8676610743647, 349.2998790800104], "111_torus": [0.007504507978219888, 3.660238365609475, -0.07064047508582538, -18.45260420097641, 6.286994976188833, 6.237900237679645, -0.6059717246423837, -0.8644299384162188, 1.9136731687603414, -6.885031644556166, -0.10705366301960328, -10.429035438850416, 4.006923196787405, -6.183958733602514, 6.965468713952014, -8.354781096305341, 1.5269349477356844, 347.9356931152152, 701.0510942925087, 350.48456557798477], "112_sphere": [-9.825976140815708, 6.913430084440613, 2.257759850615912, -11.209401667757728, 6.260158288050322, 0.5779302180631691, 11.723734296051106, 11.043365949734493, -29.374276656923477, 5.476909933349493, -9.237697985718718, 10.083625678156674, -7.721731066809011, 1.4977062735557194, -4.528156420994602, 5.493575296929553, 1.6787290314760905, 350.1518421350853, 700.5939706289886, 349.35853179169493], "113_torus": [-1.7274084284580238, -6.856704042418929, 5.1959774054313215, -5.925352980359563, -12.424720041066625, -3.3356408719046025, -16.600728792041455, -8.305439729596475, -0.6754738392875534, 3.4252107430470136, 8.566260055079926, -2.892120041525763, -5.263818053836614, -4.901853437582713, 3.211496843844303, -8.657396048112652, 1.643036070153654, 352.9939337823651, 705.6854123170981, 350.57343063219105], "114_sphere": [1.2504968301149848, -3.1161486389881983, -10.983996694161693, -19.4152324088748, -0.09478032170735373, -7.975717907819149, 16.31707442770818, 3.9123354937942576, -4.196092412499956, -14.687487909183675, -11.328989292814745, -13.812401904798007, -2.979679571288221, 9.535163073841389, -0.05210500054176581, -21.248809849274007, 1.575225275420173, 350.1140553337506, 706.2768425968686, 349.4134418374709], "115_torus": [4.569635007763238, 1.1610052937032584, -4.838780802504953, 2.464189103668172, 6.434048040888399, -6.4116882532613335, -12.085883799159555, 7.225294148595037, -0.11448728313144296, 3.3402179497483244, 9.841735171474664, 0.7626772133125301, -5.586475798424147, -8.895276947611157, 1.812053567088003, -2.084121008226476, 1.6314903491726356, 351.5683565475763, 708.9263440674043, 350.62375038723775], "116_sphere": [-24.121484307632258, 5.209117347057644, -7.028144814182371, 8.71137175164764, 4.8767945585896815, 0.008385874310781638, 7.656700749777108, 2.1356278356200047, -9.795029720376569, 2.4334096503788754, 1.3107003966857018, 8.870192661940397, -4.959340692200314, 15.498530109721063, -0.6180575384846192, -6.5834463484908685, 1.6970907178957346, 350.0791507835819, 699.2003572504392, 349.34432992230865], "117_torus": [-2.8196398957205853, -1.8731835314148135, 0.8036427571650524, -8.905392686748023, -2.8945251812502915, 6.752953443263797, -6.659004991448895, -22.02874117070489, 3.316865332482566, -6.959907878494386, 0.33067024716615245, -15.989929276608937, 9.52934909237694, -3.4424618420029187, 10.630983365420455, 12.877761648729665, 1.497723232422664, 349.01573718721426, 698.6462932262184, 350.5270600181946], "118_sphere": [-9.642043798977697, -0.6043057087964723, 14.571300661353757, -6.0233221013293905, 5.666747014920244, 14.371000826620557, -12.818119003770569, -9.678352667802894, -8.58909467563551, 4.9170153166354185, -0.7686484254199071, -0.09851322758522585, 15.693685011889045, -22.021864903672856, 1.4287049296253551, 8.784393886454254, 1.9071027165610361, 351.03140023192117, 699.8065978389394, 349.2865994777912], "119_torus": [-0.0983283698587547, -5.480748878748178, 0.34185300656731776, -5.682928758297866, -2.582490435468677, -7.1694868758807715, 8.112475020675578, 10.980353455159097, -1.7443356489762625, 6.573084865719504, 9.66597983681196, 0.34477452338627823, -1.5102490848356012, -6.763114394351077, 0.6612647097695563, 1.4505529563987052, 1.4111795097492303, 352.9350309384753, 700.8076343281767, 350.58347107495365], "120_sphere": [-1.759927875832358, -8.0963695990431, 9.847628917354102, 9.93314582197347, 10.32785395375226, 4.185316341841817, 6.383609865934099, 14.759949587222604, -4.2884934892362825, -21.229933785039748, -1.5006473629641994, -2.3426940059958454, -4.423185443551512, -0.49499771054882213, 0.4665525637400356, -3.119316089219362, 1.7434007547517119, 348.12128542665727, 700.4047731463183, 349.3291814823191], "121_torus": [2.3614682878764013, 8.67744909497503, 2.4211821603412167, -11.280762908107056, 4.409945481666084, -11.070038239089133, -19.313254318341407, -12.64278249063912, -17.935090026727543, -14.73472380829617, -5.939742471126853, -4.527255242688836, -6.453028428658367, 0.56820664486417, 15.398500785370295, -9.040641185428424, 1.5338071676901344, 352.02220471798944, 696.4591652790938, 350.4422655811456], "122_sphere": [23.75634303081273, -7.712583698107805, 6.043166655854801, 5.077822565470853, 5.898060953477524, -2.5912122029174283, 3.866330549961746, 3.7438890540678114, 6.949751440728303, 20.280510003979337, 15.659989261628393, 3.862061871917122, 14.23128297287519, -6.731520917723545, 12.347697094400965, 6.784003196280755, 1.7349346033371422, 349.9352922037579, 698.9139069130548, 349.3496800709989], "123_torus": [-7.258205907364539, 6.87146243372673, -4.0252716490585625, 1.6591790806974291, 7.068760505792391, 2.070950914435656, 3.9357822726641545, 7.340175697326, -7.518956318910442, 6.211405485778582, -9.0011004544138, 4.788866555736238, 6.898175233963844, 11.886759286288312, -3.178569495359579, 3.672751667563486, 1.485214153181379, 349.4165071423398, 696.2799134567853, 350.62109708709175], "124_sphere": [1.2706483589894848, -5.2209444425807385, 5.163036847659043, 4.7928779731178075, -2.4821197884505137, 5.30514936335254, -21.055042310244612, -2.653687533640003, 8.169951582553074, -24.395555664657007, -5.37817896847997, -9.865366004077504, -4.728737767293967, 14.126284040484933, -18.526226702627184, 1.2669245818463946, 1.8069362816589827, 350.2082841246074, 697.6094928707649, 349.316678380315], "125_torus": [11.865146570830557, -4.79138242900744, 18.88742526659682, 1.8403766725810982, -0.08952633332098259, -6.9507449859332695, -11.919357107488743, -3.5605838099059914, -1.2845132175227398, -0.5196008800603622, -1.5253271465428202, 2.2207387448898945, -5.499544082061325, -2.345513918641039, -0.7888455270625709, -3.533122264425165, 1.6201442836037734, 350.6986558983841, 706.6856941105208, 350.56685505645896], "126_sphere": [0.5605037824703416, -4.751695345811805, 2.6888892232543355, -17.892751572712363, -7.690119425676571, -21.00340062430891, -11.65966685151901, 6.686318145682464, -7.211391175668495, -5.48671876578231, -1.6223424255366017, -1.8635676861111086, -10.88296775044374, -4.336983854605917, -0.20740319156572995, -19.27522662485658, 1.8448018961930828, 344.94222329578645, 697.6554111315509, 349.3028892376544], "127_torus": [-6.141327429986867, 2.2302557166798995, -1.1175355238841993, -12.733993820290157, -5.593535535369825, -21.648485767305363, 7.90656810488996, 4.303638513452441, -0.24405584772509314, 19.736234246630225, 7.434697722893978, -7.37964708158434, -4.790552354223781, -5.185152461847184, 9.663877575482982, 4.528105572493521, 1.6608839279317837, 357.6080768674327, 700.877600668562, 350.5803077728287], "128_sphere": [0.40046939628717215, -18.37723699127671, 23.078489931337725, -5.878611353930122, -0.11343183995608541, -13.874661508578148, 3.198113653336955, 1.5897081526101302, -9.400959331264456, -7.455037847608202, -3.3445114162753438, 1.4460282975419299, 17.516772064817292, -10.173880176925811, 16.76055449600744, -13.894093388971932, 1.772459391237671, 350.25494757603064, 701.0925500294023, 349.2927617515679], "129_torus": [-3.4420721535286445, 6.84953288196755, -4.040517428059834, 3.2902776763893624, 0.8668734294129044, -0.32074278069059403, -1.1605404587725103, -8.01467009701529, 4.5378912671475415, 15.246899125861606, 1.9273479793301318, 15.148401227068163, 7.145702863587669, 12.948851201596714, -1.7456098183108337, -4.801874086340831, 1.5626184913275967, 349.120292629663, 693.8609294735527, 350.44003053308364], "130_sphere": [-3.0647351629089856, 9.278617602091597, -9.466040386812136, 14.031965606232507, -0.8072746366825475, -0.2548247308504208, 1.301979098574031, 10.045359936026058, -6.857732490532474, 2.2004739754124913, 1.271473648011522, -2.792003359669535, -15.687012321710542, -12.713540086048496, -2.220824843671058, -3.2049508317295956, 1.8392311878941203, 347.6226010336158, 698.1060777849033, 349.2902681780227], "131_torus": [-0.12315340637525374, 2.812547127701548, 0.45281347495461244, -7.893650411282895, 4.201941522890033, -0.7977791230670057, -3.623264694288333, -4.756056781492834, -23.685482917609637, 0.738030951312231, 4.773904883866097, 8.46700591386254, -1.738254061511042, -4.985518250770891, -9.089082818956008, -8.855846474881716, 1.6046759457681368, 353.0084111839804, 702.0444156001734, 350.52606270041315], "132_sphere": [3.1538139011841424, 5.076702933855813, 14.388057163731876, -0.09878524106231312, 5.836489572034076, 1.3080746382935118, -8.808615498974373, -5.5311597454187, -12.176200757016954, 9.04188415485169, 12.345416134502383, -6.31026536512203, -2.3958090785088766, -1.1937551921148937, -10.452313217445848, 3.321639400285606, 1.7937960394229362, 351.1972994194624, 700.5631359141636, 349.34233266798486], "133_torus": [-4.285332639452413, -3.7835345933120843, 13.746022876360414, -2.132114675532735, 5.954410933937014, -9.21652066114384, -3.4235287516591395, -3.503954848513061, -3.3735147215836943, 4.285075026415922, 7.9134167194879605, -1.2173011102396265, 7.215675399174383, 9.909313734770944, -0.6742885746997445, 3.3071357504954024, 1.4469044615915396, 347.94532241361554, 699.2004234823144, 350.6339428792506], "134_sphere": [1.7661605543318686, 6.798308340184118, -9.589496187677149, -8.250525586363615, -6.59635234314743, 11.080701886858343, 2.1189876470123945, 0.4532598588295389, -9.593760656328064, 7.341288408000297, -11.44857796721696, -24.325864636862473, 5.851588574919892, 13.321020673806625, 0.9580158756723609, 15.383444657549859, 1.850624776577643, 350.60161873221153, 698.0529434698833, 349.26630109363805], "135_torus": [-1.823812424257584, 5.503902505481648, -12.999297432359327, 1.8151976549964801, -1.5438105918561495, 10.298507441595444, -16.111906340508234, 1.391204672605402, -1.0087728384204873, 11.295128780811408, 6.098767144628548, 5.1372242423738115, -7.79004661882337, 2.4743295475353375, 7.623721094388554, -6.426236671486288, 1.6055719065046545, 353.44495631721736, 705.1172798250107, 350.4828246794639], "136_sphere": [-1.157914878838703, 4.12927010633582, 1.3581208248183687, -12.238959026524299, 7.313628345082629, 3.709333643119339, -11.564533439354717, -22.45131860579278, -2.4416883277753985, 8.59546330867876, -6.147038329241481, 8.760657192170921, -9.795035738393215, 3.123419984248996, -3.196924302013192, -4.188813408197653, 1.8555726799195171, 350.9759627678338, 696.6232486623485, 349.29719463700184], "137_torus": [16.0984045190418, -0.3867710745042947, 4.19152812501776, 2.399413260457716, -6.684587991016033, -3.6170668311690743, 7.239669176505704, -3.7744746060808154, -13.238696689767284, -2.561656775335553, 0.2830797233297977, -20.523210668501775, 8.125229065027584, 7.103836974388413, 5.213627948151989, -11.570127094781801, 1.442197947703955, 351.2483007031742, 705.6681673854046, 350.66765326551587], "138_sphere": [-1.439148730217191, -7.423585954228397, -13.045922523724819, 15.410812886879972, -10.758467270148227, -0.49676588119563725, 12.707548162535554, -5.385345980218125, -2.2878785103086106, -3.3101638174728203, -6.629799931781351, -4.564519213145846, 2.119191597677224, 1.1187471457100635, -3.761276951760876, 7.5978799298176725, 1.8392617325647043, 348.84620078563904, 699.3660813718275, 349.3308603146063], "139_torus": [8.980196189943035, 1.6508512664548047, 1.5965921110279542, -3.1774362539430294, 10.875134983873782, 6.6441763628576656, 4.287369451557652, 4.185188295685462, 0.6941733916980053, 5.627319244035888, 3.9079878640870636, -5.315823853641185, -4.332549233447659, -6.170034161017137, 0.26181667943712483, 7.055583489665933, 1.5668971803083591, 349.94614241806875, 698.8572390070365, 350.44873046136155], "140_sphere": [10.222719190683687, -5.5129534983330934, 8.518616369447766, 5.182813247097206, -8.376543443716718, -17.097619738485673, 2.633130281388092, 9.0576357676758, 7.780207724689741, 13.018581647985524, -4.936221456891115, -12.659810086549147, -4.855140086425779, -21.59549059290066, -0.11148247817364698, -16.43759554226319, 1.7190912183791534, 350.27619641512524, 701.3535232231234, 349.39870999582496], "141_torus": [-4.6694392298670255, 1.6155802980606815, 11.889050108399676, -4.239129497250378, -8.09347998036553, 4.685370775538686, -11.22218774452227, 10.040595204279889, -12.659101809551181, 15.860840236104428, -0.40712091150684326, 4.360463363068275, -9.365842429878416, 11.571637238885726, -2.101012877094501, -6.037324551015844, 1.5211269956882916, 351.33123627309334, 699.3813219842481, 350.4744411621337], "142_sphere": [-2.784495737930996, 11.408229727325008, -16.507765021119706, -1.2862786528253696, -1.0570644920660932, 19.954241450220763, 10.021642744418312, 5.539732153157819, -1.0563286186514866, -12.141425396869673, -4.175119604602207, -11.676569833697862, -0.5326769390596444, -21.600804466373795, -4.411596942785895, -2.1805618553720736, 1.9063444526316409, 348.6250274563361, 698.3180658251281, 349.2985277756064], "143_torus": [-8.148281323948977, 10.096521053365485, -0.37367256752036315, 2.4476403843493397, 4.800437100375574, -5.393399337493435, 1.9928297607180863, -1.7969894705688811, -3.5826349556046457, -4.769034037775472, -6.398349993951969, -12.391249165747434, -1.4266025736343932, 3.376442640771721, -4.078274690431472, 17.867074990461767, 1.4628710263747107, 349.9033980474259, 700.9550544486243, 350.66301102503706], "144_sphere": [7.372837929589201, -4.346494711749279, 1.7936215023847577, -6.516923180599219, -0.8741182954942666, -12.592954635696739, 18.676323063512218, -11.14898749784613, -5.339191593697866, -0.6866802299081431, -12.836050523937985, -3.3953617536499436, -1.4151119783308448, 6.95620461319317, -2.406894379681697, 6.6746101983785175, 1.8670616187021511, 350.590799142064, 698.4005394474555, 349.2928629388313], "145_torus": [-2.9886011272400315, -7.15274122983715, -6.2332809081754075, 0.14153122758964476, -6.10225745183801, -6.219761590410064, 9.280556670172254, -10.481614351089231, -13.923140888653354, 2.9819861495183586, -1.6747771895207686, 7.689625230604381, -8.196158713126282, -1.5455048320219005, -15.292190015747575, -23.61871893162475, 1.575176112091045, 349.79018485480907, 695.690606536541, 350.4154276851623], "146_sphere": [7.806526051787742, -26.57367701017108, -7.9548769795209875, -3.02497229588484, -8.5187200999593, 9.377785803353927, -10.064025414903824, 5.710453152867798, 0.22788045313632077, 8.971638417681156, -4.891182628080444, -12.41779412306153, -3.020995866407093, -0.8585588749892543, 15.843206078888823, -0.5470268608138098, 1.8191217930177022, 350.98983476541383, 703.022110049899, 349.3604996070808], "147_torus": [-2.8423018629631525, -5.846331487053776, 1.7709673192554822, 1.7943413206655023, -0.050636521508861776, -2.6025163127715247, 6.657683919130937, 7.457318926647032, -10.006436572824235, -17.311607844011114, 4.348437919843043, 0.2913122215858772, 0.030992390164039207, -3.9737333465601274, 3.960973990423021, 4.952669686190052, 1.436931168545805, 350.6202868684739, 699.1455148805201, 350.5051179246569], "148_sphere": [-6.388660620684671, 13.762201807718446, 7.598172096605708, -9.617499249431642, 15.711423771312136, 5.116479376804905, -4.022006033327539, 5.788257036009968, -2.7930506676829046, 4.141651915286401, -0.12453520493767067, -7.737240856897631, 2.561064944776147, -4.598169456903934, -8.554666474030157, -7.532250937126068, 1.732767972136055, 350.35845469177553, 700.744062111385, 349.3397391669117], "149_torus": [3.1191389000943177, 4.177838900401033, -3.964678640348768, 8.33488017159879, -0.1411830304675128, -4.725255840306479, 14.643459591568902, 14.900340298985576, -8.082795026798237, -1.8884877535002125, 5.176106270289156, 4.718826765205614, 19.135864511145133, -6.6795118146381025, -9.9487143416878, -5.2415079765117625, 1.4359688364153707, 349.94817896874423, 702.2922413452078, 350.63121461574826], "150_sphere": [-0.6742801480294003, 2.9798929612299565, -15.73779811715542, 19.18793182329538, -7.387318602947604, 0.3821678475138661, -24.337015359570632, 3.9251961883789264, 0.26904750315273285, -16.978288336240553, -7.059846503785465, 3.695100612002787, -8.037142620390275, 12.258284752485395, -10.3218367758458, 9.454389531559269, 1.7958548577070732, 347.9193604051521, 697.7995929272905, 349.28705686011045], "151_torus": [2.8753148067124563, -3.6900354637266295, -2.5325564818744892, -3.9005422136704655, -2.722432606442671, -3.5892753039193837, 8.132273259390942, -0.013918809747143879, -7.625698159207641, -12.030450626897439, -12.978064097711746, -20.895609103513543, -10.795140175108596, 3.6465973399463594, 6.120703037984805, 11.657604518348442, 1.5347254796769332, 351.17154549622563, 700.8604519449607, 350.5424511850474], "152_sphere": [1.9232135457208681, -13.443896210394774, -3.9308027929243363, 12.41297797978777, -12.577970346247817, -17.384688805944464, -5.617762243543117, 3.569617592421105, -24.56281766887824, 7.930136409183382, 9.555345618853746, -11.095590524099155, 7.133488254355189, 14.5337554593578, 12.267998614719525, -4.971237376520941, 1.7962235243826115, 349.1796424784215, 698.7688784466256, 349.3146083913733], "153_torus": [16.33209653556213, 3.659653215446383, -8.858069497993775, -16.63462073570545, -0.3133700145701077, -0.7031278944360162, 12.493273920450111, -19.47478831234291, -4.290691700061843, -3.2145397724052955, 5.709427078350805, -8.858871127487046, 5.343035601136396, -16.96580138142295, 3.088010026671115, -10.343233002232836, 1.5773851968955597, 349.80069078432257, 696.1253879597423, 350.41673701913464], "154_sphere": [1.1730230495490974, 7.419403632604986, -2.9995593156141016, 1.3678442579775192, 0.6602960894685177, -1.3596116102529745, -18.03094342635866, -22.721750580956428, 5.040208700883718, -16.109102485498894, -6.215091181826008, -6.33836059477225, -9.444998356073041, 6.050786841317116, 17.66247979564131, 3.217625944900293, 1.8907582391367714, 351.94400983010655, 699.8781645086616, 349.2633934690772], "155_torus": [2.442380169053453, -0.31218816010914374, 3.322107782584471, 7.740161419467851, -3.525253525093455, 2.4621726919945415, -0.9945994859754497, 13.72498748606666, -1.2599899495180962, -9.23872048108245, -13.453474231673828, -23.03525883801743, 5.0674466035725345, 1.1067521694544213, -5.344678132243308, 8.876953806864165, 1.4775989071402755, 349.9018593842322, 696.1350352148272, 350.60371690730346], "156_sphere": [-8.536336351219811, 8.455604713596964, 7.425005216026499, 19.70975278369807, 1.5970646626015652, 13.41270826028949, -1.455267889852054, 0.9703414406765836, 14.61876932448197, -1.2680176274212673, 6.9808329409962475, 1.2571238273398326, -5.308763968618591, -9.934356898422404, 2.467196180624109, -25.231826173578106, 1.6777005499617577, 349.47139280133837, 700.9302174426718, 349.34555513314234], "157_torus": [-8.245543290911339, -6.456314534639876, 11.706272596903576, -6.5869585884969215, -6.421430915726843, 3.377063418049738, 1.2708037844875841, -0.13753756800370454, 14.820386079141482, 3.504213495127991, 12.175600162538188, 11.593259809700005, 0.8504952310341087, -13.862773266525695, -1.973342698507203, -3.2621108016012914, 1.4718461832158818, 349.70072194622855, 698.8331458533112, 350.57036678406433], "158_sphere": [-9.647823308106881, -1.5756273857713565, -10.259064740376218, -0.6917673956872203, 3.532482016840306, 12.653851190191997, 2.1766784265538464, -4.948902417542678, -9.129755441854549, -2.0285022779756545, -0.2629288557718956, 2.77494490482819, 0.5986988683953802, -11.330282079390543, -23.184627073813378, -11.223368237869035, 1.6604366734350224, 350.41677924097263, 699.4156930084622, 349.34417028085494], "159_torus": [9.304131061418357, 11.046354049663183, 12.303783143357762, 1.985290450515198, -8.96177994916551, 1.3974042054693834, 1.713959434733368, 0.16841713039619854, -5.120480729475291, -0.05427094896996998, 5.308507255551475, -1.197073220201764, 1.2206515384601637, 2.703421746138755, -1.7402772449968158, -2.7935743337404073, 1.4918948726723875, 348.7336999271109, 700.1212072553246, 350.5518798881602], "160_sphere": [7.212710342416431, -4.999954733263355, 0.5125276866014032, -1.316785648908267, -10.40467641966455, -2.5197775029253955, -11.961693323674846, 0.8283999301376994, -11.087653583221119, -2.559802072557977, -7.350189166714419, 9.64289441880449, -11.46967758175338, -26.086806961406708, -4.553199968342506, -1.2247133180691936, 1.7787060315125625, 349.5436930019057, 700.3922906748588, 349.3115639085922], "161_torus": [1.8603745307081458, -4.1046132588820194, -14.067413941903228, 3.5628536614941333, -4.137498286835763, 0.03705951806536456, -7.81110343774413, -12.713777646949957, 5.654759307269769, 5.675259128510552, -14.651219955899505, -8.842267594076583, -0.1516724809951029, 16.956807985995532, 0.518527015700982, -11.920554215167687, 1.4966440803464913, 348.99292014424356, 700.8768022150247, 350.5410645774415], "162_sphere": [-1.6383978335259637, 10.412786345443205, -0.9393572268891788, 6.608908950471493, 0.3615954638640755, -0.21343400141366375, 8.885678469760212, -3.532126732600629, 4.422759812619927, -7.378134253626095, -6.677705281982962, 9.2348909290276, 4.085123355410588, -15.903416182636276, -13.894012738242521, -6.830411018609111, 1.798913207611764, 350.7680924483115, 698.3969156710028, 349.3175928091502], "163_torus": [5.802765100984062, -1.9394915144114893, 10.640905534644043, 2.107607528845075, -6.709994732714332, -7.2696729265073, -5.6694242756334505, 15.168232828305936, 5.650438870490731, 4.239091937860671, -8.647391048763133, 0.46951782377730833, -3.402094397901152, -1.2730862917937096, 3.98293952658842, -4.031278959349413, 1.5500529659236966, 347.0245393381163, 700.8958975416909, 350.5474239401888], "164_sphere": [12.951763304241766, -9.350341411878867, -10.796104913798523, 12.999975548591868, -8.73966936810062, -9.241576532488873, -12.148058923981214, -16.20047688201334, -15.074427094474178, -11.789453786681058, -10.868606002238803, -6.161433730266827, -17.04672557608741, 2.287944021427238, 1.6898952456752179, -7.77369028783221, 1.6908032084113083, 350.4489392801704, 701.0393846600832, 349.3446769282703], "165_torus": [1.9920316338865351, -11.159117414947904, -4.422099539052108, -6.026908613495751, 1.888922073247854, 11.66226138573535, -1.6896710150059553, 3.9364162785091352, -3.257152290126167, -5.560568453687923, 5.170845281383686, -9.20232124995082, -1.9603796556594455, -2.7403289273312312, 5.470695223956662, -3.472802307360384, 1.4569942727222123, 350.6691960252439, 702.6153833830112, 350.5584139955868], "166_sphere": [1.1752278564105607, -1.5042276645797612, -4.096468162058064, -8.251225090177561, -7.668696017375237, 0.015617276075738928, 0.24789528699585917, 7.141749824422541, -1.2115666347660807, -1.6318510180267918, -22.433994516813755, 21.54017485788522, 1.5339770126954262, -14.840176635932469, -6.831995653165579, -6.926004479039703, 1.8241569215025748, 349.66065156073375, 701.7229184314925, 349.29575022974956], "167_torus": [6.3043283868741264, 1.2382961330156643, -8.853068098985391, -0.04063703536236085, 10.178704444344222, 1.2588102906260268, -9.166685242136918, 7.46197924123687, 6.686315430212619, 5.051484745719106, -22.0444566848941, 3.4278961826628658, -5.624647927342863, -0.015959884831856286, 0.6686900833890901, -3.4259624391669474, 1.5084933692830385, 348.72295686429527, 699.6086441051289, 350.50450406728], "168_sphere": [0.7231478402248799, 4.200311078626615, 0.5951260700498633, 0.1932585521384147, -10.472169535858665, -0.5294296254541049, 16.050090135618277, -12.299030611430627, -3.588081866241981, 5.70477590102303, -5.537767708401283, 9.422650066603552, -14.014086207542258, -4.804472478956601, 4.264787240013493, 8.801373199135098, 1.8587180081638615, 353.2178912312585, 698.1923827399262, 349.2634322035658], "169_torus": [11.333894063899567, -2.2260355256806292, 10.172890651282394, 6.582783459302204, 5.225527138424574, -0.4148067346346842, 0.7413859433714478, 11.291071606627554, -2.7290237931399663, 1.1834123342841023, -4.7795579093612774, 1.3047544175338972, 6.328611858350726, -15.917081692244507, -15.919281659259921, 0.5868250890639466, 1.64595015875654, 353.08500646919174, 706.1559296754886, 350.5842221821235], "170_sphere": [0.5588094816136291, -10.934692847807716, 1.8429794039080885, -2.9732155418299193, 1.9459730336078094, 3.283470335528758, -1.1908366352225146, -2.8821444040577333, 6.786757123505072, -8.816313667250242, -8.775618830897285, -12.681729677418714, -7.571009640929224, 2.7168926679484393, 0.3920837742376766, 13.998033443835066, 1.8770634079969146, 352.30624891905103, 697.6608521428113, 349.2827457366941], "171_torus": [-0.15304646280978718, 9.227954139079573, 1.4753798020030708, -10.731688790004494, -7.729066932640055, 9.444077964660986, 6.700305297938857, 0.9934334950169692, 3.1619665266349672, -3.505448459364749, 3.346078727023899, 1.895806415396684, 9.05578455452591, 4.04009872649373, -2.257159271260339, -2.4044322651524133, 1.513791551938492, 345.8340584654588, 698.1184745273658, 350.5175067206695], "172_sphere": [-5.867624684280554, -12.06116692929493, -4.048236797243103, 18.654966993384406, -6.532279299550311, 23.04530079458925, -2.5511746813886718, -3.131480405724747, 0.7070622910366655, 10.587522472706839, -13.18122679442148, -12.919189977221635, 3.7789629183494053, 8.633615306162682, -1.1007005245729493, 9.983938330618653, 1.7812882936105137, 352.8357549526852, 703.1837130124856, 349.31648167244344], "173_torus": [1.944523658221185, -13.372941893796552, -14.487723028355447, -2.6535557902766618, -3.707655145101434, -9.884026511635987, 5.843835887582711, -1.8860651978995273, 0.171833367951591, 2.842601253295084, 5.908721627834179, -2.2283236270157505, -5.7172346903206, 3.5553098062950186, 6.54534117906449, 1.9506867247880653, 1.6408910222218054, 354.568878092162, 702.990608012504, 350.5885714985444], "174_sphere": [-3.929412593140638, -18.999499126256786, 7.153104608226031, -0.5116663869892015, -8.178971884213002, 1.016266421882361, -2.4022536981141576, -14.870087822726022, -3.6233422099742683, -6.260268737430889, 14.194713661038376, 19.28486705736827, 6.255406495028131, 3.815304639435238, 16.299515802356872, 2.7293457312319624, 1.6339169993731968, 349.4964603495495, 702.1211810750084, 349.3619723562306], "175_torus": [2.2801205582894353, 5.554415972314471, 1.7412563170804098, 5.30216395969663, -4.2804414119653815, 1.0642061303447417, 1.8012823393537882, 5.405384399580022, -7.668119951498484, 9.735467452218257, 4.622408317970114, 1.0068600793889066, 10.581657545944855, -11.867927715828989, -9.409881763950656, 0.7293382332268727, 1.4865856913707867, 348.2719916471841, 698.590918767026, 350.5380895487378], "176_sphere": [-5.425951920676103, 3.681607306345804, 6.594797328266294, 12.8181087802929, -1.2881100100705423, -3.0622224527195967, 5.775671918439807, -1.5887663148323523, 1.3571580935764977, -8.948713134301311, 5.861393071166683, 0.42620390812031506, -9.920312018338857, 5.718523968902885, 8.829750202876298, -1.196902502536906, 1.8813209847611432, 349.61066719492356, 699.0077904096213, 349.282038883568], "177_torus": [-3.4903801441656404, 5.110301503070667, -6.56903398264674, -28.665543545478485, 1.9575859802894366, -2.3869345235649093, 7.291209024592414, -5.018428253857039, -8.739495585533696, 13.057809304581985, 2.8522854113039458, -11.494159682238923, -4.745793433611876, 5.118590901330149, 0.6771514653861108, -1.1733634369577268, 1.6415581199206428, 352.95508591037304, 705.2682263076703, 350.57827145405827], "178_sphere": [-8.677333710906458, -4.700665472026568, -13.656904490995071, -4.396569656775212, 4.911674396464564, 6.892906719401394, -2.3896946398484005, -17.684474909435036, -21.01285567692057, -9.480510723171312, 0.7608775447614406, -11.267489581242675, -12.472369294149578, -12.71820190651864, -2.9033072493248433, 11.871835972040824, 1.8434518735042043, 351.0083940383945, 697.7385216739368, 349.29520300065917], "179_torus": [2.105268181322696, -0.7272729075610607, 9.901674173669221, -2.507029340486935, 19.679630628835177, 5.144253674776721, -5.809958796417992, 12.322122043263027, -12.627030286379705, 3.8144652483603414, -9.535268893559856, 4.1319178458672035, -6.95761998794554, -0.693270111181465, 5.3944766909884425, -1.4328003123370836, 1.5080971314509677, 349.14021599765135, 700.5180879805027, 350.53597376934283], "180_sphere": [-2.8048009594501844, -6.614710550807539, -17.17670635363219, -7.747802489081217, 6.75732825444712, -0.6191729654207316, -7.7810857406933795, -8.899167604748062, -12.043875502588932, -27.199819403248668, 14.110215537666557, -1.18591516633189, 10.77925069343444, 4.735952344431324, 9.326572491608928, 2.127313352670513, 1.8114589029854344, 347.76759647414104, 698.5495414996994, 349.285227936152], "181_torus": [7.960495651518354, 10.368304480974555, -0.5818537320291701, 3.410560207389052, -12.538107210959634, 6.776938917350561, -2.1806654464002975, 10.371605780685956, 4.47387997256425, -0.9789657204603421, -3.8946351440554388, -19.922130286187873, 2.198279387437542, -10.45322288836105, 10.115513330416537, 7.239739992241102, 1.5919865449660424, 353.7532322627016, 702.4564946910513, 350.4905120579265], "182_sphere": [-1.8743664040450696, -9.638340374160292, -7.248427153647781, 23.12897703176866, -7.5924489865687095, -1.2737269149356614, 13.236179784131478, -3.875383943377699, -0.6717681417135148, 5.006524881418651, -4.492147577495096, 2.4015226606482196, 2.7653145600118783, -9.73801943755871, 8.672262538967376, 1.9720765783607304, 1.873946281968855, 351.68654095527586, 698.1680930255352, 349.2643818634875], "183_torus": [5.504423310161657, 8.27514802257491, -3.5255313009261497, 7.8581822609083805, 5.558737604032991, 6.166449387251662, 8.544973248717888, -15.274673044693506, -2.6143848735373085, 14.8074131075536, -2.6465720892023934, -11.817770329431134, 4.387227998107583, -0.32853812519675696, -6.959243289592505, -15.250424367321735, 1.5013427062320779, 347.1356492298939, 699.7056221558538, 350.5759593269742], "184_sphere": [13.342577022819963, -6.146529677422848, 6.5572142041211965, -1.1523286048773187, 8.471437027880883, 8.124404035470794, 9.274269190425382, 25.94836004919016, 1.6599116775435654, -0.5789302066709023, 7.395335804780503, -7.224221746145899, 5.274763770046337, -8.132703580467249, -13.894087129472833, -16.840718697035502, 1.7032444322227318, 347.51655938147906, 700.9824152022816, 349.3188353408311], "185_torus": [-4.487376813849387, 3.840274909899952, -0.8913930314139966, -12.707172101318815, -5.38905410094688, -10.653393028332374, -0.24842482802490107, 8.746055398624213, 2.820294575598714, 4.512174188055206, -3.2308728176017856, 5.0013689275294295, -5.314103767291056, -1.4917257960884118, 13.508991742552817, 2.254164512655204, 1.5570152692706336, 350.0306675640714, 695.0664858553278, 350.4518388927234], "186_sphere": [-5.469525584463012, 6.205567172303237, 2.1288833526910533, 0.5651926644175749, -2.409909335012705, -9.463627527112624, 26.366241534319173, -1.9287602047528236, -7.076782461174854, -2.00321355216214, 3.5022031146210564, 15.956903047721262, 11.179678295482617, -1.378880478819056, -12.948358245300819, 2.4347792460631883, 1.6581176623817202, 350.3904558136235, 698.8547988246424, 349.3163481017556], "187_torus": [-2.003127259049518, -13.672062472990547, 7.007553127205115, -3.1359412500730888, -4.869381512524475, -0.498000627042538, 12.870216020998754, 1.4268248535276609, 22.069231773234065, 18.033046712613004, 1.5748179121665578, 3.044796019568893, 5.198489815349495, 6.23085839320823, -9.58499837039007, -2.6073252415259374, 1.5743253168077587, 349.4395189865128, 700.6109656978833, 350.4878791029079], "188_sphere": [5.58049217023415, -6.573321386115186, -3.271336350079358, 5.692650924838739, -0.7132391483280793, -3.1137057106399446, -4.799107523285994, -3.9934449822959976, -4.630676720822898, 12.930216130943776, 1.1364407919115571, 2.7705268968427825, -11.501079398170207, -1.8506472199804325, 3.8797104263056212, -9.906071910510954, 1.911981411569068, 348.63955152174685, 698.569416740817, 349.2791376393953], "189_torus": [-7.065835150542492, 10.7360307896694, -6.802511826698943, -4.008666987956527, 1.5567128116444042, 6.104834517515185, -9.359387692897691, -17.728896731867493, -6.919784729343616, 5.447343317553756, 2.83674190282645, -10.566173252501624, -7.724282259031022, -8.054851919827934, 0.7192150614483066, -3.993592023917116, 1.4179273380380586, 350.16672369365375, 703.9447759843026, 350.6468363925543], "190_sphere": [7.3671273862429185, -2.3252202657170633, -0.2799583439050899, 11.660505125855572, 4.212775292173932, -3.814284415655842, 8.352112655781024, -19.568445055089942, 6.254977611144254, -1.2489460043537006, 11.60727977106421, 8.140436333120217, -7.758384688641741, -2.946734528894119, 4.306072454683795, 0.2826911082553347, 1.9443415685139034, 347.97720030118506, 698.5315480355605, 349.2960222630765], "191_torus": [2.4147211954240744, -10.597733679048536, -5.451001273820347, -7.457827235760808, 6.384666253214176, 0.82945437726756, -0.8671712062118297, 1.8781047984949106, 8.45431945199389, -7.930259673895274, -2.5681975779690935, -14.9952863124103, 0.5341531831576172, 1.2891445709531522, 8.729957118633907, -8.477561641440548, 1.4919342692665603, 349.3944072106897, 701.7082450390342, 350.5417559788296], "192_sphere": [-8.75725631120208, -0.5049529767987322, 3.6106490691736606, -15.696038392162334, 8.000765605756147, 8.747755283594302, -1.5225348497849476, 2.1903272267720784, -0.5008873690030056, 10.246147808640389, -3.430089122449324, -12.315468737140595, 11.65364499464088, -12.146498358714087, -9.536549980578092, 15.524181852104086, 1.764144698212188, 347.3602904971033, 698.5919432870844, 349.3030710287139], "193_torus": [-3.1644874444400664, -1.4082168687559786, -6.126786010782319, -3.057340786918066, 9.19527443849226, 3.3744914771639243, -2.425840050001767, -16.169015714158405, 2.3202388203976323, -0.6950365730873184, -1.1935473590920598, 8.527498149376235, -18.109422723787954, 3.9525250329672126, -14.46972973337507, -0.6535602188052643, 1.575460129232324, 349.7983381054185, 695.7371874384235, 350.41588386595663], "194_sphere": [7.921498625341263, -2.0406313703184153, 13.722639449087092, 10.680164994322228, 11.013447197299566, -1.8726320236725762, 4.611543026309434, -0.040134573015076835, 7.120196004133813, 1.6770902039392617, 25.712530545851187, -12.426727868759748, -4.932075778795005, -10.526328113712196, -23.46644525319049, -19.819901519475444, 1.7874617680845353, 349.80762374501, 701.092404143585, 349.3112578621478], "195_torus": [1.049287202111211, 1.9612832925340271, -2.3332577255924, 4.279865423801679, -5.768403524485548, -6.945134963121206, -1.2681408936517409, 3.9357077613362357, 3.0228511289405775, 1.343492034178983, 11.70696936098037, -0.655854073096882, 6.884999369010613, 0.47700582063559605, -11.569493764550328, -1.8174708756715559, 1.5098125093347907, 347.31578968762096, 700.0088030979346, 350.57859943628875], "196_sphere": [0.926696480950179, -16.036065128617686, -3.159918067022917, -4.210371178751979, 1.5616759135019047, 3.1087500574095466, 5.000060577718706, -16.508311066757013, -7.729330764815579, 3.239246246073005, 2.5263650717384696, 13.091229104637508, -12.198924522362432, 1.1337631225599933, 12.31726507176703, -17.82700636875124, 1.8707757577108541, 350.070897118924, 696.5254597845967, 349.26700684504846], "197_torus": [0.4341962855591346, 6.186279721806835, -7.564113307402871, -4.291820946261116, -0.24373439132728336, -15.249723323708086, 2.2102684707474323, 5.83069160622691, 1.900061641222405, 4.501115333865825, 11.532720650045611, -1.3329711302306975, 3.5361974272057513, -9.42150017100716, 6.8759501160120315, -7.607783308513227, 1.628571638847391, 352.3982078898039, 704.9939939229191, 350.59105049886847], "198_sphere": [-0.6456851980009675, 0.10159057310041408, 5.8105840463028064, -5.742847138000137, 18.03340262449597, 10.571328380892101, 7.14261776576446, -8.169008584000393, -7.062664308186912, 22.139681421271455, -19.655958231824627, 4.609671779691109, 14.567261363490342, -14.944999224999147, -20.433907924494616, -1.4679889794704062, 1.7888950291047743, 348.0364864747813, 700.6370293944914, 349.27964723695635], "199_torus": [7.681928472908375, -5.635566064698859, -4.693205436554245, -3.46720549479455, -3.5963576739367706, 0.6914107007926776, 9.112960945501902, 2.0066888514185446, 12.498545452553655, 3.028246837278767, -8.445803087701314, -1.1547845718018042, 5.964368467607381, -2.24665920192858, -2.3537326070965365, -5.372705670554696, 1.5759272581793609, 348.66732306465684, 702.4705619358883, 350.5030333562862]}}


In [ ]:
%%writefile reference/glno_shapes_fingerprint_shapes_small.json
{"version": 1, "ritz_version": 1, "identity": "8bab6ef70bbe73f4597dc2ecb178f3b2be51b594e9cc4e407833f29d38f87f37", "basis": "ritz", "k_eig": 700, "shapes": {"000_sphere": [3.5466464186381463, 6.093694979253594, -11.814724591221768, -2.36012903195795, -8.61799593617241, -10.041789954367982, 7.115071145632544, -6.406707367888949, -9.889978402105285, 8.263098135344924, -9.04939060273436, -15.681301320795685, -0.9213374258232334, -2.218229765212536, 9.86643849039324, -6.636453651982074, 1.917213637465609, 349.71429182204105, 698.6963114346307, 349.28262857698013], "001_torus": [-3.5456123533197186, 12.956359237288968, -7.00817159775877, 3.1520815291362556, 2.124221426330785, -1.636311083879466, -15.69859700169699, 7.558782176156052, -12.683275356083167, -24.976885666453608, 7.195103304778811, 8.731507080260268, 16.410806690132468, -13.125154998560806, 3.954813177655324, -4.2407523233914475, 1.6508252686626868, 353.29482117020655, 706.2781117057067, 350.60067466167527], "002_sphere": [4.424618797018675, -1.2589972793487165, -16.502328821342395, 30.0130026286189, -14.179094985128339, -8.947808541655602, -4.013790179579505, -4.2471516840028585, -8.958506246638292, -8.616314510270701, -11.50964095448831, -3.987446576122912, -14.76870263239479, -5.23904175102115, 5.73580708718988, 10.062318630615643, 1.8709209728107683, 347.0029652208124, 697.5128222281387, 349.2919480402235], "003_torus": [-11.832732963202005, -5.560092858976201, -3.266473699356851, 6.053904926375425, -2.397285246733979, 4.350524086593241, -5.953915852532441, 10.649577999219659, 3.09094094700502, 5.682670863410931, 17.184538170460634, -1.5420472177426623, -0.022345253454493452, -4.87617914549236, -9.599821357491816, 3.941824297370119, 1.6362933812349165, 355.23050413156597, 703.7933272362238, 350.5394596136162], "004_sphere": [6.732887477636352, -16.679143731737362, -6.068000261552514, -4.5815450226042636, 10.650138858045619, -11.023367953261545, -13.676046660358107, 7.103435664237391, 4.989421094220859, 6.331833582409374, -1.3526311013199284, 1.9935521931715832, -9.087721411525479, -17.297958085472853, 6.588623970042969, 9.851751899492264, 1.8203613203181233, 349.5462246846122, 698.7186602766039, 349.3259965647495], "005_torus": [7.039964702838979, -1.2096522571729773, -9.562613056622919, -18.006800372739765, 6.735899587980718, 4.00792228658324, 0.8095122632959901, -4.609682200337204, 1.8981887121191185, 1.5631120714255196, 4.558922076576691, -2.7859334427328175, -4.860005907673177, 7.662201725366854, -1.0430153372404045, 0.1016372446122773, 1.532360103129293, 346.80817812766514, 701.2146407971908, 350.51337294445557], "006_sphere": [8.916573679348925, -1.9339493542045654, -12.030750840264156, 10.990412315365994, 5.874704803299607, 7.693848130795175, 16.03154543969124, 0.5906078938180621, -0.18003301999535992, 24.765678810077397, 25.261043374095415, 4.36892099493703, 2.0358043173913334, -8.154184232384644, 2.7293309938100965, -15.452234300404779, 1.6698874319045696, 349.55352103965515, 701.7322710995371, 349.31397307648365], "007_torus": [-6.348928177962587, 0.8582419689694536, 2.2836731031136415, -1.962803210955782, -18.768976153491778, -7.733627747764729, -6.275696738445625, -5.041661059231177, 18.605687222306848, 6.010758271042198, 10.216741335739707, -8.949290276617587, -11.014555678550574, -2.0470025485591403, -1.3652040790522935, 4.002108955980701, 1.4192675942618882, 351.9482605234423, 702.8565842426589, 350.5678393789484], "008_sphere": [-6.938782089669877, -0.5680862607286015, -9.28017203738417, -11.277791434740358, -0.7936181766373416, 6.8885515408143405, -11.967965568909392, -2.081483315880065, 3.9467840366380713, -7.211107294168681, -1.5200415214396148, -4.3146543134634445, -0.762129552725809, -8.329519643965128, 10.672708529304224, -13.157215277898993, 1.9430831223059908, 348.8768528074517, 700.8692005748285, 349.2943356872492], "009_torus": [1.4515591262683059, -15.893360863310606, 4.532956175165268, 0.10212198773550174, 2.821289346204228, 8.286967281035917, 10.036387027838494, 0.1048099550991104, -4.837899663044175, -2.298635835187329, -13.128639758525704, -8.991683189734818, 0.1428247595300487, -6.491093459100931, 15.470667221978808, 1.4834207821902496, 1.5824406857518531, 350.106288622024, 697.1675188541686, 350.4368854705948]}}


## Configuración

Lo que cambia entre una fase y otra del plan está en la celda siguiente, una variable por línea.

| Fase del plan | Qué cambiar |
|---|---|
| A. Prueba rápida | Nada: `QUICK_RUN = True`. Usa `shapes_small`, 2 épocas y el run `quick_<variante>` |
| C. Piloto de 50 épocas | `QUICK_RUN = False`. Quedan `VARIANT = "periodic"` y `EPOCHS = 50`: el run `shapes_glno_periodic_v1` |
| D. Extender a 300 | `QUICK_RUN = False` y `EPOCHS = 300`. Continúa el mismo run, y equivale a 300 épocas seguidas |
| E. Control | `QUICK_RUN = False`, `VARIANT = "full"` y `EPOCHS = 100` o más. Es otro run, `shapes_glno_full_v1`, que se compara en la época 100 |

- **Fase A.** Cuesta alrededor de 1 CU (5–10 min) y mide la velocidad real de la GPU: la celda de calibración proyecta las horas del dataset completo. Para ver la reanudación, suba `QUICK_EPOCHS` a 3 y ejecute todo otra vez: el entrenamiento debe decir `Resuming at epoch 2`.
- **Costos.** La tarifa de unidades de cómputo (CU) por hora no se supone: léala en Colab (*Recursos*) y póngala en `CU_PER_HOUR`. Con ella, la calibración da el costo de 50 y de 300 épocas. El plan (decisión D3) pone como techo ~60 CU para el modelo principal y ~25 CU para el control: si la proyección lo supera, no siga sin revisarlo.
- **Revisión a las épocas 20–30 del piloto.** `val_select` en $t=0.4$–$0.5$ debe estar bajando claramente, sin pasos descartados y con memoria y velocidad estables. Para verlo, interrumpa la celda de entrenamiento, ejecute la de curvas y vuelva a ejecutar la de entrenamiento: reanuda desde la última época guardada. Lo mismo está, época por época, en `history.json`, en la carpeta del run en Drive (`val_per_time`, `skipped`, `reserved_gib`, `ms_per_sample`). Si no baja, pare la sesión y revise en su computador.
- **Lo que no se cambia en un run ya empezado:** la red (`VARIANT`, `GLNO_OVERRIDES`), el lote y el resto de `SETTINGS`, salvo las épocas y el recorte de gradiente. Un cambio de esos pide otro `RUN_NAME`. En la prueba rápida el run siempre es `quick_<variante>`: para repetirla con otra red, borre antes esa carpeta en `Mi unidad/glno_schrodinger/runs`.


In [ ]:
from schrodinger_glno.glno.shapes_train import VARIANTS, ShapesTrainSettings

QUICK_RUN = True                      # fase A (~1 CU): dataset pequeño "shapes_small", 2 épocas y run "quick_..."
QUICK_EPOCHS = 2                      # épocas de la prueba rápida; con 3, una segunda ejecución muestra la reanudación
# Decisión D2 del plan. Cambiar de modelo es cambiar esta línea; el nombre del run cambia con ella.
#   "periodic": solo la rama periódica. Es el modelo principal después del ensayo local (pendiente de confirmar).
#   "full": el GLNO completo del paper, con gaussian_factor = 50. Es el control.
VARIANT = "periodic"
EPOCHS = 50                           # el piloto; subirlo a 300 continúa el mismo run
RUN_NAME = f"shapes_glno_{VARIANT}_v1"                         # carpeta de checkpoints, en DRIVE_ROOT/runs
DRIVE_ROOT = Path("/content/drive/MyDrive/glno_schrodinger")   # los tar van en DRIVE_ROOT/packed/<dataset>
IDENTITIES = {"shapes": "edba875cbb0ce818", "shapes_small": "8bab6ef70bbe73f4"}    # identidad esperada de cada dataset
CU_PER_HOUR = None                    # unidades de cómputo por hora de esta GPU: se leen en Colab, no se suponen
EVALUATE_TEST = False                 # el test se mira una sola vez, cuando la elección del modelo es definitiva
GLNO_OVERRIDES = {}                   # cambios a la red; vacío es la del paper (4 bloques, 64 canales, 1 polo)

NAME = "shapes"                       # dataset de la corrida real: sus tar se esperan en DRIVE_ROOT/packed/NAME
if QUICK_RUN:
    NAME, EPOCHS, RUN_NAME = "shapes_small", QUICK_EPOCHS, f"quick_{VARIANT}"
SETTINGS = ShapesTrainSettings(epochs=EPOCHS)   # lo demás es del paper: lotes de 4, AdamW, lr 1e-3 / 2 cada 100 épocas

if VARIANT not in VARIANTS:
    raise ValueError(f"VARIANT debe ser una de {tuple(VARIANTS)}, no {VARIANT!r}")
if not (isinstance(EPOCHS, int) and EPOCHS >= 1):
    raise ValueError(f"El número de épocas debe ser un entero mayor o igual a 1, no {EPOCHS!r}")
if CU_PER_HOUR is not None and not (isinstance(CU_PER_HOUR, (int, float)) and CU_PER_HOUR > 0):
    raise ValueError(f"CU_PER_HOUR debe ser un número positivo o None, no {CU_PER_HOUR!r}")
print(f"{QUICK_RUN=}  {VARIANT=}  {NAME=}  {RUN_NAME=}  {EPOCHS=}")
print(SETTINGS)


## Drive y dataset

Primero se monta Drive y se comprueba que quedó montado, antes de leer o escribir nada en él: sin eso, los checkpoints irían al disco de la sesión y se perderían con ella. Las celdas que usan Drive lo comprueban otra vez al empezar. Fuera de Colab todo queda en la carpeta de trabajo local.

El dataset empaquetado se busca en Drive en este orden, y se usa lo primero que exista: la carpeta `Mi unidad/glno_schrodinger/packed/<nombre>` con sus tar y `shapes_tars.json`; si no está, el archivo `glno_<nombre>.zip` en `Mi unidad/glno_schrodinger/packed`, en `Mi unidad/glno_schrodinger` o en `Mi unidad`. Con un zip, sus tar se copian al disco de la sesión y el dataset se extrae de ellos; cada paso comprueba tamaños y SHA-256. La celda imprime dónde busca.


In [ ]:
from schrodinger_glno.shapes_archive import INDEX_FILE, zip_name

MOUNT = Path("/content/drive")        # donde Colab monta Drive


def drive_name(path):
    """La ruta como se ve en la página de Drive."""
    my_drive = MOUNT / "MyDrive"
    if path == my_drive:
        return "Mi unidad"
    return f"Mi unidad/{path.relative_to(my_drive)}" if my_drive in path.parents else str(path)


def check_storage():
    """Detiene la celda si Drive ya no está montado: RUN_DIR sería entonces una carpeta del disco de la sesión."""
    if IN_COLAB and not os.path.ismount(MOUNT):
        raise RuntimeError(f"Drive no está montado en {MOUNT} (la última celda lo desmonta). Ejecute otra vez la celda "
                           "que monta Drive antes de esta: si no, los checkpoints quedarían en el disco de la sesión.")


if IN_COLAB:
    from google.colab import drive
    if MOUNT not in DRIVE_ROOT.parents:
        raise ValueError(f"DRIVE_ROOT debe estar dentro de {MOUNT}, no en {DRIVE_ROOT}")
    drive.mount(str(MOUNT))
    if not os.path.ismount(MOUNT):
        raise RuntimeError(f"Drive no quedó montado en {MOUNT}. Ejecute esta celda otra vez y autorice el acceso. "
                           "No siga sin Drive: los checkpoints quedarían en el disco de la sesión y se perderían.")
    STORAGE = DRIVE_ROOT
else:
    STORAGE = WORKDIR
TARS_DIR = STORAGE / "packed" / NAME
RUN_DIR = STORAGE / "runs" / RUN_NAME
DATA_DIR = (Path("/content/data") if IN_COLAB else WORKDIR / "data") / NAME
# El dataset empaquetado es la carpeta TARS_DIR o, si no está, el archivo ZIP_NAME, que se busca en ZIP_FOLDERS en
# este orden. La última es adonde se sube: en Colab, la raíz de Drive ("Mi unidad").
ZIP_NAME = zip_name(NAME)
ZIP_FOLDERS = list(dict.fromkeys([STORAGE / "packed", STORAGE] + ([MOUNT / "MyDrive"] if IN_COLAB else [])))
# Los tar de un zip se copian aquí, al disco de la sesión: nunca a Drive.
LOCAL_TARS = (Path("/content/packed") if IN_COLAB else WORKDIR / "unzipped") / NAME
print("Dataset empaquetado: se usa lo primero que exista, en este orden.")
print(f"  la carpeta {TARS_DIR} (los tar y {INDEX_FILE})")
print("\n".join(f"  el archivo {place / ZIP_NAME}" for place in ZIP_FOLDERS))
print("Tar de un zip:  ", LOCAL_TARS)
print("Checkpoints:    ", RUN_DIR)
print("Datos locales:  ", DATA_DIR)


Si subió el zip, sus tar se copian primero de Drive al disco local de la sesión, y cada uno debe tener el tamaño y el SHA-256 del índice; después el dataset se extrae de esos tar. Si subió la carpeta, los tar se leen directamente de Drive. En los dos casos el dataset queda en el disco local de la sesión, que es mucho más rápido para leer, y nada aparece con el nombre del dataset hasta que todos los SHA-256 coinciden. Si los tar ya están copiados, no se copian otra vez; si el dataset ya está extraído y verificado, no se hace nada. Antes de copiar o extraer se comprueba que el dataset empaquetado es el esperado y se prueba que la carpeta del run se puede escribir.

Con el dataset completo esto tarda unos minutos en cada sesión nueva. Con el zip hacen falta unos 18 GB del disco de la sesión: 8.83 GB de tar y 8.83 GB de dataset. Con la carpeta, 8.83 GB.


In [ ]:
from schrodinger_glno.shapes_archive import INDEX_FILE, read_index, read_zip_index, unpack, unzip_tars

check_storage()
UPLOAD = (f"Suba la carpeta data_packed/{NAME} de su computador, tal cual (los tar y {INDEX_FILE}), "
          f"a {drive_name(TARS_DIR)}")
UPLOAD_ZIP = (f"Suba el archivo data_packed/{ZIP_NAME} de su computador a {drive_name(ZIP_FOLDERS[-1])}, sin crear "
              "ninguna carpeta")
SYNC = "Si la subida acaba de terminar, espere a que Drive sincronice y ejecute esta celda otra vez."
AGAIN_ZIP = f"Si sigue igual, borre {ZIP_NAME} de Drive y suba de nuevo data_packed/{ZIP_NAME}."


def tar_problem(tar):
    """Qué le pasa a un tar del índice antes de extraerlo, o None si está completo."""
    path = TARS_DIR / tar["name"]
    if not path.is_file():
        return "falta"
    size = path.stat().st_size
    return None if size == tar["size"] else f"tiene {size} bytes de {tar['size']}"


ZIP = None                            # el zip solo se busca si la carpeta de los tar no está
if not (TARS_DIR / INDEX_FILE).is_file():
    ZIP = next((folder / ZIP_NAME for folder in ZIP_FOLDERS if (folder / ZIP_NAME).is_file()), None)
    if ZIP is None:
        raise FileNotFoundError(f"No encontré {INDEX_FILE} en {TARS_DIR}. Tampoco encontré {ZIP_NAME} en "
                                f"{', '.join(map(str, ZIP_FOLDERS))}. {UPLOAD_ZIP}. O, en lugar del zip: {UPLOAD}.")
    print("Zip del dataset:", ZIP)
SOURCE = TARS_DIR if ZIP is None else ZIP
UPLOADED = f"la carpeta data_packed/{NAME}" if ZIP is None else f"el archivo data_packed/{ZIP_NAME}"
try:                                  # el índice basta para saber de qué dataset son los tar: no se lee ninguno
    packed = read_index(TARS_DIR) if ZIP is None else read_zip_index(ZIP)
except ValueError as error:           # vacío o cortado: Drive puede mostrar un archivo que todavía está subiendo
    if ZIP is None:
        raise ValueError(f"No pude leer {INDEX_FILE} en {TARS_DIR} ({error}). {SYNC} Si sigue igual, súbalo de nuevo "
                         f"desde data_packed/{NAME}.") from error
    raise ValueError(f"El archivo {ZIP} no está completo o no es el zip de este dataset ({error}). {SYNC} "
                     f"{AGAIN_ZIP}") from error
if not packed["identity"].startswith(IDENTITIES[NAME]):
    raise ValueError(f"Los tar de {SOURCE} son del dataset {packed['identity'][:16]}, no del esperado "
                     f"{IDENTITIES[NAME]}. Revise que subió {UPLOADED}. Si generó el dataset de "
                     "nuevo, este notebook ya no le corresponde: lleva la identidad y la huella del anterior.")
try:                                  # una carpeta que no se puede escribir falla aquí, no después de calibrar
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    probe = RUN_DIR / "write_test.tmp"
    probe.write_bytes(b"x")
    probe.unlink()
except OSError as error:
    raise RuntimeError(f"No se puede escribir en {drive_name(RUN_DIR)} ({error}). Ahí van los checkpoints: revise "
                       "que la carpeta sea suya o que tenga permiso de edición.") from error
if ZIP is None:
    try:
        unpacked = unpack(TARS_DIR, DATA_DIR, identity=IDENTITIES[NAME], progress=print)
    except ValueError as error:
        bad = [f"{tar['name']} ({problem})" for tar in packed["tars"] if (problem := tar_problem(tar))]
        if not bad:
            raise
        raise FileNotFoundError(f"Tar que faltan o están incompletos en {TARS_DIR}: {', '.join(bad)}. {UPLOAD}. "
                                f"{SYNC}") from error
else:                                 # del zip al disco de la sesión, y de esos tar el dataset
    try:
        unzipped = unzip_tars(ZIP, LOCAL_TARS, identity=IDENTITIES[NAME], progress=print)
    except ValueError as error:
        reason = str(error)           # el motivo llega en inglés; lo que hay que hacer depende de cuál sea
        if "Not enough disk space" in reason:
            advice = ("Borre del disco de la sesión lo que no haga falta y ejecute esta celda otra vez. Si ni así "
                      f"alcanza, use la carpeta en lugar del zip, que necesita la mitad del espacio: {UPLOAD}.")
        elif "does not hold the tars of" in reason:
            advice = (f"Borre la carpeta {LOCAL_TARS} y ejecute esta celda otra vez: es una copia en el disco de la "
                      "sesión, y los tar se copian de nuevo del zip. En Drive no borre nada.")
        else:                         # el zip, o un tar dentro de él, no se pudo leer entero o llegó dañado
            advice = f"{SYNC} {AGAIN_ZIP}"
        raise ValueError(f"No pude copiar los tar de {ZIP} a {LOCAL_TARS} ({reason}). {advice}") from error
    print(f"Tar de {ZIP.name}: {unzipped['bytes'] / 1e9:.2f} GB en {LOCAL_TARS} "
          f"({'copiados del zip' if unzipped['extracted'] else 'ya estaban'}, {unzipped['seconds']:.0f} s)")
    unpacked = unpack(LOCAL_TARS, DATA_DIR, identity=IDENTITIES[NAME], progress=print)
print(f"Dataset {NAME} verificado: identidad {unpacked['identity'][:16]}, {unpacked['files']} archivos, "
      f"{unpacked['bytes'] / 1e9:.2f} GB en {DATA_DIR} "
      f"({'desempaquetado' if unpacked['extracted'] else 'ya estaba'}, {unpacked['seconds']:.0f} s)")


## Datos en la GPU y huella de la base

Se construye en la GPU lo que la red ve de cada forma: la base de Ritz (calculada aquí, en la CPU de Colab), los autovalores, la masa y los paquetes. El dataset completo ocupa 7.95 GiB.

La base se recalcula en una máquina distinta de la suya, así que se compara con la **huella de referencia** que viaja en este notebook. La comparación es con tolerancia, porque los últimos dígitos cambian con la CPU y el número de hilos. Si no coinciden, el notebook se detiene aquí, antes de calibrar y de entrenar.

Al repetir esta celda en la misma sesión con el mismo dataset, no se construye otra vez: se usa lo que ya está en la GPU.


In [ ]:
import json
import time
from schrodinger_glno.glno.shapes_data import GROUPS, SETS, ShapeStore, fingerprint_mismatch

# Después de un error, Python conserva las variables de la celda que falló, el almacén entre ellas: construir otro
# no liberaría el anterior y habría dos en la GPU. Si ya está el de este dataset, se usa ese.
if globals().get("store") is not None and store.identity == unpacked["identity"]:
    print("Los datos de este dataset ya estaban construidos en esta sesión; se usan los mismos.")
else:
    store = None                      # el de otro dataset se suelta antes de construir el nuevo
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    start = time.perf_counter()
    store = ShapeStore(DATA_DIR, device=DEVICE, progress=lambda line: print(line.ljust(72), end="\r"))
    STORE_SECONDS, STORE_GIB = time.perf_counter() - start, store.nbytes / 2**30
print(f"{len(store.shapes)} formas y {len(store)} paquetes en {DEVICE}: {STORE_GIB:.2f} GiB, "
      f"construidos en {STORE_SECONDS:.1f} s".ljust(72))
for family, label in (("sphere", "esferas"), ("torus", "toros")):
    seconds = [shape["seconds"] for shape in store.shapes if shape["family"] == family]
    if seconds:
        print(f"  {label}: {len(seconds)} formas, {np.mean(seconds):.2f} s por forma (máximo {max(seconds):.2f} s)")
print("  paquetes:", {name: len(store.rows(sets=name)) for name in SETS})

REFERENCE = WORKDIR / "reference" / f"glno_shapes_fingerprint_{NAME}.json"
reference, fingerprint = json.loads(REFERENCE.read_text()), store.fingerprint()
# Nunca con ==: fingerprint_mismatch compara con tolerancia y dice qué difiere.
reason = fingerprint_mismatch(reference, fingerprint)
if reason is None and set(reference["shapes"]) != set(fingerprint["shapes"]):
    reason = "la referencia y el dataset no tienen las mismas formas"
if reason is not None:
    store = None                      # sin almacén, las celdas siguientes no pueden entrenar con otra base
    raise RuntimeError(f"La base de Ritz calculada aquí no es la de referencia ({reason}). No entrene: los "
                       "checkpoints quedarían ligados a otra base y la evaluación en su computador los rechazaría. "
                       "Anote las versiones de NumPy y SciPy de la segunda celda y revise en su computador.")
print(f"Huella de la base de Ritz: coincide con la referencia en las {len(fingerprint['shapes'])} formas.")


## Calibración

Una red de prueba, que después se descarta, da 3 pasos de calentamiento y 10 medidos por familia, con el lote real. Salen los ms por muestra, la memoria reservada (que incluye los datos) y las horas proyectadas para 50 y 300 épocas. El entrenamiento no cambia por esto.

En la prueba rápida la proyección es para el dataset completo: es la medida que acota el gasto de las fases siguientes. Queda guardada en `calibration.json`, en la carpeta del run.


In [ ]:
from schrodinger_glno.glno.shapes_train import calibrate, shapes_glno_config

check_storage()
FULL_PACKETS = {"sphere": (400, 50), "torus": (400, 50)}   # de "shapes", por época: (entrenamiento, validación)
FULL_STORE_GIB = 7.95                                       # lo que ocupa "shapes" en la GPU


def shown(value, spec):
    return "—" if value is None else format(value, spec)


def full_dataset_hours():
    """Horas de 50 y de 300 épocas de "shapes" a la velocidad medida aquí, o None si falta medir una familia."""
    if not set(FULL_PACKETS) <= set(measured):
        return None
    # Una familia sin validación medida se evalúa al costo de entrenar: una cota por arriba.
    seconds = sum(train * measured[family]["train_ms_per_sample"]
                  + val * (measured[family]["eval_ms_per_sample"] or measured[family]["train_ms_per_sample"])
                  for family, (train, val) in FULL_PACKETS.items()) / 1000
    return {str(epochs): epochs * seconds / 3600 for epochs in (50, 300)}


CONFIG = shapes_glno_config(store, VARIANT, **GLNO_OVERRIDES)
calibration = calibrate(store, CONFIG, SETTINGS)
if DEVICE == "cuda":
    torch.cuda.empty_cache()          # la red de prueba ya no existe; se devuelve su memoria antes de entrenar
measured = calibration["families"]
print(f"{'familia':<10}{'entrenar, ms/muestra':>22}{'evaluar, ms/muestra':>22}{'reservado, GiB':>18}")
for family, values in measured.items():
    print(f"{family:<10}{shown(values['train_ms_per_sample'], '.1f'):>22}"
          f"{shown(values['eval_ms_per_sample'], '.1f'):>22}{shown(values['reserved_gib'], '.2f'):>18}")
if QUICK_RUN:
    print(f"Esta prueba: {EPOCHS} épocas de {calibration['epoch_seconds']:.1f} s cada una.")
    projection = {"dataset": "shapes", "hours": full_dataset_hours()}
    title = ('Proyección para el dataset completo "shapes" (800 paquetes de entrenamiento y 100 de validación '
             "por época):")
else:
    projection = {"dataset": NAME, "hours": calibration["hours"]}
    title = f"Proyección para este dataset ({calibration['epoch_seconds'] / 60:.1f} min por época):"
if projection["hours"] is None:
    print("Sin proyección: este dataset no entrena con las dos familias.")
else:
    print(title)
    for epochs, hours in projection["hours"].items():
        cost = "" if CU_PER_HOUR is None else f" ≈ {hours * CU_PER_HOUR:.1f} CU"
        print(f"  {epochs:>3} épocas: {hours:.2f} h{cost}")
if CU_PER_HOUR is None:
    print("Para ver el costo, ponga en CU_PER_HOUR las unidades de cómputo por hora que muestra Colab.")
if DEVICE == "cuda":
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    peak = max(values["reserved_gib"] for values in measured.values())
    expected = peak + (FULL_STORE_GIB - STORE_GIB if QUICK_RUN else 0.0)
    print(f"Memoria: máximo reservado {peak:.1f} GiB de {total:.1f}"
          + (f"; con el dataset completo serían unos {expected:.1f} GiB." if QUICK_RUN else "."))
    if expected > 0.95 * total:
        print("ATENCIÓN: la corrida real no cabe con margen en esta GPU. Use una con más memoria.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "calibration.json").write_text(json.dumps(
    {"gpu": GPU_NAME, "dataset": NAME, "variant": VARIANT, "store_gib": STORE_GIB, "store_seconds": STORE_SECONDS,
     "cu_per_hour": CU_PER_HOUR, "projection": projection, **calibration}, indent=1) + "\n")
print("Calibración guardada en", RUN_DIR / "calibration.json")


## Entrenamiento

Cada época valida con la base canónica y guarda `last.pt` (para reanudar) y, si mejora `val_select` (la validación sin toros de revolución), `best.pt`. También quedan `history.json`, `last_prev.pt` (el `last.pt` de la época anterior) y, al terminar esas épocas, `epoch_100.pt` y `epoch_200.pt`. Si la sesión se corta, ejecute de nuevo todas las celdas: continúa desde la última época guardada. Para mirar las curvas a mitad del entrenamiento, interrumpa esta celda, ejecute la de curvas y vuelva a ejecutar esta.

Un paso con pérdida o gradiente no finitos se descarta sin tocar los pesos y se cuenta (`skipped`). Si los descartados pasan del 2 % de una época, el entrenamiento se detiene.


In [ ]:
import traceback
from schrodinger_glno.glno.shapes_train import train_shapes

check_storage()
try:
    history = train_shapes(store, RUN_DIR, CONFIG, SETTINGS)
except BaseException as error:        # también la interrupción a mano
    # El error conserva las variables del paso que se cortó (la red, el lote, las activaciones) y, con ellas, su
    # memoria de GPU. Se sueltan aquí: si no, reanudar en esta sesión necesitaría el doble.
    traceback.clear_frames(error.__traceback__)
    raise


## Curvas

Del historial que el entrenamiento guarda en cada época (`history.json`), así que también sirve con un entrenamiento interrumpido: errores por época, error de validación por tiempo en la última época, norma del gradiente, y pasos descartados y recortados. La figura queda en la carpeta del run como `curves.png`.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

check_storage()
history = json.loads((RUN_DIR / "history.json").read_text())   # lo guardado, no la variable de la celda anterior
# Un color y un marcador por serie, los mismos en los dos paneles de errores.
STYLE = {"train": ("#2a78d6", "o"), "select": ("#eb6834", "s"), "round": ("#1baf7a", "^"),
         "sphere": ("#eda100", "D"), "torus": ("#e87ba4", "v")}
FIRST, SECOND = STYLE["train"], STYLE["select"]     # los dos de los paneles que solo tienen dos series
few = len(history) <= 30              # con pocas épocas, un marcador por punto; con muchas, solo la línea


def series(key):
    return np.array([np.nan if record[key] is None else record[key] for record in history], dtype=float)


def draw(ax, x, y, style, label, markers=True):
    color, marker = style
    if np.isfinite(y).any():          # una serie sin datos (un dataset sin toros de revolución) no se dibuja
        ax.plot(x, y, color=color, linewidth=2, marker=marker if markers else None, markersize=6, label=label)


def finish(ax, title, ylabel, log=False, by_epoch=True):
    ax.set(title=title, xlabel="época" if by_epoch else "t", ylabel=ylabel)
    if log:
        ax.set_yscale("log")
    if by_epoch:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3, which="both" if log else "major")
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False)


epochs = np.array([record["epoch"] + 1 for record in history])
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for key, name, label in (("train_loss", "train", "entrenamiento"), ("val_select", "select", "val_select"),
                         ("val_round", "round", "val_round (toros de revolución)")):
    draw(axes[0, 0], epochs, series(key), STYLE[name], label, markers=few)
finish(axes[0, 0], "Error por época", "error relativo L² (ponderado con M)", log=True)
last = history[-1]
for name, label in (("select", "sin toros de revolución"), ("sphere", "esferas"), ("torus", "toros"),
                    ("round", "toros de revolución")):
    values = last["val_per_time"][name]
    if values is not None:
        draw(axes[0, 1], store.times, np.asarray(values, dtype=float), STYLE[name], label)
finish(axes[0, 1], f"Validación por tiempo, época {last['epoch'] + 1}", "error relativo L²", by_epoch=False)
draw(axes[1, 0], epochs, series("grad_norm_median"), FIRST, "mediana", markers=few)
draw(axes[1, 0], epochs, series("grad_norm_max"), SECOND, "máximo", markers=few)
finish(axes[1, 0], "Norma del gradiente por época", "norma antes del recorte", log=True)
skipped, clipped = series("skipped"), series("clipped")
# Los totales van en la leyenda: lo normal es que las dos series valgan 0 y una tape a la otra.
draw(axes[1, 1], epochs, skipped, FIRST, f"descartados por no finitos ({int(skipped.sum())} en total)", markers=few)
draw(axes[1, 1], epochs, clipped, SECOND, f"recortados ({int(clipped.sum())} en total)", markers=few)
finish(axes[1, 1], f"Pasos descartados y recortados, de {last['steps']} por época", "pasos")
top = 1.05 * max(1.0, skipped.max(), clipped.max())
axes[1, 1].set_ylim(-0.03 * top, top)     # un poco bajo cero, para que los marcadores en 0 se vean enteros
fig.tight_layout()
fig.savefig(RUN_DIR / "curves.png", dpi=150)
plt.show()
print(f"Época {last['epoch'] + 1}: train {shown(last['train_loss'], '.4f')}, val_select {last['val_select']:.4f}, "
      f"val_round {shown(last['val_round'], '.4f')}; descartados {int(skipped.sum())} y recortados "
      f"{int(clipped.sum())} en {len(history)} épocas. Figura: {RUN_DIR / 'curves.png'}")


## Evaluación

Solo la **validación**, con `best.pt`: error relativo ponderado con la masa, por grupo y por tiempo. Queda en `validation.json`, en la carpeta del run.

- `select`: validación sin toros de revolución. Es el número que elige `best.pt`.
- `sphere` y `torus`: lo mismo, por familia.
- `round`: toros de revolución. Sus copias de validación son casi iguales a las de entrenamiento, así que se reportan aparte.

El test y el conjunto fuera de distribución quedan detrás de `EVALUATE_TEST = False`, para no elegir el modelo mirando el test. La evaluación completa se hace en su computador con `best.pt` (paso 10 del plan), sin gastar unidades de cómputo.


In [ ]:
from schrodinger_glno.glno.shapes_train import evaluate_rows, load_checkpoint, validation_summary

check_storage()
model, state = load_checkpoint(RUN_DIR / "best.pt", DEVICE)
reason = fingerprint_mismatch(state["fingerprint"], store.fingerprint())
if reason is not None:
    raise RuntimeError(f"best.pt se entrenó con otra base de Ritz ({reason}); no se evalúa con estos datos.")


def evaluate(name):
    """Errores de un conjunto de paquetes: cuántos hay en cada grupo, y su error total y por tiempo."""
    rows = store.rows(sets=name)
    summary = validation_summary(store, rows, evaluate_rows(model, store, rows, SETTINGS.batch_size))
    return {"packets": {group: len(positions) for group, positions in store.group_positions(rows).items()}, **summary}


def show(title, result):
    print(title)
    print(f"  {'grupo':<8}{'paquetes':>10}{'error':>10}")
    for group in GROUPS:
        print(f"  {group:<8}{result['packets'][group]:>10}{shown(result['total'][group], '.4f'):>10}")
    present = [group for group in GROUPS if result["per_time"][group] is not None]
    print(f"  {'t':<8}" + "".join(f"{group:>10}" for group in present))
    for index, t in enumerate(store.times):
        print(f"  {t:<8.3f}" + "".join(f"{result['per_time'][group][index]:>10.4f}" for group in present))


def report(sets):
    return {"run": RUN_NAME, "variant": VARIANT, "dataset": store.identity, "checkpoint": "best.pt",
            "epoch": state["epoch"] + 1, "val_select": state["record"]["val_select"],
            "times": store.times.tolist(), **{name: evaluate(name) for name in sets}}


validation = report(["val"])
show(f"Validación con best.pt (época {validation['epoch']})", validation["val"])
(RUN_DIR / "validation.json").write_text(json.dumps(validation, indent=1) + "\n")

if EVALUATE_TEST:                     # el test se mira una sola vez, cuando la elección del modelo es definitiva
    tested = report(["test", "ood"])
    show("Test", tested["test"])
    show("Fuera de distribución (k0 mayor que en entrenamiento, en las formas de test)", tested["ood"])
    (RUN_DIR / "test.json").write_text(json.dumps(tested, indent=1) + "\n")


## Cerrar

Espera a que Drive termine de subir los checkpoints y lo desmonta. Después, desconecte la sesión (*Entorno de ejecución → Desconectar y eliminar el entorno de ejecución*) para no gastar más unidades de cómputo. Si una celda anterior falló, ejecute esta de todos modos. Para seguir en la misma sesión después de esta celda, ejecute otra vez la que monta Drive.


In [ ]:
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive quedó sincronizado y desmontado. Los resultados están en", drive_name(RUN_DIR))
else:
    print("Fuera de Colab no hay nada que desmontar. Los resultados están en", RUN_DIR)
